#### Notebook 1 — Data preparation and frozen hand-off

This notebook creates the reproducible data foundation for the ESI-3 abdominal-pain binary-disposition study.

It verifies the computing environment and source paths, constructs the MIMIC-IV development cohort and MC-MED external-validation cohort, freezes the HOME-versus-ADMITTED outcome, applies the approximate MIMIC-IV temporal split derived from `anchor_year`, `anchor_year_group`, and encounter-year offset, reconstructs the harmonised S0, S1, and S2 feature blocks, and writes the raw 2,495-feature hand-off consumed by later notebooks.

The notebook performs no imputation, one-hot encoding, hyperparameter tuning, model fitting, calibration, threshold selection, or external model evaluation.


#### Environment, imports and reproducibility

All notebook-wide imports and reproducibility settings are defined once in this section. Later cells rely on these shared imports rather than repeatedly importing the same libraries.


In [ ]:
import hashlib
import html
import json
import math
import os
import platform
import random
import re
import socket
import sys
import textwrap
import unicodedata
import warnings
from collections import defaultdict
from datetime import datetime, timezone
from difflib import SequenceMatcher
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from dateutil import parser as date_parser
from IPython.display import display
from ipykernel import get_connection_file

RANDOM_STATE = 42
RANDOM_SEED = RANDOM_STATE

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 180)

warnings.simplefilter("default")

try:
    connection_file = get_connection_file()
except Exception:
    connection_file = "Unavailable"

print("Python executable :", sys.executable)
print("Python version    :", platform.python_version())
print("Operating system  :", platform.platform())
print("Remote host       :", socket.gethostname())
print("Working directory :", Path.cwd().resolve())
print("Kernel PID        :", os.getpid())
print("Connection file   :", connection_file)
print("TMUX environment  :", os.environ.get("TMUX", "Not detected"))
print("Random state      :", RANDOM_STATE)
print("UTC start time    :", datetime.now(timezone.utc).isoformat())


#### Project folders


In [ ]:
PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
LOG_DIR = PROJECT_ROOT / "logs"

PROJECT_DIRECTORIES = [
    NOTEBOOK_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    MODEL_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    LOG_DIR,
]

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

for directory in PROJECT_DIRECTORIES:
    print(f"{directory.name:12s} -> {directory}")


#### Source dataset locations


In [ ]:
DATA_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_DATA_ROOT",
        PROJECT_ROOT / "data_external",
    )
).expanduser().resolve()

SOURCE_ROOTS = {
    "mcmed": Path(
        os.environ.get(
            "MCMED_DATA_DIR",
            DATA_ROOT / "mcmed",
        )
    ).expanduser().resolve(),
    "mimiciv_ed": Path(
        os.environ.get(
            "MIMIC_IV_ED_DIR",
            DATA_ROOT / "mimic-iv-ed" / "2.2" / "ed",
        )
    ).expanduser().resolve(),
    "mimiciv_hosp": Path(
        os.environ.get(
            "MIMIC_IV_HOSP_DIR",
            DATA_ROOT / "mimic-iv" / "3.1" / "hosp",
        )
    ).expanduser().resolve(),
    "mimiciv_note": Path(
        os.environ.get(
            "MIMIC_IV_NOTE_DIR",
            DATA_ROOT / "mimic-iv-note" / "2.2" / "note",
        )
    ).expanduser().resolve(),
}

MC_MED_DIR = SOURCE_ROOTS["mcmed"]
MIMIC_IV_ED_DIR = SOURCE_ROOTS["mimiciv_ed"]
MIMIC_IV_HOSP_DIR = SOURCE_ROOTS["mimiciv_hosp"]
MIMIC_IV_NOTE_DIR = SOURCE_ROOTS["mimiciv_note"]

SOURCE_PATHS = {
    "mcmed_visits": MC_MED_DIR / "visits.csv",
    "mcmed_labs": MC_MED_DIR / "labs.csv",
    "mcmed_meds": MC_MED_DIR / "meds.csv",
    "mcmed_pmh": MC_MED_DIR / "pmh.csv",
    "mcmed_rads": MC_MED_DIR / "rads.csv",
    "mimic_edstays": MIMIC_IV_ED_DIR / "edstays.csv.gz",
    "mimic_triage": MIMIC_IV_ED_DIR / "triage.csv.gz",
    "mimic_diagnosis": MIMIC_IV_ED_DIR / "diagnosis.csv.gz",
    "mimic_medrecon": MIMIC_IV_ED_DIR / "medrecon.csv.gz",
    "mimic_patients": MIMIC_IV_HOSP_DIR / "patients.csv.gz",
    "mimic_admissions": MIMIC_IV_HOSP_DIR / "admissions.csv.gz",
    "mimic_labevents": MIMIC_IV_HOSP_DIR / "labevents.csv.gz",
    "mimic_d_labitems": MIMIC_IV_HOSP_DIR / "d_labitems.csv.gz",
    "mimic_radiology_notes": MIMIC_IV_NOTE_DIR / "radiology.csv.gz",
    "mimic_radiology_detail": MIMIC_IV_NOTE_DIR / "radiology_detail.csv.gz",
}

print("Registered source paths:", len(SOURCE_PATHS))


#### Source-path validation


In [ ]:
path_status = pd.DataFrame(
    [
        {
            "source": name,
            "path": str(path),
            "exists": path.exists(),
            "size_mb": (
                round(path.stat().st_size / (1024**2), 2)
                if path.is_file()
                else np.nan
            ),
        }
        for name, path in SOURCE_PATHS.items()
    ]
)

display(path_status)

missing_sources = path_status.loc[
    ~path_status["exists"],
    ["source", "path"]
]

if not missing_sources.empty:
    print("\nMissing or incorrectly configured paths:")
    display(missing_sources)
else:
    print("All registered source files were found.")


#### Package availability and versions


In [ ]:
PACKAGE_NAMES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "pyarrow": "pyarrow",
    "matplotlib": "matplotlib",
    "python-dateutil": "python-dateutil",
    "ipykernel": "ipykernel",
}

package_records = []

for display_name, distribution_name in PACKAGE_NAMES.items():
    try:
        installed_version = version(distribution_name)
        available = True
    except PackageNotFoundError:
        installed_version = None
        available = False

    package_records.append(
        {
            "package": display_name,
            "installed": available,
            "version": installed_version,
        }
    )

package_versions = pd.DataFrame(package_records)

display(package_versions)

missing_packages = package_versions.loc[
    ~package_versions["installed"],
    "package"
].tolist()

if missing_packages:
    print("Packages not currently detected:", missing_packages)
else:
    print("All expected packages are available.")


#### Study constants


In [ ]:
STUDY_CONFIG = {
    "study_name": "ESI-3 abdominal pain binary disposition",
    "development_dataset": "MIMIC-IV",
    "external_dataset": "MC-MED",
    "minimum_age": 18,
    "target_esi": 3,
    "target_complaint_group": "Abdominal pain",
    "negative_class": "HOME",
    "positive_class": "ADMITTED",
    "random_state": RANDOM_STATE,
    "primary_external_threshold_source": "MIMIC-IV validation only",
    "external_threshold_0_80_status": "post-hoc sensitivity analysis only",
}

display(
    pd.DataFrame(
        STUDY_CONFIG.items(),
        columns=["setting", "value"]
    )
)


#### Essential source-schema preview


In [ ]:
def preview_csv(path: Path, nrows: int = 5, header=None) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Source file not found: {path}")

    return pd.read_csv(
        path,
        nrows=nrows,
        header=header,
        low_memory=False,
    )


preview_tables = {
    "MIMIC-IV ED stays": preview_csv(
        SOURCE_PATHS["mimic_edstays"]
    ),
    "MIMIC-IV triage": preview_csv(
        SOURCE_PATHS["mimic_triage"]
    ),
    "MIMIC-IV patients": preview_csv(
        SOURCE_PATHS["mimic_patients"]
    ),
    "MC-MED visits": preview_csv(
        SOURCE_PATHS["mcmed_visits"]
    ),
}

for table_name, preview_df in preview_tables.items():
    print("\n" + "=" * 100)
    print(table_name)
    print("Preview shape:", preview_df.shape)
    print("Columns:")
    print(preview_df.columns.tolist())
    display(preview_df.head())


#### Primary ESI-3 abdominal-pain cohort construction

This section creates harmonised primary cohorts for developing the MIMIC-IV model and externally validating it using MC-MED.

The same inclusion principles are applied to both datasets:

1. begin with all emergency-department visits;
2. require valid patient, encounter and visit-order information;
3. include adults aged 18 years or older;
4. identify a strict general abdominal-pain chief complaint;
5. require a valid Emergency Severity Index value from 1 to 5;
6. restrict the cohort to ESI-3;
7. retain the first eligible ESI-3 abdominal-pain encounter per patient;
8. require a defined raw emergency-department disposition.

The binary HOME-versus-ADMITTED outcome is not created in this section. Raw dispositions are inspected first so that observation, ICU and other non-standard outcomes can be harmonised transparently across MIMIC-IV and MC-MED.

Orders, laboratory testing, imaging, repeated vital signs and continuous monitoring are not used as cohort inclusion criteria because their availability may be influenced by clinician decisions.

All figures are saved in the project figures directory at 600 DPI.


#### Essential cohort source tables

This cell loads only the source tables and columns required for constructing the MIMIC-IV development cohort and the MC-MED external-validation cohort.

Loading selected columns instead of every column reduces memory use and makes the cohort-building logic easier to audit. The resulting DataFrames are:

- `edstays`: MIMIC-IV ED encounter and disposition information;
- `triage`: MIMIC-IV chief complaint, ESI and arrival measurements;
- `patients`: MIMIC-IV age-reference information;
- `visits`: MC-MED encounter, complaint, ESI and disposition information.

Registering a file path does not load its data. These tables must be reloaded whenever the notebook kernel is restarted.


In [ ]:


def load_selected_columns(
    path: Path,
    table_name: str,
    required_columns: list[str],
    optional_columns: list[str] | None = None,
) -> pd.DataFrame:
    """
    Load required columns and any available optional columns.

    The function stops if a required column is absent, preventing the
    pipeline from silently using an incorrect table or schema.
    """

    optional_columns = optional_columns or []

    if not path.exists():
        raise FileNotFoundError(
            f"{table_name} file was not found:\n{path}"
        )

    available_columns = pd.read_csv(
        path,
        nrows=0,
    ).columns.tolist()

    missing_required = [
        column
        for column in required_columns
        if column not in available_columns
    ]

    if missing_required:
        raise ValueError(
            f"{table_name} is missing required columns:\n"
            f"{missing_required}\n\n"
            f"Available columns are:\n{available_columns}"
        )

    selected_columns = required_columns + [
        column
        for column in optional_columns
        if column in available_columns
        and column not in required_columns
    ]

    dataframe = pd.read_csv(
        path,
        usecols=selected_columns,
        low_memory=False,
    )

    print(
        f"{table_name:<22} "
        f"rows={len(dataframe):,}, "
        f"columns={dataframe.shape[1]}"
    )

    return dataframe



edstays = load_selected_columns(
    path=SOURCE_PATHS["mimic_edstays"],
    table_name="MIMIC-IV edstays",
    required_columns=[
        "subject_id",
        "stay_id",
        "intime",
        "outtime",
        "disposition",
    ],
    optional_columns=[
        "hadm_id",
        "gender",
        "race",
        "arrival_transport",
    ],
)



triage = load_selected_columns(
    path=SOURCE_PATHS["mimic_triage"],
    table_name="MIMIC-IV triage",
    required_columns=[
        "subject_id",
        "stay_id",
        "chiefcomplaint",
        "acuity",
    ],
    optional_columns=[
        "temperature",
        "heartrate",
        "resprate",
        "o2sat",
        "sbp",
        "dbp",
        "pain",
    ],
)



patients = load_selected_columns(
    path=SOURCE_PATHS["mimic_patients"],
    table_name="MIMIC-IV patients",
    required_columns=[
        "subject_id",
        "anchor_age",
        "anchor_year",
        "anchor_year_group",
    ],
    optional_columns=[
        "gender",
        "dod",
    ],
)



visits = load_selected_columns(
    path=SOURCE_PATHS["mcmed_visits"],
    table_name="MC-MED visits",
    required_columns=[
        "CSN",
        "MRN",
        "Visit_no",
        "CC",
        "Age",
        "Triage_acuity",
        "ED_dispo",
    ],
    optional_columns=[
        "Triage_Temp",
        "Triage_HR",
        "Triage_RR",
        "Triage_SpO2",
        "Triage_SBP",
        "Triage_DBP",
        "Triage_pain",
        "Sex",
        "Means_of_arrival",
    ],
)


print("\nAll four essential cohort tables were loaded successfully.")


In [ ]:

loaded_table_summary = pd.DataFrame(
    [
        {
            "variable": "edstays",
            "dataset": "MIMIC-IV",
            "rows": len(edstays),
            "columns": edstays.shape[1],
            "main_identifier": "stay_id",
            "unique_identifiers": edstays["stay_id"].nunique(),
        },
        {
            "variable": "triage",
            "dataset": "MIMIC-IV",
            "rows": len(triage),
            "columns": triage.shape[1],
            "main_identifier": "stay_id",
            "unique_identifiers": triage["stay_id"].nunique(),
        },
        {
            "variable": "patients",
            "dataset": "MIMIC-IV",
            "rows": len(patients),
            "columns": patients.shape[1],
            "main_identifier": "subject_id",
            "unique_identifiers": patients["subject_id"].nunique(),
        },
        {
            "variable": "visits",
            "dataset": "MC-MED",
            "rows": len(visits),
            "columns": visits.shape[1],
            "main_identifier": "CSN",
            "unique_identifiers": visits["CSN"].nunique(),
        },
    ]
)

display(loaded_table_summary)

print("\nMIMIC-IV edstays columns:")
print(edstays.columns.tolist())

print("\nMIMIC-IV triage columns:")
print(triage.columns.tolist())

print("\nMIMIC-IV patients columns:")
print(patients.columns.tolist())

print("\nMC-MED visits columns:")
print(visits.columns.tolist())


#### Required-column and identifier audit


In [ ]:


required_columns = {
    "MIMIC edstays": {
        "dataframe": edstays,
        "required": [
            "subject_id",
            "stay_id",
            "intime",
            "outtime",
            "disposition",
        ],
    },
    "MIMIC triage": {
        "dataframe": triage,
        "required": [
            "subject_id",
            "stay_id",
            "chiefcomplaint",
            "acuity",
        ],
    },
    "MIMIC patients": {
        "dataframe": patients,
        "required": [
            "subject_id",
            "anchor_age",
            "anchor_year",
            "anchor_year_group",
        ],
    },
    "MC-MED visits": {
        "dataframe": visits,
        "required": [
            "CSN",
            "MRN",
            "Visit_no",
            "CC",
            "Age",
            "Triage_acuity",
            "ED_dispo",
        ],
    },
}

schema_rows = []

for table_name, specification in required_columns.items():
    dataframe = specification["dataframe"]

    for column in specification["required"]:
        schema_rows.append(
            {
                "table": table_name,
                "column": column,
                "present": column in dataframe.columns,
                "dtype": (
                    str(dataframe[column].dtype)
                    if column in dataframe.columns
                    else None
                ),
            }
        )

schema_audit = pd.DataFrame(schema_rows)

print("Required-column audit:")
display(schema_audit)

missing_required_columns = schema_audit.loc[
    ~schema_audit["present"],
    ["table", "column"],
]

if not missing_required_columns.empty:
    display(missing_required_columns)
    raise ValueError(
        "Required columns are missing. Do not continue until the paths "
        "and table schemas have been corrected."
    )

identifier_audit = pd.DataFrame(
    [
        {
            "table": "MIMIC edstays",
            "rows": len(edstays),
            "identifier": "stay_id",
            "missing_identifier": int(edstays["stay_id"].isna().sum()),
            "duplicate_identifier_rows": int(
                edstays["stay_id"].duplicated(keep=False).sum()
            ),
            "unique_identifiers": int(edstays["stay_id"].nunique()),
        },
        {
            "table": "MIMIC triage",
            "rows": len(triage),
            "identifier": "stay_id",
            "missing_identifier": int(triage["stay_id"].isna().sum()),
            "duplicate_identifier_rows": int(
                triage["stay_id"].duplicated(keep=False).sum()
            ),
            "unique_identifiers": int(triage["stay_id"].nunique()),
        },
        {
            "table": "MIMIC patients",
            "rows": len(patients),
            "identifier": "subject_id",
            "missing_identifier": int(patients["subject_id"].isna().sum()),
            "duplicate_identifier_rows": int(
                patients["subject_id"].duplicated(keep=False).sum()
            ),
            "unique_identifiers": int(patients["subject_id"].nunique()),
        },
        {
            "table": "MC-MED visits",
            "rows": len(visits),
            "identifier": "CSN",
            "missing_identifier": int(visits["CSN"].isna().sum()),
            "duplicate_identifier_rows": int(
                visits["CSN"].duplicated(keep=False).sum()
            ),
            "unique_identifiers": int(visits["CSN"].nunique()),
        },
    ]
)

print("\nIdentifier uniqueness audit:")
display(identifier_audit)


#### Cohort and figure configuration


In [ ]:


FIGURE_DPI = 600

ESI_LEVELS = [1, 2, 3, 4, 5]

ESI_LABELS = {
    1: "1\nResuscitation",
    2: "2\nEmergent",
    3: "3\nUrgent",
    4: "4\nLess urgent",
    5: "5\nNon-urgent",
}

PRIMARY_ABDOMINAL_COMPLAINTS = {
    "ABDOMINAL PAIN",
    "ABD PAIN",
    "ABDO PAIN",
    "ABDOMEN PAIN",
    "BELLY PAIN",
    "STOMACH PAIN",
}

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Figure directory :", FIGURE_DIR)
print("Figure resolution:", FIGURE_DPI, "DPI")
print("Primary abdominal-pain labels:")
print(sorted(PRIMARY_ABDOMINAL_COMPLAINTS))


#### Shared cohort helper functions


In [ ]:



FIGURE_DPI = 600

FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



VALID_ESI_LEVELS = [1, 2, 3, 4, 5]

PRIMARY_ABDOMINAL_COMPLAINTS = {
    "ABDOMINAL PAIN",
    "ABD PAIN",
    "ABDO PAIN",
    "ABDOMEN PAIN",
    "BELLY PAIN",
    "STOMACH PAIN",
}



def normalise_text(
    series: pd.Series,
) -> pd.Series:
    """
    Standardise text by converting it to uppercase and removing
    unnecessary punctuation and repeated spaces.
    """

    return (
        series
        .fillna("")
        .astype(str)
        .str.upper()
        .str.replace(
            r"[^A-Z0-9]+",
            " ",
            regex=True,
        )
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )


def identify_primary_abdominal_pain(
    series: pd.Series,
) -> pd.Series:
    """
    Identify the harmonised strict general abdominal-pain phenotype.
    """

    normalised = normalise_text(series)

    return normalised.isin(
        PRIMARY_ABDOMINAL_COMPLAINTS
    )


def extract_esi_numeric(
    series: pd.Series,
) -> pd.Series:
    """
    Extract ESI values 1-5 from numeric or descriptive values.
    """

    extracted = (
        series
        .fillna("")
        .astype(str)
        .str.extract(
            r"^\s*([1-5])",
            expand=False,
        )
    )

    return pd.to_numeric(
        extracted,
        errors="coerce",
    ).astype("Int64")


def clean_disposition(
    series: pd.Series,
) -> pd.Series:
    """
    Standardise raw disposition text without mapping it to the
    binary outcome.
    """

    return (
        series
        .fillna("")
        .astype(str)
        .str.upper()
        .str.strip()
        .replace("", pd.NA)
    )


def build_flow_table(
    dataset_name: str,
    flow_steps: list[tuple[str, int]],
) -> pd.DataFrame:
    """
    Create a sequential cohort-flow table with retained and removed
    visit counts.
    """

    initial_count = int(flow_steps[0][1])
    previous_count = None
    records = []

    for step_number, (
        inclusion_criterion,
        current_count,
    ) in enumerate(
        flow_steps,
        start=1,
    ):
        current_count = int(current_count)

        removed_from_previous = (
            0
            if previous_count is None
            else previous_count - current_count
        )

        retained_from_previous = (
            100.0
            if previous_count is None
            else (
                current_count
                / previous_count
                * 100
                if previous_count > 0
                else np.nan
            )
        )

        retained_from_initial = (
            current_count
            / initial_count
            * 100
            if initial_count > 0
            else np.nan
        )

        records.append(
            {
                "dataset": dataset_name,
                "step": step_number,
                "inclusion_criterion": inclusion_criterion,
                "n_remaining": current_count,
                "n_removed_from_previous": (
                    removed_from_previous
                ),
                "retained_from_previous_pct": (
                    retained_from_previous
                ),
                "retained_from_initial_pct": (
                    retained_from_initial
                ),
            }
        )

        previous_count = current_count

    return pd.DataFrame(records)


print("Primary cohort helper functions created.")
print("Figure directory:", FIGURE_DIR)
print("Figure resolution:", FIGURE_DPI, "DPI")
print(
    "Strict abdominal-pain labels:",
    sorted(PRIMARY_ABDOMINAL_COMPLAINTS),
)


#### MIMIC-IV all-ESI cohort


In [ ]:

mimic_ed = edstays.copy()
mimic_triage = triage.copy()
mimic_patients = patients.copy()



mimic_ed["subject_id"] = pd.to_numeric(
    mimic_ed["subject_id"],
    errors="coerce",
).astype("Int64")

mimic_ed["stay_id"] = pd.to_numeric(
    mimic_ed["stay_id"],
    errors="coerce",
).astype("Int64")

mimic_triage["subject_id"] = pd.to_numeric(
    mimic_triage["subject_id"],
    errors="coerce",
).astype("Int64")

mimic_triage["stay_id"] = pd.to_numeric(
    mimic_triage["stay_id"],
    errors="coerce",
).astype("Int64")

mimic_patients["subject_id"] = pd.to_numeric(
    mimic_patients["subject_id"],
    errors="coerce",
).astype("Int64")



mimic_ed["intime"] = pd.to_datetime(
    mimic_ed["intime"],
    errors="coerce",
)

mimic_ed["outtime"] = pd.to_datetime(
    mimic_ed["outtime"],
    errors="coerce",
)



assert mimic_ed["stay_id"].is_unique, (
    "MIMIC edstays contains duplicated stay_id values."
)

assert mimic_triage["stay_id"].is_unique, (
    "MIMIC triage contains duplicated stay_id values."
)

assert mimic_patients["subject_id"].is_unique, (
    "MIMIC patients contains duplicated subject_id values."
)



mimic_working = mimic_ed.merge(
    mimic_triage,
    on=[
        "subject_id",
        "stay_id",
    ],
    how="left",
    validate="one_to_one",
    suffixes=(
        "_ed",
        "_triage",
    ),
)



mimic_age_reference = (
    mimic_patients[
        [
            "subject_id",
            "anchor_age",
            "anchor_year",
            "anchor_year_group",
        ]
    ]
    .copy()
)

mimic_working = mimic_working.merge(
    mimic_age_reference,
    on="subject_id",
    how="left",
    validate="many_to_one",
)

mimic_working["anchor_age"] = pd.to_numeric(
    mimic_working["anchor_age"],
    errors="coerce",
)

mimic_working["anchor_year"] = pd.to_numeric(
    mimic_working["anchor_year"],
    errors="coerce",
)



mimic_working["age_at_ed"] = (
    mimic_working["anchor_age"]
    + mimic_working["intime"].dt.year
    - mimic_working["anchor_year"]
)



mimic_working[
    "chief_complaint_normalised"
] = normalise_text(
    mimic_working["chiefcomplaint"]
)

mimic_working[
    "primary_abdominal_pain"
] = identify_primary_abdominal_pain(
    mimic_working["chiefcomplaint"]
)

mimic_working[
    "esi_numeric"
] = extract_esi_numeric(
    mimic_working["acuity"]
)

mimic_working[
    "disposition_clean"
] = clean_disposition(
    mimic_working["disposition"]
)



mimic_step_1 = mimic_working.copy()


mimic_step_2 = mimic_step_1.loc[
    mimic_step_1[
        [
            "subject_id",
            "stay_id",
            "intime",
        ]
    ]
    .notna()
    .all(axis=1)
].copy()


mimic_step_3 = mimic_step_2.loc[
    mimic_step_2["age_at_ed"].ge(18)
].copy()


mimic_step_4 = mimic_step_3.loc[
    mimic_step_3[
        "primary_abdominal_pain"
    ]
].copy()


mimic_step_5 = mimic_step_4.loc[
    mimic_step_4[
        "esi_numeric"
    ].isin(VALID_ESI_LEVELS)
].copy()


mimic_step_6 = mimic_step_5.loc[
    mimic_step_5[
        "esi_numeric"
    ].eq(3)
].copy()


mimic_step_7 = (
    mimic_step_6
    .sort_values(
        [
            "subject_id",
            "intime",
            "stay_id",
        ]
    )
    .drop_duplicates(
        subset="subject_id",
        keep="first",
    )
    .reset_index(drop=True)
)


mimic_step_8 = (
    mimic_step_7.loc[
        mimic_step_7[
            "disposition_clean"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)



mimic_esi3_candidate = mimic_step_8.copy()



mimic_esi3_flow_table = build_flow_table(
    dataset_name="MIMIC-IV",
    flow_steps=[
        (
            "All ED visits",
            len(mimic_step_1),
        ),
        (
            "Valid patient, encounter and visit-order information",
            len(mimic_step_2),
        ),
        (
            "Adults aged ≥18 years",
            len(mimic_step_3),
        ),
        (
            "Strict general abdominal-pain chief complaint",
            len(mimic_step_4),
        ),
        (
            "Valid ESI level 1-5",
            len(mimic_step_5),
        ),
        (
            "ESI exactly 3",
            len(mimic_step_6),
        ),
        (
            "First eligible ESI-3 visit per patient",
            len(mimic_step_7),
        ),
        (
            "Defined raw ED disposition",
            len(mimic_step_8),
        ),
    ],
)



assert mimic_esi3_candidate[
    "subject_id"
].is_unique

assert mimic_esi3_candidate[
    "stay_id"
].is_unique

assert mimic_esi3_candidate[
    "age_at_ed"
].ge(18).all()

assert mimic_esi3_candidate[
    "primary_abdominal_pain"
].all()

assert mimic_esi3_candidate[
    "esi_numeric"
].eq(3).all()

assert mimic_esi3_candidate[
    "disposition_clean"
].notna().all()


print("MIMIC-IV primary ESI-3 cohort flow:")

display(
    mimic_esi3_flow_table.round(
        {
            "retained_from_previous_pct": 2,
            "retained_from_initial_pct": 2,
        }
    )
)

print(
    "\nMIMIC-IV primary candidate cohort:",
    f"{len(mimic_esi3_candidate):,}",
)


#### MC-MED all-ESI cohort


In [ ]:

mcmed_working = visits.copy()



mcmed_working["CSN"] = pd.to_numeric(
    mcmed_working["CSN"],
    errors="coerce",
).astype("Int64")

mcmed_working["MRN"] = pd.to_numeric(
    mcmed_working["MRN"],
    errors="coerce",
).astype("Int64")

mcmed_working[
    "visit_no_numeric"
] = pd.to_numeric(
    mcmed_working["Visit_no"],
    errors="coerce",
)

mcmed_working["age_at_ed"] = pd.to_numeric(
    mcmed_working["Age"],
    errors="coerce",
)



assert mcmed_working["CSN"].is_unique, (
    "MC-MED visits contains duplicated CSN values."
)



mcmed_working[
    "chief_complaint_normalised"
] = normalise_text(
    mcmed_working["CC"]
)

mcmed_working[
    "primary_abdominal_pain"
] = identify_primary_abdominal_pain(
    mcmed_working["CC"]
)

mcmed_working[
    "esi_numeric"
] = extract_esi_numeric(
    mcmed_working["Triage_acuity"]
)

mcmed_working[
    "disposition_clean"
] = clean_disposition(
    mcmed_working["ED_dispo"]
)



mcmed_step_1 = mcmed_working.copy()


mcmed_step_2 = mcmed_step_1.loc[
    mcmed_step_1[
        [
            "MRN",
            "CSN",
            "visit_no_numeric",
        ]
    ]
    .notna()
    .all(axis=1)
].copy()


mcmed_step_3 = mcmed_step_2.loc[
    mcmed_step_2["age_at_ed"].ge(18)
].copy()


mcmed_step_4 = mcmed_step_3.loc[
    mcmed_step_3[
        "primary_abdominal_pain"
    ]
].copy()


mcmed_step_5 = mcmed_step_4.loc[
    mcmed_step_4[
        "esi_numeric"
    ].isin(VALID_ESI_LEVELS)
].copy()


mcmed_step_6 = mcmed_step_5.loc[
    mcmed_step_5[
        "esi_numeric"
    ].eq(3)
].copy()


mcmed_step_7 = (
    mcmed_step_6
    .sort_values(
        [
            "MRN",
            "visit_no_numeric",
            "CSN",
        ]
    )
    .drop_duplicates(
        subset="MRN",
        keep="first",
    )
    .reset_index(drop=True)
)


mcmed_step_8 = (
    mcmed_step_7.loc[
        mcmed_step_7[
            "disposition_clean"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)



mcmed_esi3_candidate = mcmed_step_8.copy()



mcmed_esi3_flow_table = build_flow_table(
    dataset_name="MC-MED",
    flow_steps=[
        (
            "All ED visits",
            len(mcmed_step_1),
        ),
        (
            "Valid patient, encounter and visit-order information",
            len(mcmed_step_2),
        ),
        (
            "Adults aged ≥18 years",
            len(mcmed_step_3),
        ),
        (
            "Strict general abdominal-pain chief complaint",
            len(mcmed_step_4),
        ),
        (
            "Valid ESI level 1-5",
            len(mcmed_step_5),
        ),
        (
            "ESI exactly 3",
            len(mcmed_step_6),
        ),
        (
            "First eligible ESI-3 visit per patient",
            len(mcmed_step_7),
        ),
        (
            "Defined raw ED disposition",
            len(mcmed_step_8),
        ),
    ],
)



assert mcmed_esi3_candidate[
    "MRN"
].is_unique

assert mcmed_esi3_candidate[
    "CSN"
].is_unique

assert mcmed_esi3_candidate[
    "age_at_ed"
].ge(18).all()

assert mcmed_esi3_candidate[
    "primary_abdominal_pain"
].all()

assert mcmed_esi3_candidate[
    "esi_numeric"
].eq(3).all()

assert mcmed_esi3_candidate[
    "disposition_clean"
].notna().all()


print("MC-MED primary ESI-3 cohort flow:")

display(
    mcmed_esi3_flow_table.round(
        {
            "retained_from_previous_pct": 2,
            "retained_from_initial_pct": 2,
        }
    )
)

print(
    "\nMC-MED primary candidate cohort:",
    f"{len(mcmed_esi3_candidate):,}",
)


#### Combined inclusion criteria


In [ ]:

combined_primary_flow = pd.concat(
    [
        mimic_esi3_flow_table,
        mcmed_esi3_flow_table,
    ],
    ignore_index=True,
)


combined_primary_flow_display = (
    combined_primary_flow
    .pivot(
        index=[
            "step",
            "inclusion_criterion",
        ],
        columns="dataset",
        values="n_remaining",
    )
    .reset_index()
)


combined_primary_flow_display.columns.name = None


print("Primary ESI-3 cohort inclusion summary:")

display(
    combined_primary_flow_display
)


print("\nDetailed primary cohort-flow results:")

display(
    combined_primary_flow.round(
        {
            "retained_from_previous_pct": 2,
            "retained_from_initial_pct": 2,
        }
    )
)


#### Cohort-flow figures


In [ ]:


def plot_cohort_downfall(
    flow_table: pd.DataFrame,
    dataset_name: str,
    output_filename: str,
) -> Path:
    """
    Plot the sequential decrease in eligible cohort size.
    """

    plot_data = (
        flow_table
        .sort_values("step")
        .reset_index(drop=True)
    )

    labels = [
        textwrap.fill(
            label,
            width=23,
        )
        for label in plot_data[
            "inclusion_criterion"
        ]
    ]

    counts = plot_data[
        "n_remaining"
    ].to_numpy()

    retained_percentages = plot_data[
        "retained_from_initial_pct"
    ].to_numpy()


    figure, axis = plt.subplots(
        figsize=(12.5, 7),
    )


    bars = axis.bar(
        range(len(plot_data)),
        counts,
        edgecolor="black",
        linewidth=1,
    )


    for (
        bar,
        count,
        retained_percentage,
    ) in zip(
        bars,
        counts,
        retained_percentages,
    ):
        axis.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            (
                f"{int(count):,}\n"
                f"({retained_percentage:.2f}% of initial)"
            ),
            ha="center",
            va="bottom",
            fontsize=9,
        )


    axis.set_xticks(
        range(len(labels))
    )

    axis.set_xticklabels(
        labels,
        rotation=28,
        ha="right",
    )

    axis.set_ylabel(
        "Number of ED visits"
    )

    axis.set_title(
        (
            f"{dataset_name}: Primary ESI-3 "
            "Abdominal-Pain Cohort Flow"
        ),
        weight="bold",
    )

    axis.set_ylim(
        0,
        max(counts) * 1.17,
    )

    figure.tight_layout()


    output_path = (
        FIGURE_DIR
        / output_filename
    )

    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return output_path


mimic_flow_figure = plot_cohort_downfall(
    flow_table=mimic_esi3_flow_table,
    dataset_name="MIMIC-IV",
    output_filename=(
        "mimiciv_primary_esi3_cohort_flow_600dpi.png"
    ),
)


mcmed_flow_figure = plot_cohort_downfall(
    flow_table=mcmed_esi3_flow_table,
    dataset_name="MC-MED",
    output_filename=(
        "mcmed_primary_esi3_cohort_flow_600dpi.png"
    ),
)


print("Saved:", mimic_flow_figure)
print("Saved:", mcmed_flow_figure)


#### Primary-versus-broad complaint audit


In [ ]:


def create_disposition_summary(
    dataframe: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Summarise raw ED dispositions before harmonising the binary
    outcome.
    """

    summary = (
        dataframe[
            "disposition_clean"
        ]
        .fillna("MISSING")
        .value_counts(dropna=False)
        .rename_axis(
            "raw_disposition"
        )
        .reset_index(
            name="count"
        )
    )

    summary["percentage"] = (
        summary["count"]
        / len(dataframe)
        * 100
    )

    summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    return summary


mimic_esi3_disposition = (
    create_disposition_summary(
        dataframe=mimic_esi3_candidate,
        dataset_name="MIMIC-IV",
    )
)


mcmed_esi3_disposition = (
    create_disposition_summary(
        dataframe=mcmed_esi3_candidate,
        dataset_name="MC-MED",
    )
)


print(
    "MIMIC-IV primary ESI-3 candidate cohort:",
    f"{len(mimic_esi3_candidate):,}",
)

display(
    mimic_esi3_disposition.round(
        {"percentage": 2}
    )
)


print(
    "\nMC-MED primary ESI-3 candidate cohort:",
    f"{len(mcmed_esi3_candidate):,}",
)

display(
    mcmed_esi3_disposition.round(
        {"percentage": 2}
    )
)


#### All-ESI distributions and figures


In [ ]:


def plot_raw_disposition_distribution(
    disposition_table: pd.DataFrame,
    dataset_name: str,
    output_filename: str,
) -> Path:
    """
    Plot the raw disposition distribution before binary mapping.
    """

    plot_data = (
        disposition_table
        .copy()
        .reset_index(drop=True)
    )

    labels = [
        textwrap.fill(
            str(value),
            width=18,
        )
        for value in plot_data[
            "raw_disposition"
        ]
    ]


    figure, axis = plt.subplots(
        figsize=(10.5, 6.2),
    )


    bars = axis.bar(
        range(len(plot_data)),
        plot_data["count"],
        edgecolor="black",
        linewidth=1,
    )


    for (
        bar,
        count,
        percentage,
    ) in zip(
        bars,
        plot_data["count"],
        plot_data["percentage"],
    ):
        axis.text(
            bar.get_x()
            + bar.get_width() / 2,
            bar.get_height(),
            (
                f"{int(count):,}\n"
                f"({percentage:.2f}%)"
            ),
            ha="center",
            va="bottom",
            fontsize=9,
        )


    axis.set_xticks(
        range(len(labels))
    )

    axis.set_xticklabels(
        labels,
        rotation=22,
        ha="right",
    )

    axis.set_ylabel(
        "Number of ESI-3 ED visits"
    )

    axis.set_title(
        (
            f"{dataset_name}: Raw Disposition "
            "Distribution in the Primary ESI-3 Cohort"
        ),
        weight="bold",
    )

    axis.set_ylim(
        0,
        max(
            plot_data["count"].max(),
            1,
        )
        * 1.17,
    )

    figure.tight_layout()


    output_path = (
        FIGURE_DIR
        / output_filename
    )

    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return output_path


mimic_disposition_figure = (
    plot_raw_disposition_distribution(
        disposition_table=(
            mimic_esi3_disposition
        ),
        dataset_name="MIMIC-IV",
        output_filename=(
            "mimiciv_primary_esi3_raw_disposition_600dpi.png"
        ),
    )
)


mcmed_disposition_figure = (
    plot_raw_disposition_distribution(
        disposition_table=(
            mcmed_esi3_disposition
        ),
        dataset_name="MC-MED",
        output_filename=(
            "mcmed_primary_esi3_raw_disposition_600dpi.png"
        ),
    )
)


print("Saved:", mimic_disposition_figure)
print("Saved:", mcmed_disposition_figure)


In [ ]:


def cohort_checkpoint(
    dataframe: pd.DataFrame,
    dataset_name: str,
    patient_identifier: str,
    encounter_identifier: str,
) -> dict:
    """
    Create a final quality-control summary for the candidate cohort.
    """

    return {
        "dataset": dataset_name,
        "n_visits": len(dataframe),
        "n_patients": dataframe[
            patient_identifier
        ].nunique(),
        "n_unique_encounters": dataframe[
            encounter_identifier
        ].nunique(),
        "minimum_age": dataframe[
            "age_at_ed"
        ].min(),
        "maximum_age": dataframe[
            "age_at_ed"
        ].max(),
        "esi_values": ",".join(
            map(
                str,
                sorted(
                    dataframe[
                        "esi_numeric"
                    ]
                    .dropna()
                    .astype(int)
                    .unique()
                    .tolist()
                ),
            )
        ),
        "missing_dispositions": int(
            dataframe[
                "disposition_clean"
            ].isna().sum()
        ),
        "duplicate_patient_rows": int(
            dataframe[
                patient_identifier
            ].duplicated(
                keep=False
            ).sum()
        ),
        "duplicate_encounter_rows": int(
            dataframe[
                encounter_identifier
            ].duplicated(
                keep=False
            ).sum()
        ),
    }


primary_cohort_checkpoint = pd.DataFrame(
    [
        cohort_checkpoint(
            dataframe=mimic_esi3_candidate,
            dataset_name="MIMIC-IV",
            patient_identifier="subject_id",
            encounter_identifier="stay_id",
        ),
        cohort_checkpoint(
            dataframe=mcmed_esi3_candidate,
            dataset_name="MC-MED",
            patient_identifier="MRN",
            encounter_identifier="CSN",
        ),
    ]
)


display(primary_cohort_checkpoint)


assert (
    primary_cohort_checkpoint[
        "n_visits"
    ]
    == primary_cohort_checkpoint[
        "n_patients"
    ]
).all()

assert (
    primary_cohort_checkpoint[
        "n_visits"
    ]
    == primary_cohort_checkpoint[
        "n_unique_encounters"
    ]
).all()

assert (
    primary_cohort_checkpoint[
        "minimum_age"
    ] >= 18
).all()

assert (
    primary_cohort_checkpoint[
        "esi_values"
    ] == "3"
).all()

assert (
    primary_cohort_checkpoint[
        "missing_dispositions"
    ] == 0
).all()

assert (
    primary_cohort_checkpoint[
        "duplicate_patient_rows"
    ] == 0
).all()

assert (
    primary_cohort_checkpoint[
        "duplicate_encounter_rows"
    ] == 0
).all()


expected_figure_paths = [
    mimic_flow_figure,
    mcmed_flow_figure,
    mimic_disposition_figure,
    mcmed_disposition_figure,
]


figure_status = pd.DataFrame(
    [
        {
            "figure": path.name,
            "exists": path.exists(),
            "path": str(path),
        }
        for path in expected_figure_paths
    ]
)


print(
    "\nPrimary cohort checks passed."
)

print(
    "Binary disposition mapping has not yet been applied."
)

print(
    "The next section will harmonise HOME, "
    "ADMITTED, ICU, OBSERVATION and other outcomes."
)

print(
    "\nSaved figure verification:"
)

display(figure_status)


assert figure_status["exists"].all()


In [ ]:

ESI_LABELS = {
    1: "1\nResuscitation",
    2: "2\nEmergent",
    3: "3\nUrgent",
    4: "4\nLess urgent",
    5: "5\nNon-urgent",
}


def create_pre_restriction_esi_summary(
    dataframe: pd.DataFrame,
    dataset_name: str,
    patient_identifier: str,
) -> pd.DataFrame:
    """
    Summarise all valid ESI levels immediately before restricting
    the primary cohort to ESI-3.
    """

    visit_counts = (
        dataframe["esi_numeric"]
        .value_counts()
        .reindex(VALID_ESI_LEVELS, fill_value=0)
        .astype(int)
    )

    unique_patient_counts = (
        dataframe
        .groupby("esi_numeric")[patient_identifier]
        .nunique()
        .reindex(VALID_ESI_LEVELS, fill_value=0)
        .astype(int)
    )

    summary = pd.DataFrame(
        {
            "dataset": dataset_name,
            "ESI": VALID_ESI_LEVELS,
            "n_visits": visit_counts.values,
            "n_unique_patients": unique_patient_counts.values,
        }
    )

    summary["visit_percentage"] = (
        summary["n_visits"]
        / summary["n_visits"].sum()
        * 100
    )

    return summary


def plot_pre_restriction_esi_distribution(
    summary: pd.DataFrame,
    dataset_name: str,
    output_filename: str,
) -> Path:
    """
    Plot the visit-level ESI distribution before the ESI-3
    restriction and save it at 600 DPI.
    """

    plot_data = summary.sort_values("ESI").copy()

    figure, axis = plt.subplots(
        figsize=(8.8, 5.9)
    )

    bars = axis.bar(
        range(len(plot_data)),
        plot_data["n_visits"],
        edgecolor="black",
        linewidth=1,
    )

    for bar, count, percentage in zip(
        bars,
        plot_data["n_visits"],
        plot_data["visit_percentage"],
    ):
        axis.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{int(count):,}\n({percentage:.2f}%)",
            ha="center",
            va="bottom",
            fontsize=10,
        )

    axis.set_xticks(
        range(len(VALID_ESI_LEVELS))
    )

    axis.set_xticklabels(
        [
            ESI_LABELS[level]
            for level in VALID_ESI_LEVELS
        ]
    )

    axis.set_xlabel(
        "Emergency Severity Index level"
    )

    axis.set_ylabel(
        "Number of eligible ED visits"
    )

    axis.set_title(
        (
            f"{dataset_name}: ESI Distribution Before "
            "Restriction to ESI-3"
        ),
        weight="bold",
    )

    axis.set_ylim(
        0,
        max(plot_data["n_visits"].max(), 1) * 1.18,
    )

    figure.tight_layout()

    output_path = (
        FIGURE_DIR
        / output_filename
    )

    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return output_path


mimic_pre_esi3_distribution = (
    create_pre_restriction_esi_summary(
        dataframe=mimic_step_5,
        dataset_name="MIMIC-IV",
        patient_identifier="subject_id",
    )
)

mcmed_pre_esi3_distribution = (
    create_pre_restriction_esi_summary(
        dataframe=mcmed_step_5,
        dataset_name="MC-MED",
        patient_identifier="MRN",
    )
)


print("MIMIC-IV ESI distribution before ESI-3 restriction:")

display(
    mimic_pre_esi3_distribution.round(
        {"visit_percentage": 2}
    )
)


print("\nMC-MED ESI distribution before ESI-3 restriction:")

display(
    mcmed_pre_esi3_distribution.round(
        {"visit_percentage": 2}
    )
)


mimic_pre_esi3_figure = (
    plot_pre_restriction_esi_distribution(
        summary=mimic_pre_esi3_distribution,
        dataset_name="MIMIC-IV",
        output_filename=(
            "mimiciv_esi_distribution_before_esi3_"
            "restriction_600dpi.png"
        ),
    )
)

mcmed_pre_esi3_figure = (
    plot_pre_restriction_esi_distribution(
        summary=mcmed_pre_esi3_distribution,
        dataset_name="MC-MED",
        output_filename=(
            "mcmed_esi_distribution_before_esi3_"
            "restriction_600dpi.png"
        ),
    )
)


print("Saved:", mimic_pre_esi3_figure)
print("Saved:", mcmed_pre_esi3_figure)


In [ ]:


def classify_abdominal_complaint_scope(
    series: pd.Series,
) -> pd.DataFrame:
    """
    Classify chief complaints into exact, combined-symptom,
    localised/quadrant and non-abdominal categories.
    """

    original = (
        series
        .fillna("")
        .astype(str)
    )

    normalised = normalise_text(original)


    exact_general = normalised.isin(
        PRIMARY_ABDOMINAL_COMPLAINTS
    )


    general_abdominal_pattern = (
        r"\b(?:"
        r"ABDOMINAL|ABD|ABDO|ABDOMEN|BELLY|STOMACH"
        r")\s+PAIN\b"
    )


    quadrant_abbreviation_pattern = (
        r"\b(?:RLQ|RUQ|LLQ|LUQ)\b"
    )


    quadrant_text_pattern = (
        r"\b(?:RIGHT|LEFT)\s+"
        r"(?:UPPER|LOWER)\s+"
        r"(?:QUADRANT\s+)?"
        r"(?:ABDOMINAL\s+)?"
        r"PAIN\b"
    )


    location_pattern = (
        r"\b(?:UPPER|LOWER)\s+"
        r"(?:ABDOMINAL\s+)?"
        r"PAIN\b"
        r"|"
        r"\b(?:RIGHT|LEFT)[ -]?SIDED\s+"
        r"(?:ABDOMINAL\s+)?"
        r"PAIN\b"
    )


    contains_general_abdominal_pain = (
        normalised.str.contains(
            general_abdominal_pattern,
            regex=True,
            na=False,
        )
    )


    contains_localised_abdominal_pain = (
        normalised.str.contains(
            quadrant_abbreviation_pattern,
            regex=True,
            na=False,
        )
        |
        normalised.str.contains(
            quadrant_text_pattern,
            regex=True,
            na=False,
        )
        |
        normalised.str.contains(
            location_pattern,
            regex=True,
            na=False,
        )
    )


    general_with_additional_text = (
        contains_general_abdominal_pain
        & ~exact_general
        & ~contains_localised_abdominal_pain
    )


    complaint_category = pd.Series(
        "No explicit abdominal-pain complaint",
        index=series.index,
        dtype="object",
    )


    complaint_category.loc[
        general_with_additional_text
    ] = (
        "General abdominal pain with additional "
        "text or symptoms"
    )


    complaint_category.loc[
        contains_localised_abdominal_pain
    ] = (
        "Quadrant or location-specific abdominal pain"
    )


    complaint_category.loc[
        exact_general
    ] = (
        "Exact general abdominal pain"
    )


    return pd.DataFrame(
        {
            "chief_complaint_original": original,
            "chief_complaint_normalised": normalised,
            "complaint_category": complaint_category,
            "exact_general_abdominal_pain": exact_general,
            "general_with_additional_text": (
                general_with_additional_text
            ),
            "localised_abdominal_pain": (
                contains_localised_abdominal_pain
            ),
            "any_explicit_abdominal_pain": (
                exact_general
                | general_with_additional_text
                | contains_localised_abdominal_pain
            ),
        },
        index=series.index,
    )



mimic_complaint_scope = (
    mimic_step_3[
        [
            "subject_id",
            "stay_id",
            "chiefcomplaint",
        ]
    ]
    .copy()
)

mimic_complaint_classification = (
    classify_abdominal_complaint_scope(
        mimic_complaint_scope[
            "chiefcomplaint"
        ]
    )
)

for column in mimic_complaint_classification.columns:
    mimic_complaint_scope[column] = (
        mimic_complaint_classification[column]
    )



mcmed_complaint_scope = (
    mcmed_step_3[
        [
            "MRN",
            "CSN",
            "CC",
        ]
    ]
    .copy()
)

mcmed_complaint_classification = (
    classify_abdominal_complaint_scope(
        mcmed_complaint_scope["CC"]
    )
)

for column in mcmed_complaint_classification.columns:
    mcmed_complaint_scope[column] = (
        mcmed_complaint_classification[column]
    )


def summarise_complaint_scope(
    dataframe: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """
    Summarise complaint-category prevalence among adult ED visits.
    """

    summary = (
        dataframe[
            "complaint_category"
        ]
        .value_counts(dropna=False)
        .rename_axis(
            "complaint_category"
        )
        .reset_index(
            name="n_visits"
        )
    )

    summary[
        "percentage_of_all_adult_visits"
    ] = (
        summary["n_visits"]
        / len(dataframe)
        * 100
    )

    abdominal_total = int(
        dataframe[
            "any_explicit_abdominal_pain"
        ].sum()
    )

    summary[
        "percentage_of_abdominal_pain_visits"
    ] = np.where(
        summary[
            "complaint_category"
        ].eq(
            "No explicit abdominal-pain complaint"
        ),
        np.nan,
        (
            summary["n_visits"]
            / abdominal_total
            * 100
            if abdominal_total > 0
            else np.nan
        ),
    )

    summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    return summary


mimic_complaint_scope_summary = (
    summarise_complaint_scope(
        dataframe=mimic_complaint_scope,
        dataset_name="MIMIC-IV",
    )
)

mcmed_complaint_scope_summary = (
    summarise_complaint_scope(
        dataframe=mcmed_complaint_scope,
        dataset_name="MC-MED",
    )
)


print("MIMIC-IV chief-complaint phenotype audit:")

display(
    mimic_complaint_scope_summary.round(
        {
            "percentage_of_all_adult_visits": 3,
            "percentage_of_abdominal_pain_visits": 2,
        }
    )
)


print("\nMC-MED chief-complaint phenotype audit:")

display(
    mcmed_complaint_scope_summary.round(
        {
            "percentage_of_all_adult_visits": 3,
            "percentage_of_abdominal_pain_visits": 2,
        }
    )
)


In [ ]:


def create_complaint_example_table(
    dataframe: pd.DataFrame,
    dataset_name: str,
    top_n_per_category: int = 25,
) -> pd.DataFrame:
    """
    Return the most frequent original complaint strings within each
    abdominal-pain phenotype category.
    """

    abdominal_only = dataframe.loc[
        dataframe[
            "any_explicit_abdominal_pain"
        ]
    ].copy()

    example_tables = []

    for category in [
        "Exact general abdominal pain",
        (
            "General abdominal pain with additional "
            "text or symptoms"
        ),
        (
            "Quadrant or location-specific "
            "abdominal pain"
        ),
    ]:
        category_rows = abdominal_only.loc[
            abdominal_only[
                "complaint_category"
            ].eq(category)
        ]

        category_summary = (
            category_rows[
                "chief_complaint_original"
            ]
            .fillna("MISSING")
            .value_counts(dropna=False)
            .head(top_n_per_category)
            .rename_axis(
                "original_chief_complaint"
            )
            .reset_index(
                name="count"
            )
        )

        category_summary.insert(
            0,
            "complaint_category",
            category,
        )

        category_summary.insert(
            0,
            "dataset",
            dataset_name,
        )

        example_tables.append(
            category_summary
        )

    return pd.concat(
        example_tables,
        ignore_index=True,
    )


mimic_complaint_examples = (
    create_complaint_example_table(
        dataframe=mimic_complaint_scope,
        dataset_name="MIMIC-IV",
        top_n_per_category=25,
    )
)

mcmed_complaint_examples = (
    create_complaint_example_table(
        dataframe=mcmed_complaint_scope,
        dataset_name="MC-MED",
        top_n_per_category=25,
    )
)


print("MIMIC-IV abdominal-pain complaint examples:")

display(mimic_complaint_examples)


print("\nMC-MED abdominal-pain complaint examples:")

display(mcmed_complaint_examples)


#### Disposition harmonisation

This section determines how emergency-department dispositions should be harmonised between MIMIC-IV and MC-MED before constructing the binary HOME-versus-ADMITTED outcome.

MIMIC-IV-ED records HOME and ADMITTED at the ED level. For ED visits followed by hospitalisation, the `hadm_id` can be linked to the MIMIC-IV hospital admissions table. This permits observation-type hospital admissions to be distinguished from other hospital admissions.

MC-MED directly separates DISCHARGE, INPATIENT, OBSERVATION and ICU dispositions.

Two candidate outcome definitions are evaluated:

1. **Primary strict inpatient endpoint**
   - HOME: MIMIC-IV HOME; MC-MED DISCHARGE
   - ADMITTED: MIMIC-IV non-observation hospital admission; MC-MED INPATIENT or ICU
   - OBSERVATION and non-binary outcomes are excluded

2. **Hospital-based disposition sensitivity endpoint**
   - HOME: MIMIC-IV HOME; MC-MED DISCHARGE
   - ADMITTED: all MIMIC-IV ADMITTED outcomes; MC-MED INPATIENT, ICU or OBSERVATION
   - Other non-binary outcomes are excluded

No final outcome is frozen until hospital-admission linkage, observation categories and resulting cohort sizes have been inspected.


In [ ]:



if "hadm_id" not in mimic_esi3_candidate.columns:
    edstay_link_columns = pd.read_csv(
        SOURCE_PATHS["mimic_edstays"],
        usecols=[
            "subject_id",
            "stay_id",
            "hadm_id",
        ],
        low_memory=False,
    )

    for column in [
        "subject_id",
        "stay_id",
        "hadm_id",
    ]:
        edstay_link_columns[column] = pd.to_numeric(
            edstay_link_columns[column],
            errors="coerce",
        ).astype("Int64")

    mimic_esi3_candidate = mimic_esi3_candidate.merge(
        edstay_link_columns,
        on=[
            "subject_id",
            "stay_id",
        ],
        how="left",
        validate="one_to_one",
    )



for column in [
    "subject_id",
    "stay_id",
    "hadm_id",
]:
    mimic_esi3_candidate[column] = pd.to_numeric(
        mimic_esi3_candidate[column],
        errors="coerce",
    ).astype("Int64")



mimic_admissions_path = SOURCE_PATHS[
    "mimic_admissions"
]

mimic_admissions_available_columns = pd.read_csv(
    mimic_admissions_path,
    nrows=0,
).columns.tolist()

mimic_admissions_required_columns = [
    "subject_id",
    "hadm_id",
    "admission_type",
]

mimic_admissions_optional_columns = [
    "admittime",
    "dischtime",
    "admission_location",
    "discharge_location",
    "hospital_expire_flag",
]

missing_mimic_admissions_columns = [
    column
    for column in mimic_admissions_required_columns
    if column not in mimic_admissions_available_columns
]

if missing_mimic_admissions_columns:
    raise ValueError(
        "MIMIC admissions is missing required columns: "
        f"{missing_mimic_admissions_columns}"
    )

mimic_admissions_columns_to_load = (
    mimic_admissions_required_columns
    + [
        column
        for column in mimic_admissions_optional_columns
        if column in mimic_admissions_available_columns
    ]
)

mimic_admissions_audit = pd.read_csv(
    mimic_admissions_path,
    usecols=mimic_admissions_columns_to_load,
    low_memory=False,
)

for column in [
    "subject_id",
    "hadm_id",
]:
    mimic_admissions_audit[column] = pd.to_numeric(
        mimic_admissions_audit[column],
        errors="coerce",
    ).astype("Int64")

mimic_admissions_audit["admission_type"] = (
    mimic_admissions_audit["admission_type"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
    .replace("", pd.NA)
)

assert mimic_admissions_audit["hadm_id"].is_unique



mcmed_visits_path = SOURCE_PATHS[
    "mcmed_visits"
]

mcmed_available_columns = pd.read_csv(
    mcmed_visits_path,
    nrows=0,
).columns.tolist()

mcmed_audit_optional_columns = [
    "Admit_time",
    "Admit_service",
    "Hosp_LOS",
    "DC_dispo",
    "Dispo_time",
    "Departure_time",
]

mcmed_audit_columns_to_load = [
    "CSN",
] + [
    column
    for column in mcmed_audit_optional_columns
    if column in mcmed_available_columns
]

mcmed_visit_disposition_fields = pd.read_csv(
    mcmed_visits_path,
    usecols=mcmed_audit_columns_to_load,
    low_memory=False,
)

mcmed_visit_disposition_fields["CSN"] = pd.to_numeric(
    mcmed_visit_disposition_fields["CSN"],
    errors="coerce",
).astype("Int64")

assert mcmed_visit_disposition_fields["CSN"].is_unique



mcmed_fields_to_merge = [
    column
    for column in mcmed_audit_optional_columns
    if (
        column in mcmed_visit_disposition_fields.columns
        and column not in mcmed_esi3_candidate.columns
    )
]

mcmed_esi3_disposition_audit = (
    mcmed_esi3_candidate.merge(
        mcmed_visit_disposition_fields[
            [
                "CSN",
                *mcmed_fields_to_merge,
            ]
        ],
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


print(
    "MIMIC candidate rows:",
    f"{len(mimic_esi3_candidate):,}",
)

print(
    "MIMIC admissions rows:",
    f"{len(mimic_admissions_audit):,}",
)

print(
    "MC-MED candidate rows:",
    f"{len(mcmed_esi3_disposition_audit):,}",
)

print(
    "MC-MED audit fields:",
    mcmed_fields_to_merge,
)


#### MIMIC-IV admission and observation audit

MIMIC hospital admission_type contains explicit observation categories such as AMBULATORY OBSERVATION, DIRECT OBSERVATION, EU OBSERVATION, and OBSERVATION ADMIT.


In [ ]:

MIMIC_OBSERVATION_ADMISSION_TYPES = {
    "AMBULATORY OBSERVATION",
    "DIRECT OBSERVATION",
    "EU OBSERVATION",
    "OBSERVATION ADMIT",
}



mimic_disposition_audit = (
    mimic_esi3_candidate.merge(
        mimic_admissions_audit,
        on=[
            "subject_id",
            "hadm_id",
        ],
        how="left",
        validate="many_to_one",
        indicator="hospital_admission_link",
    )
)



mimic_disposition_audit[
    "hadm_id_present"
] = mimic_disposition_audit[
    "hadm_id"
].notna()

mimic_disposition_audit[
    "hospital_record_matched"
] = mimic_disposition_audit[
    "hospital_admission_link"
].eq("both")

mimic_disposition_audit[
    "is_observation_admission"
] = mimic_disposition_audit[
    "admission_type"
].isin(
    MIMIC_OBSERVATION_ADMISSION_TYPES
)



mimic_disposition_audit[
    "resolved_hospital_category"
] = "NON_BINARY_RAW_DISPOSITION"

home_mask = mimic_disposition_audit[
    "disposition_clean"
].eq("HOME")

admitted_mask = mimic_disposition_audit[
    "disposition_clean"
].eq("ADMITTED")

observation_mask = (
    admitted_mask
    & mimic_disposition_audit[
        "is_observation_admission"
    ]
)

non_observation_admission_mask = (
    admitted_mask
    & mimic_disposition_audit[
        "hospital_record_matched"
    ]
    & ~mimic_disposition_audit[
        "is_observation_admission"
    ]
)

unresolved_admission_mask = (
    admitted_mask
    & ~mimic_disposition_audit[
        "hospital_record_matched"
    ]
)

mimic_disposition_audit.loc[
    home_mask,
    "resolved_hospital_category",
] = "HOME"

mimic_disposition_audit.loc[
    observation_mask,
    "resolved_hospital_category",
] = "OBSERVATION"

mimic_disposition_audit.loc[
    non_observation_admission_mask,
    "resolved_hospital_category",
] = "NON_OBSERVATION_HOSPITAL_ADMISSION"

mimic_disposition_audit.loc[
    unresolved_admission_mask,
    "resolved_hospital_category",
] = "UNRESOLVED_ADMITTED"



mimic_hadm_audit = (
    mimic_disposition_audit
    .groupby(
        [
            "disposition_clean",
            "hadm_id_present",
        ],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)



mimic_admitted_rows = (
    mimic_disposition_audit.loc[
        mimic_disposition_audit[
            "disposition_clean"
        ].eq("ADMITTED")
    ]
    .copy()
)

mimic_admission_type_summary = (
    mimic_admitted_rows[
        "admission_type"
    ]
    .fillna("UNMATCHED OR MISSING")
    .value_counts(dropna=False)
    .rename_axis("admission_type")
    .reset_index(name="count")
)

mimic_admission_type_summary[
    "percentage"
] = (
    mimic_admission_type_summary["count"]
    / len(mimic_admitted_rows)
    * 100
)



mimic_resolved_category_summary = (
    mimic_disposition_audit[
        "resolved_hospital_category"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "resolved_hospital_category"
    )
    .reset_index(name="count")
)

mimic_resolved_category_summary[
    "percentage"
] = (
    mimic_resolved_category_summary["count"]
    / len(mimic_disposition_audit)
    * 100
)



mimic_admission_linkage_summary = pd.DataFrame(
    [
        {
            "measure": "Raw ADMITTED visits",
            "count": len(mimic_admitted_rows),
        },
        {
            "measure": "ADMITTED with hadm_id",
            "count": int(
                mimic_admitted_rows[
                    "hadm_id_present"
                ].sum()
            ),
        },
        {
            "measure": (
                "ADMITTED matched to hospital admissions"
            ),
            "count": int(
                mimic_admitted_rows[
                    "hospital_record_matched"
                ].sum()
            ),
        },
        {
            "measure": (
                "ADMITTED resolved as observation"
            ),
            "count": int(
                mimic_admitted_rows[
                    "is_observation_admission"
                ].sum()
            ),
        },
        {
            "measure": (
                "ADMITTED unresolved after linkage"
            ),
            "count": int(
                (
                    mimic_admitted_rows[
                        "resolved_hospital_category"
                    ]
                    == "UNRESOLVED_ADMITTED"
                ).sum()
            ),
        },
    ]
)


print("MIMIC-IV raw disposition versus hadm_id availability:")

display(mimic_hadm_audit)


print("\nMIMIC-IV admission types among raw ADMITTED visits:")

display(
    mimic_admission_type_summary.round(
        {"percentage": 2}
    )
)


print("\nMIMIC-IV resolved hospital categories:")

display(
    mimic_resolved_category_summary.round(
        {"percentage": 2}
    )
)


print("\nMIMIC-IV hospital-admission linkage quality:")

display(mimic_admission_linkage_summary)


#### MC-MED observation and hospital-field audit

MC-MED directly reports four ED dispositions and provides fields such as Admit_time, Admit_service, Hosp_LOS, and DC_dispo for hospital-based care


In [ ]:

MC_MED_HOSPITAL_AUDIT_FIELDS = [
    column
    for column in [
        "Admit_time",
        "Admit_service",
        "Hosp_LOS",
        "DC_dispo",
    ]
    if column in mcmed_esi3_disposition_audit.columns
]



mcmed_hospital_field_records = []

for (
    disposition,
    disposition_rows,
) in mcmed_esi3_disposition_audit.groupby(
    "disposition_clean",
    dropna=False,
):
    record = {
        "raw_disposition": disposition,
        "n_visits": len(disposition_rows),
    }

    for field in MC_MED_HOSPITAL_AUDIT_FIELDS:
        non_missing_count = int(
            disposition_rows[field]
            .notna()
            .sum()
        )

        record[
            f"{field}_available_n"
        ] = non_missing_count

        record[
            f"{field}_available_pct"
        ] = (
            non_missing_count
            / len(disposition_rows)
            * 100
            if len(disposition_rows) > 0
            else np.nan
        )

    mcmed_hospital_field_records.append(record)


mcmed_hospital_field_summary = pd.DataFrame(
    mcmed_hospital_field_records
)



if "Hosp_LOS" in mcmed_esi3_disposition_audit.columns:
    mcmed_esi3_disposition_audit[
        "Hosp_LOS_numeric"
    ] = pd.to_numeric(
        mcmed_esi3_disposition_audit[
            "Hosp_LOS"
        ],
        errors="coerce",
    )

    mcmed_hospital_los_summary = (
        mcmed_esi3_disposition_audit
        .groupby(
            "disposition_clean",
            dropna=False,
        )[
            "Hosp_LOS_numeric"
        ]
        .agg(
            n_non_missing="count",
            mean="mean",
            median="median",
            minimum="min",
            maximum="max",
        )
        .reset_index()
    )
else:
    mcmed_hospital_los_summary = pd.DataFrame()



if "Admit_service" in mcmed_esi3_disposition_audit.columns:
    mcmed_admit_service_summary = (
        mcmed_esi3_disposition_audit.loc[
            mcmed_esi3_disposition_audit[
                "disposition_clean"
            ].isin(
                [
                    "INPATIENT",
                    "OBSERVATION",
                    "ICU",
                ]
            )
        ]
        .groupby(
            [
                "disposition_clean",
                "Admit_service",
            ],
            dropna=False,
        )
        .size()
        .rename("count")
        .reset_index()
        .sort_values(
            [
                "disposition_clean",
                "count",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )
else:
    mcmed_admit_service_summary = pd.DataFrame()


print("MC-MED hospital-field availability by disposition:")

display(
    mcmed_hospital_field_summary.round(2)
)


print("\nMC-MED hospital length-of-stay summary:")

display(
    mcmed_hospital_los_summary.round(2)
)


print("\nMC-MED top admitting-service values by disposition:")

if not mcmed_admit_service_summary.empty:
    display(
        mcmed_admit_service_summary
        .groupby(
            "disposition_clean",
            group_keys=False,
        )
        .head(10)
    )
else:
    print("Admit_service was not available.")


#### Candidate binary-outcome mappings

This cell creates proposed labels, but it does not overwrite your candidate cohorts.


In [ ]:



mimic_disposition_audit[
    "outcome_primary_strict"
] = pd.NA

mimic_disposition_audit.loc[
    mimic_disposition_audit[
        "resolved_hospital_category"
    ].eq("HOME"),
    "outcome_primary_strict",
] = "HOME"

mimic_disposition_audit.loc[
    mimic_disposition_audit[
        "resolved_hospital_category"
    ].eq(
        "NON_OBSERVATION_HOSPITAL_ADMISSION"
    ),
    "outcome_primary_strict",
] = "ADMITTED"



mimic_disposition_audit[
    "outcome_sensitivity_hospital_based"
] = pd.NA

mimic_disposition_audit.loc[
    mimic_disposition_audit[
        "disposition_clean"
    ].eq("HOME"),
    "outcome_sensitivity_hospital_based",
] = "HOME"

mimic_disposition_audit.loc[
    mimic_disposition_audit[
        "disposition_clean"
    ].eq("ADMITTED"),
    "outcome_sensitivity_hospital_based",
] = "ADMITTED"



mcmed_esi3_disposition_audit[
    "outcome_primary_strict"
] = pd.NA

mcmed_esi3_disposition_audit.loc[
    mcmed_esi3_disposition_audit[
        "disposition_clean"
    ].eq("DISCHARGE"),
    "outcome_primary_strict",
] = "HOME"

mcmed_esi3_disposition_audit.loc[
    mcmed_esi3_disposition_audit[
        "disposition_clean"
    ].isin(
        [
            "INPATIENT",
            "ICU",
        ]
    ),
    "outcome_primary_strict",
] = "ADMITTED"



mcmed_esi3_disposition_audit[
    "outcome_sensitivity_hospital_based"
] = pd.NA

mcmed_esi3_disposition_audit.loc[
    mcmed_esi3_disposition_audit[
        "disposition_clean"
    ].eq("DISCHARGE"),
    "outcome_sensitivity_hospital_based",
] = "HOME"

mcmed_esi3_disposition_audit.loc[
    mcmed_esi3_disposition_audit[
        "disposition_clean"
    ].isin(
        [
            "INPATIENT",
            "ICU",
            "OBSERVATION",
        ]
    ),
    "outcome_sensitivity_hospital_based",
] = "ADMITTED"



def summarise_candidate_mapping(
    dataframe: pd.DataFrame,
    dataset_name: str,
    mapping_column: str,
    mapping_name: str,
) -> pd.DataFrame:
    total_candidate_rows = len(dataframe)

    included_rows = dataframe.loc[
        dataframe[mapping_column].notna()
    ].copy()

    summary = (
        included_rows[mapping_column]
        .value_counts(dropna=False)
        .reindex(
            [
                "HOME",
                "ADMITTED",
            ],
            fill_value=0,
        )
        .rename_axis("outcome")
        .reset_index(name="count")
    )

    summary["percentage_among_included"] = (
        summary["count"]
        / len(included_rows)
        * 100
        if len(included_rows) > 0
        else np.nan
    )

    summary["total_candidate_rows"] = (
        total_candidate_rows
    )

    summary["total_binary_rows"] = (
        len(included_rows)
    )

    summary["excluded_rows"] = (
        total_candidate_rows
        - len(included_rows)
    )

    summary.insert(
        0,
        "mapping",
        mapping_name,
    )

    summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    return summary


disposition_mapping_comparison = pd.concat(
    [
        summarise_candidate_mapping(
            dataframe=mimic_disposition_audit,
            dataset_name="MIMIC-IV",
            mapping_column=(
                "outcome_primary_strict"
            ),
            mapping_name=(
                "Primary strict inpatient endpoint"
            ),
        ),
        summarise_candidate_mapping(
            dataframe=mcmed_esi3_disposition_audit,
            dataset_name="MC-MED",
            mapping_column=(
                "outcome_primary_strict"
            ),
            mapping_name=(
                "Primary strict inpatient endpoint"
            ),
        ),
        summarise_candidate_mapping(
            dataframe=mimic_disposition_audit,
            dataset_name="MIMIC-IV",
            mapping_column=(
                "outcome_sensitivity_hospital_based"
            ),
            mapping_name=(
                "Hospital-based sensitivity endpoint"
            ),
        ),
        summarise_candidate_mapping(
            dataframe=mcmed_esi3_disposition_audit,
            dataset_name="MC-MED",
            mapping_column=(
                "outcome_sensitivity_hospital_based"
            ),
            mapping_name=(
                "Hospital-based sensitivity endpoint"
            ),
        ),
    ],
    ignore_index=True,
)


display(
    disposition_mapping_comparison.round(
        {
            "percentage_among_included": 2,
        }
    )
)



TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

mimic_admission_type_summary.to_csv(
    TABLE_DIR
    / "mimiciv_admission_type_audit.csv",
    index=False,
)

mcmed_hospital_field_summary.to_csv(
    TABLE_DIR
    / "mcmed_disposition_hospital_field_audit.csv",
    index=False,
)

disposition_mapping_comparison.to_csv(
    TABLE_DIR
    / "disposition_mapping_comparison.csv",
    index=False,
)


print(
    "\nDisposition audit tables saved to:",
    TABLE_DIR,
)

print(
    "\nNo final binary cohort has been frozen yet."
)


In [ ]:

mimic_home_with_hadm = (
    mimic_disposition_audit.loc[
        mimic_disposition_audit[
            "disposition_clean"
        ].eq("HOME")
        &
        mimic_disposition_audit[
            "hadm_id_present"
        ]
    ]
    .copy()
)



for timestamp_column in [
    "intime",
    "outtime",
    "admittime",
    "dischtime",
]:
    if timestamp_column in mimic_home_with_hadm.columns:
        mimic_home_with_hadm[
            timestamp_column
        ] = pd.to_datetime(
            mimic_home_with_hadm[
                timestamp_column
            ],
            errors="coerce",
        )



if {
    "admittime",
    "intime",
}.issubset(
    mimic_home_with_hadm.columns
):
    mimic_home_with_hadm[
        "hours_from_ed_arrival_to_hospital_admission"
    ] = (
        mimic_home_with_hadm["admittime"]
        - mimic_home_with_hadm["intime"]
    ).dt.total_seconds() / 3600


if {
    "admittime",
    "outtime",
}.issubset(
    mimic_home_with_hadm.columns
):
    mimic_home_with_hadm[
        "hours_from_ed_departure_to_hospital_admission"
    ] = (
        mimic_home_with_hadm["admittime"]
        - mimic_home_with_hadm["outtime"]
    ).dt.total_seconds() / 3600


if {
    "admittime",
    "dischtime",
}.issubset(
    mimic_home_with_hadm.columns
):
    mimic_home_with_hadm[
        "hospital_los_days"
    ] = (
        mimic_home_with_hadm["dischtime"]
        - mimic_home_with_hadm["admittime"]
    ).dt.total_seconds() / 86400



mimic_home_hadm_admission_types = (
    mimic_home_with_hadm[
        "admission_type"
    ]
    .fillna("UNMATCHED OR MISSING")
    .value_counts(dropna=False)
    .rename_axis("admission_type")
    .reset_index(name="count")
)

mimic_home_hadm_admission_types[
    "percentage"
] = (
    mimic_home_hadm_admission_types["count"]
    / len(mimic_home_with_hadm)
    * 100
)



mimic_home_hadm_category_summary = (
    mimic_home_with_hadm.assign(
        linked_category=np.select(
            [
                ~mimic_home_with_hadm[
                    "hospital_record_matched"
                ],
                mimic_home_with_hadm[
                    "is_observation_admission"
                ],
            ],
            [
                "UNMATCHED",
                "OBSERVATION",
            ],
            default="NON_OBSERVATION_ADMISSION",
        )
    )
    ["linked_category"]
    .value_counts(dropna=False)
    .rename_axis("linked_category")
    .reset_index(name="count")
)

mimic_home_hadm_category_summary[
    "percentage"
] = (
    mimic_home_hadm_category_summary["count"]
    / len(mimic_home_with_hadm)
    * 100
)



timing_columns = [
    column
    for column in [
        "hours_from_ed_arrival_to_hospital_admission",
        "hours_from_ed_departure_to_hospital_admission",
        "hospital_los_days",
    ]
    if column in mimic_home_with_hadm.columns
]

if timing_columns:
    mimic_home_hadm_timing_summary = (
        mimic_home_with_hadm[
            timing_columns
        ]
        .describe(
            percentiles=[
                0.25,
                0.50,
                0.75,
            ]
        )
        .T
        .reset_index()
        .rename(
            columns={
                "index": "measure",
            }
        )
    )
else:
    mimic_home_hadm_timing_summary = (
        pd.DataFrame()
    )


print(
    "MIMIC HOME encounters with hadm_id:",
    f"{len(mimic_home_with_hadm):,}",
)

print(
    "\nAdmission types among HOME encounters "
    "with hadm_id:"
)

display(
    mimic_home_hadm_admission_types.round(
        {"percentage": 2}
    )
)

print(
    "\nLinked hospital categories among HOME "
    "encounters with hadm_id:"
)

display(
    mimic_home_hadm_category_summary.round(
        {"percentage": 2}
    )
)

print(
    "\nTiming summary:"
)

display(
    mimic_home_hadm_timing_summary.round(2)
)



example_columns = [
    column
    for column in [
        "subject_id",
        "stay_id",
        "hadm_id",
        "intime",
        "outtime",
        "admittime",
        "dischtime",
        "admission_type",
        "admission_location",
        "discharge_location",
        "hours_from_ed_arrival_to_hospital_admission",
        "hours_from_ed_departure_to_hospital_admission",
        "hospital_los_days",
    ]
    if column in mimic_home_with_hadm.columns
]

print(
    "\nExample discordant HOME encounters:"
)

display(
    mimic_home_with_hadm[
        example_columns
    ]
    .sort_values(
        "admittime",
        na_position="last",
    )
    .head(25)
)


#### MIMIC-IV ICU and hospital length-of-stay audit

Before freezing the final HOME-versus-ADMITTED outcome, non-observation MIMIC-IV hospital admissions are examined for ICU utilisation.

A hospitalisation is identified as ICU-associated when its `hadm_id` is present in the MIMIC-IV `icustays` table. Because an ICU episode may begin immediately after ED departure or later during the hospitalisation, two related measures are reported:

- any ICU stay occurring during the hospital admission;
- timing of the first ICU entry relative to ED arrival and ED departure.

MIMIC-IV hospitalisations are then classified into three mutually exclusive categories:

- observation admission;
- non-observation inpatient admission without a recorded ICU stay;
- non-observation hospital admission with at least one recorded ICU stay.

Hospital length of stay is calculated from hospital admission time to hospital discharge time. Mean, median, interquartile range and the mode after rounding to the nearest whole day are reported to support comparison with MC-MED.

This section is an audit only. It does not yet modify or save the final binary cohorts.


In [ ]:



if "mimic_icustays" in SOURCE_PATHS:
    mimic_icustays_path = SOURCE_PATHS["mimic_icustays"]
else:
    mimic_icustays_path = (
        MIMIC_IV_HOSP_DIR.parent
        / "icu"
        / "icustays.csv.gz"
    )

print("MIMIC-IV ICU stays path:")
print(mimic_icustays_path)

if not mimic_icustays_path.exists():
    raise FileNotFoundError(
        "The MIMIC-IV icustays table was not found:\n"
        f"{mimic_icustays_path}\n\n"
        "Expected location:\n"
        ".../mimiciv/3.1/icu/icustays.csv.gz"
    )



mimic_icustays_available_columns = pd.read_csv(
    mimic_icustays_path,
    nrows=0,
).columns.tolist()

mimic_icustays_required_columns = [
    "subject_id",
    "hadm_id",
    "stay_id",
    "first_careunit",
    "last_careunit",
    "intime",
    "outtime",
    "los",
]

missing_icu_columns = [
    column
    for column in mimic_icustays_required_columns
    if column not in mimic_icustays_available_columns
]

if missing_icu_columns:
    raise ValueError(
        "MIMIC-IV icustays is missing required columns:\n"
        f"{missing_icu_columns}"
    )

mimic_icustays = pd.read_csv(
    mimic_icustays_path,
    usecols=mimic_icustays_required_columns,
    low_memory=False,
)



for column in [
    "subject_id",
    "hadm_id",
    "stay_id",
]:
    mimic_icustays[column] = pd.to_numeric(
        mimic_icustays[column],
        errors="coerce",
    ).astype("Int64")

for column in [
    "intime",
    "outtime",
]:
    mimic_icustays[column] = pd.to_datetime(
        mimic_icustays[column],
        errors="coerce",
    )

mimic_icustays["los"] = pd.to_numeric(
    mimic_icustays["los"],
    errors="coerce",
)



assert mimic_icustays["stay_id"].notna().all()
assert mimic_icustays["stay_id"].is_unique
assert mimic_icustays["hadm_id"].notna().all()

print(
    "MIMIC-IV ICU rows:",
    f"{len(mimic_icustays):,}",
)

print(
    "Unique ICU hospital admissions:",
    f"{mimic_icustays['hadm_id'].nunique():,}",
)



mimic_icustays_sorted = (
    mimic_icustays
    .sort_values(
        [
            "subject_id",
            "hadm_id",
            "intime",
            "stay_id",
        ]
    )
    .copy()
)



mimic_icu_by_admission = (
    mimic_icustays_sorted
    .groupby(
        [
            "subject_id",
            "hadm_id",
        ],
        as_index=False,
    )
    .agg(
        n_icu_stays=(
            "stay_id",
            "nunique",
        ),
        first_icu_stay_id=(
            "stay_id",
            "first",
        ),
        first_icu_careunit=(
            "first_careunit",
            "first",
        ),
        last_icu_careunit=(
            "last_careunit",
            "last",
        ),
        first_icu_intime=(
            "intime",
            "min",
        ),
        final_icu_outtime=(
            "outtime",
            "max",
        ),
        total_recorded_icu_los_days=(
            "los",
            "sum",
        ),
    )
)


assert not mimic_icu_by_admission[
    [
        "subject_id",
        "hadm_id",
    ]
].duplicated().any()


print(
    "\nAggregated hospital admissions with ICU records:",
    f"{len(mimic_icu_by_admission):,}",
)

display(
    mimic_icu_by_admission.head()
)


#### Observation, inpatient and ICU classification

This cell examines only:

confirmed observation admissions;
confirmed non-observation hospital admissions.

The unresolved 21 ADMITTED cases and non-binary ED dispositions remain outside this audit.


In [ ]:


mimic_hospital_los_audit = (
    mimic_disposition_audit.loc[
        mimic_disposition_audit[
            "resolved_hospital_category"
        ].isin(
            [
                "OBSERVATION",
                "NON_OBSERVATION_HOSPITAL_ADMISSION",
            ]
        )
    ]
    .copy()
)



mimic_hospital_los_audit = (
    mimic_hospital_los_audit.merge(
        mimic_icu_by_admission,
        on=[
            "subject_id",
            "hadm_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


mimic_hospital_los_audit[
    "has_recorded_icu_stay"
] = mimic_hospital_los_audit[
    "first_icu_stay_id"
].notna()



for column in [
    "admittime",
    "dischtime",
    "intime",
    "outtime",
    "first_icu_intime",
    "final_icu_outtime",
]:
    if column in mimic_hospital_los_audit.columns:
        mimic_hospital_los_audit[column] = pd.to_datetime(
            mimic_hospital_los_audit[column],
            errors="coerce",
        )


mimic_hospital_los_audit[
    "hospital_los_days"
] = (
    mimic_hospital_los_audit["dischtime"]
    - mimic_hospital_los_audit["admittime"]
).dt.total_seconds() / 86400



mimic_hospital_los_audit[
    "hospital_disposition_group"
] = pd.Series(
    pd.NA,
    index=mimic_hospital_los_audit.index,
    dtype="string",
)


observation_mask = mimic_hospital_los_audit[
    "resolved_hospital_category"
].eq("OBSERVATION")


non_observation_mask = mimic_hospital_los_audit[
    "resolved_hospital_category"
].eq(
    "NON_OBSERVATION_HOSPITAL_ADMISSION"
)


mimic_hospital_los_audit.loc[
    observation_mask,
    "hospital_disposition_group",
] = "OBSERVATION"


mimic_hospital_los_audit.loc[
    non_observation_mask
    & ~mimic_hospital_los_audit[
        "has_recorded_icu_stay"
    ],
    "hospital_disposition_group",
] = "INPATIENT_NO_ICU"


mimic_hospital_los_audit.loc[
    non_observation_mask
    & mimic_hospital_los_audit[
        "has_recorded_icu_stay"
    ],
    "hospital_disposition_group",
] = "ICU_ASSOCIATED"


assert mimic_hospital_los_audit[
    "hospital_disposition_group"
].notna().all()



mimic_non_observation_only = (
    mimic_hospital_los_audit.loc[
        non_observation_mask
    ]
    .copy()
)


mimic_non_observation_icu_summary = pd.DataFrame(
    [
        {
            "category": "Non-observation admissions without ICU",
            "count": int(
                (
                    ~mimic_non_observation_only[
                        "has_recorded_icu_stay"
                    ]
                ).sum()
            ),
        },
        {
            "category": "Non-observation admissions with ICU",
            "count": int(
                mimic_non_observation_only[
                    "has_recorded_icu_stay"
                ].sum()
            ),
        },
    ]
)


mimic_non_observation_icu_summary[
    "percentage"
] = (
    mimic_non_observation_icu_summary["count"]
    / len(mimic_non_observation_only)
    * 100
)



MIMIC_DISPOSITION_ORDER = [
    "OBSERVATION",
    "INPATIENT_NO_ICU",
    "ICU_ASSOCIATED",
]


mimic_three_group_counts = (
    mimic_hospital_los_audit[
        "hospital_disposition_group"
    ]
    .value_counts()
    .reindex(
        MIMIC_DISPOSITION_ORDER,
        fill_value=0,
    )
    .rename_axis(
        "hospital_disposition_group"
    )
    .reset_index(name="count")
)


mimic_three_group_counts[
    "percentage"
] = (
    mimic_three_group_counts["count"]
    / len(mimic_hospital_los_audit)
    * 100
)


print(
    "Confirmed non-observation MIMIC admissions:",
    f"{len(mimic_non_observation_only):,}",
)

print(
    "\nICU status among confirmed non-observation admissions:"
)

display(
    mimic_non_observation_icu_summary.round(
        {"percentage": 2}
    )
)


print(
    "\nMIMIC three-category hospital disposition audit:"
)

display(
    mimic_three_group_counts.round(
        {"percentage": 2}
    )
)


#### Hospital length of stay across outcome groups


In [ ]:


def rounded_day_mode(
    series: pd.Series,
) -> str | float:
    """
    Return mode(s) after rounding non-negative LOS values to the
    nearest whole day.

    Multiple modes are returned as a comma-separated string.
    """

    valid_values = pd.to_numeric(
        series,
        errors="coerce",
    ).dropna()

    valid_values = valid_values.loc[
        valid_values.ge(0)
    ]

    if valid_values.empty:
        return np.nan

    rounded_days = np.floor(
        valid_values + 0.5
    ).astype(int)

    modes = rounded_days.mode().sort_values()

    return ", ".join(
        modes.astype(str).tolist()
    )


def create_los_summary(
    dataframe: pd.DataFrame,
    group_column: str,
    los_column: str,
    group_order: list[str],
    dataset_name: str,
) -> pd.DataFrame:
    """
    Summarise LOS using mean, median, IQR and rounded-day mode.
    """

    records = []

    for group_name in group_order:
        group_rows = dataframe.loc[
            dataframe[group_column].eq(group_name)
        ].copy()

        los_values = pd.to_numeric(
            group_rows[los_column],
            errors="coerce",
        )

        valid_los = los_values.loc[
            los_values.notna()
            & los_values.ge(0)
        ]

        records.append(
            {
                "dataset": dataset_name,
                "disposition_group": group_name,
                "n_visits": len(group_rows),
                "n_non_missing_los": len(valid_los),
                "mean_los_days": (
                    valid_los.mean()
                    if len(valid_los) > 0
                    else np.nan
                ),
                "median_los_days": (
                    valid_los.median()
                    if len(valid_los) > 0
                    else np.nan
                ),
                "mode_rounded_los_days": (
                    rounded_day_mode(valid_los)
                ),
                "q1_los_days": (
                    valid_los.quantile(0.25)
                    if len(valid_los) > 0
                    else np.nan
                ),
                "q3_los_days": (
                    valid_los.quantile(0.75)
                    if len(valid_los) > 0
                    else np.nan
                ),
                "minimum_los_days": (
                    valid_los.min()
                    if len(valid_los) > 0
                    else np.nan
                ),
                "maximum_los_days": (
                    valid_los.max()
                    if len(valid_los) > 0
                    else np.nan
                ),
                "negative_los_values_excluded": int(
                    los_values.lt(0).sum()
                ),
            }
        )

    return pd.DataFrame(records)



mimic_hospital_los_summary_three_groups = (
    create_los_summary(
        dataframe=mimic_hospital_los_audit,
        group_column="hospital_disposition_group",
        los_column="hospital_los_days",
        group_order=MIMIC_DISPOSITION_ORDER,
        dataset_name="MIMIC-IV",
    )
)


print(
    "MIMIC-IV hospital LOS by disposition group:"
)

display(
    mimic_hospital_los_summary_three_groups.round(
        {
            "mean_los_days": 2,
            "median_los_days": 2,
            "q1_los_days": 2,
            "q3_los_days": 2,
            "minimum_los_days": 2,
            "maximum_los_days": 2,
        }
    )
)



mimic_icu_associated_rows = (
    mimic_hospital_los_audit.loc[
        mimic_hospital_los_audit[
            "hospital_disposition_group"
        ].eq("ICU_ASSOCIATED")
    ]
    .copy()
)


mimic_icu_los_summary = pd.DataFrame(
    [
        {
            "measure": "Hospital LOS days",
            "n_non_missing": int(
                mimic_icu_associated_rows[
                    "hospital_los_days"
                ].notna().sum()
            ),
            "mean": mimic_icu_associated_rows[
                "hospital_los_days"
            ].mean(),
            "median": mimic_icu_associated_rows[
                "hospital_los_days"
            ].median(),
            "mode_rounded_days": rounded_day_mode(
                mimic_icu_associated_rows[
                    "hospital_los_days"
                ]
            ),
        },
        {
            "measure": "Total recorded ICU LOS days",
            "n_non_missing": int(
                mimic_icu_associated_rows[
                    "total_recorded_icu_los_days"
                ].notna().sum()
            ),
            "mean": mimic_icu_associated_rows[
                "total_recorded_icu_los_days"
            ].mean(),
            "median": mimic_icu_associated_rows[
                "total_recorded_icu_los_days"
            ].median(),
            "mode_rounded_days": rounded_day_mode(
                mimic_icu_associated_rows[
                    "total_recorded_icu_los_days"
                ]
            ),
        },
    ]
)


print(
    "\nMIMIC-IV ICU-associated hospital and ICU LOS:"
)

display(
    mimic_icu_los_summary.round(
        {
            "mean": 2,
            "median": 2,
        }
    )
)


#### Timing of ICU admission

This is important because an ICU episode later in the hospitalisation is not equivalent to an ED disposition of ICU.


In [ ]:

mimic_icu_timing_audit = (
    mimic_icu_associated_rows.copy()
)



mimic_icu_timing_audit[
    "hours_ed_arrival_to_first_icu"
] = (
    mimic_icu_timing_audit[
        "first_icu_intime"
    ]
    - mimic_icu_timing_audit[
        "intime"
    ]
).dt.total_seconds() / 3600


mimic_icu_timing_audit[
    "hours_ed_departure_to_first_icu"
] = (
    mimic_icu_timing_audit[
        "first_icu_intime"
    ]
    - mimic_icu_timing_audit[
        "outtime"
    ]
).dt.total_seconds() / 3600


mimic_icu_timing_audit[
    "hours_hospital_admission_to_first_icu"
] = (
    mimic_icu_timing_audit[
        "first_icu_intime"
    ]
    - mimic_icu_timing_audit[
        "admittime"
    ]
).dt.total_seconds() / 3600



icu_before_ed_arrival = (
    mimic_icu_timing_audit[
        "first_icu_intime"
    ]
    < mimic_icu_timing_audit[
        "intime"
    ]
)

icu_during_recorded_ed_stay = (
    mimic_icu_timing_audit[
        "first_icu_intime"
    ].ge(
        mimic_icu_timing_audit[
            "intime"
        ]
    )
    &
    mimic_icu_timing_audit[
        "first_icu_intime"
    ].le(
        mimic_icu_timing_audit[
            "outtime"
        ]
    )
)

icu_within_6h_after_ed = (
    mimic_icu_timing_audit[
        "hours_ed_departure_to_first_icu"
    ].gt(0)
    &
    mimic_icu_timing_audit[
        "hours_ed_departure_to_first_icu"
    ].le(6)
)

icu_6_to_24h_after_ed = (
    mimic_icu_timing_audit[
        "hours_ed_departure_to_first_icu"
    ].gt(6)
    &
    mimic_icu_timing_audit[
        "hours_ed_departure_to_first_icu"
    ].le(24)
)

icu_more_than_24h_after_ed = (
    mimic_icu_timing_audit[
        "hours_ed_departure_to_first_icu"
    ].gt(24)
)


mimic_icu_timing_audit[
    "first_icu_timing_category"
] = np.select(
    [
        icu_before_ed_arrival,
        icu_during_recorded_ed_stay,
        icu_within_6h_after_ed,
        icu_6_to_24h_after_ed,
        icu_more_than_24h_after_ed,
    ],
    [
        "ICU start before recorded ED arrival",
        "ICU start during recorded ED stay",
        "ICU start within 6 hours after ED departure",
        "ICU start 6-24 hours after ED departure",
        "ICU start more than 24 hours after ED departure",
    ],
    default="Missing or unresolved timing",
)



mimic_icu_timing_summary = (
    mimic_icu_timing_audit[
        "first_icu_timing_category"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "first_icu_timing_category"
    )
    .reset_index(name="count")
)


mimic_icu_timing_summary[
    "percentage"
] = (
    mimic_icu_timing_summary["count"]
    / len(mimic_icu_timing_audit)
    * 100
)



mimic_first_icu_careunit_summary = (
    mimic_icu_timing_audit[
        "first_icu_careunit"
    ]
    .fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis(
        "first_icu_careunit"
    )
    .reset_index(name="count")
)


mimic_first_icu_careunit_summary[
    "percentage"
] = (
    mimic_first_icu_careunit_summary["count"]
    / len(mimic_icu_timing_audit)
    * 100
)



mimic_icu_numeric_timing_summary = (
    mimic_icu_timing_audit[
        [
            "hours_ed_arrival_to_first_icu",
            "hours_ed_departure_to_first_icu",
            "hours_hospital_admission_to_first_icu",
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "timing_measure",
        }
    )
)


print(
    "MIMIC-IV timing of first ICU entry:"
)

display(
    mimic_icu_timing_summary.round(
        {"percentage": 2}
    )
)


print(
    "\nMIMIC-IV first ICU care units:"
)

display(
    mimic_first_icu_careunit_summary.round(
        {"percentage": 2}
    )
)


print(
    "\nMIMIC-IV numeric ICU timing summary:"
)

display(
    mimic_icu_numeric_timing_summary.round(2)
)


#### Freeze the primary HOME-versus-non-ICU-inpatient cohorts

This section freezes the final primary cohorts used in all subsequent notebooks.

The primary outcome is non-ICU inpatient admission versus emergency-department discharge home.

For MIMIC-IV:

- HOME includes encounters with raw ED disposition HOME and no linked hospital admission identifier;
- ADMITTED includes confirmed non-observation hospital admissions without a recorded ICU stay during the hospitalisation;
- observation admissions are excluded;
- ICU-associated admissions are excluded;
- HOME encounters linked to a hospital admission are excluded as outcome-discordant;
- unresolved ADMITTED encounters and all non-binary ED dispositions are excluded.

For MC-MED:

- DISCHARGE is mapped to HOME;
- INPATIENT is mapped to ADMITTED;
- OBSERVATION and ICU are excluded.

Each patient contributes only their first eligible ESI-3 general abdominal-pain encounter. The resulting cohort rows and outcome labels are saved as immutable Parquet files and row manifests for use in later notebooks.

No sensitivity outcome is created in this pipeline.


In [ ]:


mimic_hospital_group_lookup = (
    mimic_hospital_los_audit[
        [
            "subject_id",
            "stay_id",
            "hospital_disposition_group",
            "has_recorded_icu_stay",
        ]
    ]
    .drop_duplicates(
        subset=[
            "subject_id",
            "stay_id",
        ]
    )
    .copy()
)

assert not mimic_hospital_group_lookup[
    [
        "subject_id",
        "stay_id",
    ]
].duplicated().any()


mimic_outcome_source = mimic_disposition_audit.copy()

columns_to_remove_before_merge = [
    column
    for column in [
        "hospital_disposition_group",
        "has_recorded_icu_stay",
        "outcome",
        "outcome_binary",
    ]
    if column in mimic_outcome_source.columns
]

if columns_to_remove_before_merge:
    mimic_outcome_source = mimic_outcome_source.drop(
        columns=columns_to_remove_before_merge
    )


mimic_outcome_source = mimic_outcome_source.merge(
    mimic_hospital_group_lookup,
    on=[
        "subject_id",
        "stay_id",
    ],
    how="left",
    validate="one_to_one",
)



mimic_outcome_source["outcome"] = pd.Series(
    pd.NA,
    index=mimic_outcome_source.index,
    dtype="string",
)

mimic_clean_home_mask = (
    mimic_outcome_source[
        "disposition_clean"
    ].eq("HOME")
    &
    ~mimic_outcome_source[
        "hadm_id_present"
    ]
)

mimic_non_icu_inpatient_mask = (
    mimic_outcome_source[
        "hospital_disposition_group"
    ].eq("INPATIENT_NO_ICU")
)

mimic_outcome_source.loc[
    mimic_clean_home_mask,
    "outcome",
] = "HOME"

mimic_outcome_source.loc[
    mimic_non_icu_inpatient_mask,
    "outcome",
] = "ADMITTED"


mimic_primary_binary = (
    mimic_outcome_source.loc[
        mimic_outcome_source[
            "outcome"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

mimic_primary_binary["outcome_binary"] = (
    mimic_primary_binary["outcome"]
    .map(
        {
            "HOME": 0,
            "ADMITTED": 1,
        }
    )
    .astype("int8")
)



mcmed_outcome_source = (
    mcmed_esi3_disposition_audit.copy()
)

for column in [
    "outcome",
    "outcome_binary",
]:
    if column in mcmed_outcome_source.columns:
        mcmed_outcome_source = (
            mcmed_outcome_source.drop(
                columns=column
            )
        )


mcmed_outcome_source["outcome"] = pd.Series(
    pd.NA,
    index=mcmed_outcome_source.index,
    dtype="string",
)

mcmed_outcome_source.loc[
    mcmed_outcome_source[
        "disposition_clean"
    ].eq("DISCHARGE"),
    "outcome",
] = "HOME"

mcmed_outcome_source.loc[
    mcmed_outcome_source[
        "disposition_clean"
    ].eq("INPATIENT"),
    "outcome",
] = "ADMITTED"


mcmed_primary_binary = (
    mcmed_outcome_source.loc[
        mcmed_outcome_source[
            "outcome"
        ].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

mcmed_primary_binary["outcome_binary"] = (
    mcmed_primary_binary["outcome"]
    .map(
        {
            "HOME": 0,
            "ADMITTED": 1,
        }
    )
    .astype("int8")
)



assert mimic_primary_binary[
    "subject_id"
].is_unique

assert mimic_primary_binary[
    "stay_id"
].is_unique

assert mcmed_primary_binary[
    "MRN"
].is_unique

assert mcmed_primary_binary[
    "CSN"
].is_unique

assert set(
    mimic_primary_binary[
        "outcome"
    ].unique()
) == {
    "HOME",
    "ADMITTED",
}

assert set(
    mcmed_primary_binary[
        "outcome"
    ].unique()
) == {
    "HOME",
    "ADMITTED",
}

assert set(
    mimic_primary_binary[
        "outcome_binary"
    ].unique()
) == {
    0,
    1,
}

assert set(
    mcmed_primary_binary[
        "outcome_binary"
    ].unique()
) == {
    0,
    1,
}


print(
    "MIMIC-IV frozen primary cohort:",
    f"{len(mimic_primary_binary):,}",
)

print(
    "MC-MED frozen primary cohort:",
    f"{len(mcmed_primary_binary):,}",
)


In [ ]:

EXPECTED_PRIMARY_COUNTS = {
    "MIMIC-IV": {
        "HOME": 6254,
        "ADMITTED": 2296,
        "TOTAL": 8550,
    },
    "MC-MED": {
        "HOME": 4872,
        "ADMITTED": 1835,
        "TOTAL": 6707,
    },
}


def create_final_cohort_summary(
    dataframe: pd.DataFrame,
    dataset_name: str,
    candidate_count: int,
    patient_identifier: str,
    encounter_identifier: str,
) -> pd.DataFrame:
    """Create a compact summary of the frozen primary cohort."""

    outcome_counts = (
        dataframe["outcome"]
        .value_counts()
        .reindex(
            [
                "HOME",
                "ADMITTED",
            ],
            fill_value=0,
        )
    )

    summary = pd.DataFrame(
        {
            "dataset": [
                dataset_name,
                dataset_name,
            ],
            "outcome": [
                "HOME",
                "ADMITTED",
            ],
            "count": [
                int(outcome_counts["HOME"]),
                int(outcome_counts["ADMITTED"]),
            ],
        }
    )

    summary["percentage"] = (
        summary["count"]
        / len(dataframe)
        * 100
    )

    summary["candidate_rows"] = (
        candidate_count
    )

    summary["included_rows"] = (
        len(dataframe)
    )

    summary["excluded_rows"] = (
        candidate_count
        - len(dataframe)
    )

    summary["n_unique_patients"] = (
        dataframe[
            patient_identifier
        ].nunique()
    )

    summary["n_unique_encounters"] = (
        dataframe[
            encounter_identifier
        ].nunique()
    )

    return summary


mimic_final_outcome_summary = (
    create_final_cohort_summary(
        dataframe=mimic_primary_binary,
        dataset_name="MIMIC-IV",
        candidate_count=len(
            mimic_esi3_candidate
        ),
        patient_identifier="subject_id",
        encounter_identifier="stay_id",
    )
)

mcmed_final_outcome_summary = (
    create_final_cohort_summary(
        dataframe=mcmed_primary_binary,
        dataset_name="MC-MED",
        candidate_count=len(
            mcmed_esi3_candidate
        ),
        patient_identifier="MRN",
        encounter_identifier="CSN",
    )
)

final_primary_outcome_summary = pd.concat(
    [
        mimic_final_outcome_summary,
        mcmed_final_outcome_summary,
    ],
    ignore_index=True,
)



for dataset_name, expected in (
    EXPECTED_PRIMARY_COUNTS.items()
):
    dataset_summary = (
        final_primary_outcome_summary.loc[
            final_primary_outcome_summary[
                "dataset"
            ].eq(dataset_name)
        ]
    )

    observed_home = int(
        dataset_summary.loc[
            dataset_summary[
                "outcome"
            ].eq("HOME"),
            "count",
        ].iloc[0]
    )

    observed_admitted = int(
        dataset_summary.loc[
            dataset_summary[
                "outcome"
            ].eq("ADMITTED"),
            "count",
        ].iloc[0]
    )

    observed_total = (
        observed_home
        + observed_admitted
    )

    assert observed_home == expected["HOME"], (
        f"{dataset_name} HOME count changed: "
        f"expected {expected['HOME']}, "
        f"observed {observed_home}."
    )

    assert (
        observed_admitted
        == expected["ADMITTED"]
    ), (
        f"{dataset_name} ADMITTED count changed: "
        f"expected {expected['ADMITTED']}, "
        f"observed {observed_admitted}."
    )

    assert observed_total == expected["TOTAL"], (
        f"{dataset_name} total count changed: "
        f"expected {expected['TOTAL']}, "
        f"observed {observed_total}."
    )


display(
    final_primary_outcome_summary.round(
        {
            "percentage": 2,
        }
    )
)

print(
    "\nAll expected primary cohort counts "
    "were reproduced successfully."
)


In [ ]:

primary_disposition_mapping = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "source_status": (
                "Raw HOME with no linked hadm_id"
            ),
            "final_status": "HOME",
            "reason": (
                "Clean ED discharge without a linked "
                "hospital admission"
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "source_status": (
                "Confirmed non-observation inpatient "
                "admission without an ICU stay"
            ),
            "final_status": "ADMITTED",
            "reason": (
                "Comparable non-ICU inpatient outcome"
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "source_status": "Observation admission",
            "final_status": "EXCLUDED",
            "reason": (
                "Observation is distinct from routine "
                "inpatient admission"
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "source_status": (
                "Hospital admission with any recorded "
                "ICU stay"
            ),
            "final_status": "EXCLUDED",
            "reason": (
                "ICU severity and timing are not directly "
                "comparable with MC-MED ED disposition"
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "source_status": (
                "Raw HOME with linked hadm_id"
            ),
            "final_status": "EXCLUDED",
            "reason": (
                "Discordance between ED disposition and "
                "linked hospital admission"
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "source_status": (
                "Unresolved ADMITTED or other non-binary "
                "raw disposition"
            ),
            "final_status": "EXCLUDED",
            "reason": (
                "Outcome could not be assigned reliably "
                "to HOME or non-ICU inpatient admission"
            ),
        },
        {
            "dataset": "MC-MED",
            "source_status": "DISCHARGE",
            "final_status": "HOME",
            "reason": "ED discharge",
        },
        {
            "dataset": "MC-MED",
            "source_status": "INPATIENT",
            "final_status": "ADMITTED",
            "reason": (
                "Non-ICU inpatient hospital admission"
            ),
        },
        {
            "dataset": "MC-MED",
            "source_status": "OBSERVATION",
            "final_status": "EXCLUDED",
            "reason": (
                "Observation is distinct from routine "
                "inpatient admission"
            ),
        },
        {
            "dataset": "MC-MED",
            "source_status": "ICU",
            "final_status": "EXCLUDED",
            "reason": (
                "ICU is a clinically distinct high-severity "
                "disposition"
            ),
        },
    ]
)



mimic_exclusion_summary = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "exclusion": "Observation admission",
            "count": int(
                mimic_outcome_source[
                    "hospital_disposition_group"
                ]
                .eq("OBSERVATION")
                .sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "exclusion": "ICU-associated admission",
            "count": int(
                mimic_outcome_source[
                    "hospital_disposition_group"
                ]
                .eq("ICU_ASSOCIATED")
                .sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "exclusion": (
                "HOME with linked hadm_id"
            ),
            "count": int(
                (
                    mimic_outcome_source[
                        "disposition_clean"
                    ].eq("HOME")
                    &
                    mimic_outcome_source[
                        "hadm_id_present"
                    ]
                ).sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "exclusion": (
                "Unresolved ADMITTED"
            ),
            "count": int(
                mimic_outcome_source[
                    "resolved_hospital_category"
                ]
                .eq("UNRESOLVED_ADMITTED")
                .sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "exclusion": (
                "Other non-binary ED dispositions"
            ),
            "count": int(
                mimic_outcome_source[
                    "resolved_hospital_category"
                ]
                .eq(
                    "NON_BINARY_RAW_DISPOSITION"
                )
                .sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "exclusion": "Observation",
            "count": int(
                mcmed_outcome_source[
                    "disposition_clean"
                ]
                .eq("OBSERVATION")
                .sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "exclusion": "ICU",
            "count": int(
                mcmed_outcome_source[
                    "disposition_clean"
                ]
                .eq("ICU")
                .sum()
            ),
        },
    ]
)


print("Primary disposition mapping:")

display(primary_disposition_mapping)


print("\nExcluded candidate encounters:")

display(mimic_exclusion_summary)


In [ ]:


def append_final_flow_step(
    flow_table: pd.DataFrame,
    criterion: str,
    final_count: int,
) -> pd.DataFrame:
    """Append the frozen primary-outcome step."""

    updated_flow = (
        flow_table
        .sort_values("step")
        .reset_index(drop=True)
        .copy()
    )

    initial_count = int(
        updated_flow.iloc[0][
            "n_remaining"
        ]
    )

    previous_count = int(
        updated_flow.iloc[-1][
            "n_remaining"
        ]
    )

    final_count = int(final_count)

    new_step = pd.DataFrame(
        [
            {
                "dataset": (
                    updated_flow[
                        "dataset"
                    ].iloc[0]
                ),
                "step": int(
                    updated_flow[
                        "step"
                    ].max()
                ) + 1,
                "inclusion_criterion": criterion,
                "n_remaining": final_count,
                "n_removed_from_previous": (
                    previous_count
                    - final_count
                ),
                "retained_from_previous_pct": (
                    final_count
                    / previous_count
                    * 100
                ),
                "retained_from_initial_pct": (
                    final_count
                    / initial_count
                    * 100
                ),
            }
        ]
    )

    return pd.concat(
        [
            updated_flow,
            new_step,
        ],
        ignore_index=True,
    )


mimic_final_flow_table = append_final_flow_step(
    flow_table=mimic_esi3_flow_table,
    criterion=(
        "Eligible for HOME versus non-ICU "
        "inpatient outcome"
    ),
    final_count=len(
        mimic_primary_binary
    ),
)

mcmed_final_flow_table = append_final_flow_step(
    flow_table=mcmed_esi3_flow_table,
    criterion=(
        "Eligible for HOME versus non-ICU "
        "inpatient outcome"
    ),
    final_count=len(
        mcmed_primary_binary
    ),
)


print("Final MIMIC-IV cohort flow:")

display(
    mimic_final_flow_table.round(
        {
            "retained_from_previous_pct": 2,
            "retained_from_initial_pct": 2,
        }
    )
)


print("\nFinal MC-MED cohort flow:")

display(
    mcmed_final_flow_table.round(
        {
            "retained_from_previous_pct": 2,
            "retained_from_initial_pct": 2,
        }
    )
)


In [ ]:

mimic_final_flow_figure = (
    plot_cohort_downfall(
        flow_table=mimic_final_flow_table,
        dataset_name="MIMIC-IV",
        output_filename=(
            "mimiciv_final_home_vs_"
            "non_icu_inpatient_cohort_flow_600dpi.png"
        ),
    )
)

mcmed_final_flow_figure = (
    plot_cohort_downfall(
        flow_table=mcmed_final_flow_table,
        dataset_name="MC-MED",
        output_filename=(
            "mcmed_final_home_vs_"
            "non_icu_inpatient_cohort_flow_600dpi.png"
        ),
    )
)


print("Saved:", mimic_final_flow_figure)
print("Saved:", mcmed_final_flow_figure)


In [ ]:


def plot_final_binary_outcome(
    dataframe: pd.DataFrame,
    dataset_name: str,
    output_filename: str,
) -> Path:
    """Plot the frozen primary binary outcome."""

    outcome_order = [
        "HOME",
        "ADMITTED",
    ]

    counts = (
        dataframe["outcome"]
        .value_counts()
        .reindex(
            outcome_order,
            fill_value=0,
        )
    )

    percentages = (
        counts
        / counts.sum()
        * 100
    )

    figure, axis = plt.subplots(
        figsize=(7.5, 5.5)
    )

    bars = axis.bar(
        outcome_order,
        counts.values,
        edgecolor="black",
        linewidth=1,
    )

    for bar, count, percentage in zip(
        bars,
        counts.values,
        percentages.values,
    ):
        axis.text(
            (
                bar.get_x()
                + bar.get_width() / 2
            ),
            bar.get_height(),
            (
                f"{int(count):,}\n"
                f"({percentage:.2f}%)"
            ),
            ha="center",
            va="bottom",
            fontsize=11,
        )

    axis.set_xlabel(
        "Primary binary outcome"
    )

    axis.set_ylabel(
        "Number of ESI-3 ED encounters"
    )

    axis.set_title(
        (
            f"{dataset_name}: HOME Versus "
            "Non-ICU Inpatient Admission"
        ),
        weight="bold",
    )

    axis.set_ylim(
        0,
        max(counts.max(), 1) * 1.18,
    )

    figure.tight_layout()

    output_path = (
        FIGURE_DIR
        / output_filename
    )

    figure.savefig(
        output_path,
        dpi=FIGURE_DPI,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return output_path


mimic_final_outcome_figure = (
    plot_final_binary_outcome(
        dataframe=mimic_primary_binary,
        dataset_name="MIMIC-IV",
        output_filename=(
            "mimiciv_final_home_vs_"
            "non_icu_inpatient_outcome_600dpi.png"
        ),
    )
)

mcmed_final_outcome_figure = (
    plot_final_binary_outcome(
        dataframe=mcmed_primary_binary,
        dataset_name="MC-MED",
        output_filename=(
            "mcmed_final_home_vs_"
            "non_icu_inpatient_outcome_600dpi.png"
        ),
    )
)


print("Saved:", mimic_final_outcome_figure)
print("Saved:", mcmed_final_outcome_figure)


In [ ]:


PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



MIMIC_PRIMARY_COHORT_PATH = (
    PROCESSED_DIR
    / (
        "mimiciv_primary_esi3_home_vs_"
        "non_icu_inpatient_cohort.parquet"
    )
)

MCMED_PRIMARY_COHORT_PATH = (
    PROCESSED_DIR
    / (
        "mcmed_primary_esi3_home_vs_"
        "non_icu_inpatient_cohort.parquet"
    )
)


mimic_primary_binary.to_parquet(
    MIMIC_PRIMARY_COHORT_PATH,
    index=False,
)

mcmed_primary_binary.to_parquet(
    MCMED_PRIMARY_COHORT_PATH,
    index=False,
)



MIMIC_ROW_MANIFEST_PATH = (
    PROCESSED_DIR
    / "mimiciv_primary_row_manifest.parquet"
)

MCMED_ROW_MANIFEST_PATH = (
    PROCESSED_DIR
    / "mcmed_primary_row_manifest.parquet"
)


mimic_primary_binary[
    [
        "subject_id",
        "stay_id",
        "outcome",
        "outcome_binary",
    ]
].to_parquet(
    MIMIC_ROW_MANIFEST_PATH,
    index=False,
)

mcmed_primary_binary[
    [
        "MRN",
        "CSN",
        "outcome",
        "outcome_binary",
    ]
].to_parquet(
    MCMED_ROW_MANIFEST_PATH,
    index=False,
)



final_primary_outcome_summary.to_csv(
    TABLE_DIR
    / "final_primary_outcome_summary.csv",
    index=False,
)

primary_disposition_mapping.to_csv(
    TABLE_DIR
    / "primary_disposition_mapping.csv",
    index=False,
)

mimic_exclusion_summary.to_csv(
    TABLE_DIR
    / "primary_disposition_exclusion_counts.csv",
    index=False,
)

mimic_final_flow_table.to_csv(
    TABLE_DIR
    / "mimiciv_final_primary_cohort_flow.csv",
    index=False,
)

mcmed_final_flow_table.to_csv(
    TABLE_DIR
    / "mcmed_final_primary_cohort_flow.csv",
    index=False,
)



cohort_freeze_metadata = {
    "study_population": (
        "Adults aged 18 years or older with ESI-3 "
        "and a strict normalised general abdominal-pain "
        "chief complaint"
    ),
    "primary_outcome": (
        "Non-ICU inpatient admission versus ED "
        "discharge home"
    ),
    "one_visit_per_patient": True,
    "mimiciv": {
        "home_definition": (
            "Raw HOME with no linked hadm_id"
        ),
        "admitted_definition": (
            "Confirmed non-observation inpatient "
            "admission without any recorded ICU stay"
        ),
        "home_count": 6254,
        "admitted_count": 2296,
        "total_count": 8550,
    },
    "mcmed": {
        "home_definition": "DISCHARGE",
        "admitted_definition": "INPATIENT",
        "home_count": 4872,
        "admitted_count": 1835,
        "total_count": 6707,
    },
    "excluded_primary_categories": [
        "Observation",
        "ICU",
        "Outcome-discordant MIMIC HOME with hadm_id",
        "Unresolved MIMIC admission",
        "Transfer",
        "Eloped",
        "Left without being seen",
        "Left against medical advice",
        "Other non-binary disposition",
    ],
    "random_state": RANDOM_STATE,
    "frozen_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


COHORT_FREEZE_METADATA_PATH = (
    PROCESSED_DIR
    / "primary_cohort_freeze_metadata.json"
)

with open(
    COHORT_FREEZE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        cohort_freeze_metadata,
        metadata_file,
        indent=2,
    )



saved_primary_outputs = [
    MIMIC_PRIMARY_COHORT_PATH,
    MCMED_PRIMARY_COHORT_PATH,
    MIMIC_ROW_MANIFEST_PATH,
    MCMED_ROW_MANIFEST_PATH,
    COHORT_FREEZE_METADATA_PATH,
    TABLE_DIR
    / "final_primary_outcome_summary.csv",
    TABLE_DIR
    / "primary_disposition_mapping.csv",
    TABLE_DIR
    / "primary_disposition_exclusion_counts.csv",
    TABLE_DIR
    / "mimiciv_final_primary_cohort_flow.csv",
    TABLE_DIR
    / "mcmed_final_primary_cohort_flow.csv",
]


saved_primary_output_status = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if path.exists()
                else np.nan
            ),
            "path": str(path),
        }
        for path in saved_primary_outputs
    ]
)


display(saved_primary_output_status)

assert saved_primary_output_status[
    "exists"
].all()

print(
    "\nThe primary cohorts have been frozen successfully."
)

print(
    "MIMIC-IV:",
    f"{len(mimic_primary_binary):,}",
    "rows",
)

print(
    "MC-MED:",
    f"{len(mcmed_primary_binary):,}",
    "rows",
)


In [ ]:


def final_frozen_cohort_check(
    dataframe: pd.DataFrame,
    dataset_name: str,
    patient_identifier: str,
    encounter_identifier: str,
) -> dict:
    """Create the final reproducibility checkpoint."""

    outcome_counts = (
        dataframe["outcome"]
        .value_counts()
    )

    return {
        "dataset": dataset_name,
        "n_rows": len(dataframe),
        "n_unique_patients": dataframe[
            patient_identifier
        ].nunique(),
        "n_unique_encounters": dataframe[
            encounter_identifier
        ].nunique(),
        "home_count": int(
            outcome_counts.get(
                "HOME",
                0,
            )
        ),
        "admitted_count": int(
            outcome_counts.get(
                "ADMITTED",
                0,
            )
        ),
        "admitted_percentage": (
            outcome_counts.get(
                "ADMITTED",
                0,
            )
            / len(dataframe)
            * 100
        ),
        "minimum_age": dataframe[
            "age_at_ed"
        ].min(),
        "maximum_age": dataframe[
            "age_at_ed"
        ].max(),
        "esi_values": ",".join(
            map(
                str,
                sorted(
                    dataframe[
                        "esi_numeric"
                    ]
                    .dropna()
                    .astype(int)
                    .unique()
                    .tolist()
                ),
            )
        ),
        "missing_outcomes": int(
            dataframe[
                "outcome"
            ].isna().sum()
        ),
        "duplicate_patient_rows": int(
            dataframe[
                patient_identifier
            ].duplicated(
                keep=False
            ).sum()
        ),
        "duplicate_encounter_rows": int(
            dataframe[
                encounter_identifier
            ].duplicated(
                keep=False
            ).sum()
        ),
    }


final_frozen_cohort_checkpoint = pd.DataFrame(
    [
        final_frozen_cohort_check(
            dataframe=mimic_primary_binary,
            dataset_name="MIMIC-IV",
            patient_identifier="subject_id",
            encounter_identifier="stay_id",
        ),
        final_frozen_cohort_check(
            dataframe=mcmed_primary_binary,
            dataset_name="MC-MED",
            patient_identifier="MRN",
            encounter_identifier="CSN",
        ),
    ]
)


display(
    final_frozen_cohort_checkpoint.round(
        {
            "admitted_percentage": 2,
        }
    )
)


assert (
    final_frozen_cohort_checkpoint[
        "n_rows"
    ]
    == final_frozen_cohort_checkpoint[
        "n_unique_patients"
    ]
).all()

assert (
    final_frozen_cohort_checkpoint[
        "n_rows"
    ]
    == final_frozen_cohort_checkpoint[
        "n_unique_encounters"
    ]
).all()

assert (
    final_frozen_cohort_checkpoint[
        "esi_values"
    ] == "3"
).all()

assert (
    final_frozen_cohort_checkpoint[
        "minimum_age"
    ] >= 18
).all()

assert (
    final_frozen_cohort_checkpoint[
        "missing_outcomes"
    ] == 0
).all()

assert (
    final_frozen_cohort_checkpoint[
        "duplicate_patient_rows"
    ] == 0
).all()

assert (
    final_frozen_cohort_checkpoint[
        "duplicate_encounter_rows"
    ] == 0
).all()


print(
    "\nFINAL PRIMARY COHORT FREEZE PASSED"
)

print(
    "Outcome: HOME versus non-ICU inpatient admission"
)

print(
    "MIMIC-IV: 6,254 HOME and 2,296 ADMITTED"
)

print(
    "MC-MED: 4,872 HOME and 1,835 ADMITTED"
)

print(
    "These row manifests must be used by all later notebooks."
)


#### MIMIC-IV approximate encounter-year alignment and temporal split

MIMIC-IV shifts dates independently for each patient, so shifted ED timestamps cannot directly order different patients. The split is therefore based on an approximate original encounter year.

For each encounter, the code parses the patient-level `anchor_year_group`, calculates the offset between the shifted encounter year and `anchor_year`, and applies that offset to the three-year anchor range. The midpoint of the aligned range is used only to assign broad development periods:

- training: approximate encounter years 2008–2013;
- validation: approximate encounter years 2014–2016;
- internal temporal test: approximate encounter years 2017–2022.

The split is approximate because MIMIC-IV provides year ranges rather than exact original dates. The cohort, outcome definition, feature vocabulary, and MC-MED external cohort are unchanged.


In [ ]:
required_split_columns = {
    "subject_id",
    "stay_id",
    "intime",
    "outtime",
    "outcome",
    "outcome_binary",
    "anchor_year",
    "anchor_year_group",
}

missing_split_columns = required_split_columns - set(mimic_primary_binary.columns)
if missing_split_columns:
    raise ValueError(
        "The MIMIC-IV binary cohort is missing temporal metadata:\n"
        f"{sorted(missing_split_columns)}"
    )

mimic_split_source = mimic_primary_binary.copy()

mimic_split_source["intime"] = pd.to_datetime(
    mimic_split_source["intime"],
    errors="coerce",
)
mimic_split_source["outtime"] = pd.to_datetime(
    mimic_split_source["outtime"],
    errors="coerce",
)
mimic_split_source["anchor_year"] = pd.to_numeric(
    mimic_split_source["anchor_year"],
    errors="coerce",
).astype("Int64")

assert len(mimic_split_source) == 8550
assert mimic_split_source["subject_id"].is_unique
assert mimic_split_source["stay_id"].is_unique
assert mimic_split_source["intime"].notna().all()
assert mimic_split_source["anchor_year"].notna().all()
assert mimic_split_source["anchor_year_group"].notna().all()
assert set(mimic_split_source["outcome_binary"].unique()) == {0, 1}

anchor_year_parts = (
    mimic_split_source["anchor_year_group"]
    .astype("string")
    .str.extract(
        r"(?P<anchor_group_start>\d{4})\s*-\s*(?P<anchor_group_end>\d{4})"
    )
)

if anchor_year_parts.isna().any().any():
    invalid_groups = (
        mimic_split_source.loc[
            anchor_year_parts.isna().any(axis=1),
            "anchor_year_group",
        ]
        .astype("string")
        .drop_duplicates()
        .tolist()
    )
    raise ValueError(
        "Some anchor_year_group values could not be parsed:\n"
        f"{invalid_groups[:20]}"
    )

mimic_split_source["anchor_group_start"] = pd.to_numeric(
    anchor_year_parts["anchor_group_start"],
    errors="raise",
).astype("int16")
mimic_split_source["anchor_group_end"] = pd.to_numeric(
    anchor_year_parts["anchor_group_end"],
    errors="raise",
).astype("int16")

mimic_split_source["shifted_encounter_year"] = (
    mimic_split_source["intime"].dt.year.astype("int16")
)
mimic_split_source["anchor_to_encounter_year_offset"] = (
    mimic_split_source["shifted_encounter_year"]
    - mimic_split_source["anchor_year"].astype("int16")
)
mimic_split_source["approx_event_year_start"] = (
    mimic_split_source["anchor_group_start"]
    + mimic_split_source["anchor_to_encounter_year_offset"]
).astype("int16")
mimic_split_source["approx_event_year_end"] = (
    mimic_split_source["anchor_group_end"]
    + mimic_split_source["anchor_to_encounter_year_offset"]
).astype("int16")
mimic_split_source["approx_event_year"] = (
    (
        mimic_split_source["approx_event_year_start"]
        + mimic_split_source["approx_event_year_end"]
    )
    // 2
).astype("int16")

unexpected_year_mask = ~mimic_split_source["approx_event_year"].between(
    2008,
    2022,
    inclusive="both",
)
if unexpected_year_mask.any():
    display(
        mimic_split_source.loc[
            unexpected_year_mask,
            [
                "subject_id",
                "stay_id",
                "intime",
                "anchor_year",
                "anchor_year_group",
                "approx_event_year_start",
                "approx_event_year_end",
                "approx_event_year",
            ],
        ].head(20)
    )
    raise ValueError(
        "Aligned approximate encounter years fall outside 2008–2022."
    )

mimic_split_source["approx_event_period"] = pd.cut(
    mimic_split_source["approx_event_year"],
    bins=[2007, 2010, 2013, 2016, 2019, 2022],
    labels=[
        "2008 - 2010",
        "2011 - 2013",
        "2014 - 2016",
        "2017 - 2019",
        "2020 - 2022",
    ],
    include_lowest=True,
    right=True,
).astype("string")

assert mimic_split_source["approx_event_period"].notna().all()

mimic_split_source["split"] = np.select(
    [
        mimic_split_source["approx_event_year"].between(
            2008, 2013, inclusive="both"
        ),
        mimic_split_source["approx_event_year"].between(
            2014, 2016, inclusive="both"
        ),
        mimic_split_source["approx_event_year"].between(
            2017, 2022, inclusive="both"
        ),
    ],
    ["train", "validation", "test"],
    default="UNASSIGNED",
)

if mimic_split_source["split"].eq("UNASSIGNED").any():
    raise ValueError("At least one MIMIC-IV encounter was not assigned.")

split_order = ["train", "validation", "test"]

mimic_split_summary = (
    mimic_split_source.groupby("split", observed=True)["outcome_binary"]
    .agg(rows="size", admitted="sum")
    .reindex(split_order)
    .reset_index()
)
mimic_split_summary["home"] = (
    mimic_split_summary["rows"] - mimic_split_summary["admitted"]
)
mimic_split_summary["admitted_prevalence"] = (
    mimic_split_summary["admitted"] / mimic_split_summary["rows"]
)
mimic_split_summary = mimic_split_summary[
    ["split", "rows", "home", "admitted", "admitted_prevalence"]
]

mimic_split_year_audit = (
    mimic_split_source.groupby("split", observed=True)["approx_event_year"]
    .agg(
        approximate_year_min="min",
        approximate_year_max="max",
    )
    .reindex(split_order)
    .reset_index()
)

mimic_split_period_audit = (
    mimic_split_source.groupby(
        ["split", "approx_event_period"],
        observed=True,
    )["outcome_binary"]
    .agg(rows="size", admitted="sum")
    .reset_index()
)
mimic_split_period_audit["home"] = (
    mimic_split_period_audit["rows"]
    - mimic_split_period_audit["admitted"]
)
mimic_split_period_audit["admitted_prevalence"] = (
    mimic_split_period_audit["admitted"]
    / mimic_split_period_audit["rows"]
)

assert int(mimic_split_summary["rows"].sum()) == 8550
assert int(mimic_split_summary["home"].sum()) == 6254
assert int(mimic_split_summary["admitted"].sum()) == 2296
assert mimic_split_summary["home"].gt(0).all()
assert mimic_split_summary["admitted"].gt(0).all()

train_max = int(
    mimic_split_year_audit.loc[
        mimic_split_year_audit["split"].eq("train"),
        "approximate_year_max",
    ].iloc[0]
)
validation_min = int(
    mimic_split_year_audit.loc[
        mimic_split_year_audit["split"].eq("validation"),
        "approximate_year_min",
    ].iloc[0]
)
validation_max = int(
    mimic_split_year_audit.loc[
        mimic_split_year_audit["split"].eq("validation"),
        "approximate_year_max",
    ].iloc[0]
)
test_min = int(
    mimic_split_year_audit.loc[
        mimic_split_year_audit["split"].eq("test"),
        "approximate_year_min",
    ].iloc[0]
)

assert train_max < validation_min
assert validation_max < test_min

mimic_split_assignments = mimic_split_source[
    [
        "subject_id",
        "stay_id",
        "split",
        "outcome",
        "outcome_binary",
        "anchor_year",
        "anchor_year_group",
        "shifted_encounter_year",
        "anchor_to_encounter_year_offset",
        "approx_event_year_start",
        "approx_event_year_end",
        "approx_event_year",
        "approx_event_period",
    ]
].copy()

mimic_temporal_split_policy = {
    "method": "approximate encounter-year temporal split",
    "alignment_method": (
        "midpoint(anchor_year_group) + year(intime) - anchor_year"
    ),
    "training_years": "2008-2013",
    "validation_years": "2014-2016",
    "test_years": "2017-2022",
    "exact_original_dates_available": False,
    "split_is_approximate": True,
    "shifted_dates_comparable_across_patients": False,
    "training_use": "model fitting and training-only cross-validation",
    "validation_use": (
        "algorithm comparison, calibration, and threshold selection"
    ),
    "test_use": "single locked internal temporal evaluation",
}

print("Corrected approximate temporal split:")
display(mimic_split_summary.round({"admitted_prevalence": 4}))

print("\nApproximate year range by split:")
display(mimic_split_year_audit)

print("\nOutcome support by approximate period:")
display(
    mimic_split_period_audit.round({"admitted_prevalence": 4})
)

if mimic_split_summary["admitted"].min() < 100:
    warnings.warn(
        "At least one split has fewer than 100 ADMITTED encounters; "
        "final confidence intervals may be wide.",
        RuntimeWarning,
    )

print("\nMIMIC-IV approximate temporal split created.")
print("No file was saved and no model was fitted.")


#### Feature timing and eligibility registry

This section defines which predictor families may be used in the primary disposition model before individual features are constructed.

The study uses three cumulative information stages:

1. **S0 — Triage**
   - age;
   - sex or gender;
   - arrival transport;
   - initial triage vital signs;
   - patient-reported pain score.

2. **S1 — History-enhanced**
   - all S0 information;
   - diagnoses from hospital encounters completed before the current ED visit;
   - medications documented as previous or home medications rather than treatments administered during the current ED visit.

3. **S2 — Objective work-up results**
   - all S1 information;
   - laboratory result abnormalities recorded during the eligible ED encounter;
   - imaging or radiology abnormal-result evidence recorded during the eligible ED encounter.

S2 is the intended full primary feature stage.

Provider orders, treatment orders, intravenous-fluid orders, consultations, time-to-first-event variables and repeated-monitoring summaries are excluded from the primary model because they directly encode clinician actions, workflow intensity and local hospital practice.

Laboratory and imaging results are objective clinical findings, but their availability depends on whether a clinician requested the investigation. They are therefore classified as objective but workflow-contingent, not completely clinician-independent.

Race and ethnicity are retained only for cohort description and fairness auditing. They are not used as primary predictors because coding systems and population distributions can differ substantially between hospitals.

Chief complaint and ESI are excluded as predictors because the cohort has already been restricted to a strict general abdominal-pain complaint and ESI-3. They are constant by design and cannot provide meaningful discrimination.

Identifiers, timestamps, split labels, disposition, hospital admission type, observation status, ICU status, hospital length of stay and post-disposition information are metadata or prohibited outcome-related variables.

HPI, physical-examination text and locally generated LLM interpretations are reserved for downstream novelty and plausibility analysis. They are not used to train the primary disposition model.


#### Feature-family policy


In [ ]:

feature_family_policy_rows = [
    {
        "family_id": "age",
        "feature_family": "Age at ED arrival",
        "availability_stage": "S0_TRIAGE",
        "information_type": "patient_context",
        "clinician_influence": "none_or_minimal",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "age_at_ed",
        "methodological_note": (
            "Available at arrival and harmonisable between datasets."
        ),
    },
    {
        "family_id": "sex_gender",
        "feature_family": "Sex or gender",
        "availability_stage": "S0_TRIAGE",
        "information_type": "patient_context",
        "clinician_influence": "none_or_minimal",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "gender, Sex",
        "methodological_note": (
            "Dataset terminology must be documented rather than "
            "silently treating sex and gender as identical."
        ),
    },
    {
        "family_id": "arrival_transport",
        "feature_family": "Arrival transport",
        "availability_stage": "S0_TRIAGE",
        "information_type": "arrival_context",
        "clinician_influence": "pre_ed_system_influence",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "arrival_transport, Means_of_arrival",
        "methodological_note": (
            "Known at arrival; categories require cross-dataset grouping."
        ),
    },
    {
        "family_id": "race_ethnicity",
        "feature_family": "Race or ethnicity",
        "availability_stage": "S0_TRIAGE",
        "information_type": "demographic_audit",
        "clinician_influence": "administrative_and_social_construct",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "FAIRNESS_AUDIT_ONLY",
        "examples": "race, Ethnoracial_group",
        "methodological_note": (
            "Retained for description and subgroup auditing, "
            "not primary prediction."
        ),
    },
    {
        "family_id": "chief_complaint",
        "feature_family": "Chief complaint",
        "availability_stage": "S0_TRIAGE",
        "information_type": "cohort_definition",
        "clinician_influence": "documentation_mediated",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "CONSTANT_BY_DESIGN",
        "examples": "chiefcomplaint, CC",
        "methodological_note": (
            "All primary-cohort rows already satisfy the strict "
            "general abdominal-pain definition."
        ),
    },
    {
        "family_id": "esi",
        "feature_family": "Emergency Severity Index",
        "availability_stage": "S0_TRIAGE",
        "information_type": "cohort_definition",
        "clinician_influence": "nurse_assigned",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "CONSTANT_BY_DESIGN",
        "examples": "acuity, Triage_acuity, esi_numeric",
        "methodological_note": (
            "All primary-cohort rows are ESI-3."
        ),
    },
    {
        "family_id": "triage_vitals",
        "feature_family": "Initial triage vital signs",
        "availability_stage": "S0_TRIAGE",
        "information_type": "objective_patient_state",
        "clinician_influence": "measurement_and_documentation",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": (
            "temperature, heart rate, respiratory rate, "
            "SpO2, SBP, DBP"
        ),
        "methodological_note": (
            "Use only the initial triage measurements, "
            "not repeated monitoring summaries."
        ),
    },
    {
        "family_id": "pain_score",
        "feature_family": "Initial pain score",
        "availability_stage": "S0_TRIAGE",
        "information_type": "patient_reported_state",
        "clinician_influence": "documentation_mediated",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "pain, pain_score, Triage_pain",
        "methodological_note": (
            "Patient-reported and subjective, but not a clinician "
            "treatment or ordering decision."
        ),
    },
    {
        "family_id": "prior_diagnoses",
        "feature_family": "Previous diagnosis history",
        "availability_stage": "S1_HISTORY",
        "information_type": "pre_existing_history",
        "clinician_influence": "historical_coding_mediated",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "pmh_icd_title__*",
        "methodological_note": (
            "Only diagnoses from encounters completed before the "
            "current ED arrival are eligible."
        ),
    },
    {
        "family_id": "prior_medications",
        "feature_family": "Previous or home medications",
        "availability_stage": "S1_HISTORY",
        "information_type": "pre_existing_history",
        "clinician_influence": "historical_prescribing_and_documentation",
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "pmh_med_class__*, pmh_med_name__*",
        "methodological_note": (
            "Current-encounter treatment and medication orders "
            "must not be included."
        ),
    },
    {
        "family_id": "lab_results",
        "feature_family": "Laboratory abnormal-result features",
        "availability_stage": "S2_OBJECTIVE_WORKUP",
        "information_type": "objective_result",
        "clinician_influence": (
            "objective_value_but_availability_clinician_mediated"
        ),
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "poe_result__lab__*",
        "methodological_note": (
            "Include result evidence only. Do not include order labels "
            "or time-to-result variables."
        ),
    },
    {
        "family_id": "imaging_results",
        "feature_family": "Imaging abnormal-result features",
        "availability_stage": "S2_OBJECTIVE_WORKUP",
        "information_type": "objective_result",
        "clinician_influence": (
            "objective_value_but_availability_clinician_mediated"
        ),
        "primary_model_eligible": True,
        "external_harmonisation_required": True,
        "planned_use": "PRIMARY_MODEL",
        "examples": "poe_result__imaging_radiology__*",
        "methodological_note": (
            "Include report-derived abnormality evidence only. "
            "Do not include imaging order indicators."
        ),
    },
    {
        "family_id": "poe_orders",
        "feature_family": "Provider orders",
        "availability_stage": "CURRENT_ED_WORKFLOW",
        "information_type": "clinician_action",
        "clinician_influence": "direct_clinician_action",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "EXCLUDED_WORKFLOW",
        "examples": "poe_order__*, poe_order_type__*",
        "methodological_note": (
            "Strongly reflects clinical decisions and local workflow."
        ),
    },
    {
        "family_id": "treatment_orders",
        "feature_family": "Medication and IV-fluid treatment orders",
        "availability_stage": "CURRENT_ED_WORKFLOW",
        "information_type": "treatment_decision",
        "clinician_influence": "direct_clinician_action",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "EXCLUDED_WORKFLOW",
        "examples": "medication orders, IV-fluid orders",
        "methodological_note": (
            "Treatment may be a response to the same severity that "
            "determines disposition."
        ),
    },
    {
        "family_id": "consult_orders",
        "feature_family": "Consultation and care-team orders",
        "availability_stage": "CURRENT_ED_WORKFLOW",
        "information_type": "clinician_action",
        "clinician_influence": "direct_clinician_action",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "EXCLUDED_WORKFLOW",
        "examples": "consult_care_team_order",
        "methodological_note": (
            "Directly captures escalation and intended disposition."
        ),
    },
    {
        "family_id": "downstream_timing",
        "feature_family": "Time-to-first-result or intervention",
        "availability_stage": "CURRENT_ED_WORKFLOW",
        "information_type": "process_timing",
        "clinician_influence": "workflow_and_resource_mediated",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "EXCLUDED_WORKFLOW",
        "examples": (
            "time_to_first_lab_result_min, "
            "time_to_first_radiology_report_min"
        ),
        "methodological_note": (
            "Reflects workflow intensity, crowding and resource availability."
        ),
    },
    {
        "family_id": "repeated_monitoring",
        "feature_family": "Repeated vital monitoring and deltas",
        "availability_stage": "CURRENT_ED_WORKFLOW",
        "information_type": "dynamic_monitoring",
        "clinician_influence": "monitoring_frequency_clinician_mediated",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "EXCLUDED_WORKFLOW",
        "examples": (
            "n_vitals_taken, monitoring span, "
            "vital-sign deltas"
        ),
        "methodological_note": (
            "Measurement frequency itself can encode perceived severity."
        ),
    },
    {
        "family_id": "hpi_pe_text",
        "feature_family": "HPI and physical-examination text",
        "availability_stage": "DOCUMENTATION_LAYER",
        "information_type": "clinical_narrative",
        "clinician_influence": "documentation_mediated",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "NOVELTY_ANALYSIS_ONLY",
        "examples": "HPI text, physical-examination text",
        "methodological_note": (
            "Reserved for explaining candidate features, not prediction."
        ),
    },
    {
        "family_id": "llm_hpi_pe_features",
        "feature_family": "Locally extracted LLM HPI/PE concepts",
        "availability_stage": "DOWNSTREAM_RESEARCH_ANALYSIS",
        "information_type": "derived_narrative_interpretation",
        "clinician_influence": "documentation_and_model_mediated",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "NOVELTY_ANALYSIS_ONLY",
        "examples": "local Qwen HPI/PE extraction",
        "methodological_note": (
            "Must not influence model development or primary labels."
        ),
    },
    {
        "family_id": "identifiers",
        "feature_family": "Patient and encounter identifiers",
        "availability_stage": "METADATA",
        "information_type": "identifier",
        "clinician_influence": "none",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "METADATA_ONLY",
        "examples": "subject_id, stay_id, hadm_id, MRN, CSN",
        "methodological_note": "Used only for linkage and audit.",
    },
    {
        "family_id": "split_metadata",
        "feature_family": "Split and approximate temporal metadata",
        "availability_stage": "METADATA",
        "information_type": "analysis_metadata",
        "clinician_influence": "none",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "METADATA_ONLY",
        "examples": "split, approx_event_year, approx_event_period, intime",
        "methodological_note": (
            "Used to assign datasets, never as model input."
        ),
    },
    {
        "family_id": "outcome_fields",
        "feature_family": "Disposition and outcome fields",
        "availability_stage": "POST_OUTCOME",
        "information_type": "target",
        "clinician_influence": "outcome",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "PROHIBITED",
        "examples": "outcome, outcome_binary, disposition",
        "methodological_note": "Direct target leakage.",
    },
    {
        "family_id": "hospital_course",
        "feature_family": "Hospital-course and severity outcomes",
        "availability_stage": "POST_OUTCOME",
        "information_type": "post_disposition_outcome",
        "clinician_influence": "outcome_and_treatment",
        "primary_model_eligible": False,
        "external_harmonisation_required": False,
        "planned_use": "PROHIBITED",
        "examples": (
            "admission_type, ICU status, hospital LOS, "
            "discharge location"
        ),
        "methodological_note": (
            "Only used to define or validate the outcome."
        ),
    },
]


feature_family_policy = pd.DataFrame(
    feature_family_policy_rows
)


assert feature_family_policy["family_id"].is_unique


print("Feature-family policy:")

display(feature_family_policy)


print("\nPolicy summary:")

feature_policy_summary = (
    feature_family_policy
    .groupby(
        [
            "planned_use",
            "primary_model_eligible",
        ],
        dropna=False,
    )
    .size()
    .rename("n_feature_families")
    .reset_index()
)

display(feature_policy_summary)


#### Cumulative prediction stages


In [ ]:

prediction_stage_rows = [
    {
        "stage_order": 0,
        "stage_id": "S0_TRIAGE",
        "stage_name": "Triage-stage model",
        "families_added": [
            "age",
            "sex_gender",
            "arrival_transport",
            "triage_vitals",
            "pain_score",
        ],
        "description": (
            "Information available at or immediately after triage."
        ),
        "intended_role": (
            "Arrival-time baseline and comparison model."
        ),
    },
    {
        "stage_order": 1,
        "stage_id": "S1_HISTORY",
        "stage_name": "History-enhanced model",
        "families_added": [
            "prior_diagnoses",
            "prior_medications",
        ],
        "description": (
            "Adds patient information available before the current "
            "ED encounter."
        ),
        "intended_role": (
            "Tests the value of longitudinal history beyond triage."
        ),
    },
    {
        "stage_order": 2,
        "stage_id": "S2_OBJECTIVE_WORKUP",
        "stage_name": "Objective work-up model",
        "families_added": [
            "lab_results",
            "imaging_results",
        ],
        "description": (
            "Adds objective laboratory and imaging result evidence "
            "from the current ED encounter."
        ),
        "intended_role": (
            "Full primary clinician-action-restricted model."
        ),
    },
]


prediction_stage_registry = pd.DataFrame(
    prediction_stage_rows
)



cumulative_families = []
cumulative_family_records = []

for _, stage_row in (
    prediction_stage_registry
    .sort_values("stage_order")
    .iterrows()
):
    cumulative_families.extend(
        stage_row["families_added"]
    )

    cumulative_families = list(
        dict.fromkeys(cumulative_families)
    )

    cumulative_family_records.append(
        cumulative_families.copy()
    )


prediction_stage_registry[
    "cumulative_families"
] = cumulative_family_records



policy_lookup = (
    feature_family_policy
    .set_index("family_id")
)


for family_list in prediction_stage_registry[
    "cumulative_families"
]:
    unknown_families = [
        family_id
        for family_id in family_list
        if family_id not in policy_lookup.index
    ]

    if unknown_families:
        raise ValueError(
            "Unknown families in prediction-stage registry: "
            f"{unknown_families}"
        )

    non_eligible_families = [
        family_id
        for family_id in family_list
        if not bool(
            policy_lookup.loc[
                family_id,
                "primary_model_eligible",
            ]
        )
    ]

    if non_eligible_families:
        raise ValueError(
            "Non-eligible feature families were assigned to "
            f"a primary stage: {non_eligible_families}"
        )


print("Prediction-stage registry:")

display(
    prediction_stage_registry[
        [
            "stage_order",
            "stage_id",
            "stage_name",
            "families_added",
            "cumulative_families",
            "description",
            "intended_role",
        ]
    ]
)


#### Direct cohort-column audit


In [ ]:

direct_feature_candidates = [
    {
        "canonical_feature": "age",
        "family_id": "age",
        "mimic_candidates": [
            "age_at_ed",
        ],
        "mcmed_candidates": [
            "age_at_ed",
            "Age",
        ],
    },
    {
        "canonical_feature": "sex_gender",
        "family_id": "sex_gender",
        "mimic_candidates": [
            "gender",
            "sex",
        ],
        "mcmed_candidates": [
            "Sex",
            "sex",
            "Gender",
            "gender",
        ],
    },
    {
        "canonical_feature": "arrival_transport",
        "family_id": "arrival_transport",
        "mimic_candidates": [
            "arrival_transport",
            "arrival_transport_grouped",
        ],
        "mcmed_candidates": [
            "Means_of_arrival",
            "Means_of_transport",
            "means_of_arrival",
            "arrival_transport",
        ],
    },
    {
        "canonical_feature": "temperature",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "temperature",
        ],
        "mcmed_candidates": [
            "Triage_Temp",
        ],
    },
    {
        "canonical_feature": "heart_rate",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "heartrate",
            "heart_rate",
        ],
        "mcmed_candidates": [
            "Triage_HR",
        ],
    },
    {
        "canonical_feature": "respiratory_rate",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "resprate",
            "resp_rate",
        ],
        "mcmed_candidates": [
            "Triage_RR",
        ],
    },
    {
        "canonical_feature": "oxygen_saturation",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "o2sat",
            "spo2",
        ],
        "mcmed_candidates": [
            "Triage_SpO2",
        ],
    },
    {
        "canonical_feature": "systolic_blood_pressure",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "sbp",
        ],
        "mcmed_candidates": [
            "Triage_SBP",
        ],
    },
    {
        "canonical_feature": "diastolic_blood_pressure",
        "family_id": "triage_vitals",
        "mimic_candidates": [
            "dbp",
        ],
        "mcmed_candidates": [
            "Triage_DBP",
        ],
    },
    {
        "canonical_feature": "pain_score",
        "family_id": "pain_score",
        "mimic_candidates": [
            "pain",
            "pain_score",
        ],
        "mcmed_candidates": [
            "Triage_pain",
            "Triage_Pain",
            "Pain",
            "pain_score",
        ],
    },
    {
        "canonical_feature": "race_ethnicity",
        "family_id": "race_ethnicity",
        "mimic_candidates": [
            "race",
            "Ethnoracial_group",
        ],
        "mcmed_candidates": [
            "Race",
            "race",
            "Ethnicity",
            "ethnicity",
        ],
    },
    {
        "canonical_feature": "chief_complaint",
        "family_id": "chief_complaint",
        "mimic_candidates": [
            "chiefcomplaint",
            "chief_complaint_normalised",
        ],
        "mcmed_candidates": [
            "CC",
            "chief_complaint_normalised",
        ],
    },
    {
        "canonical_feature": "esi",
        "family_id": "esi",
        "mimic_candidates": [
            "esi_numeric",
            "acuity",
        ],
        "mcmed_candidates": [
            "esi_numeric",
            "Triage_acuity",
        ],
    },
]


def first_existing_column(
    dataframe: pd.DataFrame,
    candidate_columns: list[str],
) -> str | None:
    """Return the first candidate column present in a dataframe."""

    for column in candidate_columns:
        if column in dataframe.columns:
            return column

    return None


direct_feature_availability_rows = []


for feature_specification in direct_feature_candidates:
    for (
        dataset_name,
        dataframe,
        candidate_key,
    ) in [
        (
            "MIMIC-IV",
            mimic_primary_binary,
            "mimic_candidates",
        ),
        (
            "MC-MED",
            mcmed_primary_binary,
            "mcmed_candidates",
        ),
    ]:
        selected_column = first_existing_column(
            dataframe=dataframe,
            candidate_columns=feature_specification[
                candidate_key
            ],
        )

        column_present = selected_column is not None

        direct_feature_availability_rows.append(
            {
                "dataset": dataset_name,
                "canonical_feature": (
                    feature_specification[
                        "canonical_feature"
                    ]
                ),
                "family_id": (
                    feature_specification[
                        "family_id"
                    ]
                ),
                "selected_source_column": (
                    selected_column
                ),
                "column_present": (
                    column_present
                ),
                "dtype": (
                    str(
                        dataframe[
                            selected_column
                        ].dtype
                    )
                    if column_present
                    else None
                ),
                "missing_n": (
                    int(
                        dataframe[
                            selected_column
                        ].isna().sum()
                    )
                    if column_present
                    else None
                ),
                "missing_pct": (
                    dataframe[
                        selected_column
                    ].isna().mean()
                    * 100
                    if column_present
                    else None
                ),
                "n_unique_values": (
                    int(
                        dataframe[
                            selected_column
                        ].nunique(
                            dropna=True
                        )
                    )
                    if column_present
                    else None
                ),
            }
        )


direct_feature_availability = pd.DataFrame(
    direct_feature_availability_rows
)


direct_feature_availability = (
    direct_feature_availability.merge(
        feature_family_policy[
            [
                "family_id",
                "primary_model_eligible",
                "planned_use",
            ]
        ],
        on="family_id",
        how="left",
        validate="many_to_one",
    )
)


print("Direct feature availability in frozen cohorts:")

display(
    direct_feature_availability.round(
        {
            "missing_pct": 2,
        }
    )
)


print("\nMissing direct source columns:")

display(
    direct_feature_availability.loc[
        ~direct_feature_availability[
            "column_present"
        ]
    ]
)


#### Feature-construction source registry


In [ ]:


SOURCE_PATHS.setdefault(
    "mimic_diagnoses_icd",
    MIMIC_IV_HOSP_DIR
    / "diagnoses_icd.csv.gz",
)

SOURCE_PATHS.setdefault(
    "mimic_d_icd_diagnoses",
    MIMIC_IV_HOSP_DIR
    / "d_icd_diagnoses.csv.gz",
)



feature_source_rows = [
    {
        "family_id": "age",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_patients",
        "temporal_rule": (
            "Age derived at current ED arrival."
        ),
        "expected_output": "age",
    },
    {
        "family_id": "age",
        "dataset": "MC-MED",
        "source_key": "mcmed_visits",
        "temporal_rule": (
            "Age recorded for current ED visit."
        ),
        "expected_output": "age",
    },
    {
        "family_id": "sex_gender",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_edstays",
        "temporal_rule": "Available at current ED arrival.",
        "expected_output": "sex_gender",
    },
    {
        "family_id": "sex_gender",
        "dataset": "MC-MED",
        "source_key": "mcmed_visits",
        "temporal_rule": "Available at current ED arrival.",
        "expected_output": "sex_gender",
    },
    {
        "family_id": "arrival_transport",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_edstays",
        "temporal_rule": "Available at current ED arrival.",
        "expected_output": "arrival_transport_grouped",
    },
    {
        "family_id": "arrival_transport",
        "dataset": "MC-MED",
        "source_key": "mcmed_visits",
        "temporal_rule": "Available at current ED arrival.",
        "expected_output": "arrival_transport_grouped",
    },
    {
        "family_id": "triage_vitals",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_triage",
        "temporal_rule": (
            "Use only initial triage measurements."
        ),
        "expected_output": (
            "temperature, heart_rate, respiratory_rate, "
            "oxygen_saturation, SBP, DBP"
        ),
    },
    {
        "family_id": "triage_vitals",
        "dataset": "MC-MED",
        "source_key": "mcmed_visits",
        "temporal_rule": (
            "Use only initial triage measurements."
        ),
        "expected_output": (
            "temperature, heart_rate, respiratory_rate, "
            "oxygen_saturation, SBP, DBP"
        ),
    },
    {
        "family_id": "pain_score",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_triage",
        "temporal_rule": "Initial triage pain score only.",
        "expected_output": "pain_score",
    },
    {
        "family_id": "pain_score",
        "dataset": "MC-MED",
        "source_key": "mcmed_visits",
        "temporal_rule": "Initial triage pain score only.",
        "expected_output": "pain_score",
    },
    {
        "family_id": "prior_diagnoses",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_admissions",
        "temporal_rule": (
            "Previous hospital discharge time must be earlier "
            "than current ED intime."
        ),
        "expected_output": "previous diagnosis history",
    },
    {
        "family_id": "prior_diagnoses",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_diagnoses_icd",
        "temporal_rule": (
            "Use diagnoses linked only to eligible previous admissions."
        ),
        "expected_output": "previous ICD diagnosis codes",
    },
    {
        "family_id": "prior_diagnoses",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_d_icd_diagnoses",
        "temporal_rule": (
            "Map previous ICD codes to readable titles."
        ),
        "expected_output": "previous diagnosis titles",
    },
    {
        "family_id": "prior_diagnoses",
        "dataset": "MC-MED",
        "source_key": "mcmed_pmh",
        "temporal_rule": (
            "Use previous-history records available before or at "
            "current arrival; current final diagnoses are prohibited."
        ),
        "expected_output": "harmonised previous diagnosis features",
    },
    {
        "family_id": "prior_medications",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_medrecon",
        "temporal_rule": (
            "Use previous/home medication reconciliation only; "
            "exclude current treatment orders."
        ),
        "expected_output": (
            "previous medication name and class features"
        ),
    },
    {
        "family_id": "prior_medications",
        "dataset": "MC-MED",
        "source_key": "mcmed_meds",
        "temporal_rule": (
            "Use previous/home medications only; "
            "exclude current ED treatment."
        ),
        "expected_output": (
            "harmonised previous medication features"
        ),
    },
    {
        "family_id": "lab_results",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_labevents",
        "temporal_rule": (
            "Result chart time must fall within the current ED "
            "intime-outtime interval."
        ),
        "expected_output": "laboratory abnormal-result features",
    },
    {
        "family_id": "lab_results",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_d_labitems",
        "temporal_rule": "Map itemid to laboratory labels.",
        "expected_output": "laboratory item dictionary",
    },
    {
        "family_id": "lab_results",
        "dataset": "MC-MED",
        "source_key": "mcmed_labs",
        "temporal_rule": (
            "Use laboratory results associated with the eligible "
            "current ED encounter."
        ),
        "expected_output": (
            "harmonised laboratory abnormal-result features"
        ),
    },
    {
        "family_id": "imaging_results",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_radiology_notes",
        "temporal_rule": (
            "Use reports associated with the eligible ED encounter "
            "and available before ED departure."
        ),
        "expected_output": (
            "radiology abnormal-result evidence"
        ),
    },
    {
        "family_id": "imaging_results",
        "dataset": "MIMIC-IV",
        "source_key": "mimic_radiology_detail",
        "temporal_rule": (
            "Use report/exam metadata for safe encounter and "
            "timestamp linkage."
        ),
        "expected_output": "radiology linkage metadata",
    },
    {
        "family_id": "imaging_results",
        "dataset": "MC-MED",
        "source_key": "mcmed_rads",
        "temporal_rule": (
            "Use radiology results associated with the eligible "
            "current ED encounter."
        ),
        "expected_output": (
            "harmonised radiology abnormal-result features"
        ),
    },
]


feature_source_registry = pd.DataFrame(
    feature_source_rows
)



source_status_records = []


for _, source_row in feature_source_registry.iterrows():
    source_key = source_row["source_key"]

    source_path = SOURCE_PATHS.get(
        source_key
    )

    source_exists = (
        source_path is not None
        and Path(source_path).exists()
    )

    source_status_records.append(
        {
            **source_row.to_dict(),
            "source_path": (
                str(source_path)
                if source_path is not None
                else None
            ),
            "source_registered": (
                source_path is not None
            ),
            "source_exists": source_exists,
            "size_mb": (
                round(
                    Path(source_path).stat().st_size
                    / (1024 ** 2),
                    2,
                )
                if source_exists
                else np.nan
            ),
        }
    )


feature_source_status = pd.DataFrame(
    source_status_records
)


print("Feature source registry:")

display(feature_source_status)


print("\nMissing or unregistered required feature sources:")

display(
    feature_source_status.loc[
        ~feature_source_status[
            "source_exists"
        ]
    ]
)


#### Registry validation


In [ ]:



incorrectly_eligible = (
    feature_family_policy.loc[
        feature_family_policy[
            "planned_use"
        ].isin(
            [
                "EXCLUDED_WORKFLOW",
                "PROHIBITED",
                "METADATA_ONLY",
                "CONSTANT_BY_DESIGN",
                "NOVELTY_ANALYSIS_ONLY",
            ]
        )
        &
        feature_family_policy[
            "primary_model_eligible"
        ]
    ]
)


assert incorrectly_eligible.empty, (
    "One or more excluded/prohibited feature families "
    "were incorrectly marked as primary-model eligible."
)



primary_family_ids = set(
    feature_family_policy.loc[
        feature_family_policy[
            "primary_model_eligible"
        ],
        "family_id",
    ]
)


source_dataset_coverage = (
    feature_source_registry.loc[
        feature_source_registry[
            "family_id"
        ].isin(primary_family_ids)
    ]
    .groupby("family_id")[
        "dataset"
    ]
    .agg(
        lambda values: sorted(
            set(values)
        )
    )
    .reset_index(
        name="datasets_available"
    )
)


source_dataset_coverage[
    "available_in_both_datasets"
] = (
    source_dataset_coverage[
        "datasets_available"
    ]
    .apply(
        lambda values: set(values)
        == {
            "MIMIC-IV",
            "MC-MED",
        }
    )
)


display(source_dataset_coverage)


missing_cross_dataset_sources = (
    source_dataset_coverage.loc[
        ~source_dataset_coverage[
            "available_in_both_datasets"
        ]
    ]
)


assert missing_cross_dataset_sources.empty, (
    "At least one primary feature family does not have "
    "a registered source in both datasets."
)



age_availability = (
    direct_feature_availability.loc[
        direct_feature_availability[
            "canonical_feature"
        ].eq("age")
    ]
)


assert (
    age_availability[
        "column_present"
    ].all()
), (
    "Age was not found in both frozen cohort tables."
)



FEATURE_POLICY_PATH = (
    TABLE_DIR
    / "feature_family_policy.csv"
)

PREDICTION_STAGE_PATH = (
    TABLE_DIR
    / "prediction_stage_registry.csv"
)

DIRECT_FEATURE_AUDIT_PATH = (
    TABLE_DIR
    / "direct_feature_availability.csv"
)

FEATURE_SOURCE_PATH = (
    TABLE_DIR
    / "feature_source_registry.csv"
)


feature_family_policy.to_csv(
    FEATURE_POLICY_PATH,
    index=False,
)


prediction_stage_registry_csv = (
    prediction_stage_registry.copy()
)

for list_column in [
    "families_added",
    "cumulative_families",
]:
    prediction_stage_registry_csv[
        list_column
    ] = prediction_stage_registry_csv[
        list_column
    ].apply(json.dumps)


prediction_stage_registry_csv.to_csv(
    PREDICTION_STAGE_PATH,
    index=False,
)


direct_feature_availability.to_csv(
    DIRECT_FEATURE_AUDIT_PATH,
    index=False,
)


feature_source_status.to_csv(
    FEATURE_SOURCE_PATH,
    index=False,
)



FEATURE_REGISTRY_JSON_PATH = (
    PROCESSED_DIR
    / "feature_timing_and_eligibility_registry.json"
)


feature_registry_json = {
    "primary_full_stage": "S2_OBJECTIVE_WORKUP",
    "study_description": (
        "Clinician-action-restricted disposition model using "
        "triage, previous history and objective work-up results."
    ),
    "feature_family_policy": (
        feature_family_policy.to_dict(
            orient="records"
        )
    ),
    "prediction_stages": (
        prediction_stage_registry.to_dict(
            orient="records"
        )
    ),
    "excluded_from_primary_model": (
        feature_family_policy.loc[
            ~feature_family_policy[
                "primary_model_eligible"
            ],
            [
                "family_id",
                "planned_use",
                "methodological_note",
            ],
        ]
        .to_dict(
            orient="records"
        )
    ),
}


with open(
    FEATURE_REGISTRY_JSON_PATH,
    "w",
    encoding="utf-8",
) as registry_file:
    json.dump(
        feature_registry_json,
        registry_file,
        indent=2,
    )



registry_output_paths = [
    FEATURE_POLICY_PATH,
    PREDICTION_STAGE_PATH,
    DIRECT_FEATURE_AUDIT_PATH,
    FEATURE_SOURCE_PATH,
    FEATURE_REGISTRY_JSON_PATH,
]


registry_output_status = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "path": str(path),
        }
        for path in registry_output_paths
    ]
)


display(registry_output_status)


assert registry_output_status[
    "exists"
].all()


print(
    "\nFEATURE TIMING AND ELIGIBILITY REGISTRY PASSED"
)

print(
    "Primary full stage: S2_OBJECTIVE_WORKUP"
)

print(
    "Clinician orders, treatments, timing and repeated "
    "monitoring are excluded."
)

print(
    "HPI/PE and LLM-derived variables are reserved for "
    "downstream novelty analysis."
)


#### Canonical S0 triage feature block

This section creates the first harmonised predictor block for MIMIC-IV model development and MC-MED external validation.

The S0 triage block contains:

- age at ED arrival;
- sex or gender as represented by each source dataset;
- grouped arrival transport;
- initial triage temperature;
- initial heart rate;
- initial respiratory rate;
- initial oxygen saturation;
- initial systolic blood pressure;
- initial diastolic blood pressure.

Pain score is excluded from the primary cross-dataset model. MIMIC-IV provides a patient-reported triage pain field, whereas MC-MED does not provide an equivalent visit-level triage pain variable. Pain measurements extracted from the MC-MED longitudinal numerics table would represent repeated or post-arrival monitoring rather than the same triage measurement.

The MC-MED `Gender` column is recovered directly from `visits.csv` because it was not included in the earlier source-table loading cell.

Temperature is harmonised to degrees Celsius. Values above 50 are interpreted as Fahrenheit and converted to Celsius, while values at or below 50 are retained as Celsius. The source-unit interpretation is audited before the feature block is saved.

No imputation, scaling, one-hot encoding, feature selection or model fitting is performed in this section.


#### MC-MED gender recovery and policy correction


In [ ]:



pain_policy_mask = (
    feature_family_policy["family_id"]
    .eq("pain_score")
)

assert pain_policy_mask.sum() == 1

feature_family_policy.loc[
    pain_policy_mask,
    "primary_model_eligible",
] = False

feature_family_policy.loc[
    pain_policy_mask,
    "external_harmonisation_required",
] = False

feature_family_policy.loc[
    pain_policy_mask,
    "planned_use",
] = "EXCLUDED_NON_EQUIVALENT"

feature_family_policy.loc[
    pain_policy_mask,
    "clinician_influence",
] = (
    "patient_reported_but_cross_dataset_measurement_not_equivalent"
)

feature_family_policy.loc[
    pain_policy_mask,
    "methodological_note",
] = (
    "MIMIC-IV provides an initial free-text triage pain score, "
    "whereas MC-MED has no equivalent visit-level triage pain "
    "field. Longitudinal MC-MED Pain measurements are therefore "
    "not used in the primary external-validation model."
)



def ensure_python_list(value):
    """Convert a stored list or JSON-list string into a list."""

    if isinstance(value, list):
        return value.copy()

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, str):
        try:
            parsed = json.loads(value)

            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass

    raise TypeError(
        "Prediction-stage family values must be lists "
        "or JSON-formatted lists."
    )


prediction_stage_registry[
    "families_added"
] = prediction_stage_registry[
    "families_added"
].apply(ensure_python_list)


prediction_stage_registry[
    "families_added"
] = prediction_stage_registry[
    "families_added"
].apply(
    lambda families: [
        family
        for family in families
        if family != "pain_score"
    ]
)


cumulative_families = []
updated_cumulative_lists = []

for _, stage_row in (
    prediction_stage_registry
    .sort_values("stage_order")
    .iterrows()
):
    cumulative_families.extend(
        stage_row["families_added"]
    )

    cumulative_families = list(
        dict.fromkeys(cumulative_families)
    )

    updated_cumulative_lists.append(
        cumulative_families.copy()
    )


prediction_stage_registry = (
    prediction_stage_registry
    .sort_values("stage_order")
    .reset_index(drop=True)
)

prediction_stage_registry[
    "cumulative_families"
] = updated_cumulative_lists



mcmed_visits_path = SOURCE_PATHS[
    "mcmed_visits"
]

mcmed_visits_header = pd.read_csv(
    mcmed_visits_path,
    nrows=0,
).columns.tolist()


def normalise_column_name(column_name):
    """Normalise a column name for case-insensitive matching."""

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(column_name).lower(),
    )


mcmed_normalised_header = {
    normalise_column_name(column): column
    for column in mcmed_visits_header
}


def find_header_column(candidate_names):
    """Return the first matching source column."""

    for candidate in candidate_names:
        candidate_normalised = normalise_column_name(
            candidate
        )

        if candidate_normalised in mcmed_normalised_header:
            return mcmed_normalised_header[
                candidate_normalised
            ]

    return None


mcmed_gender_source_column = find_header_column(
    [
        "Gender",
        "Sex",
        "Patient_gender",
        "Patient_sex",
    ]
)


if mcmed_gender_source_column is None:
    raise ValueError(
        "No MC-MED gender/sex field was found in visits.csv.\n"
        f"Available columns:\n{mcmed_visits_header}"
    )



mcmed_gender_lookup = pd.read_csv(
    mcmed_visits_path,
    usecols=[
        "CSN",
        mcmed_gender_source_column,
    ],
    low_memory=False,
)

mcmed_gender_lookup["CSN"] = pd.to_numeric(
    mcmed_gender_lookup["CSN"],
    errors="coerce",
).astype("Int64")

assert mcmed_gender_lookup["CSN"].notna().all()
assert mcmed_gender_lookup["CSN"].is_unique


mcmed_gender_lookup = (
    mcmed_gender_lookup.rename(
        columns={
            mcmed_gender_source_column:
                "gender_source_mcmed"
        }
    )
)


mcmed_primary_with_gender = (
    mcmed_primary_binary.copy()
)

for column in [
    "Gender",
    "gender",
    "Sex",
    "sex",
    "gender_source_mcmed",
]:
    if column in mcmed_primary_with_gender.columns:
        mcmed_primary_with_gender = (
            mcmed_primary_with_gender.drop(
                columns=column
            )
        )


mcmed_primary_with_gender = (
    mcmed_primary_with_gender.merge(
        mcmed_gender_lookup,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)



updated_policy_summary = (
    feature_family_policy
    .groupby(
        [
            "planned_use",
            "primary_model_eligible",
        ],
        dropna=False,
    )
    .size()
    .rename("n_feature_families")
    .reset_index()
)


print("Updated feature-policy summary:")

display(updated_policy_summary)


print(
    "\nMC-MED gender source column:",
    mcmed_gender_source_column,
)

print("\nMC-MED recovered Gender distribution:")

display(
    mcmed_primary_with_gender[
        "gender_source_mcmed"
    ]
    .fillna("MISSING")
    .value_counts(dropna=False)
    .rename_axis("source_value")
    .reset_index(name="count")
)


#### Construct the canonical S0 blocks


In [ ]:


def map_sex_gender(value):
    """
    Map source sex/gender values into a shared descriptive
    representation without claiming biological equivalence.
    """

    if pd.isna(value):
        return "OTHER_OR_UNKNOWN"

    normalised = str(value).strip().upper()

    female_values = {
        "F",
        "FEMALE",
        "WOMAN",
    }

    male_values = {
        "M",
        "MALE",
        "MAN",
    }

    if normalised in female_values:
        return "F"

    if normalised in male_values:
        return "M"

    return "OTHER_OR_UNKNOWN"


def map_arrival_transport(value):
    """Map source arrival modes into shared categories."""

    if pd.isna(value):
        return "OTHER_OR_UNKNOWN"

    normalised = str(value).strip().lower()

    if normalised in {
        "",
        "nan",
        "none",
        "null",
        "unknown",
        "other",
        "not specified",
    }:
        return "OTHER_OR_UNKNOWN"

    ems_terms = [
        "ambulance",
        "ems",
        "emergency medical",
        "paramedic",
        "medic",
        "helicopter",
        "air ambulance",
    ]

    self_terms = [
        "self",
        "walk",
        "walk-in",
        "walk in",
        "private",
        "car",
        "taxi",
        "cab",
        "bus",
        "family",
        "friend",
    ]

    if any(
        term in normalised
        for term in ems_terms
    ):
        return "EMS"

    if any(
        term in normalised
        for term in self_terms
    ):
        return "SELF"

    return "OTHER_OR_UNKNOWN"


def harmonise_temperature_to_celsius(series):
    """
    Convert Fahrenheit-like values to Celsius.

    Values greater than 50 are treated as Fahrenheit.
    Values at or below 50 are treated as Celsius.
    """

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    fahrenheit_mask = numeric.gt(50)

    temperature_c = numeric.copy()

    temperature_c.loc[
        fahrenheit_mask
    ] = (
        numeric.loc[fahrenheit_mask]
        - 32
    ) * 5 / 9

    return temperature_c, fahrenheit_mask



mimic_s0_source = (
    mimic_split_source.copy()
)

required_mimic_s0_columns = [
    "subject_id",
    "stay_id",
    "intime",
    "split",
    "outcome",
    "outcome_binary",
    "anchor_year",
    "anchor_year_group",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
    "approx_event_period",
    "age_at_ed",
    "gender",
    "arrival_transport",
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
]

missing_mimic_s0_columns = [
    column
    for column in required_mimic_s0_columns
    if column not in mimic_s0_source.columns
]

if missing_mimic_s0_columns:
    raise ValueError(
        "MIMIC S0 source is missing columns:\n"
        f"{missing_mimic_s0_columns}"
    )


mimic_temperature_c, mimic_temp_f_mask = (
    harmonise_temperature_to_celsius(
        mimic_s0_source["temperature"]
    )
)


mimic_s0_features = pd.DataFrame(
    {
        "source_dataset": "MIMIC-IV",
        "subject_id": (
            mimic_s0_source["subject_id"]
        ),
        "stay_id": (
            mimic_s0_source["stay_id"]
        ),
        "split": (
            mimic_s0_source["split"]
            .astype("string")
        ),
        "intime": pd.to_datetime(
            mimic_s0_source["intime"],
            errors="coerce",
        ),
        "outcome": (
            mimic_s0_source["outcome"]
            .astype("string")
        ),
        "outcome_binary": (
            mimic_s0_source[
                "outcome_binary"
            ].astype("int8")
        ),
        "anchor_year": pd.to_numeric(
            mimic_s0_source["anchor_year"],
            errors="raise",
        ).astype("Int64"),
        "anchor_year_group": (
            mimic_s0_source["anchor_year_group"]
            .astype("string")
        ),
        "shifted_encounter_year": pd.to_numeric(
            mimic_s0_source["shifted_encounter_year"],
            errors="raise",
        ).astype("Int64"),
        "anchor_to_encounter_year_offset": pd.to_numeric(
            mimic_s0_source["anchor_to_encounter_year_offset"],
            errors="raise",
        ).astype("Int64"),
        "approx_event_year_start": pd.to_numeric(
            mimic_s0_source["approx_event_year_start"],
            errors="raise",
        ).astype("Int64"),
        "approx_event_year_end": pd.to_numeric(
            mimic_s0_source["approx_event_year_end"],
            errors="raise",
        ).astype("Int64"),
        "approx_event_year": pd.to_numeric(
            mimic_s0_source["approx_event_year"],
            errors="raise",
        ).astype("Int64"),
        "approx_event_period": (
            mimic_s0_source["approx_event_period"]
            .astype("string")
        ),
        "age": pd.to_numeric(
            mimic_s0_source["age_at_ed"],
            errors="coerce",
        ),
        "sex_gender": (
            mimic_s0_source["gender"]
            .apply(map_sex_gender)
            .astype("string")
        ),
        "arrival_transport_grouped": (
            mimic_s0_source[
                "arrival_transport"
            ]
            .apply(map_arrival_transport)
            .astype("string")
        ),
        "temperature_c": mimic_temperature_c,
        "heart_rate": pd.to_numeric(
            mimic_s0_source["heartrate"],
            errors="coerce",
        ),
        "respiratory_rate": pd.to_numeric(
            mimic_s0_source["resprate"],
            errors="coerce",
        ),
        "oxygen_saturation": pd.to_numeric(
            mimic_s0_source["o2sat"],
            errors="coerce",
        ),
        "systolic_blood_pressure": pd.to_numeric(
            mimic_s0_source["sbp"],
            errors="coerce",
        ),
        "diastolic_blood_pressure": pd.to_numeric(
            mimic_s0_source["dbp"],
            errors="coerce",
        ),
    }
)



mcmed_s0_source = (
    mcmed_primary_with_gender.copy()
)

required_mcmed_s0_columns = [
    "MRN",
    "CSN",
    "outcome",
    "outcome_binary",
    "age_at_ed",
    "gender_source_mcmed",
    "Means_of_arrival",
    "Triage_Temp",
    "Triage_HR",
    "Triage_RR",
    "Triage_SpO2",
    "Triage_SBP",
    "Triage_DBP",
]

missing_mcmed_s0_columns = [
    column
    for column in required_mcmed_s0_columns
    if column not in mcmed_s0_source.columns
]

if missing_mcmed_s0_columns:
    raise ValueError(
        "MC-MED S0 source is missing columns:\n"
        f"{missing_mcmed_s0_columns}"
    )


mcmed_temperature_c, mcmed_temp_f_mask = (
    harmonise_temperature_to_celsius(
        mcmed_s0_source["Triage_Temp"]
    )
)


mcmed_s0_features = pd.DataFrame(
    {
        "source_dataset": "MC-MED",
        "MRN": (
            mcmed_s0_source["MRN"]
        ),
        "CSN": (
            mcmed_s0_source["CSN"]
        ),
        "split": pd.Series(
            "external",
            index=mcmed_s0_source.index,
            dtype="string",
        ),
        "outcome": (
            mcmed_s0_source["outcome"]
            .astype("string")
        ),
        "outcome_binary": (
            mcmed_s0_source[
                "outcome_binary"
            ].astype("int8")
        ),
        "age": pd.to_numeric(
            mcmed_s0_source["age_at_ed"],
            errors="coerce",
        ),
        "sex_gender": (
            mcmed_s0_source[
                "gender_source_mcmed"
            ]
            .apply(map_sex_gender)
            .astype("string")
        ),
        "arrival_transport_grouped": (
            mcmed_s0_source[
                "Means_of_arrival"
            ]
            .apply(map_arrival_transport)
            .astype("string")
        ),
        "temperature_c": mcmed_temperature_c,
        "heart_rate": pd.to_numeric(
            mcmed_s0_source["Triage_HR"],
            errors="coerce",
        ),
        "respiratory_rate": pd.to_numeric(
            mcmed_s0_source["Triage_RR"],
            errors="coerce",
        ),
        "oxygen_saturation": pd.to_numeric(
            mcmed_s0_source["Triage_SpO2"],
            errors="coerce",
        ),
        "systolic_blood_pressure": pd.to_numeric(
            mcmed_s0_source["Triage_SBP"],
            errors="coerce",
        ),
        "diastolic_blood_pressure": pd.to_numeric(
            mcmed_s0_source["Triage_DBP"],
            errors="coerce",
        ),
    }
)



S0_FEATURE_COLUMNS = [
    "age",
    "sex_gender",
    "arrival_transport_grouped",
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
]


S0_NUMERIC_FEATURES = [
    "age",
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
]


S0_CATEGORICAL_FEATURES = [
    "sex_gender",
    "arrival_transport_grouped",
]


print(
    "MIMIC-IV S0 shape:",
    mimic_s0_features.shape,
)

print(
    "MC-MED S0 shape:",
    mcmed_s0_features.shape,
)

print(
    "Canonical S0 model features:",
    S0_FEATURE_COLUMNS,
)


#### S0 units, categories and distributions


In [ ]:


temperature_unit_audit = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "n_non_missing_temperature": int(
                mimic_s0_source[
                    "temperature"
                ].notna().sum()
            ),
            "n_interpreted_as_fahrenheit": int(
                mimic_temp_f_mask.sum()
            ),
            "n_interpreted_as_celsius": int(
                (
                    mimic_s0_source[
                        "temperature"
                    ].notna()
                    & ~mimic_temp_f_mask
                ).sum()
            ),
            "raw_median": pd.to_numeric(
                mimic_s0_source[
                    "temperature"
                ],
                errors="coerce",
            ).median(),
            "harmonised_median_c": (
                mimic_s0_features[
                    "temperature_c"
                ].median()
            ),
        },
        {
            "dataset": "MC-MED",
            "n_non_missing_temperature": int(
                mcmed_s0_source[
                    "Triage_Temp"
                ].notna().sum()
            ),
            "n_interpreted_as_fahrenheit": int(
                mcmed_temp_f_mask.sum()
            ),
            "n_interpreted_as_celsius": int(
                (
                    mcmed_s0_source[
                        "Triage_Temp"
                    ].notna()
                    & ~mcmed_temp_f_mask
                ).sum()
            ),
            "raw_median": pd.to_numeric(
                mcmed_s0_source[
                    "Triage_Temp"
                ],
                errors="coerce",
            ).median(),
            "harmonised_median_c": (
                mcmed_s0_features[
                    "temperature_c"
                ].median()
            ),
        },
    ]
)


print("Temperature-unit audit:")

display(
    temperature_unit_audit.round(2)
)



categorical_distribution_tables = []

for (
    dataset_name,
    dataframe,
) in [
    (
        "MIMIC-IV",
        mimic_s0_features,
    ),
    (
        "MC-MED",
        mcmed_s0_features,
    ),
]:
    for feature in S0_CATEGORICAL_FEATURES:
        feature_summary = (
            dataframe[feature]
            .fillna("MISSING")
            .value_counts(dropna=False)
            .rename_axis("category")
            .reset_index(name="count")
        )

        feature_summary["percentage"] = (
            feature_summary["count"]
            / len(dataframe)
            * 100
        )

        feature_summary.insert(
            0,
            "feature",
            feature,
        )

        feature_summary.insert(
            0,
            "dataset",
            dataset_name,
        )

        categorical_distribution_tables.append(
            feature_summary
        )


s0_categorical_distribution = pd.concat(
    categorical_distribution_tables,
    ignore_index=True,
)


print("\nS0 categorical distributions:")

display(
    s0_categorical_distribution.round(
        {"percentage": 2}
    )
)



numeric_distribution_tables = []

for (
    dataset_name,
    dataframe,
) in [
    (
        "MIMIC-IV",
        mimic_s0_features,
    ),
    (
        "MC-MED",
        mcmed_s0_features,
    ),
]:
    numeric_summary = (
        dataframe[
            S0_NUMERIC_FEATURES
        ]
        .describe(
            percentiles=[
                0.01,
                0.25,
                0.50,
                0.75,
                0.99,
            ]
        )
        .T
        .reset_index()
        .rename(
            columns={
                "index": "feature",
            }
        )
    )

    numeric_summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    numeric_distribution_tables.append(
        numeric_summary
    )


s0_numeric_distribution = pd.concat(
    numeric_distribution_tables,
    ignore_index=True,
)


print("\nS0 numeric distributions:")

display(
    s0_numeric_distribution.round(2)
)


#### S0 missingness and feature alignment


In [ ]:


def create_feature_missingness_summary(
    dataframe,
    dataset_name,
    split_column="split",
):
    """Summarise raw missingness without performing imputation."""

    records = []

    split_values = (
        dataframe[split_column]
        .astype("string")
        .dropna()
        .unique()
        .tolist()
    )

    for split_name in split_values:
        split_rows = dataframe.loc[
            dataframe[
                split_column
            ].astype("string").eq(
                split_name
            )
        ]

        for feature in S0_FEATURE_COLUMNS:
            missing_count = int(
                split_rows[
                    feature
                ].isna().sum()
            )

            records.append(
                {
                    "dataset": dataset_name,
                    "split": split_name,
                    "feature": feature,
                    "n_rows": len(split_rows),
                    "missing_n": missing_count,
                    "missing_pct": (
                        missing_count
                        / len(split_rows)
                        * 100
                        if len(split_rows) > 0
                        else np.nan
                    ),
                    "n_unique_non_missing": int(
                        split_rows[
                            feature
                        ].nunique(
                            dropna=True
                        )
                    ),
                }
            )

    return pd.DataFrame(records)


mimic_s0_missingness = (
    create_feature_missingness_summary(
        dataframe=mimic_s0_features,
        dataset_name="MIMIC-IV",
    )
)

mcmed_s0_missingness = (
    create_feature_missingness_summary(
        dataframe=mcmed_s0_features,
        dataset_name="MC-MED",
    )
)

s0_missingness_summary = pd.concat(
    [
        mimic_s0_missingness,
        mcmed_s0_missingness,
    ],
    ignore_index=True,
)


print("S0 missingness before imputation:")

display(
    s0_missingness_summary.round(
        {"missing_pct": 2}
    )
)



mimic_s0_model_columns = (
    mimic_s0_features[
        S0_FEATURE_COLUMNS
    ].columns.tolist()
)

mcmed_s0_model_columns = (
    mcmed_s0_features[
        S0_FEATURE_COLUMNS
    ].columns.tolist()
)

assert (
    mimic_s0_model_columns
    == mcmed_s0_model_columns
)



assert len(mimic_s0_features) == 8550
assert len(mcmed_s0_features) == 6707

assert mimic_s0_features[
    "subject_id"
].is_unique

assert mimic_s0_features[
    "stay_id"
].is_unique

assert mcmed_s0_features[
    "MRN"
].is_unique

assert mcmed_s0_features[
    "CSN"
].is_unique



PROHIBITED_MODEL_COLUMNS = {
    "outcome",
    "outcome_binary",
    "disposition",
    "disposition_clean",
    "hadm_id",
    "admission_type",
    "hospital_los_days",
    "has_recorded_icu_stay",
    "hospital_disposition_group",
    "split",
    "anchor_year",
    "anchor_year_group",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
    "approx_event_period",
    "intime",
    "outtime",
}


prohibited_s0_features = (
    set(S0_FEATURE_COLUMNS)
    & PROHIBITED_MODEL_COLUMNS
)

assert not prohibited_s0_features, (
    "Prohibited columns entered the S0 model feature list: "
    f"{sorted(prohibited_s0_features)}"
)


print(
    "\nS0 feature names align exactly between datasets."
)

print(
    "Patient and encounter uniqueness checks passed."
)

print(
    "No outcome, hospital-course or split variables "
    "are present in the S0 model feature list."
)


#### S0 feature manifest


In [ ]:

s0_feature_manifest_rows = [
    {
        "canonical_feature": "age",
        "family_id": "age",
        "data_type": "numeric",
        "unit_or_categories": "years",
        "mimic_source": "age_at_ed",
        "mcmed_source": "Age / age_at_ed",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
    {
        "canonical_feature": "sex_gender",
        "family_id": "sex_gender",
        "data_type": "categorical",
        "unit_or_categories": (
            "F, M, OTHER_OR_UNKNOWN"
        ),
        "mimic_source": "edstays.gender",
        "mcmed_source": "visits.Gender",
        "transformation": (
            "Shared descriptive category mapping"
        ),
        "imputation_applied": False,
    },
    {
        "canonical_feature": (
            "arrival_transport_grouped"
        ),
        "family_id": "arrival_transport",
        "data_type": "categorical",
        "unit_or_categories": (
            "SELF, EMS, OTHER_OR_UNKNOWN"
        ),
        "mimic_source": (
            "edstays.arrival_transport"
        ),
        "mcmed_source": (
            "visits.Means_of_arrival"
        ),
        "transformation": (
            "Cross-dataset category grouping"
        ),
        "imputation_applied": False,
    },
    {
        "canonical_feature": "temperature_c",
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": "degrees Celsius",
        "mimic_source": "triage.temperature",
        "mcmed_source": "visits.Triage_Temp",
        "transformation": (
            "Values >50 converted from Fahrenheit "
            "to Celsius"
        ),
        "imputation_applied": False,
    },
    {
        "canonical_feature": "heart_rate",
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": "beats per minute",
        "mimic_source": "triage.heartrate",
        "mcmed_source": "visits.Triage_HR",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
    {
        "canonical_feature": "respiratory_rate",
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": (
            "breaths per minute"
        ),
        "mimic_source": "triage.resprate",
        "mcmed_source": "visits.Triage_RR",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
    {
        "canonical_feature": "oxygen_saturation",
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": "percent",
        "mimic_source": "triage.o2sat",
        "mcmed_source": "visits.Triage_SpO2",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
    {
        "canonical_feature": (
            "systolic_blood_pressure"
        ),
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": "mmHg",
        "mimic_source": "triage.sbp",
        "mcmed_source": "visits.Triage_SBP",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
    {
        "canonical_feature": (
            "diastolic_blood_pressure"
        ),
        "family_id": "triage_vitals",
        "data_type": "numeric",
        "unit_or_categories": "mmHg",
        "mimic_source": "triage.dbp",
        "mcmed_source": "visits.Triage_DBP",
        "transformation": "Numeric conversion only",
        "imputation_applied": False,
    },
]


s0_feature_manifest = pd.DataFrame(
    s0_feature_manifest_rows
)


assert (
    s0_feature_manifest[
        "canonical_feature"
    ].tolist()
    == S0_FEATURE_COLUMNS
)


display(s0_feature_manifest)


#### Save the raw S0 blocks


In [ ]:


MIMIC_S0_PATH = (
    PROCESSED_DIR
    / "mimiciv_s0_triage_features_raw.parquet"
)

MCMED_S0_PATH = (
    PROCESSED_DIR
    / "mcmed_s0_triage_features_raw.parquet"
)

S0_MANIFEST_PATH = (
    TABLE_DIR
    / "s0_feature_manifest.csv"
)

S0_MISSINGNESS_PATH = (
    TABLE_DIR
    / "s0_missingness_before_imputation.csv"
)

S0_NUMERIC_AUDIT_PATH = (
    TABLE_DIR
    / "s0_numeric_distribution_audit.csv"
)

S0_CATEGORICAL_AUDIT_PATH = (
    TABLE_DIR
    / "s0_categorical_distribution_audit.csv"
)

TEMPERATURE_UNIT_AUDIT_PATH = (
    TABLE_DIR
    / "s0_temperature_unit_audit.csv"
)



mimic_s0_features.to_parquet(
    MIMIC_S0_PATH,
    index=False,
)

mcmed_s0_features.to_parquet(
    MCMED_S0_PATH,
    index=False,
)



s0_feature_manifest.to_csv(
    S0_MANIFEST_PATH,
    index=False,
)

s0_missingness_summary.to_csv(
    S0_MISSINGNESS_PATH,
    index=False,
)

s0_numeric_distribution.to_csv(
    S0_NUMERIC_AUDIT_PATH,
    index=False,
)

s0_categorical_distribution.to_csv(
    S0_CATEGORICAL_AUDIT_PATH,
    index=False,
)

temperature_unit_audit.to_csv(
    TEMPERATURE_UNIT_AUDIT_PATH,
    index=False,
)



feature_family_policy.to_csv(
    FEATURE_POLICY_PATH,
    index=False,
)


prediction_stage_registry_to_save = (
    prediction_stage_registry.copy()
)

for list_column in [
    "families_added",
    "cumulative_families",
]:
    prediction_stage_registry_to_save[
        list_column
    ] = prediction_stage_registry_to_save[
        list_column
    ].apply(json.dumps)


prediction_stage_registry_to_save.to_csv(
    PREDICTION_STAGE_PATH,
    index=False,
)


corrected_feature_registry_json = {
    "primary_full_stage": (
        "S2_OBJECTIVE_WORKUP"
    ),
    "primary_s0_features": (
        S0_FEATURE_COLUMNS
    ),
    "pain_score_primary_model_eligible": False,
    "pain_score_exclusion_reason": (
        "No equivalent visit-level MC-MED triage "
        "pain variable"
    ),
    "feature_family_policy": (
        feature_family_policy.to_dict(
            orient="records"
        )
    ),
    "prediction_stages": (
        prediction_stage_registry.to_dict(
            orient="records"
        )
    ),
}


with open(
    FEATURE_REGISTRY_JSON_PATH,
    "w",
    encoding="utf-8",
) as registry_file:
    json.dump(
        corrected_feature_registry_json,
        registry_file,
        indent=2,
    )



s0_output_paths = [
    MIMIC_S0_PATH,
    MCMED_S0_PATH,
    S0_MANIFEST_PATH,
    S0_MISSINGNESS_PATH,
    S0_NUMERIC_AUDIT_PATH,
    S0_CATEGORICAL_AUDIT_PATH,
    TEMPERATURE_UNIT_AUDIT_PATH,
    FEATURE_POLICY_PATH,
    PREDICTION_STAGE_PATH,
    FEATURE_REGISTRY_JSON_PATH,
]


s0_output_status = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if path.exists()
                else np.nan
            ),
            "path": str(path),
        }
        for path in s0_output_paths
    ]
)


display(s0_output_status)

assert s0_output_status[
    "exists"
].all()


print(
    "\nS0 TRIAGE FEATURE BLOCK PASSED"
)

print(
    "MIMIC-IV rows:",
    f"{len(mimic_s0_features):,}",
)

print(
    "MC-MED rows:",
    f"{len(mcmed_s0_features):,}",
)

print(
    "Number of aligned S0 features:",
    len(S0_FEATURE_COLUMNS),
)

print(
    "Pain score excluded from the primary "
    "cross-dataset feature set."
)


#### S0 physiological-validity audit

The raw S0 triage feature block is retained unchanged for audit. A second validated S0 block is created by identifying values outside broad physiological plausibility limits.

These limits are data-quality rules rather than definitions of clinically normal values. Values outside the limits are treated as invalid measurements and replaced with missing values. They are not clipped to the nearest boundary, winsorised or used to exclude encounters.

The predefined validity ranges are:

- temperature: 25–45°C;
- heart rate: 20–250 beats per minute;
- respiratory rate: 3–80 breaths per minute;
- oxygen saturation: 50–100%;
- systolic blood pressure: 40–300 mmHg;
- diastolic blood pressure: 20–200 mmHg.

When both blood-pressure measurements are available but diastolic pressure is greater than or equal to systolic pressure, both values are treated as an unresolved measurement pair and replaced with missing values.

No imputation is performed in Notebook 1. Numeric imputation parameters will later be fitted using the MIMIC-IV training partition only and applied unchanged to validation, internal test and MC-MED.


In [ ]:



S0_PHYSIOLOGICAL_BOUNDS = {
    "temperature_c": (25.0, 45.0),
    "heart_rate": (20.0, 250.0),
    "respiratory_rate": (3.0, 80.0),
    "oxygen_saturation": (50.0, 100.0),
    "systolic_blood_pressure": (40.0, 300.0),
    "diastolic_blood_pressure": (20.0, 200.0),
}


def format_invalid_examples(
    series: pd.Series,
    maximum_examples: int = 10,
) -> str:
    """
    Return compact examples of invalid numeric values.
    """

    values = (
        pd.to_numeric(
            series,
            errors="coerce",
        )
        .dropna()
        .drop_duplicates()
        .sort_values()
        .head(maximum_examples)
        .tolist()
    )

    return ", ".join(
        f"{value:g}"
        for value in values
    )


def validate_s0_physiology(
    dataframe: pd.DataFrame,
    dataset_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Replace values outside predefined physiological validity
    limits with missing values and return an audit table.
    """

    validated = dataframe.copy()
    audit_records = []


    for feature, (
        lower_bound,
        upper_bound,
    ) in S0_PHYSIOLOGICAL_BOUNDS.items():

        if feature not in validated.columns:
            raise KeyError(
                f"{dataset_name} is missing the required "
                f"S0 feature: {feature}"
            )

        numeric_values = pd.to_numeric(
            validated[feature],
            errors="coerce",
        )

        invalid_mask = (
            numeric_values.notna()
            & ~numeric_values.between(
                lower_bound,
                upper_bound,
                inclusive="both",
            )
        )

        invalid_values = numeric_values.loc[
            invalid_mask
        ]

        audit_records.append(
            {
                "dataset": dataset_name,
                "rule": "individual_feature_range",
                "feature": feature,
                "lower_bound": lower_bound,
                "upper_bound": upper_bound,
                "n_rows": len(validated),
                "n_non_missing_before": int(
                    numeric_values.notna().sum()
                ),
                "invalid_n": int(
                    invalid_mask.sum()
                ),
                "invalid_pct_of_all_rows": (
                    invalid_mask.mean() * 100
                ),
                "invalid_examples": (
                    format_invalid_examples(
                        invalid_values
                    )
                ),
            }
        )

        validated[feature] = numeric_values.mask(
            invalid_mask,
            np.nan,
        )


    sbp = validated[
        "systolic_blood_pressure"
    ]

    dbp = validated[
        "diastolic_blood_pressure"
    ]

    inconsistent_bp_mask = (
        sbp.notna()
        & dbp.notna()
        & dbp.ge(sbp)
    )

    bp_example_strings = [
        (
            f"SBP={row.systolic_blood_pressure:g}, "
            f"DBP={row.diastolic_blood_pressure:g}"
        )
        for row in (
            validated.loc[
                inconsistent_bp_mask,
                [
                    "systolic_blood_pressure",
                    "diastolic_blood_pressure",
                ],
            ]
            .drop_duplicates()
            .head(10)
            .itertuples(index=False)
        )
    ]

    audit_records.append(
        {
            "dataset": dataset_name,
            "rule": "blood_pressure_consistency",
            "feature": (
                "systolic_blood_pressure + "
                "diastolic_blood_pressure"
            ),
            "lower_bound": np.nan,
            "upper_bound": np.nan,
            "n_rows": len(validated),
            "n_non_missing_before": int(
                (
                    sbp.notna()
                    & dbp.notna()
                ).sum()
            ),
            "invalid_n": int(
                inconsistent_bp_mask.sum()
            ),
            "invalid_pct_of_all_rows": (
                inconsistent_bp_mask.mean() * 100
            ),
            "invalid_examples": "; ".join(
                bp_example_strings
            ),
        }
    )

    validated.loc[
        inconsistent_bp_mask,
        [
            "systolic_blood_pressure",
            "diastolic_blood_pressure",
        ],
    ] = np.nan

    audit_table = pd.DataFrame(
        audit_records
    )

    return validated, audit_table



(
    mimic_s0_validated,
    mimic_s0_outlier_audit,
) = validate_s0_physiology(
    dataframe=mimic_s0_features,
    dataset_name="MIMIC-IV",
)

(
    mcmed_s0_validated,
    mcmed_s0_outlier_audit,
) = validate_s0_physiology(
    dataframe=mcmed_s0_features,
    dataset_name="MC-MED",
)


s0_outlier_audit = pd.concat(
    [
        mimic_s0_outlier_audit,
        mcmed_s0_outlier_audit,
    ],
    ignore_index=True,
)



assert len(mimic_s0_validated) == len(
    mimic_s0_features
)

assert len(mcmed_s0_validated) == len(
    mcmed_s0_features
)

assert mimic_s0_validated[
    "subject_id"
].is_unique

assert mimic_s0_validated[
    "stay_id"
].is_unique

assert mcmed_s0_validated[
    "MRN"
].is_unique

assert mcmed_s0_validated[
    "CSN"
].is_unique


for feature, (
    lower_bound,
    upper_bound,
) in S0_PHYSIOLOGICAL_BOUNDS.items():

    mimic_valid_values = (
        mimic_s0_validated[feature]
        .dropna()
    )

    mcmed_valid_values = (
        mcmed_s0_validated[feature]
        .dropna()
    )

    assert mimic_valid_values.between(
        lower_bound,
        upper_bound,
        inclusive="both",
    ).all()

    assert mcmed_valid_values.between(
        lower_bound,
        upper_bound,
        inclusive="both",
    ).all()


assert not (
    mimic_s0_validated[
        "diastolic_blood_pressure"
    ].notna()
    & mimic_s0_validated[
        "systolic_blood_pressure"
    ].notna()
    & mimic_s0_validated[
        "diastolic_blood_pressure"
    ].ge(
        mimic_s0_validated[
            "systolic_blood_pressure"
        ]
    )
).any()


assert not (
    mcmed_s0_validated[
        "diastolic_blood_pressure"
    ].notna()
    & mcmed_s0_validated[
        "systolic_blood_pressure"
    ].notna()
    & mcmed_s0_validated[
        "diastolic_blood_pressure"
    ].ge(
        mcmed_s0_validated[
            "systolic_blood_pressure"
        ]
    )
).any()


print("S0 physiological-validity audit:")

display(
    s0_outlier_audit.round(
        {
            "invalid_pct_of_all_rows": 3,
        }
    )
)


print(
    "\nMIMIC-IV validated S0 rows:",
    f"{len(mimic_s0_validated):,}",
)

print(
    "MC-MED validated S0 rows:",
    f"{len(mcmed_s0_validated):,}",
)

print(
    "\nInvalid measurements were converted to missing values."
)

print(
    "No patient encounters were removed."
)


#### Save the validated S0 blocks


In [ ]:

validated_s0_missingness = pd.concat(
    [
        create_feature_missingness_summary(
            dataframe=mimic_s0_validated,
            dataset_name="MIMIC-IV",
        ),
        create_feature_missingness_summary(
            dataframe=mcmed_s0_validated,
            dataset_name="MC-MED",
        ),
    ],
    ignore_index=True,
)


print(
    "S0 missingness after invalid values "
    "were converted to missing:"
)

display(
    validated_s0_missingness.round(
        {"missing_pct": 2}
    )
)



validated_numeric_summaries = []

for dataset_name, dataframe in [
    ("MIMIC-IV", mimic_s0_validated),
    ("MC-MED", mcmed_s0_validated),
]:
    summary = (
        dataframe[
            S0_NUMERIC_FEATURES
        ]
        .describe(
            percentiles=[
                0.01,
                0.25,
                0.50,
                0.75,
                0.99,
            ]
        )
        .T
        .reset_index()
        .rename(
            columns={"index": "feature"}
        )
    )

    summary.insert(
        0,
        "dataset",
        dataset_name,
    )

    validated_numeric_summaries.append(
        summary
    )


s0_validated_numeric_distribution = (
    pd.concat(
        validated_numeric_summaries,
        ignore_index=True,
    )
)


print("\nValidated S0 numeric distributions:")

display(
    s0_validated_numeric_distribution.round(2)
)



MIMIC_S0_VALIDATED_PATH = (
    PROCESSED_DIR
    / "mimiciv_s0_triage_features_validated.parquet"
)

MCMED_S0_VALIDATED_PATH = (
    PROCESSED_DIR
    / "mcmed_s0_triage_features_validated.parquet"
)

S0_OUTLIER_AUDIT_PATH = (
    TABLE_DIR
    / "s0_physiological_validity_audit.csv"
)

S0_VALIDATED_MISSINGNESS_PATH = (
    TABLE_DIR
    / "s0_validated_missingness_before_imputation.csv"
)

S0_VALIDATED_NUMERIC_PATH = (
    TABLE_DIR
    / "s0_validated_numeric_distribution.csv"
)


mimic_s0_validated.to_parquet(
    MIMIC_S0_VALIDATED_PATH,
    index=False,
)

mcmed_s0_validated.to_parquet(
    MCMED_S0_VALIDATED_PATH,
    index=False,
)

s0_outlier_audit.to_csv(
    S0_OUTLIER_AUDIT_PATH,
    index=False,
)

validated_s0_missingness.to_csv(
    S0_VALIDATED_MISSINGNESS_PATH,
    index=False,
)

s0_validated_numeric_distribution.to_csv(
    S0_VALIDATED_NUMERIC_PATH,
    index=False,
)


s0_validated_output_status = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "path": str(path),
        }
        for path in [
            MIMIC_S0_VALIDATED_PATH,
            MCMED_S0_VALIDATED_PATH,
            S0_OUTLIER_AUDIT_PATH,
            S0_VALIDATED_MISSINGNESS_PATH,
            S0_VALIDATED_NUMERIC_PATH,
        ]
    ]
)


display(s0_validated_output_status)

assert s0_validated_output_status[
    "exists"
].all()


print(
    "\nS0 PHYSIOLOGICAL VALIDATION PASSED"
)

print(
    "Invalid measurements were converted to missing."
)

print(
    "No rows were removed and no imputation was applied."
)


#### S1 previous-diagnosis history
Construct temporally valid previous-diagnosis history

S1 extends the validated S0 block with diagnosis history that was available before the current eligible ED encounter.

For MIMIC-IV, diagnoses are taken only from hospital admissions whose hospital discharge time occurred before the current ED arrival. Diagnoses assigned during the current ED encounter or its resulting hospital admission are excluded.

For MC-MED, PMH records are linked by patient identifier. Records whose `Noted_date` is on or before the current ED arrival are considered temporally confirmed previous history. Records with a missing `Noted_date` are audited separately and are not silently combined with temporally confirmed records.

Each diagnosis is counted at most once for each current ED encounter. The final diagnosis features will be binary presence indicators rather than counts of repeated coding occurrences.

The previous notebook used a 0.05% training-prevalence threshold, equivalent to approximately three encounters in the current training cohort. Because this is too permissive for a stable externally validated model, thresholds of 0.05%, 0.25%, 0.5% and 1% are compared before the S1 vocabulary is frozen.

Feature-support thresholds are calculated using the MIMIC-IV training partition only. Validation, internal test and MC-MED are not used to decide which diagnosis features are retained.


In [ ]:



MIMIC_ADMISSIONS_PATH = SOURCE_PATHS[
    "mimic_admissions"
]

MIMIC_DIAGNOSES_PATH = SOURCE_PATHS[
    "mimic_diagnoses_icd"
]

MIMIC_DIAGNOSIS_DICTIONARY_PATH = SOURCE_PATHS[
    "mimic_d_icd_diagnoses"
]

MCMED_PMH_PATH = SOURCE_PATHS[
    "mcmed_pmh"
]

MCMED_VISITS_PATH = SOURCE_PATHS[
    "mcmed_visits"
]



mimic_admissions_header = pd.read_csv(
    MIMIC_ADMISSIONS_PATH,
    nrows=0,
).columns.tolist()

mimic_diagnoses_header = pd.read_csv(
    MIMIC_DIAGNOSES_PATH,
    nrows=0,
).columns.tolist()

mimic_diagnosis_dictionary_header = (
    pd.read_csv(
        MIMIC_DIAGNOSIS_DICTIONARY_PATH,
        nrows=0,
    ).columns.tolist()
)

mcmed_pmh_header = pd.read_csv(
    MCMED_PMH_PATH,
    nrows=0,
).columns.tolist()

mcmed_visits_header = pd.read_csv(
    MCMED_VISITS_PATH,
    nrows=0,
).columns.tolist()



required_source_columns = {
    "MIMIC admissions": {
        "available": mimic_admissions_header,
        "required": [
            "subject_id",
            "hadm_id",
            "dischtime",
        ],
    },
    "MIMIC diagnoses_icd": {
        "available": mimic_diagnoses_header,
        "required": [
            "subject_id",
            "hadm_id",
            "icd_code",
            "icd_version",
        ],
    },
    "MIMIC diagnosis dictionary": {
        "available": (
            mimic_diagnosis_dictionary_header
        ),
        "required": [
            "icd_code",
            "icd_version",
            "long_title",
        ],
    },
    "MC-MED PMH": {
        "available": mcmed_pmh_header,
        "required": [
            "MRN",
            "Noted_date",
            "Code",
            "Desc10",
        ],
    },
    "MC-MED visits": {
        "available": mcmed_visits_header,
        "required": [
            "CSN",
        ],
    },
}


schema_audit_records = []

for source_name, source_specification in (
    required_source_columns.items()
):
    for required_column in (
        source_specification["required"]
    ):
        schema_audit_records.append(
            {
                "source": source_name,
                "column": required_column,
                "present": (
                    required_column
                    in source_specification[
                        "available"
                    ]
                ),
            }
        )


diagnosis_source_schema_audit = pd.DataFrame(
    schema_audit_records
)


print("Diagnosis-history source schema audit:")

display(diagnosis_source_schema_audit)


assert diagnosis_source_schema_audit[
    "present"
].all(), (
    "At least one required diagnosis-history source "
    "column is missing."
)



def normalise_source_column_name(
    column_name: str,
) -> str:
    """
    Normalise a source column name for robust matching.
    """

    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(column_name).lower(),
    )



mcmed_visits_header_lookup = {
    normalise_source_column_name(column): column
    for column in mcmed_visits_header
}


MC_ARRIVAL_TIME_CANDIDATES = [
    "Arrival_time",
    "Arrival time",
    "arrival_time",
    "ED_arrival_time",
    "ED arrival time",
]


mcmed_arrival_source_column = next(
    (
        mcmed_visits_header_lookup[
            normalise_source_column_name(
                candidate
            )
        ]
        for candidate
        in MC_ARRIVAL_TIME_CANDIDATES
        if normalise_source_column_name(
            candidate
        )
        in mcmed_visits_header_lookup
    ),
    None,
)


if mcmed_arrival_source_column is None:
    raise ValueError(
        "No MC-MED ED arrival-time column was found "
        "in visits.csv.\n\n"
        f"Available columns:\n"
        f"{mcmed_visits_header}"
    )



MISSING_DATE_STRINGS = {
    "",
    "nan",
    "nat",
    "none",
    "null",
    "missing",
}


def parse_datetime_to_seconds(
    series: pd.Series,
) -> tuple[pd.Series, pd.DataFrame]:
    """
    Parse dates into datetime64 second-level resolution.

    Second-level resolution supports MC-MED shifted dates
    beyond the pandas datetime64 nanosecond limit in 2262.

    The function first attempts fast NumPy parsing. If the
    source contains mixed formats, it falls back to dateutil
    only for non-missing values.
    """

    cleaned = (
        series
        .astype("string")
        .str.strip()
    )

    source_missing_mask = (
        cleaned.isna()
        |
        cleaned.str.lower().isin(
            MISSING_DATE_STRINGS
        )
    )

    parsed_values = np.full(
        len(cleaned),
        np.datetime64("NaT", "s"),
        dtype="datetime64[s]",
    )

    valid_positions = np.flatnonzero(
        ~source_missing_mask.to_numpy()
    )

    valid_strings = (
        cleaned.loc[
            ~source_missing_mask
        ]
        .astype(str)
        .tolist()
    )

    failure_records = []


    if valid_strings:
        try:
            parsed_values[
                valid_positions
            ] = np.asarray(
                valid_strings,
                dtype="datetime64[s]",
            )

        except (TypeError, ValueError):

            for position, (
                source_index,
                raw_value,
            ) in zip(
                valid_positions,
                cleaned.loc[
                    ~source_missing_mask
                ].items(),
            ):
                try:
                    parsed_datetime = (
                        date_parser.parse(
                            str(raw_value),
                            dayfirst=False,
                            fuzzy=False,
                        )
                    )

                    if (
                        parsed_datetime.tzinfo
                        is not None
                    ):
                        parsed_datetime = (
                            parsed_datetime
                            .astimezone(
                                timezone.utc
                            )
                            .replace(
                                tzinfo=None
                            )
                        )

                    parsed_datetime = (
                        parsed_datetime.replace(
                            microsecond=0
                        )
                    )

                    parsed_values[
                        position
                    ] = np.datetime64(
                        parsed_datetime,
                        "s",
                    )

                except Exception as error:
                    failure_records.append(
                        {
                            "source_index": (
                                source_index
                            ),
                            "raw_value": (
                                raw_value
                            ),
                            "error": str(error),
                        }
                    )

    parsed_series = pd.Series(
        parsed_values,
        index=series.index,
        name=series.name,
    )

    failure_table = pd.DataFrame(
        failure_records
    )

    return parsed_series, failure_table



mcmed_arrival_lookup = pd.read_csv(
    MCMED_VISITS_PATH,
    usecols=[
        "CSN",
        mcmed_arrival_source_column,
    ],
    dtype={
        mcmed_arrival_source_column:
            "string",
    },
    low_memory=False,
)


mcmed_arrival_lookup["CSN"] = pd.to_numeric(
    mcmed_arrival_lookup["CSN"],
    errors="coerce",
).astype("Int64")


(
    mcmed_arrival_lookup[
        "current_arrival_time"
    ],
    mcmed_arrival_parse_failures,
) = parse_datetime_to_seconds(
    mcmed_arrival_lookup[
        mcmed_arrival_source_column
    ]
)



arrival_raw_values = (
    mcmed_arrival_lookup[
        mcmed_arrival_source_column
    ]
    .astype("string")
    .str.strip()
)

arrival_source_missing_mask = (
    arrival_raw_values.isna()
    |
    arrival_raw_values.str.lower().isin(
        MISSING_DATE_STRINGS
    )
)

arrival_parse_failure_mask = (
    ~arrival_source_missing_mask
    &
    mcmed_arrival_lookup[
        "current_arrival_time"
    ].isna()
)


mcmed_arrival_source_audit = pd.DataFrame(
    [
        {
            "measure": "Rows in visits.csv",
            "value": len(
                mcmed_arrival_lookup
            ),
        },
        {
            "measure": (
                "Raw Arrival_time missing"
            ),
            "value": int(
                arrival_source_missing_mask.sum()
            ),
        },
        {
            "measure": (
                "Non-missing Arrival_time parsed"
            ),
            "value": int(
                (
                    ~arrival_source_missing_mask
                    &
                    mcmed_arrival_lookup[
                        "current_arrival_time"
                    ].notna()
                ).sum()
            ),
        },
        {
            "measure": (
                "Non-missing Arrival_time "
                "parse failures"
            ),
            "value": int(
                arrival_parse_failure_mask.sum()
            ),
        },
        {
            "measure": (
                "Earliest parsed Arrival_time"
            ),
            "value": (
                mcmed_arrival_lookup[
                    "current_arrival_time"
                ].min()
            ),
        },
        {
            "measure": (
                "Latest parsed Arrival_time"
            ),
            "value": (
                mcmed_arrival_lookup[
                    "current_arrival_time"
                ].max()
            ),
        },
    ]
)


print(
    "MC-MED arrival-time source column:",
    mcmed_arrival_source_column,
)

print(
    "\nMC-MED visits.csv arrival-time "
    "parsing audit:"
)

display(mcmed_arrival_source_audit)


if not mcmed_arrival_parse_failures.empty:
    print(
        "\nArrival-time parsing failures:"
    )

    display(
        mcmed_arrival_parse_failures.head(25)
    )


assert int(
    arrival_parse_failure_mask.sum()
) == 0, (
    "Some non-missing MC-MED Arrival_time values "
    "could not be parsed."
)



mcmed_arrival_lookup = (
    mcmed_arrival_lookup[
        [
            "CSN",
            "current_arrival_time",
        ]
    ]
    .copy()
)


assert mcmed_arrival_lookup[
    "CSN"
].notna().all()

assert mcmed_arrival_lookup[
    "CSN"
].is_unique



mcmed_primary_history_source = (
    mcmed_primary_binary[
        [
            "MRN",
            "CSN",
        ]
    ]
    .copy()
    .merge(
        mcmed_arrival_lookup,
        on="CSN",
        how="left",
        validate="one_to_one",
        indicator="arrival_merge_status",
    )
)



mcmed_cohort_arrival_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen MC-MED cohort encounters"
            ),
            "value": len(
                mcmed_primary_history_source
            ),
        },
        {
            "measure": (
                "CSNs matched to visits.csv"
            ),
            "value": int(
                mcmed_primary_history_source[
                    "arrival_merge_status"
                ].eq("both").sum()
            ),
        },
        {
            "measure": (
                "Matched encounters with "
                "parsed arrival time"
            ),
            "value": int(
                mcmed_primary_history_source[
                    "current_arrival_time"
                ].notna().sum()
            ),
        },
        {
            "measure": (
                "Matched encounters with "
                "missing source arrival time"
            ),
            "value": int(
                mcmed_primary_history_source[
                    "current_arrival_time"
                ].isna().sum()
            ),
        },
        {
            "measure": (
                "Earliest cohort ED arrival"
            ),
            "value": (
                mcmed_primary_history_source[
                    "current_arrival_time"
                ].min()
            ),
        },
        {
            "measure": (
                "Latest cohort ED arrival"
            ),
            "value": (
                mcmed_primary_history_source[
                    "current_arrival_time"
                ].max()
            ),
        },
    ]
)


print(
    "\nMC-MED frozen-cohort arrival-time "
    "recovery:"
)

display(mcmed_cohort_arrival_audit)


assert len(
    mcmed_primary_history_source
) == len(
    mcmed_primary_binary
)

assert mcmed_primary_history_source[
    "arrival_merge_status"
].eq("both").all(), (
    "At least one frozen MC-MED CSN was "
    "not found in visits.csv."
)

assert mcmed_primary_history_source[
    "current_arrival_time"
].notna().all(), (
    "At least one frozen MC-MED encounter "
    "has a genuinely missing source Arrival_time."
)

assert mcmed_primary_history_source[
    "MRN"
].is_unique

assert mcmed_primary_history_source[
    "CSN"
].is_unique


mcmed_primary_history_source = (
    mcmed_primary_history_source.drop(
        columns="arrival_merge_status"
    )
)



mimic_history_admissions = pd.read_csv(
    MIMIC_ADMISSIONS_PATH,
    usecols=[
        "subject_id",
        "hadm_id",
        "dischtime",
    ],
    low_memory=False,
)


mimic_history_diagnoses = pd.read_csv(
    MIMIC_DIAGNOSES_PATH,
    usecols=[
        column
        for column in [
            "subject_id",
            "hadm_id",
            "seq_num",
            "icd_code",
            "icd_version",
        ]
        if column in mimic_diagnoses_header
    ],
    low_memory=False,
)


mimic_diagnosis_dictionary = pd.read_csv(
    MIMIC_DIAGNOSIS_DICTIONARY_PATH,
    usecols=[
        "icd_code",
        "icd_version",
        "long_title",
    ],
    low_memory=False,
)



mcmed_pmh_columns_to_load = [
    column
    for column in [
        "MRN",
        "Noted_date",
        "Code",
        "Desc10",
        "CCS",
        "DescCCS",
    ]
    if column in mcmed_pmh_header
]


mcmed_pmh_history = pd.read_csv(
    MCMED_PMH_PATH,
    usecols=mcmed_pmh_columns_to_load,
    dtype={
        "Noted_date": "string",
    },
    low_memory=False,
)



print(
    "\nMIMIC admissions rows:",
    f"{len(mimic_history_admissions):,}",
)

print(
    "MIMIC diagnosis rows:",
    f"{len(mimic_history_diagnoses):,}",
)

print(
    "MIMIC diagnosis dictionary rows:",
    f"{len(mimic_diagnosis_dictionary):,}",
)

print(
    "MC-MED PMH rows:",
    f"{len(mcmed_pmh_history):,}",
)

print(
    "\nMC-MED cohort arrival-time dtype:",
    mcmed_primary_history_source[
        "current_arrival_time"
    ].dtype,
)


#### MIMIC-IV previous-diagnosis reconstruction


In [ ]:


for dataframe in [
    mimic_history_admissions,
    mimic_history_diagnoses,
]:
    dataframe["subject_id"] = pd.to_numeric(
        dataframe["subject_id"],
        errors="coerce",
    ).astype("Int64")

    dataframe["hadm_id"] = pd.to_numeric(
        dataframe["hadm_id"],
        errors="coerce",
    ).astype("Int64")


mimic_history_admissions[
    "dischtime"
] = pd.to_datetime(
    mimic_history_admissions["dischtime"],
    errors="coerce",
    utc=True,
)

mimic_current_history_cohort = (
    mimic_split_source[
        [
            "subject_id",
            "stay_id",
            "intime",
            "split",
        ]
    ]
    .copy()
)

mimic_current_history_cohort[
    "current_intime"
] = pd.to_datetime(
    mimic_current_history_cohort["intime"],
    errors="coerce",
    utc=True,
)

mimic_current_history_cohort = (
    mimic_current_history_cohort.drop(
        columns="intime"
    )
)



mimic_history_subject_ids = set(
    mimic_current_history_cohort[
        "subject_id"
    ]
    .dropna()
    .tolist()
)

mimic_history_admissions_restricted = (
    mimic_history_admissions.loc[
        mimic_history_admissions[
            "subject_id"
        ].isin(mimic_history_subject_ids)
    ]
    .rename(
        columns={
            "hadm_id": "prior_hadm_id",
            "dischtime": (
                "prior_hospital_dischtime"
            ),
        }
    )
    .copy()
)



mimic_previous_admission_links = (
    mimic_current_history_cohort.merge(
        mimic_history_admissions_restricted,
        on="subject_id",
        how="left",
        validate="one_to_many",
    )
)


mimic_previous_admission_links = (
    mimic_previous_admission_links.loc[
        mimic_previous_admission_links[
            "prior_hospital_dischtime"
        ].notna()
        &
        mimic_previous_admission_links[
            "prior_hospital_dischtime"
        ].lt(
            mimic_previous_admission_links[
                "current_intime"
            ]
        )
    ]
    .copy()
)



mimic_history_diagnoses_for_merge = (
    mimic_history_diagnoses.rename(
        columns={
            "hadm_id": "prior_hadm_id",
        }
    )
)


mimic_previous_diagnosis_long = (
    mimic_previous_admission_links.merge(
        mimic_history_diagnoses_for_merge,
        on=[
            "subject_id",
            "prior_hadm_id",
        ],
        how="inner",
        validate="one_to_many",
    )
)


mimic_previous_diagnosis_long = (
    mimic_previous_diagnosis_long.merge(
        mimic_diagnosis_dictionary,
        on=[
            "icd_code",
            "icd_version",
        ],
        how="left",
        validate="many_to_one",
    )
)



mimic_previous_diagnosis_long = (
    mimic_previous_diagnosis_long.loc[
        mimic_previous_diagnosis_long[
            "long_title"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "stay_id",
            "icd_code",
            "icd_version",
            "long_title",
        ]
    )
    .reset_index(drop=True)
)


mimic_history_availability = pd.DataFrame(
    [
        {
            "measure": (
                "Current cohort encounters"
            ),
            "count": len(
                mimic_current_history_cohort
            ),
        },
        {
            "measure": (
                "Encounters with >=1 previous admission"
            ),
            "count": (
                mimic_previous_admission_links[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 previous "
                "diagnosis title"
            ),
            "count": (
                mimic_previous_diagnosis_long[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique previous diagnosis titles"
            ),
            "count": (
                mimic_previous_diagnosis_long[
                    "long_title"
                ].nunique()
            ),
        },
    ]
)


print(
    "MIMIC-IV previous-history availability:"
)

display(mimic_history_availability)

print(
    "\nMIMIC previous-diagnosis long-table shape:",
    mimic_previous_diagnosis_long.shape,
)


#### MC-MED previous-diagnosis reconstruction


In [ ]:


mcmed_pmh_history["MRN"] = pd.to_numeric(
    mcmed_pmh_history["MRN"],
    errors="coerce",
).astype("Int64")



(
    mcmed_pmh_history[
        "Noted_date_parsed"
    ],
    mcmed_noted_date_parse_failures,
) = parse_datetime_to_seconds(
    mcmed_pmh_history[
        "Noted_date"
    ]
)



mcmed_noted_date_raw = (
    mcmed_pmh_history[
        "Noted_date"
    ]
    .astype("string")
    .str.strip()
)


mcmed_noted_date_source_missing_mask = (
    mcmed_noted_date_raw.isna()
    |
    mcmed_noted_date_raw.str.lower().isin(
        MISSING_DATE_STRINGS
    )
)


mcmed_noted_date_parse_failure_mask = (
    ~mcmed_noted_date_source_missing_mask
    &
    mcmed_pmh_history[
        "Noted_date_parsed"
    ].isna()
)


mcmed_noted_date_parsing_audit = pd.DataFrame(
    [
        {
            "measure": "PMH source rows",
            "value": len(
                mcmed_pmh_history
            ),
        },
        {
            "measure": (
                "Raw Noted_date missing"
            ),
            "value": int(
                mcmed_noted_date_source_missing_mask.sum()
            ),
        },
        {
            "measure": (
                "Non-missing Noted_date parsed"
            ),
            "value": int(
                (
                    ~mcmed_noted_date_source_missing_mask
                    &
                    mcmed_pmh_history[
                        "Noted_date_parsed"
                    ].notna()
                ).sum()
            ),
        },
        {
            "measure": (
                "Non-missing Noted_date "
                "parse failures"
            ),
            "value": int(
                mcmed_noted_date_parse_failure_mask.sum()
            ),
        },
        {
            "measure": (
                "Earliest parsed Noted_date"
            ),
            "value": (
                mcmed_pmh_history[
                    "Noted_date_parsed"
                ].min()
            ),
        },
        {
            "measure": (
                "Latest parsed Noted_date"
            ),
            "value": (
                mcmed_pmh_history[
                    "Noted_date_parsed"
                ].max()
            ),
        },
    ]
)


print(
    "MC-MED PMH Noted_date parsing audit:"
)

display(mcmed_noted_date_parsing_audit)


if not mcmed_noted_date_parse_failures.empty:
    print(
        "\nExamples of PMH Noted_date "
        "parsing failures:"
    )

    display(
        mcmed_noted_date_parse_failures.head(25)
    )


assert int(
    mcmed_noted_date_parse_failure_mask.sum()
) == 0, (
    "Some non-missing MC-MED PMH Noted_date "
    "values could not be parsed."
)



mcmed_current_history_cohort = (
    mcmed_primary_history_source.copy()
)


assert len(
    mcmed_current_history_cohort
) == 6707

assert mcmed_current_history_cohort[
    "MRN"
].notna().all()

assert mcmed_current_history_cohort[
    "CSN"
].notna().all()

assert mcmed_current_history_cohort[
    "current_arrival_time"
].notna().all()

assert mcmed_current_history_cohort[
    "MRN"
].is_unique

assert mcmed_current_history_cohort[
    "CSN"
].is_unique


print(
    "\nMC-MED current arrival dtype:",
    mcmed_current_history_cohort[
        "current_arrival_time"
    ].dtype,
)

print(
    "MC-MED PMH Noted_date dtype:",
    mcmed_pmh_history[
        "Noted_date_parsed"
    ].dtype,
)



mcmed_history_mrns = set(
    mcmed_current_history_cohort[
        "MRN"
    ]
    .dropna()
    .tolist()
)


mcmed_pmh_restricted = (
    mcmed_pmh_history.loc[
        mcmed_pmh_history[
            "MRN"
        ].isin(
            mcmed_history_mrns
        )
    ]
    .copy()
)



mcmed_pmh_linked = (
    mcmed_current_history_cohort.merge(
        mcmed_pmh_restricted,
        on="MRN",
        how="left",
        validate="one_to_many",
        indicator="pmh_merge_status",
    )
)



mcmed_pmh_linked[
    "pmh_record_present"
] = (
    mcmed_pmh_linked[
        "pmh_merge_status"
    ].eq("both")
)



mcmed_pmh_linked[
    "temporal_status"
] = np.select(
    [
        ~mcmed_pmh_linked[
            "pmh_record_present"
        ],

        (
            mcmed_pmh_linked[
                "pmh_record_present"
            ]
            &
            mcmed_pmh_linked[
                "Noted_date_parsed"
            ].isna()
        ),

        (
            mcmed_pmh_linked[
                "pmh_record_present"
            ]
            &
            mcmed_pmh_linked[
                "Noted_date_parsed"
            ].notna()
            &
            mcmed_pmh_linked[
                "Noted_date_parsed"
            ].le(
                mcmed_pmh_linked[
                    "current_arrival_time"
                ]
            )
        ),
    ],
    [
        "NO_PMH_RECORD",
        "PMH_RECORD_WITH_MISSING_NOTED_DATE",
        "CONFIRMED_ON_OR_BEFORE_ARRIVAL",
    ],
    default="RECORDED_AFTER_CURRENT_ARRIVAL",
)



mcmed_pmh_temporal_audit = (
    mcmed_pmh_linked[
        "temporal_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("temporal_status")
    .reset_index(
        name="n_linked_rows"
    )
)


mcmed_pmh_temporal_audit[
    "percentage_of_linked_rows"
] = (
    mcmed_pmh_temporal_audit[
        "n_linked_rows"
    ]
    / len(mcmed_pmh_linked)
    * 100
)



mcmed_pmh_encounter_temporal_audit = (
    mcmed_pmh_linked
    .groupby(
        "temporal_status",
        dropna=False,
    )["CSN"]
    .nunique()
    .rename(
        "n_unique_encounters"
    )
    .reset_index()
)


mcmed_pmh_encounter_temporal_audit[
    "percentage_of_cohort_encounters"
] = (
    mcmed_pmh_encounter_temporal_audit[
        "n_unique_encounters"
    ]
    / len(
        mcmed_current_history_cohort
    )
    * 100
)



mcmed_previous_diagnosis_long_strict = (
    mcmed_pmh_linked.loc[
        mcmed_pmh_linked[
            "temporal_status"
        ].eq(
            "CONFIRMED_ON_OR_BEFORE_ARRIVAL"
        )
        &
        mcmed_pmh_linked[
            "Desc10"
        ].notna()
    ]
    .copy()
)



mcmed_previous_diagnosis_long_strict[
    "Code"
] = (
    mcmed_previous_diagnosis_long_strict[
        "Code"
    ]
    .astype("string")
    .str.strip()
)


mcmed_previous_diagnosis_long_strict[
    "Desc10"
] = (
    mcmed_previous_diagnosis_long_strict[
        "Desc10"
    ]
    .astype("string")
    .str.strip()
)



mcmed_previous_diagnosis_long_strict = (
    mcmed_previous_diagnosis_long_strict
    .drop_duplicates(
        subset=[
            "CSN",
            "Code",
            "Desc10",
        ]
    )
    .reset_index(drop=True)
)



confirmed_history_csns = set(
    mcmed_previous_diagnosis_long_strict[
        "CSN"
    ]
    .dropna()
    .tolist()
)


mcmed_history_availability = pd.DataFrame(
    [
        {
            "measure": (
                "Current frozen cohort encounters"
            ),
            "count": len(
                mcmed_current_history_cohort
            ),
        },
        {
            "measure": (
                "Encounters with any linked "
                "PMH source record"
            ),
            "count": (
                mcmed_pmh_linked.loc[
                    mcmed_pmh_linked[
                        "pmh_record_present"
                    ],
                    "CSN",
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 temporally "
                "confirmed PMH diagnosis"
            ),
            "count": len(
                confirmed_history_csns
            ),
        },
        {
            "measure": (
                "Encounters with PMH records "
                "but missing Noted_date"
            ),
            "count": (
                mcmed_pmh_linked.loc[
                    mcmed_pmh_linked[
                        "temporal_status"
                    ].eq(
                        "PMH_RECORD_WITH_"
                        "MISSING_NOTED_DATE"
                    ),
                    "CSN",
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with PMH recorded "
                "after current arrival"
            ),
            "count": (
                mcmed_pmh_linked.loc[
                    mcmed_pmh_linked[
                        "temporal_status"
                    ].eq(
                        "RECORDED_AFTER_"
                        "CURRENT_ARRIVAL"
                    ),
                    "CSN",
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique temporally confirmed "
                "diagnosis codes"
            ),
            "count": (
                mcmed_previous_diagnosis_long_strict[
                    "Code"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique temporally confirmed "
                "diagnosis descriptions"
            ),
            "count": (
                mcmed_previous_diagnosis_long_strict[
                    "Desc10"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Temporally confirmed "
                "encounter-diagnosis rows"
            ),
            "count": len(
                mcmed_previous_diagnosis_long_strict
            ),
        },
    ]
)



assert (
    mcmed_previous_diagnosis_long_strict[
        "Noted_date_parsed"
    ]
    <=
    mcmed_previous_diagnosis_long_strict[
        "current_arrival_time"
    ]
).all(), (
    "At least one retained MC-MED PMH diagnosis "
    "was recorded after the current ED arrival."
)


assert not (
    mcmed_previous_diagnosis_long_strict[
        [
            "CSN",
            "Code",
            "Desc10",
        ]
    ]
    .duplicated()
    .any()
), (
    "Duplicate encounter-diagnosis rows remain "
    "after PMH deduplication."
)



print(
    "MC-MED PMH linked-row temporal-status audit:"
)

display(
    mcmed_pmh_temporal_audit.round(
        {
            "percentage_of_linked_rows": 2,
        }
    )
)


print(
    "\nMC-MED PMH encounter-level "
    "temporal-status audit:"
)

display(
    mcmed_pmh_encounter_temporal_audit.round(
        {
            "percentage_of_cohort_encounters": 2,
        }
    )
)


print(
    "\nMC-MED strict previous-history availability:"
)

display(mcmed_history_availability)


print(
    "\nMC-MED strict previous-diagnosis "
    "long-table shape:",
    mcmed_previous_diagnosis_long_strict.shape,
)


#### Diagnosis normalisation and support thresholds


In [ ]:


def normalise_diagnosis_title(
    value,
) -> str:
    """
    Create a conservative diagnosis-title matching key.

    This standardises case, punctuation and spacing but does
    not merge clinically different diagnoses.
    """

    if pd.isna(value):
        return ""

    text = str(value).lower().strip()

    text = text.replace(
        "&",
        " and ",
    )

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if text in {
        "",
        "unknown",
        "none",
        "nan",
        "not specified",
    }:
        return ""

    return text



mimic_previous_diagnosis_long[
    "diagnosis_key"
] = mimic_previous_diagnosis_long[
    "long_title"
].apply(
    normalise_diagnosis_title
)

mcmed_previous_diagnosis_long_strict[
    "diagnosis_key"
] = (
    mcmed_previous_diagnosis_long_strict[
        "Desc10"
    ].apply(
        normalise_diagnosis_title
    )
)


mimic_previous_diagnosis_long = (
    mimic_previous_diagnosis_long.loc[
        mimic_previous_diagnosis_long[
            "diagnosis_key"
        ].ne("")
    ]
    .drop_duplicates(
        subset=[
            "stay_id",
            "diagnosis_key",
        ]
    )
    .reset_index(drop=True)
)


mcmed_previous_diagnosis_long_strict = (
    mcmed_previous_diagnosis_long_strict.loc[
        mcmed_previous_diagnosis_long_strict[
            "diagnosis_key"
        ].ne("")
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "diagnosis_key",
        ]
    )
    .reset_index(drop=True)
)



mimic_train_stay_ids = set(
    mimic_split_source.loc[
        mimic_split_source[
            "split"
        ].astype(str).eq("train"),
        "stay_id",
    ]
)


mimic_train_previous_diagnoses = (
    mimic_previous_diagnosis_long.loc[
        mimic_previous_diagnosis_long[
            "stay_id"
        ].isin(mimic_train_stay_ids)
    ]
    .copy()
)


n_mimic_train_encounters = len(
    mimic_train_stay_ids
)


mimic_train_diagnosis_support = (
    mimic_train_previous_diagnoses
    .groupby("diagnosis_key")[
        "stay_id"
    ]
    .nunique()
    .rename(
        "n_train_encounters_with_diagnosis"
    )
    .reset_index()
)


mimic_train_diagnosis_support[
    "train_prevalence_pct"
] = (
    mimic_train_diagnosis_support[
        "n_train_encounters_with_diagnosis"
    ]
    / n_mimic_train_encounters
    * 100
)


mcmed_external_diagnosis_keys = set(
    mcmed_previous_diagnosis_long_strict[
        "diagnosis_key"
    ].unique()
)



DIAGNOSIS_THRESHOLD_CANDIDATES_PCT = [
    0.05,
    0.25,
    0.50,
    1.00,
]


threshold_audit_records = []


for threshold_pct in (
    DIAGNOSIS_THRESHOLD_CANDIDATES_PCT
):
    minimum_train_encounters = max(
        1,
        int(
            math.ceil(
                n_mimic_train_encounters
                * threshold_pct
                / 100
            )
        ),
    )

    retained = (
        mimic_train_diagnosis_support.loc[
            mimic_train_diagnosis_support[
                "n_train_encounters_with_diagnosis"
            ].ge(
                minimum_train_encounters
            )
        ]
        .copy()
    )

    retained_keys = set(
        retained["diagnosis_key"]
    )

    exact_mcmed_matches = (
        retained_keys
        & mcmed_external_diagnosis_keys
    )

    threshold_audit_records.append(
        {
            "threshold_pct": threshold_pct,
            "n_mimic_train_encounters": (
                n_mimic_train_encounters
            ),
            "minimum_train_encounters": (
                minimum_train_encounters
            ),
            "n_diagnosis_features_retained": (
                len(retained_keys)
            ),
            "n_exact_title_keys_found_in_mcmed": (
                len(exact_mcmed_matches)
            ),
            "exact_title_match_pct": (
                len(exact_mcmed_matches)
                / len(retained_keys)
                * 100
                if retained_keys
                else np.nan
            ),
        }
    )


diagnosis_threshold_audit = pd.DataFrame(
    threshold_audit_records
)


print(
    "Train-only previous-diagnosis "
    "threshold comparison:"
)

display(
    diagnosis_threshold_audit.round(
        {"exact_title_match_pct": 2}
    )
)


print(
    "\nTop MIMIC training diagnosis titles:"
)

display(
    mimic_train_diagnosis_support
    .sort_values(
        "n_train_encounters_with_diagnosis",
        ascending=False,
    )
    .head(30)
    .round(
        {"train_prevalence_pct": 2}
    )
)


PLANNED_DIAGNOSIS_THRESHOLD_PCT = 0.50

planned_minimum_train_encounters = int(
    math.ceil(
        n_mimic_train_encounters
        * PLANNED_DIAGNOSIS_THRESHOLD_PCT
        / 100
    )
)


print(
    "\nPlanned publication threshold:",
    PLANNED_DIAGNOSIS_THRESHOLD_PCT,
    "%",
)

print(
    "Equivalent minimum MIMIC training encounters:",
    planned_minimum_train_encounters,
)


#### Previous-history support-threshold audit

This audit compares minimum-support values around the provisional threshold of 10 MIMIC-IV training encounters.

The comparison is performed separately for:

- previous diagnosis titles;
- previous medication names;
- previous medication classes.

Diagnosis support uses the temporally valid previous-diagnosis table already constructed.

Medication support follows the original pipeline logic. Medication-reconciliation records are linked to the current ED encounter, restricted to documentation occurring between ED arrival and ED departure, cleaned, and counted once per medication or medication class per encounter.

The historical reference feature counts are:

- 879 diagnosis-title features;
- 812 medication-name features;
- 364 medication-class features.

Thresholds are calculated using the 5,985 MIMIC-IV training encounters only. No validation, test, MC-MED, outcome, model-performance or SHAP information is used in this audit.

This cell does not select or save the final vocabulary. It only shows how closely each candidate minimum-support value reproduces the previous feature-space dimensions.


In [ ]:



required_objects = [
    "mimic_split_source",
    "mimic_train_diagnosis_support",
    "SOURCE_PATHS",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the previous diagnosis-history cells first. "
        f"Missing objects: {missing_objects}"
    )



REFERENCE_FEATURE_COUNTS = {
    "diagnosis_titles": 879,
    "medication_names": 812,
    "medication_classes": 364,
}


SUPPORT_VALUES_TO_CHECK = list(
    range(8, 13)
)

PROVISIONAL_MINIMUM_SUPPORT = 10



required_train_columns = [
    "subject_id",
    "stay_id",
    "intime",
    "outtime",
    "split",
]

missing_train_columns = [
    column
    for column in required_train_columns
    if column not in mimic_split_source.columns
]

if missing_train_columns:
    raise ValueError(
        "mimic_split_source is missing required columns: "
        f"{missing_train_columns}"
    )


mimic_train_windows = (
    mimic_split_source.loc[
        mimic_split_source[
            "split"
        ].astype("string").eq("train"),
        [
            "subject_id",
            "stay_id",
            "intime",
            "outtime",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)


def convert_to_comparable_datetime(
    series: pd.Series,
) -> pd.Series:
    """
    Convert timestamps into timezone-naive pandas datetimes so
    source and cohort dates can be compared consistently.
    """

    return (
        pd.to_datetime(
            series,
            errors="coerce",
            utc=True,
        )
        .dt.tz_convert(None)
    )


mimic_train_windows["intime"] = (
    convert_to_comparable_datetime(
        mimic_train_windows["intime"]
    )
)

mimic_train_windows["outtime"] = (
    convert_to_comparable_datetime(
        mimic_train_windows["outtime"]
    )
)


expected_mimic_training_rows = int(
    mimic_split_source["split"].astype("string").eq("train").sum()
)
assert len(mimic_train_windows) == expected_mimic_training_rows

assert mimic_train_windows[
    "subject_id"
].is_unique

assert mimic_train_windows[
    "stay_id"
].is_unique

assert mimic_train_windows[
    [
        "intime",
        "outtime",
    ]
].notna().all().all()


n_mimic_train_encounters = len(
    mimic_train_windows
)



mimic_medrecon_path = SOURCE_PATHS[
    "mimic_medrecon"
]

medrecon_header = pd.read_csv(
    mimic_medrecon_path,
    nrows=0,
).columns.tolist()


required_medrecon_columns = [
    "subject_id",
    "stay_id",
    "charttime",
    "name",
    "etcdescription",
]

missing_medrecon_columns = [
    column
    for column in required_medrecon_columns
    if column not in medrecon_header
]

if missing_medrecon_columns:
    raise ValueError(
        "The MIMIC-IV-ED medrecon table is missing "
        f"required columns: {missing_medrecon_columns}"
    )


mimic_medrecon_support_source = pd.read_csv(
    mimic_medrecon_path,
    usecols=required_medrecon_columns,
    low_memory=False,
)



for identifier_column in [
    "subject_id",
    "stay_id",
]:
    mimic_medrecon_support_source[
        identifier_column
    ] = pd.to_numeric(
        mimic_medrecon_support_source[
            identifier_column
        ],
        errors="coerce",
    ).astype("Int64")


mimic_medrecon_support_source[
    "charttime"
] = convert_to_comparable_datetime(
    mimic_medrecon_support_source[
        "charttime"
    ]
)



mimic_train_medrecon = (
    mimic_medrecon_support_source.merge(
        mimic_train_windows,
        on=[
            "subject_id",
            "stay_id",
        ],
        how="inner",
        validate="many_to_one",
    )
)


mimic_train_medrecon[
    "inside_current_ed_window"
] = (
    mimic_train_medrecon[
        "charttime"
    ].notna()
    &
    mimic_train_medrecon[
        "charttime"
    ].ge(
        mimic_train_medrecon[
            "intime"
        ]
    )
    &
    mimic_train_medrecon[
        "charttime"
    ].le(
        mimic_train_medrecon[
            "outtime"
        ]
    )
)


mimic_train_medrecon = (
    mimic_train_medrecon.loc[
        mimic_train_medrecon[
            "inside_current_ed_window"
        ]
    ]
    .copy()
)



def clean_medication_text(
    value,
) -> str:
    """
    Reproduce the conservative cleaning used in the earlier
    medication-reconciliation pipeline.
    """

    if pd.isna(value):
        return ""

    text = str(value).upper().strip()

    text = text.replace(
        '"',
        "",
    ).replace(
        "'",
        "",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if text in {
        "",
        "NAN",
        "NONE",
        "NULL",
        "UNKNOWN",
        "NOT SPECIFIED",
    }:
        return ""

    return text


mimic_train_medrecon[
    "medication_name_key"
] = mimic_train_medrecon[
    "name"
].apply(
    clean_medication_text
)


mimic_train_medrecon[
    "medication_class_key"
] = mimic_train_medrecon[
    "etcdescription"
].apply(
    clean_medication_text
)



mimic_train_medication_names = (
    mimic_train_medrecon.loc[
        mimic_train_medrecon[
            "medication_name_key"
        ].ne(""),
        [
            "stay_id",
            "medication_name_key",
        ],
    ]
    .drop_duplicates()
)


mimic_train_medication_classes = (
    mimic_train_medrecon.loc[
        mimic_train_medrecon[
            "medication_class_key"
        ].ne(""),
        [
            "stay_id",
            "medication_class_key",
        ],
    ]
    .drop_duplicates()
)


mimic_train_medication_name_support = (
    mimic_train_medication_names
    .groupby(
        "medication_name_key"
    )["stay_id"]
    .nunique()
    .rename(
        "n_train_encounters"
    )
    .reset_index()
)


mimic_train_medication_class_support = (
    mimic_train_medication_classes
    .groupby(
        "medication_class_key"
    )["stay_id"]
    .nunique()
    .rename(
        "n_train_encounters"
    )
    .reset_index()
)



diagnosis_support_column = (
    "n_train_encounters_with_diagnosis"
)

if diagnosis_support_column not in (
    mimic_train_diagnosis_support.columns
):
    raise ValueError(
        "mimic_train_diagnosis_support does not contain "
        f"'{diagnosis_support_column}'."
    )



support_comparison_records = []


for minimum_support in SUPPORT_VALUES_TO_CHECK:

    diagnosis_count = int(
        mimic_train_diagnosis_support[
            diagnosis_support_column
        ].ge(
            minimum_support
        ).sum()
    )

    medication_name_count = int(
        mimic_train_medication_name_support[
            "n_train_encounters"
        ].ge(
            minimum_support
        ).sum()
    )

    medication_class_count = int(
        mimic_train_medication_class_support[
            "n_train_encounters"
        ].ge(
            minimum_support
        ).sum()
    )

    diagnosis_difference = abs(
        diagnosis_count
        - REFERENCE_FEATURE_COUNTS[
            "diagnosis_titles"
        ]
    )

    medication_name_difference = abs(
        medication_name_count
        - REFERENCE_FEATURE_COUNTS[
            "medication_names"
        ]
    )

    medication_class_difference = abs(
        medication_class_count
        - REFERENCE_FEATURE_COUNTS[
            "medication_classes"
        ]
    )

    support_comparison_records.append(
        {
            "minimum_train_encounters": (
                minimum_support
            ),
            "equivalent_train_prevalence_pct": (
                minimum_support
                / n_mimic_train_encounters
                * 100
            ),
            "diagnosis_features": (
                diagnosis_count
            ),
            "diagnosis_reference": (
                REFERENCE_FEATURE_COUNTS[
                    "diagnosis_titles"
                ]
            ),
            "diagnosis_absolute_difference": (
                diagnosis_difference
            ),
            "medication_name_features": (
                medication_name_count
            ),
            "medication_name_reference": (
                REFERENCE_FEATURE_COUNTS[
                    "medication_names"
                ]
            ),
            "medication_name_absolute_difference": (
                medication_name_difference
            ),
            "medication_class_features": (
                medication_class_count
            ),
            "medication_class_reference": (
                REFERENCE_FEATURE_COUNTS[
                    "medication_classes"
                ]
            ),
            "medication_class_absolute_difference": (
                medication_class_difference
            ),
            "total_history_features": (
                diagnosis_count
                + medication_name_count
                + medication_class_count
            ),
            "total_absolute_difference": (
                diagnosis_difference
                + medication_name_difference
                + medication_class_difference
            ),
        }
    )


history_support_comparison = pd.DataFrame(
    support_comparison_records
)



history_support_comparison[
    "closest_common_support"
] = False


closest_support_index = (
    history_support_comparison[
        "total_absolute_difference"
    ].idxmin()
)


history_support_comparison.loc[
    closest_support_index,
    "closest_common_support",
] = True


history_support_comparison[
    "provisional_support_10"
] = (
    history_support_comparison[
        "minimum_train_encounters"
    ].eq(
        PROVISIONAL_MINIMUM_SUPPORT
    )
)


print(
    "History feature-count comparison:"
)

display(
    history_support_comparison.round(
        {
            "equivalent_train_prevalence_pct": 3,
        }
    )
)



support_10_summary = (
    history_support_comparison.loc[
        history_support_comparison[
            "minimum_train_encounters"
        ].eq(
            PROVISIONAL_MINIMUM_SUPPORT
        )
    ]
    .copy()
)


print(
    "\nProvisional minimum support of 10 "
    "training encounters:"
)

display(
    support_10_summary.round(
        {
            "equivalent_train_prevalence_pct": 3,
        }
    )
)



history_source_availability = pd.DataFrame(
    [
        {
            "measure": (
                "MIMIC training encounters"
            ),
            "count": (
                n_mimic_train_encounters
            ),
        },
        {
            "measure": (
                "Training encounters with at least "
                "one medrecon row inside ED window"
            ),
            "count": (
                mimic_train_medrecon[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Training encounters with at least "
                "one medication name"
            ),
            "count": (
                mimic_train_medication_names[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Training encounters with at least "
                "one medication class"
            ),
            "count": (
                mimic_train_medication_classes[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique raw medication names"
            ),
            "count": (
                mimic_train_medication_name_support[
                    "medication_name_key"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique raw medication classes"
            ),
            "count": (
                mimic_train_medication_class_support[
                    "medication_class_key"
                ].nunique()
            ),
        },
    ]
)


print(
    "\nTraining medication-history availability:"
)

display(history_source_availability)



closest_common_support_row = (
    history_support_comparison.loc[
        closest_support_index
    ]
)


print(
    "\nClosest common support among tested values:",
    int(
        closest_common_support_row[
            "minimum_train_encounters"
        ]
    ),
)

print(
    "Equivalent prevalence:",
    round(
        closest_common_support_row[
            "equivalent_train_prevalence_pct"
        ],
        3,
    ),
    "%",
)

print(
    "\nThis is an audit only. "
    "No S1 vocabulary has been frozen or saved."
)


#### Prespecified and frozen S1/S2 feature vocabularies

Candidate previous-history features were prespecified during the preliminary development phase using the original MIMIC-IV development partition and a minimum support of five training encounters. The diagnosis, medication-name, and medication-class vocabulary subsequently underwent clinician review.

The previous-history, laboratory-result, and imaging-result vocabularies are therefore treated as prespecified and frozen during reconstruction of the refined cohorts. The revised approximate temporal split and MC-MED cohort do not determine which feature names are retained.

This section loads and verifies feature metadata only. Patient-level values are reconstructed from the current source tables using the current cohort identifiers and temporal eligibility rules.


In [ ]:



VOCABULARY_DIR = (
    PROJECT_ROOT
    / "config"
    / "feature_vocabularies"
)


if not VOCABULARY_DIR.exists():
    raise FileNotFoundError(
        "The frozen feature-vocabulary directory does not "
        f"exist:\n{VOCABULARY_DIR}\n\n"
        "Run the export cell in abd_event_log.ipynb first."
    )



CHECKSUM_PATH = (
    VOCABULARY_DIR
    / "checksums.sha256"
)


def calculate_sha256(
    file_path: Path,
) -> str:
    """Calculate SHA-256 for a local vocabulary file."""

    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as input_file:
        for chunk in iter(
            lambda: input_file.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


checksum_records = []

if CHECKSUM_PATH.exists():
    with open(
        CHECKSUM_PATH,
        "r",
        encoding="utf-8",
    ) as checksum_file:
        for line in checksum_file:
            line = line.strip()

            if not line:
                continue

            expected_hash, filename = (
                line.split(
                    None,
                    1,
                )
            )

            filename = filename.strip()

            file_path = (
                VOCABULARY_DIR
                / filename
            )

            actual_hash = (
                calculate_sha256(
                    file_path
                )
                if file_path.exists()
                else None
            )

            checksum_records.append(
                {
                    "filename": filename,
                    "exists": (
                        file_path.exists()
                    ),
                    "expected_sha256": (
                        expected_hash
                    ),
                    "actual_sha256": (
                        actual_hash
                    ),
                    "checksum_match": (
                        actual_hash
                        == expected_hash
                    ),
                }
            )


checksum_audit = pd.DataFrame(
    checksum_records
)


if not checksum_audit.empty:
    print("Vocabulary checksum audit:")

    display(
        checksum_audit[
            [
                "filename",
                "exists",
                "checksum_match",
            ]
        ]
    )

    assert checksum_audit[
        "exists"
    ].all()

    assert checksum_audit[
        "checksum_match"
    ].all(), (
        "At least one vocabulary file has changed since "
        "its checksum was generated."
    )



FROZEN_MANIFEST_PATH = (
    VOCABULARY_DIR
    / "frozen_s1_s2_feature_manifest.csv"
)


frozen_s1_s2_manifest = pd.read_csv(
    FROZEN_MANIFEST_PATH
)


required_manifest_columns = {
    "global_s1_s2_position",
    "family_position",
    "stage",
    "family",
    "feature_name",
}


missing_manifest_columns = (
    required_manifest_columns
    - set(
        frozen_s1_s2_manifest.columns
    )
)


if missing_manifest_columns:
    raise ValueError(
        "Frozen feature manifest is missing columns: "
        f"{sorted(missing_manifest_columns)}"
    )


frozen_s1_s2_manifest = (
    frozen_s1_s2_manifest
    .sort_values(
        "global_s1_s2_position"
    )
    .reset_index(drop=True)
)


assert frozen_s1_s2_manifest[
    "feature_name"
].is_unique



def get_frozen_family_features(
    family_name: str,
) -> list[str]:
    """Return one frozen family in its stored order."""

    return (
        frozen_s1_s2_manifest.loc[
            frozen_s1_s2_manifest[
                "family"
            ].eq(family_name)
        ]
        .sort_values(
            "family_position"
        )["feature_name"]
        .astype(str)
        .tolist()
    )


FROZEN_PMH_DIAGNOSIS_FEATURES = (
    get_frozen_family_features(
        "s1_diagnosis"
    )
)

FROZEN_PMH_MEDICATION_NAME_FEATURES = (
    get_frozen_family_features(
        "s1_medication_name"
    )
)

FROZEN_PMH_MEDICATION_CLASS_FEATURES = (
    get_frozen_family_features(
        "s1_medication_class"
    )
)

FROZEN_LAB_RESULT_FEATURES = (
    get_frozen_family_features(
        "s2_lab_result"
    )
)

FROZEN_IMAGING_RESULT_FEATURES = (
    get_frozen_family_features(
        "s2_imaging_result"
    )
)



S1_FROZEN_FEATURE_COLUMNS = (
    FROZEN_PMH_DIAGNOSIS_FEATURES
    +
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


S2_FROZEN_FEATURE_COLUMNS = (
    FROZEN_LAB_RESULT_FEATURES
    +
    FROZEN_IMAGING_RESULT_FEATURES
)


PRIMARY_MODEL_FEATURE_COLUMNS_PLANNED = (
    S0_FEATURE_COLUMNS
    +
    S1_FROZEN_FEATURE_COLUMNS
    +
    S2_FROZEN_FEATURE_COLUMNS
)



EXPECTED_COUNTS = {
    "S0 current reconstruction": 9,
    "S1 diagnoses": 879,
    "S1 medication names": 812,
    "S1 medication classes": 364,
    "S2 laboratory results": 351,
    "S2 imaging results": 80,
}


assert len(
    S0_FEATURE_COLUMNS
) == EXPECTED_COUNTS[
    "S0 current reconstruction"
]

assert len(
    FROZEN_PMH_DIAGNOSIS_FEATURES
) == EXPECTED_COUNTS[
    "S1 diagnoses"
]

assert len(
    FROZEN_PMH_MEDICATION_NAME_FEATURES
) == EXPECTED_COUNTS[
    "S1 medication names"
]

assert len(
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
) == EXPECTED_COUNTS[
    "S1 medication classes"
]

assert len(
    FROZEN_LAB_RESULT_FEATURES
) == EXPECTED_COUNTS[
    "S2 laboratory results"
]

assert len(
    FROZEN_IMAGING_RESULT_FEATURES
) == EXPECTED_COUNTS[
    "S2 imaging results"
]

assert len(
    S1_FROZEN_FEATURE_COLUMNS
) == 2055

assert len(
    S2_FROZEN_FEATURE_COLUMNS
) == 431

assert len(
    PRIMARY_MODEL_FEATURE_COLUMNS_PLANNED
) == 2495


assert len(
    set(
        PRIMARY_MODEL_FEATURE_COLUMNS_PLANNED
    )
) == len(
    PRIMARY_MODEL_FEATURE_COLUMNS_PLANNED
), (
    "Duplicate feature names exist across S0, S1 or S2."
)



PMH_LOOKUP_PATH = (
    VOCABULARY_DIR
    / "s1_pmh_feature_lookup_detailed.csv"
)

LAB_PLAN_PATH = (
    VOCABULARY_DIR
    / "s2_lab_feature_plan.csv"
)

IMAGING_SUMMARY_PATH = (
    VOCABULARY_DIR
    / "s2_imaging_feature_summary.csv"
)


frozen_pmh_feature_lookup = (
    pd.read_csv(PMH_LOOKUP_PATH)
    if PMH_LOOKUP_PATH.exists()
    else pd.DataFrame()
)

frozen_lab_feature_plan = (
    pd.read_csv(LAB_PLAN_PATH)
    if LAB_PLAN_PATH.exists()
    else pd.DataFrame()
)

frozen_imaging_feature_summary = (
    pd.read_csv(
        IMAGING_SUMMARY_PATH
    )
    if IMAGING_SUMMARY_PATH.exists()
    else pd.DataFrame()
)



frozen_vocabulary_load_audit = pd.DataFrame(
    [
        {
            "stage": "S0",
            "feature_family": (
                "Current reconstructed triage block"
            ),
            "n_features": len(
                S0_FEATURE_COLUMNS
            ),
            "vocabulary_source": (
                "Current Notebook 1"
            ),
        },
        {
            "stage": "S1",
            "feature_family": (
                "Previous diagnoses"
            ),
            "n_features": len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
            "vocabulary_source": (
                "Frozen development inventory"
            ),
        },
        {
            "stage": "S1",
            "feature_family": (
                "Previous medication names"
            ),
            "n_features": len(
                FROZEN_PMH_MEDICATION_NAME_FEATURES
            ),
            "vocabulary_source": (
                "Frozen development inventory"
            ),
        },
        {
            "stage": "S1",
            "feature_family": (
                "Previous medication classes"
            ),
            "n_features": len(
                FROZEN_PMH_MEDICATION_CLASS_FEATURES
            ),
            "vocabulary_source": (
                "Frozen development inventory"
            ),
        },
        {
            "stage": "S2",
            "feature_family": (
                "Abnormal laboratory results"
            ),
            "n_features": len(
                FROZEN_LAB_RESULT_FEATURES
            ),
            "vocabulary_source": (
                "Frozen development inventory"
            ),
        },
        {
            "stage": "S2",
            "feature_family": (
                "Abnormal imaging results"
            ),
            "n_features": len(
                FROZEN_IMAGING_RESULT_FEATURES
            ),
            "vocabulary_source": (
                "Frozen development inventory"
            ),
        },
    ]
)


print("Frozen vocabulary load audit:")

display(frozen_vocabulary_load_audit)


print(
    "\nS0 features:",
    len(S0_FEATURE_COLUMNS),
)

print(
    "S1 features:",
    len(S1_FROZEN_FEATURE_COLUMNS),
)

print(
    "S2 features:",
    len(S2_FROZEN_FEATURE_COLUMNS),
)

print(
    "Planned primary feature count:",
    len(
        PRIMARY_MODEL_FEATURE_COLUMNS_PLANNED
    ),
)


print(
    "\nDetailed PMH lookup rows:",
    len(frozen_pmh_feature_lookup),
)

print(
    "Laboratory plan rows:",
    len(frozen_lab_feature_plan),
)

print(
    "Imaging summary rows:",
    len(frozen_imaging_feature_summary),
)


print(
    "\nFROZEN S1/S2 VOCABULARY LOAD PASSED"
)


#### Frozen previous-diagnosis feature block

This section reconstructs the exact 879 previous-diagnosis features that were defined during the original MIMIC-IV development analysis and subsequently reviewed by clinicians.

The frozen feature dictionary determines which diagnosis variables exist. The revised training, validation, test and external-validation cohorts do not select new diagnosis features and do not remove previously reviewed features.

For MIMIC-IV, diagnosis values are reconstructed from hospital admissions completed before the current ED arrival. Each frozen diagnosis is encoded once per current ED encounter when its normalised historical diagnosis title is found.

For MC-MED, only PMH diagnoses documented on or before the current ED arrival are eligible. Exact canonical-title matching is attempted first. A conservative harmonisation rule equivalent to the previous external-validation logic is then used for minor differences in punctuation and wording. Every harmonised match is retained in an audit table.

Binary value 1 means that the frozen diagnosis was identified in the available previous-history records. Binary value 0 means that it was not identified in the available records; it does not prove that the patient has never had the condition.

No outcome labels, model results, validation data, external-validation outcomes, SHAP values or clinician importance judgements are used to construct these features.


In [ ]:



required_objects = [
    "frozen_pmh_feature_lookup",
    "FROZEN_PMH_DIAGNOSIS_FEATURES",
    "mimic_previous_diagnosis_long",
    "mcmed_previous_diagnosis_long_strict",
    "mimic_s0_validated",
    "mcmed_s0_validated",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding cohort, history and frozen-vocabulary "
        f"cells first. Missing objects: {missing_objects}"
    )



FEATURE_NAME_COLUMN_CANDIDATES = [
    "final_feature_name",
    "feature_name_cleaned",
    "feature_name",
]

frozen_lookup_feature_name_column = next(
    (
        column
        for column in FEATURE_NAME_COLUMN_CANDIDATES
        if column in frozen_pmh_feature_lookup.columns
    ),
    None,
)

if frozen_lookup_feature_name_column is None:
    raise ValueError(
        "The detailed PMH lookup does not contain a recognised "
        "final feature-name column."
    )


if "normalised_item" not in frozen_pmh_feature_lookup.columns:
    raise ValueError(
        "The detailed PMH lookup does not contain the required "
        "'normalised_item' column."
    )


print(
    "Authoritative frozen feature-name column:",
    frozen_lookup_feature_name_column,
)



def legacy_normalise_diagnosis_item(
    value,
) -> str:
    """
    Reproduce the diagnosis-title normalisation used when the
    frozen MIMIC-IV diagnosis vocabulary was originally created.
    """

    if pd.isna(value):
        return ""

    text = str(value).strip()

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if text in {
        "",
        "nan",
        "none",
        "null",
        "missing",
    }:
        return ""

    return text



frozen_lookup_working = (
    frozen_pmh_feature_lookup.copy()
)


frozen_lookup_working[
    "_resolved_feature_name"
] = (
    frozen_lookup_working[
        frozen_lookup_feature_name_column
    ]
    .astype("string")
    .str.strip()
)


frozen_diagnosis_feature_set = set(
    FROZEN_PMH_DIAGNOSIS_FEATURES
)


frozen_lookup_working = (
    frozen_lookup_working.loc[
        frozen_lookup_working[
            "_resolved_feature_name"
        ].isin(
            frozen_diagnosis_feature_set
        )
    ]
    .copy()
)


frozen_lookup_working[
    "_resolved_normalised_item"
] = (
    frozen_lookup_working[
        "normalised_item"
    ].apply(
        legacy_normalise_diagnosis_item
    )
)



frozen_diagnosis_lookup = pd.DataFrame(
    {
        "feature_name": (
            frozen_lookup_working[
                "_resolved_feature_name"
            ]
            .astype(str)
            .to_numpy()
        ),
        "normalised_item": (
            frozen_lookup_working[
                "_resolved_normalised_item"
            ]
            .astype(str)
            .to_numpy()
        ),
    }
)


OPTIONAL_LOOKUP_COLUMNS = [
    "source_column",
    "source_group",
    "n_reference_stays_with_item",
    "pct_reference_stays_with_item",
]

for optional_column in OPTIONAL_LOOKUP_COLUMNS:
    if optional_column in frozen_lookup_working.columns:
        frozen_diagnosis_lookup[
            optional_column
        ] = frozen_lookup_working[
            optional_column
        ].to_numpy()



duplicate_feature_rows = (
    frozen_diagnosis_lookup.loc[
        frozen_diagnosis_lookup[
            "feature_name"
        ].duplicated(
            keep=False
        )
    ]
    .sort_values(
        [
            "feature_name",
            "normalised_item",
        ]
    )
)


if not duplicate_feature_rows.empty:
    duplicate_feature_conflicts = (
        duplicate_feature_rows
        .groupby("feature_name")[
            "normalised_item"
        ]
        .nunique()
        .rename(
            "n_distinct_normalised_items"
        )
        .reset_index()
    )

    conflicting_features = (
        duplicate_feature_conflicts.loc[
            duplicate_feature_conflicts[
                "n_distinct_normalised_items"
            ].gt(1)
        ]
    )

    if not conflicting_features.empty:
        print(
            "Conflicting duplicate diagnosis mappings:"
        )

        display(
            duplicate_feature_rows.loc[
                duplicate_feature_rows[
                    "feature_name"
                ].isin(
                    conflicting_features[
                        "feature_name"
                    ]
                )
            ]
        )

        raise ValueError(
            "At least one frozen diagnosis feature maps to "
            "multiple different canonical diagnosis items."
        )


frozen_diagnosis_lookup = (
    frozen_diagnosis_lookup
    .drop_duplicates(
        subset=[
            "feature_name",
            "normalised_item",
        ]
    )
    .reset_index(drop=True)
)



observed_feature_set = set(
    frozen_diagnosis_lookup[
        "feature_name"
    ]
)

missing_frozen_features = (
    frozen_diagnosis_feature_set
    - observed_feature_set
)

unexpected_lookup_features = (
    observed_feature_set
    - frozen_diagnosis_feature_set
)


if missing_frozen_features:
    print(
        "Missing frozen diagnosis features:"
    )

    display(
        pd.DataFrame(
            {
                "missing_feature_name": sorted(
                    missing_frozen_features
                )
            }
        ).head(50)
    )

    raise ValueError(
        f"{len(missing_frozen_features)} frozen diagnosis "
        "features were not recovered from the detailed lookup."
    )


if unexpected_lookup_features:
    raise ValueError(
        "Unexpected diagnosis features were found after "
        f"filtering: {sorted(unexpected_lookup_features)[:20]}"
    )



duplicate_canonical_items = (
    frozen_diagnosis_lookup.loc[
        frozen_diagnosis_lookup[
            "normalised_item"
        ].duplicated(
            keep=False
        )
    ]
    .sort_values(
        [
            "normalised_item",
            "feature_name",
        ]
    )
)


if not duplicate_canonical_items.empty:
    print(
        "Canonical diagnosis items linked to more than "
        "one frozen feature:"
    )

    display(
        duplicate_canonical_items.head(50)
    )

    raise ValueError(
        "More than one frozen feature maps to the same "
        "normalised diagnosis item."
    )



frozen_diagnosis_lookup = (
    frozen_diagnosis_lookup
    .set_index("feature_name")
    .loc[
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ]
    .reset_index()
)


frozen_diagnosis_lookup.insert(
    0,
    "family_position",
    range(
        1,
        len(frozen_diagnosis_lookup) + 1,
    ),
)



assert len(
    FROZEN_PMH_DIAGNOSIS_FEATURES
) == 879

assert len(
    frozen_diagnosis_lookup
) == 879

assert frozen_diagnosis_lookup[
    "feature_name"
].is_unique

assert frozen_diagnosis_lookup[
    "normalised_item"
].is_unique

assert frozen_diagnosis_lookup[
    "normalised_item"
].ne("").all()

assert frozen_diagnosis_lookup[
    "feature_name"
].tolist() == (
    FROZEN_PMH_DIAGNOSIS_FEATURES
)



frozen_diagnosis_lookup_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen diagnosis features expected"
            ),
            "value": 879,
        },
        {
            "measure": (
                "Frozen diagnosis lookup rows recovered"
            ),
            "value": len(
                frozen_diagnosis_lookup
            ),
        },
        {
            "measure": (
                "Unique frozen feature names"
            ),
            "value": (
                frozen_diagnosis_lookup[
                    "feature_name"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique canonical diagnosis items"
            ),
            "value": (
                frozen_diagnosis_lookup[
                    "normalised_item"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Missing canonical diagnosis items"
            ),
            "value": int(
                frozen_diagnosis_lookup[
                    "normalised_item"
                ].eq("").sum()
            ),
        },
    ]
)


print(
    "\nFrozen diagnosis lookup audit:"
)

display(frozen_diagnosis_lookup_audit)


print(
    "\nExample frozen diagnosis mappings:"
)

display(
    frozen_diagnosis_lookup[
        [
            "family_position",
            "normalised_item",
            "feature_name",
        ]
    ].head(20)
)


print(
    "\nFROZEN 879-FEATURE DIAGNOSIS LOOKUP PASSED"
)


#### MIMIC-IV diagnosis features


In [ ]:


def build_binary_feature_matrix(
    mapped_long_dataframe: pd.DataFrame,
    identifier_column: str,
    feature_column: str,
    all_identifiers: pd.Series,
    ordered_feature_columns: list[str],
) -> pd.DataFrame:
    """
    Build a one-row-per-encounter binary feature matrix.

    All frozen columns are returned in their prespecified order,
    including features with no positive values.
    """

    identifier_index = pd.Index(
        all_identifiers.tolist(),
        name=identifier_column,
    )

    if mapped_long_dataframe.empty:
        wide_matrix = pd.DataFrame(
            0,
            index=identifier_index,
            columns=ordered_feature_columns,
            dtype="int8",
        )

    else:
        unique_mapping = (
            mapped_long_dataframe[
                [
                    identifier_column,
                    feature_column,
                ]
            ]
            .dropna()
            .drop_duplicates()
        )

        wide_matrix = pd.crosstab(
            unique_mapping[
                identifier_column
            ],
            unique_mapping[
                feature_column
            ],
        )

        wide_matrix = (
            wide_matrix.gt(0)
            .astype("int8")
        )

        wide_matrix = wide_matrix.reindex(
            index=identifier_index,
            columns=ordered_feature_columns,
            fill_value=0,
        )

        wide_matrix = wide_matrix.astype(
            "int8"
        )

    return wide_matrix.reset_index()



required_mimic_history_columns = {
    "subject_id",
    "stay_id",
    "long_title",
}

missing_mimic_history_columns = (
    required_mimic_history_columns
    - set(
        mimic_previous_diagnosis_long.columns
    )
)

if missing_mimic_history_columns:
    raise ValueError(
        "The MIMIC previous-diagnosis table is missing: "
        f"{sorted(missing_mimic_history_columns)}"
    )


if {
    "prior_hospital_dischtime",
    "current_intime",
}.issubset(
    mimic_previous_diagnosis_long.columns
):
    assert (
        mimic_previous_diagnosis_long[
            "prior_hospital_dischtime"
        ]
        <
        mimic_previous_diagnosis_long[
            "current_intime"
        ]
    ).all(), (
        "A retained MIMIC diagnosis originates from an "
        "admission that did not finish before current ED arrival."
    )



mimic_diagnosis_mapping_source = (
    mimic_previous_diagnosis_long[
        [
            "subject_id",
            "stay_id",
            "long_title",
        ]
    ]
    .copy()
)


mimic_diagnosis_mapping_source[
    "normalised_item"
] = mimic_diagnosis_mapping_source[
    "long_title"
].apply(
    legacy_normalise_diagnosis_item
)


mimic_diagnosis_mapping_source = (
    mimic_diagnosis_mapping_source.loc[
        mimic_diagnosis_mapping_source[
            "normalised_item"
        ].ne("")
    ]
    .drop_duplicates(
        subset=[
            "stay_id",
            "normalised_item",
        ]
    )
    .reset_index(drop=True)
)



mimic_frozen_diagnosis_mapping = (
    mimic_diagnosis_mapping_source.merge(
        frozen_diagnosis_lookup[
            [
                "normalised_item",
                "feature_name",
            ]
        ],
        on="normalised_item",
        how="left",
        validate="many_to_one",
        indicator="frozen_mapping_status",
    )
)


mimic_frozen_diagnosis_positive_long = (
    mimic_frozen_diagnosis_mapping.loc[
        mimic_frozen_diagnosis_mapping[
            "feature_name"
        ].notna(),
        [
            "subject_id",
            "stay_id",
            "normalised_item",
            "feature_name",
        ],
    ]
    .drop_duplicates(
        subset=[
            "stay_id",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mimic_diagnosis_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mimic_frozen_diagnosis_positive_long
        ),
        identifier_column="stay_id",
        feature_column="feature_name",
        all_identifiers=(
            mimic_s0_validated[
                "stay_id"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_DIAGNOSIS_FEATURES
        ),
    )
)


mimic_s1_diagnosis_features = (
    mimic_s0_validated[
        [
            "subject_id",
            "stay_id",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mimic_diagnosis_wide,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
)


for feature_name in (
    FROZEN_PMH_DIAGNOSIS_FEATURES
):
    mimic_s1_diagnosis_features[
        feature_name
    ] = (
        mimic_s1_diagnosis_features[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )



assert len(
    mimic_s1_diagnosis_features
) == 8550

assert mimic_s1_diagnosis_features[
    "subject_id"
].is_unique

assert mimic_s1_diagnosis_features[
    "stay_id"
].is_unique

mimic_diagnosis_feature_columns_observed = [
    column
    for column in mimic_s1_diagnosis_features.columns
    if column.startswith(
        "pmh_icd_title__"
    )
]

assert (
    mimic_diagnosis_feature_columns_observed
    ==
    FROZEN_PMH_DIAGNOSIS_FEATURES
), (
    "MIMIC diagnosis columns do not exactly match the "
    "frozen feature order."
)

assert len(
    mimic_diagnosis_feature_columns_observed
) == 879

mimic_binary_values = np.unique(
    mimic_s1_diagnosis_features[
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ].to_numpy()
)

assert set(
    mimic_binary_values.tolist()
).issubset(
    {
        0,
        1,
    }
)



mimic_diagnosis_reconstruction_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen MIMIC cohort encounters"
            ),
            "value": len(
                mimic_s1_diagnosis_features
            ),
        },
        {
            "measure": (
                "Temporally valid historical "
                "encounter-title pairs"
            ),
            "value": len(
                mimic_diagnosis_mapping_source
            ),
        },
        {
            "measure": (
                "Historical title pairs mapped "
                "to frozen vocabulary"
            ),
            "value": int(
                mimic_frozen_diagnosis_mapping[
                    "feature_name"
                ].notna().sum()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "frozen diagnosis"
            ),
            "value": (
                mimic_frozen_diagnosis_positive_long[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns"
            ),
            "value": len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                ">=1 positive value"
            ),
            "value": int(
                (
                    mimic_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                "zero positive values"
            ),
            "value": int(
                (
                    mimic_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    == 0
                ).sum()
            ),
        },
    ]
)


print(
    "MIMIC-IV frozen diagnosis reconstruction audit:"
)

display(mimic_diagnosis_reconstruction_audit)


print(
    "\nMIMIC-IV diagnosis matrix shape:",
    mimic_s1_diagnosis_features.shape,
)

print(
    "MIMIC-IV frozen diagnosis columns:",
    len(
        mimic_diagnosis_feature_columns_observed
    ),
)


#### MC-MED diagnosis features


In [ ]:



def normalise_external_diagnosis_slug(
    value,
) -> str:
    """
    Reproduce the diagnosis-title slug used in the original
    MC-MED external harmonisation.
    """

    if pd.isna(value):
        return ""

    text = str(value).lower().strip()

    text = text.replace(
        "&",
        " and ",
    )

    text = text.replace(
        "+",
        " plus ",
    )

    text = re.sub(
        r"\([^)]*\)",
        " ",
        text,
    )

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text,
    )

    text = re.sub(
        r"_+",
        "_",
        text,
    ).strip("_")

    return text


def diagnosis_token_overlap(
    first_slug: str,
    second_slug: str,
) -> float:
    """Calculate conservative token overlap."""

    first_tokens = {
        token
        for token in str(
            first_slug
        ).split("_")
        if len(token) > 1
    }

    second_tokens = {
        token
        for token in str(
            second_slug
        ).split("_")
        if len(token) > 1
    }

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(
            first_tokens
            & second_tokens
        )
        /
        max(
            len(first_tokens),
            len(second_tokens),
        )
    )


def diagnosis_string_similarity(
    first_slug: str,
    second_slug: str,
) -> float:
    """Calculate character-level similarity."""

    return SequenceMatcher(
        None,
        str(first_slug),
        str(second_slug),
    ).ratio()



mcmed_diagnosis_targets = (
    frozen_diagnosis_lookup[
        [
            "feature_name",
            "normalised_item",
        ]
    ]
    .copy()
)


mcmed_diagnosis_targets[
    "feature_slug"
] = mcmed_diagnosis_targets[
    "feature_name"
].str.replace(
    r"^pmh_icd_title__",
    "",
    regex=True,
)


mcmed_diagnosis_targets[
    "canonical_slug"
] = mcmed_diagnosis_targets[
    "normalised_item"
].apply(
    normalise_external_diagnosis_slug
)


assert mcmed_diagnosis_targets[
    "feature_name"
].is_unique

assert mcmed_diagnosis_targets[
    "normalised_item"
].is_unique

assert mcmed_diagnosis_targets[
    "feature_slug"
].is_unique, (
    "Duplicate external feature slugs were found. "
    "Inspect retained technical-hash feature names."
)


canonical_item_to_feature = dict(
    zip(
        mcmed_diagnosis_targets[
            "normalised_item"
        ],
        mcmed_diagnosis_targets[
            "feature_name"
        ],
    )
)

feature_slug_to_feature = dict(
    zip(
        mcmed_diagnosis_targets[
            "feature_slug"
        ],
        mcmed_diagnosis_targets[
            "feature_name"
        ],
    )
)


target_feature_records = (
    mcmed_diagnosis_targets
    .set_index("feature_name")
    .to_dict(orient="index")
)



target_token_index = defaultdict(set)

for feature_name, target_record in (
    target_feature_records.items()
):
    target_slug = (
        target_record[
            "feature_slug"
        ]
    )

    for token in set(
        target_slug.split("_")
    ):
        if len(token) > 1:
            target_token_index[
                token
            ].add(
                feature_name
            )



def map_mcmed_diagnosis_to_frozen_feature(
    description,
) -> dict:
    """
    Map one MC-MED diagnosis description to a frozen feature.

    Matching order:
    1. exact canonical normalised title;
    2. exact external feature slug;
    3. conservative fuzzy match using the old thresholds.
    """

    canonical_item = (
        legacy_normalise_diagnosis_item(
            description
        )
    )

    description_slug = (
        normalise_external_diagnosis_slug(
            description
        )
    )

    empty_result = {
        "normalised_mcmed_item": (
            canonical_item
        ),
        "mcmed_description_slug": (
            description_slug
        ),
        "matched_feature": pd.NA,
        "match_method": "UNMATCHED",
        "combined_score": np.nan,
        "token_overlap": np.nan,
        "string_similarity": np.nan,
    }

    if not canonical_item:
        return empty_result

    if canonical_item in (
        canonical_item_to_feature
    ):
        return {
            **empty_result,
            "matched_feature": (
                canonical_item_to_feature[
                    canonical_item
                ]
            ),
            "match_method": (
                "EXACT_CANONICAL_LOOKUP"
            ),
            "combined_score": 1.0,
            "token_overlap": 1.0,
            "string_similarity": 1.0,
        }

    if description_slug in (
        feature_slug_to_feature
    ):
        return {
            **empty_result,
            "matched_feature": (
                feature_slug_to_feature[
                    description_slug
                ]
            ),
            "match_method": (
                "EXACT_FEATURE_SLUG"
            ),
            "combined_score": 1.0,
            "token_overlap": 1.0,
            "string_similarity": 1.0,
        }

    if len(description_slug) < 8:
        return empty_result

    description_tokens = {
        token
        for token in description_slug.split("_")
        if len(token) > 1
    }

    candidate_features = set()

    for token in description_tokens:
        candidate_features.update(
            target_token_index.get(
                token,
                set(),
            )
        )

    if not candidate_features:
        return empty_result

    best_feature = None
    best_score = 0.0
    best_overlap = 0.0
    best_similarity = 0.0

    for feature_name in sorted(
        candidate_features
    ):
        target_slug = (
            target_feature_records[
                feature_name
            ]["feature_slug"]
        )

        overlap = (
            diagnosis_token_overlap(
                description_slug,
                target_slug,
            )
        )

        similarity = (
            diagnosis_string_similarity(
                description_slug,
                target_slug,
            )
        )

        combined_score = (
            0.70 * overlap
            +
            0.30 * similarity
        )

        if combined_score > best_score:
            best_feature = feature_name
            best_score = combined_score
            best_overlap = overlap
            best_similarity = similarity

    if (
        best_feature is not None
        and best_score >= 0.88
        and best_overlap >= 0.75
        and best_similarity >= 0.82
    ):
        return {
            **empty_result,
            "matched_feature": best_feature,
            "match_method": (
                "CONSERVATIVE_FUZZY"
            ),
            "combined_score": best_score,
            "token_overlap": best_overlap,
            "string_similarity": (
                best_similarity
            ),
        }

    return {
        **empty_result,
        "combined_score": best_score,
        "token_overlap": best_overlap,
        "string_similarity": best_similarity,
    }



required_mcmed_history_columns = {
    "MRN",
    "CSN",
    "Desc10",
    "Noted_date_parsed",
    "current_arrival_time",
}

missing_mcmed_history_columns = (
    required_mcmed_history_columns
    - set(
        mcmed_previous_diagnosis_long_strict.columns
    )
)

if missing_mcmed_history_columns:
    raise ValueError(
        "The strict MC-MED PMH table is missing: "
        f"{sorted(missing_mcmed_history_columns)}"
    )


assert (
    mcmed_previous_diagnosis_long_strict[
        "Noted_date_parsed"
    ]
    <=
    mcmed_previous_diagnosis_long_strict[
        "current_arrival_time"
    ]
).all()



mcmed_unique_diagnosis_descriptions = (
    mcmed_previous_diagnosis_long_strict[
        [
            "Desc10",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)


mcmed_mapping_result_records = []

for description in (
    mcmed_unique_diagnosis_descriptions[
        "Desc10"
    ]
):
    mapping_result = (
        map_mcmed_diagnosis_to_frozen_feature(
            description
        )
    )

    mcmed_mapping_result_records.append(
        {
            "Desc10": description,
            **mapping_result,
        }
    )


mcmed_frozen_diagnosis_mapping_audit = (
    pd.DataFrame(
        mcmed_mapping_result_records
    )
)



mcmed_diagnosis_mapping_source = (
    mcmed_previous_diagnosis_long_strict.merge(
        mcmed_frozen_diagnosis_mapping_audit,
        on="Desc10",
        how="left",
        validate="many_to_one",
    )
)


mcmed_frozen_diagnosis_positive_long = (
    mcmed_diagnosis_mapping_source.loc[
        mcmed_diagnosis_mapping_source[
            "matched_feature"
        ].notna(),
        [
            "MRN",
            "CSN",
            "Desc10",
            "matched_feature",
            "match_method",
            "combined_score",
        ],
    ]
    .rename(
        columns={
            "matched_feature":
                "feature_name"
        }
    )
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mcmed_diagnosis_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mcmed_frozen_diagnosis_positive_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated[
                "CSN"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_DIAGNOSIS_FEATURES
        ),
    )
)


mcmed_s1_diagnosis_features = (
    mcmed_s0_validated[
        [
            "MRN",
            "CSN",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mcmed_diagnosis_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


for feature_name in (
    FROZEN_PMH_DIAGNOSIS_FEATURES
):
    mcmed_s1_diagnosis_features[
        feature_name
    ] = (
        mcmed_s1_diagnosis_features[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )



assert len(
    mcmed_s1_diagnosis_features
) == 6707

assert mcmed_s1_diagnosis_features[
    "MRN"
].is_unique

assert mcmed_s1_diagnosis_features[
    "CSN"
].is_unique

mcmed_diagnosis_feature_columns_observed = [
    column
    for column
    in mcmed_s1_diagnosis_features.columns
    if column.startswith(
        "pmh_icd_title__"
    )
]

assert (
    mcmed_diagnosis_feature_columns_observed
    ==
    FROZEN_PMH_DIAGNOSIS_FEATURES
)

assert len(
    mcmed_diagnosis_feature_columns_observed
) == 879

mcmed_binary_values = np.unique(
    mcmed_s1_diagnosis_features[
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ].to_numpy()
)

assert set(
    mcmed_binary_values.tolist()
).issubset(
    {
        0,
        1,
    }
)



mcmed_mapping_method_summary = (
    mcmed_frozen_diagnosis_mapping_audit[
        "match_method"
    ]
    .value_counts(dropna=False)
    .rename_axis("match_method")
    .reset_index(
        name="n_unique_descriptions"
    )
)

mcmed_mapping_method_summary[
    "percentage"
] = (
    mcmed_mapping_method_summary[
        "n_unique_descriptions"
    ]
    /
    len(
        mcmed_frozen_diagnosis_mapping_audit
    )
    * 100
)


mcmed_diagnosis_reconstruction_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen MC-MED cohort encounters"
            ),
            "value": len(
                mcmed_s1_diagnosis_features
            ),
        },
        {
            "measure": (
                "Temporally confirmed PMH "
                "encounter-diagnosis rows"
            ),
            "value": len(
                mcmed_previous_diagnosis_long_strict
            ),
        },
        {
            "measure": (
                "Unique MC-MED diagnosis "
                "descriptions audited"
            ),
            "value": len(
                mcmed_frozen_diagnosis_mapping_audit
            ),
        },
        {
            "measure": (
                "Unique descriptions mapped "
                "to frozen vocabulary"
            ),
            "value": int(
                mcmed_frozen_diagnosis_mapping_audit[
                    "matched_feature"
                ].notna().sum()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "frozen diagnosis"
            ),
            "value": (
                mcmed_frozen_diagnosis_positive_long[
                    "CSN"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns"
            ),
            "value": len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                ">=1 positive value"
            ),
            "value": int(
                (
                    mcmed_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                "zero positive values"
            ),
            "value": int(
                (
                    mcmed_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    == 0
                ).sum()
            ),
        },
    ]
)


print(
    "MC-MED frozen diagnosis reconstruction audit:"
)

display(mcmed_diagnosis_reconstruction_audit)


print(
    "\nMC-MED diagnosis matching methods:"
)

display(
    mcmed_mapping_method_summary.round(
        {
            "percentage": 2,
        }
    )
)


print(
    "\nConservative fuzzy mappings for review:"
)

display(
    mcmed_frozen_diagnosis_mapping_audit.loc[
        mcmed_frozen_diagnosis_mapping_audit[
            "match_method"
        ].eq(
            "CONSERVATIVE_FUZZY"
        )
    ]
    .sort_values(
        "combined_score"
    )
    .head(30)
    .round(
        {
            "combined_score": 3,
            "token_overlap": 3,
            "string_similarity": 3,
        }
    )
)


print(
    "\nMC-MED diagnosis matrix shape:",
    mcmed_s1_diagnosis_features.shape,
)

print(
    "MC-MED frozen diagnosis columns:",
    len(
        mcmed_diagnosis_feature_columns_observed
    ),
)


In [ ]:



if (
    "mcmed_frozen_diagnosis_mapping_audit"
    not in globals()
):
    raise NameError(
        "Run Cell 75 before applying the reviewed "
        "MC-MED diagnosis mappings."
    )



APPROVED_FUZZY_DIAGNOSIS_DESCRIPTIONS = {
    "Acquired absence of left breast and nipple",
    "Spinal stenosis, lumbar region without neurogenic claud",
    "Migraine, unsp, not intractable, without status migrainosus",
    "Other forms of chronic ischemic heart disease",
    "Asymptomatic human immunodeficiency virus infection status",
    "Secondary malignant neoplasm of retroperiton and peritoneum",
    "Personal history of other venous thrombosis and embolism",
    "Personal history of other diseases of the digestive system",
    "Malignant neoplasm of upper lobe, right bronchus or lung",
    "Malignant neoplasm of upper lobe, left bronchus or lung",
    "Secondary malig neoplasm of liver and intrahepatic bile duct",
    "Chronic kidney disease, stage 5",
    "Body mass index (BMI) 50-59.9, adult",
}



REJECTED_FUZZY_DIAGNOSIS_REASONS = {
    (
        "Malignant neoplasm of middle lobe, "
        "bronchus or lung"
    ): (
        "Different anatomical site: middle lobe "
        "cannot be mapped to upper lobe."
    ),

    (
        "Personal history of other malignant "
        "neoplasm of kidney"
    ): (
        "Different organ: kidney malignancy cannot "
        "be mapped to skin malignancy."
    ),

    (
        "Spinal stenosis, lumbar region with "
        "neurogenic claudication"
    ): (
        "Opposing qualifier: with neurogenic "
        "claudication cannot map to without."
    ),

    "Chronic kidney disease, stage 2 (mild)": (
        "Different disease stage: stage 2 cannot "
        "be mapped to stage V."
    ),

    "Chronic kidney disease, stage 4 (severe)": (
        "Different disease stage: stage 4 cannot "
        "be mapped to stage V."
    ),

    "Chronic kidney disease, stage 1": (
        "Different disease stage: stage 1 cannot "
        "be mapped to stage V."
    ),

    "Vitamin deficiency, unspecified": (
        "Unspecified vitamin deficiency cannot be "
        "assumed to be vitamin D deficiency."
    ),

    "Vitamin A deficiency, unspecified": (
        "Different vitamin subtype: vitamin A "
        "cannot be mapped to vitamin D."
    ),

    "Vitamin B deficiency, unspecified": (
        "Different vitamin subtype: vitamin B "
        "cannot be mapped to vitamin D."
    ),

    "Type 1 diabetes mellitus without complications": (
        "Different diabetes type: type 1 cannot "
        "be mapped to type 2."
    ),

    "Unspecified viral hepatitis without hepatic coma": (
        "Unspecified viral hepatitis cannot be "
        "assumed to be hepatitis C."
    ),

    "Unspecified viral hepatitis B without hepatic coma": (
        "Different viral subtype: hepatitis B "
        "cannot be mapped to hepatitis C."
    ),
}



fuzzy_mapping_rows = (
    mcmed_frozen_diagnosis_mapping_audit.loc[
        mcmed_frozen_diagnosis_mapping_audit[
            "match_method"
        ].eq("CONSERVATIVE_FUZZY")
    ]
    .copy()
)


observed_fuzzy_descriptions = set(
    fuzzy_mapping_rows["Desc10"].astype(str)
)

reviewed_fuzzy_descriptions = (
    APPROVED_FUZZY_DIAGNOSIS_DESCRIPTIONS
    |
    set(
        REJECTED_FUZZY_DIAGNOSIS_REASONS
    )
)


unreviewed_fuzzy_descriptions = (
    observed_fuzzy_descriptions
    - reviewed_fuzzy_descriptions
)

reviewed_but_not_observed = (
    reviewed_fuzzy_descriptions
    - observed_fuzzy_descriptions
)


if unreviewed_fuzzy_descriptions:
    display(
        pd.DataFrame(
            {
                "unreviewed_description": sorted(
                    unreviewed_fuzzy_descriptions
                )
            }
        )
    )

    raise ValueError(
        "At least one fuzzy diagnosis mapping has not "
        "received an explicit clinical review decision."
    )


if reviewed_but_not_observed:
    print(
        "Warning: some stored review descriptions were not "
        "present in the current fuzzy output:"
    )

    display(
        pd.DataFrame(
            {
                "not_observed_description": sorted(
                    reviewed_but_not_observed
                )
            }
        )
    )



mcmed_fuzzy_diagnosis_review = (
    fuzzy_mapping_rows[
        [
            "Desc10",
            "normalised_mcmed_item",
            "matched_feature",
            "combined_score",
            "token_overlap",
            "string_similarity",
        ]
    ]
    .copy()
)


mcmed_fuzzy_diagnosis_review[
    "review_decision"
] = np.where(
    mcmed_fuzzy_diagnosis_review[
        "Desc10"
    ].isin(
        APPROVED_FUZZY_DIAGNOSIS_DESCRIPTIONS
    ),
    "APPROVED",
    "REJECTED",
)


mcmed_fuzzy_diagnosis_review[
    "review_reason"
] = mcmed_fuzzy_diagnosis_review[
    "Desc10"
].map(
    REJECTED_FUZZY_DIAGNOSIS_REASONS
)


mcmed_fuzzy_diagnosis_review.loc[
    mcmed_fuzzy_diagnosis_review[
        "review_decision"
    ].eq("APPROVED"),
    "review_reason",
] = (
    "Clinically equivalent wording, abbreviation, "
    "truncation or acceptable specific-to-general mapping."
)


mcmed_fuzzy_diagnosis_review = (
    mcmed_fuzzy_diagnosis_review
    .sort_values(
        [
            "review_decision",
            "Desc10",
        ]
    )
    .reset_index(drop=True)
)


print("Reviewed fuzzy diagnosis mappings:")

display(
    mcmed_fuzzy_diagnosis_review.round(
        {
            "combined_score": 3,
            "token_overlap": 3,
            "string_similarity": 3,
        }
    )
)


print(
    "\nApproved fuzzy mappings:",
    int(
        mcmed_fuzzy_diagnosis_review[
            "review_decision"
        ].eq("APPROVED").sum()
    ),
)

print(
    "Rejected fuzzy mappings:",
    int(
        mcmed_fuzzy_diagnosis_review[
            "review_decision"
        ].eq("REJECTED").sum()
    ),
)


assert (
    mcmed_fuzzy_diagnosis_review[
        "review_decision"
    ].eq("APPROVED").sum()
    == 13
)

assert (
    mcmed_fuzzy_diagnosis_review[
        "review_decision"
    ].eq("REJECTED").sum()
    == 12
)



mcmed_frozen_diagnosis_mapping_audit = (
    mcmed_frozen_diagnosis_mapping_audit.copy()
)


rejected_fuzzy_mask = (
    mcmed_frozen_diagnosis_mapping_audit[
        "match_method"
    ].eq("CONSERVATIVE_FUZZY")
    &
    mcmed_frozen_diagnosis_mapping_audit[
        "Desc10"
    ].isin(
        REJECTED_FUZZY_DIAGNOSIS_REASONS
    )
)


approved_fuzzy_mask = (
    mcmed_frozen_diagnosis_mapping_audit[
        "match_method"
    ].eq("CONSERVATIVE_FUZZY")
    &
    mcmed_frozen_diagnosis_mapping_audit[
        "Desc10"
    ].isin(
        APPROVED_FUZZY_DIAGNOSIS_DESCRIPTIONS
    )
)


mcmed_frozen_diagnosis_mapping_audit.loc[
    approved_fuzzy_mask,
    "match_method",
] = "REVIEWED_FUZZY_APPROVED"


mcmed_frozen_diagnosis_mapping_audit.loc[
    rejected_fuzzy_mask,
    "matched_feature",
] = pd.NA


mcmed_frozen_diagnosis_mapping_audit.loc[
    rejected_fuzzy_mask,
    "match_method",
] = "REVIEWED_FUZZY_REJECTED"



mcmed_diagnosis_mapping_source = (
    mcmed_previous_diagnosis_long_strict.merge(
        mcmed_frozen_diagnosis_mapping_audit,
        on="Desc10",
        how="left",
        validate="many_to_one",
    )
)


mcmed_frozen_diagnosis_positive_long = (
    mcmed_diagnosis_mapping_source.loc[
        mcmed_diagnosis_mapping_source[
            "matched_feature"
        ].notna(),
        [
            "MRN",
            "CSN",
            "Desc10",
            "matched_feature",
            "match_method",
            "combined_score",
        ],
    ]
    .rename(
        columns={
            "matched_feature": "feature_name"
        }
    )
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mcmed_diagnosis_wide = build_binary_feature_matrix(
    mapped_long_dataframe=(
        mcmed_frozen_diagnosis_positive_long
    ),
    identifier_column="CSN",
    feature_column="feature_name",
    all_identifiers=mcmed_s0_validated["CSN"],
    ordered_feature_columns=(
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ),
)


mcmed_s1_diagnosis_features = (
    mcmed_s0_validated[
        [
            "MRN",
            "CSN",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mcmed_diagnosis_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


for feature_name in FROZEN_PMH_DIAGNOSIS_FEATURES:
    mcmed_s1_diagnosis_features[
        feature_name
    ] = (
        mcmed_s1_diagnosis_features[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )



mcmed_mapping_method_summary = (
    mcmed_frozen_diagnosis_mapping_audit[
        "match_method"
    ]
    .value_counts(dropna=False)
    .rename_axis("match_method")
    .reset_index(
        name="n_unique_descriptions"
    )
)


mcmed_mapping_method_summary[
    "percentage"
] = (
    mcmed_mapping_method_summary[
        "n_unique_descriptions"
    ]
    / len(
        mcmed_frozen_diagnosis_mapping_audit
    )
    * 100
)


n_strict_rows = len(
    mcmed_previous_diagnosis_long_strict
)

n_mapped_strict_rows = int(
    mcmed_diagnosis_mapping_source[
        "matched_feature"
    ].notna().sum()
)


n_history_positive_encounters = (
    mcmed_previous_diagnosis_long_strict[
        "CSN"
    ].nunique()
)

n_encounters_with_mapped_diagnosis = (
    mcmed_frozen_diagnosis_positive_long[
        "CSN"
    ].nunique()
)


mcmed_diagnosis_reconstruction_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen MC-MED cohort encounters"
            ),
            "value": len(
                mcmed_s1_diagnosis_features
            ),
        },
        {
            "measure": (
                "Temporally confirmed PMH "
                "encounter-diagnosis rows"
            ),
            "value": n_strict_rows,
        },
        {
            "measure": (
                "Mapped temporally confirmed "
                "encounter-diagnosis rows"
            ),
            "value": n_mapped_strict_rows,
        },
        {
            "measure": (
                "Row-level frozen-vocabulary "
                "coverage percentage"
            ),
            "value": (
                n_mapped_strict_rows
                / n_strict_rows
                * 100
                if n_strict_rows
                else np.nan
            ),
        },
        {
            "measure": (
                "Unique MC-MED diagnosis "
                "descriptions audited"
            ),
            "value": len(
                mcmed_frozen_diagnosis_mapping_audit
            ),
        },
        {
            "measure": (
                "Unique descriptions mapped "
                "after clinical review"
            ),
            "value": int(
                mcmed_frozen_diagnosis_mapping_audit[
                    "matched_feature"
                ].notna().sum()
            ),
        },
        {
            "measure": (
                "Encounters with temporally "
                "confirmed diagnosis history"
            ),
            "value": n_history_positive_encounters,
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "frozen diagnosis"
            ),
            "value": (
                n_encounters_with_mapped_diagnosis
            ),
        },
        {
            "measure": (
                "Encounter-level mapping "
                "coverage percentage"
            ),
            "value": (
                n_encounters_with_mapped_diagnosis
                / n_history_positive_encounters
                * 100
                if n_history_positive_encounters
                else np.nan
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns"
            ),
            "value": len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                ">=1 positive external value"
            ),
            "value": int(
                (
                    mcmed_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
        {
            "measure": (
                "Frozen diagnosis columns with "
                "zero external values"
            ),
            "value": int(
                (
                    mcmed_s1_diagnosis_features[
                        FROZEN_PMH_DIAGNOSIS_FEATURES
                    ].sum(axis=0)
                    == 0
                ).sum()
            ),
        },
    ]
)



assert len(
    mcmed_s1_diagnosis_features
) == 6707

assert mcmed_s1_diagnosis_features[
    "CSN"
].is_unique

assert (
    [
        column
        for column
        in mcmed_s1_diagnosis_features.columns
        if column.startswith(
            "pmh_icd_title__"
        )
    ]
    ==
    FROZEN_PMH_DIAGNOSIS_FEATURES
)


assert set(
    np.unique(
        mcmed_s1_diagnosis_features[
            FROZEN_PMH_DIAGNOSIS_FEATURES
        ].to_numpy()
    ).tolist()
).issubset(
    {
        0,
        1,
    }
)



FUZZY_REVIEW_PATH = (
    VOCABULARY_DIR
    / "s1_mcmed_diagnosis_fuzzy_review.csv"
)


mcmed_fuzzy_diagnosis_review.to_csv(
    FUZZY_REVIEW_PATH,
    index=False,
)


print(
    "\nCorrected MC-MED diagnosis reconstruction audit:"
)

display(
    mcmed_diagnosis_reconstruction_audit.round(2)
)


print(
    "\nCorrected diagnosis matching methods:"
)

display(
    mcmed_mapping_method_summary.round(
        {
            "percentage": 2,
        }
    )
)


print(
    "\nSaved reviewed mapping configuration:"
)

display(
    pd.DataFrame(
        [
            {
                "file": FUZZY_REVIEW_PATH.name,
                "exists": (
                    FUZZY_REVIEW_PATH.exists()
                ),
                "path": str(
                    FUZZY_REVIEW_PATH
                ),
            }
        ]
    )
)


print(
    "\nMC-MED DIAGNOSIS HARMONISATION REVIEW PASSED"
)

print(
    "Rerun Cell 76 to regenerate support tables "
    "and overwrite the saved MC-MED diagnosis matrix."
)


In [ ]:


def create_binary_feature_support_table(
    dataframe: pd.DataFrame,
    dataset_name: str,
    split_name: str,
    feature_columns: list[str],
) -> pd.DataFrame:
    """Summarise positive support for binary features."""

    support_counts = (
        dataframe[
            feature_columns
        ]
        .sum(axis=0)
        .astype("int64")
    )

    support_table = pd.DataFrame(
        {
            "feature_name": (
                feature_columns
            ),
            "positive_count": (
                support_counts.reindex(
                    feature_columns
                ).to_numpy()
            ),
        }
    )

    support_table[
        "positive_prevalence_pct"
    ] = (
        support_table[
            "positive_count"
        ]
        / len(dataframe)
        * 100
    )

    support_table.insert(
        0,
        "n_rows",
        len(dataframe),
    )

    support_table.insert(
        0,
        "split",
        split_name,
    )

    support_table.insert(
        0,
        "dataset",
        dataset_name,
    )

    return support_table



diagnosis_support_tables = []

for split_name in [
    "train",
    "validation",
    "test",
]:
    split_rows = (
        mimic_s1_diagnosis_features.loc[
            mimic_s1_diagnosis_features[
                "split"
            ].astype("string").eq(
                split_name
            )
        ]
    )

    diagnosis_support_tables.append(
        create_binary_feature_support_table(
            dataframe=split_rows,
            dataset_name="MIMIC-IV",
            split_name=split_name,
            feature_columns=(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
        )
    )


diagnosis_support_tables.append(
    create_binary_feature_support_table(
        dataframe=(
            mcmed_s1_diagnosis_features
        ),
        dataset_name="MC-MED",
        split_name="external",
        feature_columns=(
            FROZEN_PMH_DIAGNOSIS_FEATURES
        ),
    )
)


frozen_diagnosis_feature_support = pd.concat(
    diagnosis_support_tables,
    ignore_index=True,
)



diagnosis_support_wide = (
    frozen_diagnosis_feature_support
    .pivot(
        index="feature_name",
        columns=[
            "dataset",
            "split",
        ],
        values="positive_count",
    )
    .fillna(0)
)


diagnosis_support_wide.columns = [
    (
        f"{dataset.lower().replace('-', '')}_"
        f"{split_name}_positive_count"
    )
    for dataset, split_name
    in diagnosis_support_wide.columns
]


diagnosis_support_wide = (
    diagnosis_support_wide
    .reindex(
        FROZEN_PMH_DIAGNOSIS_FEATURES
    )
    .reset_index()
)


expected_support_columns = {
    "mimiciv_train_positive_count",
    "mimiciv_validation_positive_count",
    "mimiciv_test_positive_count",
    "mcmed_external_positive_count",
}

missing_support_columns = (
    expected_support_columns
    - set(
        diagnosis_support_wide.columns
    )
)

if missing_support_columns:
    raise ValueError(
        "Diagnosis support table is missing: "
        f"{sorted(missing_support_columns)}"
    )


diagnosis_support_wide[
    "zero_in_mimic_training"
] = (
    diagnosis_support_wide[
        "mimiciv_train_positive_count"
    ].eq(0)
)

diagnosis_support_wide[
    "supported_1_to_4_in_mimic_training"
] = (
    diagnosis_support_wide[
        "mimiciv_train_positive_count"
    ].between(
        1,
        4,
        inclusive="both",
    )
)

diagnosis_support_wide[
    "supported_5_to_9_in_mimic_training"
] = (
    diagnosis_support_wide[
        "mimiciv_train_positive_count"
    ].between(
        5,
        9,
        inclusive="both",
    )
)

diagnosis_support_wide[
    "supported_at_least_10_in_mimic_training"
] = (
    diagnosis_support_wide[
        "mimiciv_train_positive_count"
    ].ge(10)
)

diagnosis_support_wide[
    "present_in_mcmed"
] = (
    diagnosis_support_wide[
        "mcmed_external_positive_count"
    ].gt(0)
)



diagnosis_inventory_summary = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen diagnosis features"
            ),
            "count": len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            ),
        },
        {
            "measure": (
                "Supported >=10 times in "
                "MIMIC training"
            ),
            "count": int(
                diagnosis_support_wide[
                    "supported_at_least_10_"
                    "in_mimic_training"
                ].sum()
            ),
        },
        {
            "measure": (
                "Supported 5-9 times in "
                "MIMIC training"
            ),
            "count": int(
                diagnosis_support_wide[
                    "supported_5_to_9_"
                    "in_mimic_training"
                ].sum()
            ),
        },
        {
            "measure": (
                "Supported 1-4 times in "
                "MIMIC training"
            ),
            "count": int(
                diagnosis_support_wide[
                    "supported_1_to_4_"
                    "in_mimic_training"
                ].sum()
            ),
        },
        {
            "measure": (
                "Zero in MIMIC training"
            ),
            "count": int(
                diagnosis_support_wide[
                    "zero_in_mimic_training"
                ].sum()
            ),
        },
        {
            "measure": (
                "Present at least once "
                "in MC-MED"
            ),
            "count": int(
                diagnosis_support_wide[
                    "present_in_mcmed"
                ].sum()
            ),
        },
    ]
)


print(
    "Frozen diagnosis inventory support:"
)

display(diagnosis_inventory_summary)


print(
    "\nLowest-support frozen diagnosis features:"
)

display(
    diagnosis_support_wide.sort_values(
        [
            "mimiciv_train_positive_count",
            "mcmed_external_positive_count",
        ],
        ascending=[
            True,
            True,
        ],
    ).head(30)
)



mimic_feature_order = [
    column
    for column
    in mimic_s1_diagnosis_features.columns
    if column.startswith(
        "pmh_icd_title__"
    )
]

mcmed_feature_order = [
    column
    for column
    in mcmed_s1_diagnosis_features.columns
    if column.startswith(
        "pmh_icd_title__"
    )
]


assert (
    mimic_feature_order
    ==
    FROZEN_PMH_DIAGNOSIS_FEATURES
)

assert (
    mcmed_feature_order
    ==
    FROZEN_PMH_DIAGNOSIS_FEATURES
)

assert mimic_feature_order == mcmed_feature_order



MIMIC_S1_DIAGNOSIS_PATH = (
    PROCESSED_DIR
    / (
        "mimiciv_s1_frozen_previous_"
        "diagnosis_features.parquet"
    )
)

MCMED_S1_DIAGNOSIS_PATH = (
    PROCESSED_DIR
    / (
        "mcmed_s1_frozen_previous_"
        "diagnosis_features.parquet"
    )
)


mimic_s1_diagnosis_features.to_parquet(
    MIMIC_S1_DIAGNOSIS_PATH,
    index=False,
)

mcmed_s1_diagnosis_features.to_parquet(
    MCMED_S1_DIAGNOSIS_PATH,
    index=False,
)



MIMIC_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH = (
    TABLE_DIR
    / (
        "mimiciv_frozen_diagnosis_"
        "reconstruction_audit.csv"
    )
)

MCMED_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH = (
    TABLE_DIR
    / (
        "mcmed_frozen_diagnosis_"
        "reconstruction_audit.csv"
    )
)

MCMED_DIAGNOSIS_MAPPING_AUDIT_PATH = (
    TABLE_DIR
    / (
        "mcmed_frozen_diagnosis_"
        "mapping_audit.csv"
    )
)

DIAGNOSIS_FEATURE_SUPPORT_PATH = (
    TABLE_DIR
    / (
        "frozen_diagnosis_feature_"
        "support_by_split.csv"
    )
)

DIAGNOSIS_SUPPORT_STATUS_PATH = (
    TABLE_DIR
    / (
        "frozen_diagnosis_feature_"
        "support_status.csv"
    )
)


mimic_diagnosis_reconstruction_audit.to_csv(
    MIMIC_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH,
    index=False,
)

mcmed_diagnosis_reconstruction_audit.to_csv(
    MCMED_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH,
    index=False,
)

mcmed_frozen_diagnosis_mapping_audit.to_csv(
    MCMED_DIAGNOSIS_MAPPING_AUDIT_PATH,
    index=False,
)

frozen_diagnosis_feature_support.to_csv(
    DIAGNOSIS_FEATURE_SUPPORT_PATH,
    index=False,
)

diagnosis_support_wide.to_csv(
    DIAGNOSIS_SUPPORT_STATUS_PATH,
    index=False,
)



diagnosis_output_paths = [
    MIMIC_S1_DIAGNOSIS_PATH,
    MCMED_S1_DIAGNOSIS_PATH,
    MIMIC_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH,
    MCMED_DIAGNOSIS_RECONSTRUCTION_AUDIT_PATH,
    MCMED_DIAGNOSIS_MAPPING_AUDIT_PATH,
    DIAGNOSIS_FEATURE_SUPPORT_PATH,
    DIAGNOSIS_SUPPORT_STATUS_PATH,
]


diagnosis_output_status = pd.DataFrame(
    [
        {
            "file": path.name,
            "exists": path.exists(),
            "size_mb": (
                round(
                    path.stat().st_size
                    / (1024 ** 2),
                    3,
                )
                if path.exists()
                else np.nan
            ),
            "path": str(path),
        }
        for path in diagnosis_output_paths
    ]
)


display(diagnosis_output_status)

assert diagnosis_output_status[
    "exists"
].all()


print(
    "\nFROZEN S1 DIAGNOSIS RECONSTRUCTION PASSED"
)

print(
    "MIMIC-IV rows:",
    f"{len(mimic_s1_diagnosis_features):,}",
)

print(
    "MC-MED rows:",
    f"{len(mcmed_s1_diagnosis_features):,}",
)

print(
    "Aligned frozen diagnosis features:",
    len(
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ),
)

print(
    "The exact clinician-reviewed 879-feature "
    "inventory has been preserved."
)


#### Frozen previous-medication feature block

This section reconstructs the exact 812 previous-medication-name features and 364 previous-medication-class features defined during the original MIMIC-IV development analysis and subsequently reviewed by clinicians.

For MIMIC-IV, home or previous medications are taken from medication reconciliation records belonging to the current eligible ED encounter. Only records documented between ED arrival and ED departure are used. Medication reconciliation represents the patient's recorded pre-encounter medication history rather than medication administered as current ED treatment.

For MC-MED, medication records are linked by patient identifier and restricted to medications recorded as active at the current ED arrival. A medication is eligible when:

- `Active` equals `Y`;
- `Entry_date` is missing or on/before current arrival;
- `Start_date` is missing or on/before current arrival;
- `End_date` is missing or on/after current arrival.

The frozen MIMIC-IV vocabulary determines which medication variables exist. MC-MED does not introduce new features.

External medication-name mapping follows a conservative hierarchy:

1. exact normalised medication name or generic name;
2. exact medication component;
3. exact NDC-based crosswalk where the NDC uniquely identifies one frozen feature;
4. unique canonical medication-name mapping after removal of dose, route and formulation text.

Medication-class mapping uses exact or unique canonical class matching. Automatic fuzzy matches are not accepted into the feature matrix. Unmatched and ambiguous medication descriptions are retained in audit tables for subsequent review.

Binary value 1 means that the frozen medication was identified in the available medication-history records. Binary value 0 means it was not identified; it does not prove that the patient was never prescribed that medication.


In [ ]:



required_objects = [
    "frozen_pmh_feature_lookup",
    "FROZEN_PMH_MEDICATION_NAME_FEATURES",
    "FROZEN_PMH_MEDICATION_CLASS_FEATURES",
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "mimic_split_source",
    "SOURCE_PATHS",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding cohort and frozen-vocabulary "
        f"cells first. Missing objects: {missing_objects}"
    )



MEDICATION_FEATURE_NAME_CANDIDATES = [
    "final_feature_name",
    "feature_name_cleaned",
    "feature_name",
]

frozen_medication_feature_name_column = next(
    (
        column
        for column in MEDICATION_FEATURE_NAME_CANDIDATES
        if column in frozen_pmh_feature_lookup.columns
    ),
    None,
)

if frozen_medication_feature_name_column is None:
    raise ValueError(
        "No recognised feature-name column exists in the "
        "frozen PMH lookup."
    )

if "normalised_item" not in frozen_pmh_feature_lookup.columns:
    raise ValueError(
        "The frozen PMH lookup is missing normalised_item."
    )



def legacy_normalise_medication_item(
    value,
) -> str:
    """
    Reproduce the original MIMIC medication normalisation.

    Medication names and classes preserve dose and formulation.
    Only case and whitespace are standardised.
    """

    if pd.isna(value):
        return ""

    text = str(value).replace(
        "\xa0",
        " ",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip().upper()

    if text in {
        "",
        "NAN",
        "NONE",
        "NULL",
        "MISSING",
        "UNKNOWN",
    }:
        return ""

    return text



medication_lookup_working = (
    frozen_pmh_feature_lookup.copy()
)

medication_lookup_working[
    "_resolved_feature_name"
] = (
    medication_lookup_working[
        frozen_medication_feature_name_column
    ]
    .astype("string")
    .str.strip()
)

medication_lookup_working[
    "_resolved_normalised_item"
] = (
    medication_lookup_working[
        "normalised_item"
    ].apply(
        legacy_normalise_medication_item
    )
)


def create_frozen_medication_lookup(
    ordered_features: list[str],
    expected_prefix: str,
) -> pd.DataFrame:
    """
    Recover one frozen medication family without duplicate
    column-name collisions.
    """

    ordered_feature_set = set(
        ordered_features
    )

    working = (
        medication_lookup_working.loc[
            medication_lookup_working[
                "_resolved_feature_name"
            ].isin(
                ordered_feature_set
            )
        ]
        .copy()
    )

    result = pd.DataFrame(
        {
            "feature_name": (
                working[
                    "_resolved_feature_name"
                ]
                .astype(str)
                .to_numpy()
            ),
            "normalised_item": (
                working[
                    "_resolved_normalised_item"
                ]
                .astype(str)
                .to_numpy()
            ),
        }
    )

    for optional_column in [
        "source_column",
        "source_group",
        "n_reference_stays_with_item",
        "pct_reference_stays_with_item",
    ]:
        if optional_column in working.columns:
            result[
                optional_column
            ] = working[
                optional_column
            ].to_numpy()

    result = (
        result
        .drop_duplicates(
            subset=[
                "feature_name",
                "normalised_item",
            ]
        )
        .reset_index(drop=True)
    )

    conflicting_feature_counts = (
        result
        .groupby("feature_name")[
            "normalised_item"
        ]
        .nunique()
    )

    if conflicting_feature_counts.gt(1).any():
        conflicting_names = (
            conflicting_feature_counts.loc[
                conflicting_feature_counts.gt(1)
            ].index.tolist()
        )

        display(
            result.loc[
                result[
                    "feature_name"
                ].isin(
                    conflicting_names
                )
            ]
        )

        raise ValueError(
            "A frozen medication feature maps to multiple "
            "different canonical items."
        )

    result = (
        result
        .drop_duplicates(
            subset="feature_name"
        )
        .set_index("feature_name")
    )

    missing_features = (
        ordered_feature_set
        - set(result.index)
    )

    if missing_features:
        display(
            pd.DataFrame(
                {
                    "missing_feature": sorted(
                        missing_features
                    )
                }
            ).head(50)
        )

        raise ValueError(
            f"{len(missing_features)} frozen medication "
            "features were not found in the detailed lookup."
        )

    result = (
        result
        .loc[ordered_features]
        .reset_index()
    )

    result.insert(
        0,
        "family_position",
        range(
            1,
            len(result) + 1,
        ),
    )

    assert result[
        "feature_name"
    ].str.startswith(
        expected_prefix
    ).all()

    assert result[
        "feature_name"
    ].is_unique

    assert result[
        "normalised_item"
    ].ne("").all()

    assert result[
        "normalised_item"
    ].is_unique

    return result


frozen_medication_name_lookup = (
    create_frozen_medication_lookup(
        ordered_features=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
        expected_prefix="pmh_med_name__",
    )
)

frozen_medication_class_lookup = (
    create_frozen_medication_lookup(
        ordered_features=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
        expected_prefix="pmh_med_class__",
    )
)



assert len(
    frozen_medication_name_lookup
) == 812

assert len(
    frozen_medication_class_lookup
) == 364

assert (
    frozen_medication_name_lookup[
        "feature_name"
    ].tolist()
    ==
    FROZEN_PMH_MEDICATION_NAME_FEATURES
)

assert (
    frozen_medication_class_lookup[
        "feature_name"
    ].tolist()
    ==
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


frozen_medication_lookup_audit = pd.DataFrame(
    [
        {
            "feature_family": (
                "Previous medication names"
            ),
            "expected_features": 812,
            "recovered_features": len(
                frozen_medication_name_lookup
            ),
            "unique_items": (
                frozen_medication_name_lookup[
                    "normalised_item"
                ].nunique()
            ),
        },
        {
            "feature_family": (
                "Previous medication classes"
            ),
            "expected_features": 364,
            "recovered_features": len(
                frozen_medication_class_lookup
            ),
            "unique_items": (
                frozen_medication_class_lookup[
                    "normalised_item"
                ].nunique()
            ),
        },
    ]
)


print(
    "Authoritative feature-name column:",
    frozen_medication_feature_name_column,
)

print(
    "\nFrozen medication lookup audit:"
)

display(frozen_medication_lookup_audit)


print(
    "\nExample frozen medication-name mappings:"
)

display(
    frozen_medication_name_lookup[
        [
            "family_position",
            "normalised_item",
            "feature_name",
        ]
    ].head(15)
)


print(
    "\nExample frozen medication-class mappings:"
)

display(
    frozen_medication_class_lookup[
        [
            "family_position",
            "normalised_item",
            "feature_name",
        ]
    ].head(15)
)


print(
    "\nFROZEN 1,176-FEATURE MEDICATION LOOKUP PASSED"
)


#### MIMIC-IV medication features


In [ ]:


MIMIC_MEDRECON_PATH = SOURCE_PATHS[
    "mimic_medrecon"
]

mimic_medrecon_header = pd.read_csv(
    MIMIC_MEDRECON_PATH,
    nrows=0,
).columns.tolist()


required_mimic_medrecon_columns = [
    "subject_id",
    "stay_id",
    "charttime",
    "name",
    "etcdescription",
]

missing_mimic_medrecon_columns = [
    column
    for column in required_mimic_medrecon_columns
    if column not in mimic_medrecon_header
]

if missing_mimic_medrecon_columns:
    raise ValueError(
        "MIMIC medrecon is missing required columns: "
        f"{missing_mimic_medrecon_columns}"
    )


mimic_medrecon_columns_to_load = [
    column
    for column in [
        "subject_id",
        "stay_id",
        "charttime",
        "name",
        "gsn",
        "ndc",
        "etccode",
        "etcdescription",
    ]
    if column in mimic_medrecon_header
]


mimic_medrecon_source = pd.read_csv(
    MIMIC_MEDRECON_PATH,
    usecols=mimic_medrecon_columns_to_load,
    low_memory=False,
)



for identifier_column in [
    "subject_id",
    "stay_id",
]:
    mimic_medrecon_source[
        identifier_column
    ] = pd.to_numeric(
        mimic_medrecon_source[
            identifier_column
        ],
        errors="coerce",
    ).astype("Int64")


def convert_mimic_datetime(
    series: pd.Series,
) -> pd.Series:
    """Create comparable timezone-naive MIMIC timestamps."""

    return (
        pd.to_datetime(
            series,
            errors="coerce",
            utc=True,
        )
        .dt.tz_convert(None)
    )


mimic_medrecon_source[
    "charttime"
] = convert_mimic_datetime(
    mimic_medrecon_source[
        "charttime"
    ]
)


mimic_medication_windows = (
    mimic_split_source[
        [
            "subject_id",
            "stay_id",
            "intime",
            "outtime",
            "split",
        ]
    ]
    .copy()
)


mimic_medication_windows[
    "intime"
] = convert_mimic_datetime(
    mimic_medication_windows[
        "intime"
    ]
)

mimic_medication_windows[
    "outtime"
] = convert_mimic_datetime(
    mimic_medication_windows[
        "outtime"
    ]
)



mimic_medrecon_linked = (
    mimic_medrecon_source.merge(
        mimic_medication_windows,
        on=[
            "subject_id",
            "stay_id",
        ],
        how="inner",
        validate="many_to_one",
    )
)


mimic_medrecon_linked[
    "inside_current_ed_window"
] = (
    mimic_medrecon_linked[
        "charttime"
    ].notna()
    &
    mimic_medrecon_linked[
        "charttime"
    ].ge(
        mimic_medrecon_linked[
            "intime"
        ]
    )
    &
    mimic_medrecon_linked[
        "charttime"
    ].le(
        mimic_medrecon_linked[
            "outtime"
        ]
    )
)


mimic_medrecon_eligible = (
    mimic_medrecon_linked.loc[
        mimic_medrecon_linked[
            "inside_current_ed_window"
        ]
    ]
    .copy()
)



medication_name_exact_map = dict(
    zip(
        frozen_medication_name_lookup[
            "normalised_item"
        ],
        frozen_medication_name_lookup[
            "feature_name"
        ],
    )
)

medication_class_exact_map = dict(
    zip(
        frozen_medication_class_lookup[
            "normalised_item"
        ],
        frozen_medication_class_lookup[
            "feature_name"
        ],
    )
)


mimic_medrecon_eligible[
    "medication_name_normalised"
] = (
    mimic_medrecon_eligible[
        "name"
    ].apply(
        legacy_normalise_medication_item
    )
)

mimic_medrecon_eligible[
    "medication_class_normalised"
] = (
    mimic_medrecon_eligible[
        "etcdescription"
    ].apply(
        legacy_normalise_medication_item
    )
)


mimic_medrecon_eligible[
    "medication_name_feature"
] = (
    mimic_medrecon_eligible[
        "medication_name_normalised"
    ].map(
        medication_name_exact_map
    )
)

mimic_medrecon_eligible[
    "medication_class_feature"
] = (
    mimic_medrecon_eligible[
        "medication_class_normalised"
    ].map(
        medication_class_exact_map
    )
)



mimic_medication_name_positive_long = (
    mimic_medrecon_eligible.loc[
        mimic_medrecon_eligible[
            "medication_name_feature"
        ].notna(),
        [
            "subject_id",
            "stay_id",
            "medication_name_feature",
        ],
    ]
    .rename(
        columns={
            "medication_name_feature":
                "feature_name"
        }
    )
    .drop_duplicates(
        subset=[
            "stay_id",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)


mimic_medication_class_positive_long = (
    mimic_medrecon_eligible.loc[
        mimic_medrecon_eligible[
            "medication_class_feature"
        ].notna(),
        [
            "subject_id",
            "stay_id",
            "medication_class_feature",
        ],
    ]
    .rename(
        columns={
            "medication_class_feature":
                "feature_name"
        }
    )
    .drop_duplicates(
        subset=[
            "stay_id",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mimic_medication_name_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mimic_medication_name_positive_long
        ),
        identifier_column="stay_id",
        feature_column="feature_name",
        all_identifiers=(
            mimic_s0_validated[
                "stay_id"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


mimic_medication_class_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mimic_medication_class_positive_long
        ),
        identifier_column="stay_id",
        feature_column="feature_name",
        all_identifiers=(
            mimic_s0_validated[
                "stay_id"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
    )
)


mimic_s1_medication_features = (
    mimic_s0_validated[
        [
            "subject_id",
            "stay_id",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mimic_medication_name_wide,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mimic_medication_class_wide,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
)


MIMIC_FROZEN_MEDICATION_FEATURES = (
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


for feature_name in (
    MIMIC_FROZEN_MEDICATION_FEATURES
):
    mimic_s1_medication_features[
        feature_name
    ] = (
        mimic_s1_medication_features[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )



assert len(
    mimic_s1_medication_features
) == 8550

assert mimic_s1_medication_features[
    "stay_id"
].is_unique

observed_mimic_medication_columns = [
    column
    for column
    in mimic_s1_medication_features.columns
    if (
        column.startswith(
            "pmh_med_name__"
        )
        or
        column.startswith(
            "pmh_med_class__"
        )
    )
]

assert (
    observed_mimic_medication_columns
    ==
    MIMIC_FROZEN_MEDICATION_FEATURES
)

assert len(
    observed_mimic_medication_columns
) == 1176

assert set(
    np.unique(
        mimic_s1_medication_features[
            MIMIC_FROZEN_MEDICATION_FEATURES
        ].to_numpy()
    ).tolist()
).issubset(
    {
        0,
        1,
    }
)



mimic_medication_reconstruction_audit = pd.DataFrame(
    [
        {
            "measure": (
                "Frozen MIMIC cohort encounters"
            ),
            "value": len(
                mimic_s1_medication_features
            ),
        },
        {
            "measure": (
                "Medrecon source rows linked "
                "to cohort encounters"
            ),
            "value": len(
                mimic_medrecon_linked
            ),
        },
        {
            "measure": (
                "Medrecon rows inside current "
                "ED window"
            ),
            "value": len(
                mimic_medrecon_eligible
            ),
        },
        {
            "measure": (
                "Encounters with >=1 eligible "
                "medrecon row"
            ),
            "value": (
                mimic_medrecon_eligible[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "medication name"
            ),
            "value": (
                mimic_medication_name_positive_long[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "medication class"
            ),
            "value": (
                mimic_medication_class_positive_long[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Frozen medication-name columns"
            ),
            "value": 812,
        },
        {
            "measure": (
                "Medication-name columns with "
                ">=1 positive value"
            ),
            "value": int(
                (
                    mimic_s1_medication_features[
                        FROZEN_PMH_MEDICATION_NAME_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
        {
            "measure": (
                "Frozen medication-class columns"
            ),
            "value": 364,
        },
        {
            "measure": (
                "Medication-class columns with "
                ">=1 positive value"
            ),
            "value": int(
                (
                    mimic_s1_medication_features[
                        FROZEN_PMH_MEDICATION_CLASS_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
    ]
)


print(
    "MIMIC-IV medication reconstruction audit:"
)

display(mimic_medication_reconstruction_audit)


print(
    "\nMIMIC-IV medication matrix shape:",
    mimic_s1_medication_features.shape,
)

print(
    "Aligned medication features:",
    len(
        observed_mimic_medication_columns
    ),
)


#### MC-MED active-at-arrival medication records


In [ ]:


for required_helper in [
    "parse_datetime_to_seconds",
    "MISSING_DATE_STRINGS",
    "mcmed_primary_history_source",
]:
    if required_helper not in globals():
        raise NameError(
            f"{required_helper} is missing. "
            "Run Cell 68 before this section."
        )



MCMED_MEDS_PATH = SOURCE_PATHS[
    "mcmed_meds"
]

mcmed_meds_header = pd.read_csv(
    MCMED_MEDS_PATH,
    nrows=0,
).columns.tolist()


required_mcmed_meds_columns = [
    "MRN",
    "Active",
]

missing_mcmed_meds_columns = [
    column
    for column in required_mcmed_meds_columns
    if column not in mcmed_meds_header
]

if missing_mcmed_meds_columns:
    raise ValueError(
        "MC-MED meds.csv is missing required columns: "
        f"{missing_mcmed_meds_columns}"
    )


mcmed_meds_columns_to_load = [
    column
    for column in [
        "MRN",
        "Med_ID",
        "NDC",
        "Name",
        "Generic_name",
        "Med_class",
        "Med_subclass",
        "Active",
        "Entry_date",
        "Start_date",
        "End_date",
    ]
    if column in mcmed_meds_header
]


mcmed_meds_source = pd.read_csv(
    MCMED_MEDS_PATH,
    usecols=mcmed_meds_columns_to_load,
    dtype={
        date_column: "string"
        for date_column in [
            "Entry_date",
            "Start_date",
            "End_date",
        ]
        if date_column in mcmed_meds_header
    },
    low_memory=False,
)


for optional_column in [
    "Med_ID",
    "NDC",
    "Name",
    "Generic_name",
    "Med_class",
    "Med_subclass",
    "Entry_date",
    "Start_date",
    "End_date",
]:
    if optional_column not in mcmed_meds_source.columns:
        mcmed_meds_source[
            optional_column
        ] = pd.NA


mcmed_meds_source["MRN"] = pd.to_numeric(
    mcmed_meds_source["MRN"],
    errors="coerce",
).astype("Int64")



mcmed_medication_date_audit_records = []

for source_date_column in [
    "Entry_date",
    "Start_date",
    "End_date",
]:
    parsed_column = (
        source_date_column
        + "_parsed"
    )

    (
        mcmed_meds_source[
            parsed_column
        ],
        parse_failure_table,
    ) = parse_datetime_to_seconds(
        mcmed_meds_source[
            source_date_column
        ]
    )

    raw_values = (
        mcmed_meds_source[
            source_date_column
        ]
        .astype("string")
        .str.strip()
    )

    source_missing_mask = (
        raw_values.isna()
        |
        raw_values.str.lower().isin(
            MISSING_DATE_STRINGS
        )
    )

    parse_failure_mask = (
        ~source_missing_mask
        &
        mcmed_meds_source[
            parsed_column
        ].isna()
    )

    mcmed_medication_date_audit_records.append(
        {
            "date_field": source_date_column,
            "source_rows": len(
                mcmed_meds_source
            ),
            "raw_missing_n": int(
                source_missing_mask.sum()
            ),
            "parsed_non_missing_n": int(
                mcmed_meds_source[
                    parsed_column
                ].notna().sum()
            ),
            "non_missing_parse_failure_n": int(
                parse_failure_mask.sum()
            ),
            "earliest_parsed": (
                mcmed_meds_source[
                    parsed_column
                ].min()
            ),
            "latest_parsed": (
                mcmed_meds_source[
                    parsed_column
                ].max()
            ),
        }
    )

    if parse_failure_mask.any():
        display(
            pd.DataFrame(
                {
                    "raw_value": (
                        mcmed_meds_source.loc[
                            parse_failure_mask,
                            source_date_column,
                        ].head(25)
                    )
                }
            )
        )

        raise ValueError(
            f"Non-missing {source_date_column} "
            "values failed parsing."
        )


mcmed_medication_date_audit = (
    pd.DataFrame(
        mcmed_medication_date_audit_records
    )
)


print(
    "MC-MED medication date parsing audit:"
)

display(mcmed_medication_date_audit)



mcmed_medication_current_cohort = (
    mcmed_primary_history_source[
        [
            "MRN",
            "CSN",
            "current_arrival_time",
        ]
    ]
    .copy()
)


mcmed_visit_medications = (
    mcmed_medication_current_cohort.merge(
        mcmed_meds_source,
        on="MRN",
        how="left",
        validate="one_to_many",
        indicator="medication_merge_status",
    )
)


mcmed_visit_medications[
    "medication_record_present"
] = (
    mcmed_visit_medications[
        "medication_merge_status"
    ].eq("both")
)



active_status_clean = (
    mcmed_visit_medications[
        "Active"
    ]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)


active_y = (
    active_status_clean.eq("Y")
    .fillna(False)
    .astype(bool)
)


medication_record_present = (
    mcmed_visit_medications[
        "medication_record_present"
    ]
    .fillna(False)
    .astype(bool)
)


arrival_time = (
    mcmed_visit_medications[
        "current_arrival_time"
    ]
)

entry_time = (
    mcmed_visit_medications[
        "Entry_date_parsed"
    ]
)

start_time = (
    mcmed_visit_medications[
        "Start_date_parsed"
    ]
)

end_time = (
    mcmed_visit_medications[
        "End_date_parsed"
    ]
)



entry_eligible = (
    entry_time.isna()
    |
    entry_time.le(
        arrival_time
    )
).fillna(False).astype(bool)


start_eligible = (
    start_time.isna()
    |
    start_time.le(
        arrival_time
    )
).fillna(False).astype(bool)


end_eligible = (
    end_time.isna()
    |
    end_time.ge(
        arrival_time
    )
).fillna(False).astype(bool)


entry_after_arrival = (
    entry_time.notna()
    &
    entry_time.gt(
        arrival_time
    )
).fillna(False).astype(bool)


start_after_arrival = (
    start_time.notna()
    &
    start_time.gt(
        arrival_time
    )
).fillna(False).astype(bool)


ended_before_arrival = (
    end_time.notna()
    &
    end_time.lt(
        arrival_time
    )
).fillna(False).astype(bool)


strict_active_at_arrival = (
    medication_record_present
    &
    active_y
    &
    entry_eligible
    &
    start_eligible
    &
    end_eligible
).fillna(False).astype(bool)



temporal_conditions = [
    (
        ~medication_record_present
    ).to_numpy(
        dtype=bool
    ),

    strict_active_at_arrival.to_numpy(
        dtype=bool
    ),

    (
        medication_record_present
        &
        ~active_y
    ).fillna(False).astype(bool).to_numpy(
        dtype=bool
    ),

    (
        medication_record_present
        &
        active_y
        &
        entry_after_arrival
    ).fillna(False).astype(bool).to_numpy(
        dtype=bool
    ),

    (
        medication_record_present
        &
        active_y
        &
        entry_eligible
        &
        start_after_arrival
    ).fillna(False).astype(bool).to_numpy(
        dtype=bool
    ),

    (
        medication_record_present
        &
        active_y
        &
        entry_eligible
        &
        start_eligible
        &
        ended_before_arrival
    ).fillna(False).astype(bool).to_numpy(
        dtype=bool
    ),
]


temporal_choices = [
    "NO_MEDICATION_RECORD",
    "STRICT_ACTIVE_AT_ARRIVAL",
    "NOT_MARKED_ACTIVE",
    "ENTRY_AFTER_CURRENT_ARRIVAL",
    "START_AFTER_CURRENT_ARRIVAL",
    "ENDED_BEFORE_CURRENT_ARRIVAL",
]


mcmed_visit_medications[
    "medication_temporal_status"
] = np.select(
    temporal_conditions,
    temporal_choices,
    default="OTHER_NOT_ELIGIBLE",
)



assert strict_active_at_arrival.dtype == bool

assert len(
    strict_active_at_arrival
) == len(
    mcmed_visit_medications
)

assert mcmed_visit_medications[
    "medication_temporal_status"
].notna().all()


assert (
    mcmed_visit_medications.loc[
        strict_active_at_arrival,
        "medication_temporal_status",
    ]
    .eq(
        "STRICT_ACTIVE_AT_ARRIVAL"
    )
    .all()
)


mcmed_active_field_audit = (
    active_status_clean
    .replace(
        "",
        "<MISSING>",
    )
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "active_source_value"
    )
    .reset_index(
        name="n_rows"
    )
)


mcmed_active_field_audit[
    "percentage"
] = (
    mcmed_active_field_audit[
        "n_rows"
    ]
    / len(
        active_status_clean
    )
    * 100
)


print(
    "MC-MED medication Active-field audit:"
)

display(
    mcmed_active_field_audit.round(
        {
            "percentage": 2,
        }
    )
)



has_medication_identity = (
    mcmed_visit_medications[
        [
            "Name",
            "Generic_name",
            "NDC",
            "Med_class",
            "Med_subclass",
        ]
    ]
    .notna()
    .any(axis=1)
)


mcmed_active_medications_strict = (
    mcmed_visit_medications.loc[
        strict_active_at_arrival
        &
        has_medication_identity
    ]
    .copy()
)



mcmed_medication_temporal_audit = (
    mcmed_visit_medications[
        "medication_temporal_status"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "medication_temporal_status"
    )
    .reset_index(
        name="n_linked_rows"
    )
)


mcmed_medication_temporal_audit[
    "percentage_of_linked_rows"
] = (
    mcmed_medication_temporal_audit[
        "n_linked_rows"
    ]
    / len(
        mcmed_visit_medications
    )
    * 100
)


mcmed_medication_encounter_audit = (
    mcmed_visit_medications
    .groupby(
        "medication_temporal_status",
        dropna=False,
    )["CSN"]
    .nunique()
    .rename(
        "n_unique_encounters"
    )
    .reset_index()
)


mcmed_medication_encounter_audit[
    "percentage_of_cohort_encounters"
] = (
    mcmed_medication_encounter_audit[
        "n_unique_encounters"
    ]
    / len(
        mcmed_medication_current_cohort
    )
    * 100
)


print(
    "MC-MED medication linked-row temporal audit:"
)

display(
    mcmed_medication_temporal_audit.round(
        {
            "percentage_of_linked_rows": 2,
        }
    )
)


print(
    "\nMC-MED medication encounter-level audit:"
)

display(
    mcmed_medication_encounter_audit.round(
        {
            "percentage_of_cohort_encounters": 2,
        }
    )
)


print(
    "\nStrict active-at-arrival medication rows:",
    f"{len(mcmed_active_medications_strict):,}",
)

print(
    "Encounters with >=1 strict active medication:",
    f"{mcmed_active_medications_strict['CSN'].nunique():,}",
)


#### High-confidence MC-MED medication mapping


In [ ]:



MEDICATION_FORM_TERMS = {
    "TABLET",
    "TABLETS",
    "TAB",
    "TABS",
    "CAPSULE",
    "CAPSULES",
    "CAP",
    "CAPS",
    "SOLUTION",
    "SUSPENSION",
    "SYRUP",
    "INJECTION",
    "INJECTABLE",
    "CREAM",
    "OINTMENT",
    "GEL",
    "PATCH",
    "POWDER",
    "SPRAY",
    "DROP",
    "DROPS",
    "ORAL",
    "PO",
    "IV",
    "INTRAVENOUS",
    "IM",
    "SUBCUTANEOUS",
    "SC",
    "CHEWABLE",
    "DISINTEGRATING",
    "ENTERIC",
    "COATED",
    "DAILY",
    "BID",
    "TID",
    "QID",
    "QHS",
    "PRN",
}


def canonical_medication_key(
    value,
) -> str:
    """
    Create a conservative dose/form-independent medication key.

    This key is used only when it maps to exactly one frozen
    medication feature.
    """

    text = legacy_normalise_medication_item(
        value
    )

    if not text:
        return ""

    text = text.replace(
        "&",
        " AND ",
    )

    text = re.sub(
        r"\([^)]*\)",
        " ",
        text,
    )

    text = re.sub(
        (
            r"\b\d+(?:\.\d+)?\s*"
            r"(?:MG|MCG|G|GRAMS?|ML|UNITS?|IU|MEQ|%)\b"
        ),
        " ",
        text,
    )

    text = re.sub(
        r"\b\d+(?:\.\d+)?\b",
        " ",
        text,
    )

    text = re.sub(
        r"[^A-Z0-9]+",
        " ",
        text,
    )

    tokens = [
        token
        for token in text.split()
        if token not in MEDICATION_FORM_TERMS
    ]

    return " ".join(tokens).strip()


def canonical_medication_class_key(
    value,
) -> str:
    """Create a conservative medication-class text key."""

    text = legacy_normalise_medication_item(
        value
    )

    if not text:
        return ""

    text = text.replace(
        "&",
        " AND ",
    )

    text = re.sub(
        r"[^A-Z0-9]+",
        " ",
        text,
    )

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def medication_text_candidates(
    *values,
) -> list[str]:
    """
    Return full medication strings and exact delimiter-separated
    components without applying fuzzy matching.
    """

    candidates = []

    for value in values:
        normalised_full = (
            legacy_normalise_medication_item(
                value
            )
        )

        if not normalised_full:
            continue

        candidates.append(
            normalised_full
        )

        for component in re.split(
            r"[;/+]",
            normalised_full,
        ):
            component = (
                legacy_normalise_medication_item(
                    component
                )
            )

            if component:
                candidates.append(
                    component
                )

    return list(
        dict.fromkeys(
            candidates
        )
    )


def normalise_ndc(
    value,
) -> str:
    """
    Create a conservative numeric NDC key.

    Leading zeros are padded to 11 digits where possible.
    """

    if pd.isna(value):
        return ""

    text = str(value).strip()

    text = re.sub(
        r"\.0+$",
        "",
        text,
    )

    digits = re.sub(
        r"[^0-9]",
        "",
        text,
    )

    if not digits:
        return ""

    digits = digits.lstrip("0") or "0"

    if len(digits) <= 11:
        digits = digits.zfill(11)

    return digits



frozen_name_exact_map = dict(
    zip(
        frozen_medication_name_lookup[
            "normalised_item"
        ],
        frozen_medication_name_lookup[
            "feature_name"
        ],
    )
)

frozen_class_exact_map = dict(
    zip(
        frozen_medication_class_lookup[
            "normalised_item"
        ],
        frozen_medication_class_lookup[
            "feature_name"
        ],
    )
)


frozen_name_canonical_groups = defaultdict(
    list
)

for _, row in (
    frozen_medication_name_lookup.iterrows()
):
    key = canonical_medication_key(
        row["normalised_item"]
    )

    if key:
        frozen_name_canonical_groups[
            key
        ].append(
            row["feature_name"]
        )


frozen_class_canonical_groups = defaultdict(
    list
)

for _, row in (
    frozen_medication_class_lookup.iterrows()
):
    key = canonical_medication_class_key(
        row["normalised_item"]
    )

    if key:
        frozen_class_canonical_groups[
            key
        ].append(
            row["feature_name"]
        )


frozen_name_unique_canonical_map = {
    key: features[0]
    for key, features
    in frozen_name_canonical_groups.items()
    if len(set(features)) == 1
}

frozen_class_unique_canonical_map = {
    key: features[0]
    for key, features
    in frozen_class_canonical_groups.items()
    if len(set(features)) == 1
}



mimic_ndc_mapping_source = (
    mimic_medrecon_source.copy()
)

mimic_ndc_mapping_source[
    "medication_name_normalised"
] = (
    mimic_ndc_mapping_source[
        "name"
    ].apply(
        legacy_normalise_medication_item
    )
)

mimic_ndc_mapping_source[
    "feature_name"
] = (
    mimic_ndc_mapping_source[
        "medication_name_normalised"
    ].map(
        frozen_name_exact_map
    )
)

mimic_ndc_mapping_source[
    "ndc_key"
] = (
    mimic_ndc_mapping_source[
        "ndc"
    ].apply(
        normalise_ndc
    )
)


mimic_ndc_feature_pairs = (
    mimic_ndc_mapping_source.loc[
        mimic_ndc_mapping_source[
            "feature_name"
        ].notna()
        &
        mimic_ndc_mapping_source[
            "ndc_key"
        ].ne(""),
        [
            "ndc_key",
            "feature_name",
        ],
    ]
    .drop_duplicates()
)


ndc_feature_counts = (
    mimic_ndc_feature_pairs
    .groupby("ndc_key")[
        "feature_name"
    ]
    .nunique()
)


unique_ndc_keys = set(
    ndc_feature_counts.loc[
        ndc_feature_counts.eq(1)
    ].index
)


frozen_name_ndc_map = (
    mimic_ndc_feature_pairs.loc[
        mimic_ndc_feature_pairs[
            "ndc_key"
        ].isin(
            unique_ndc_keys
        )
    ]
    .set_index("ndc_key")[
        "feature_name"
    ]
    .to_dict()
)



def map_mcmed_medication_record(
    row: pd.Series,
) -> dict:
    """
    Map one unique MC-MED medication record without automatic
    fuzzy matching.
    """

    name_candidates = (
        medication_text_candidates(
            row.get(
                "Generic_name"
            ),
            row.get(
                "Name"
            ),
        )
    )


    exact_name_features = {
        frozen_name_exact_map[
            candidate
        ]
        for candidate in name_candidates
        if candidate
        in frozen_name_exact_map
    }

    if exact_name_features:
        name_features = sorted(
            exact_name_features
        )

        name_method = (
            "EXACT_NAME_OR_COMPONENT"
        )

    else:
        ndc_key = normalise_ndc(
            row.get("NDC")
        )

        if ndc_key in frozen_name_ndc_map:
            name_features = [
                frozen_name_ndc_map[
                    ndc_key
                ]
            ]

            name_method = (
                "UNIQUE_NDC_CROSSWALK"
            )

        else:
            canonical_name_features = set()

            ambiguous_name_keys = []

            for candidate in name_candidates:
                canonical_key = (
                    canonical_medication_key(
                        candidate
                    )
                )

                if (
                    canonical_key
                    in
                    frozen_name_unique_canonical_map
                ):
                    canonical_name_features.add(
                        frozen_name_unique_canonical_map[
                            canonical_key
                        ]
                    )

                elif (
                    canonical_key
                    and canonical_key
                    in frozen_name_canonical_groups
                ):
                    ambiguous_name_keys.append(
                        canonical_key
                    )

            if canonical_name_features:
                name_features = sorted(
                    canonical_name_features
                )

                name_method = (
                    "UNIQUE_CANONICAL_NAME"
                )

            elif ambiguous_name_keys:
                name_features = []

                name_method = (
                    "AMBIGUOUS_CANONICAL_NAME"
                )

            else:
                name_features = []

                name_method = (
                    "UNMATCHED_NAME"
                )


    subclass_value = (
        legacy_normalise_medication_item(
            row.get(
                "Med_subclass"
            )
        )
    )

    class_value = (
        legacy_normalise_medication_item(
            row.get(
                "Med_class"
            )
        )
    )

    class_features = []
    class_method = (
        "UNMATCHED_CLASS"
    )

    if (
        subclass_value
        in frozen_class_exact_map
    ):
        class_features = [
            frozen_class_exact_map[
                subclass_value
            ]
        ]

        class_method = (
            "EXACT_MED_SUBCLASS"
        )

    elif (
        class_value
        in frozen_class_exact_map
    ):
        class_features = [
            frozen_class_exact_map[
                class_value
            ]
        ]

        class_method = (
            "EXACT_MED_CLASS"
        )

    else:
        subclass_key = (
            canonical_medication_class_key(
                subclass_value
            )
        )

        class_key = (
            canonical_medication_class_key(
                class_value
            )
        )

        if (
            subclass_key
            in
            frozen_class_unique_canonical_map
        ):
            class_features = [
                frozen_class_unique_canonical_map[
                    subclass_key
                ]
            ]

            class_method = (
                "UNIQUE_CANONICAL_SUBCLASS"
            )

        elif (
            class_key
            in
            frozen_class_unique_canonical_map
        ):
            class_features = [
                frozen_class_unique_canonical_map[
                    class_key
                ]
            ]

            class_method = (
                "UNIQUE_CANONICAL_CLASS"
            )

        elif (
            subclass_key
            in frozen_class_canonical_groups
            or
            class_key
            in frozen_class_canonical_groups
        ):
            class_method = (
                "AMBIGUOUS_CANONICAL_CLASS"
            )

    return {
        "mapped_name_features": (
            name_features
        ),
        "name_match_method": (
            name_method
        ),
        "mapped_class_features": (
            class_features
        ),
        "class_match_method": (
            class_method
        ),
        "n_name_features": len(
            name_features
        ),
        "n_class_features": len(
            class_features
        ),
    }



mcmed_medication_identity_columns = [
    "NDC",
    "Name",
    "Generic_name",
    "Med_class",
    "Med_subclass",
]


mcmed_unique_medication_records = (
    mcmed_active_medications_strict[
        mcmed_medication_identity_columns
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


mcmed_medication_mapping_records = []

for mapping_key, row in (
    mcmed_unique_medication_records.iterrows()
):
    result = map_mcmed_medication_record(
        row
    )

    mcmed_medication_mapping_records.append(
        {
            "medication_mapping_key": (
                mapping_key
            ),
            **{
                column: row[column]
                for column
                in mcmed_medication_identity_columns
            },
            **result,
        }
    )


mcmed_medication_mapping_audit = pd.DataFrame(
    mcmed_medication_mapping_records
)


mcmed_unique_medication_records[
    "medication_mapping_key"
] = range(
    len(
        mcmed_unique_medication_records
    )
)



mcmed_active_medications_mapped = (
    mcmed_active_medications_strict.merge(
        mcmed_unique_medication_records,
        on=mcmed_medication_identity_columns,
        how="left",
        validate="many_to_one",
    )
    .merge(
        mcmed_medication_mapping_audit[
            [
                "medication_mapping_key",
                "mapped_name_features",
                "name_match_method",
                "mapped_class_features",
                "class_match_method",
                "n_name_features",
                "n_class_features",
            ]
        ],
        on="medication_mapping_key",
        how="left",
        validate="many_to_one",
    )
)



mcmed_medication_name_positive_long = (
    mcmed_active_medications_mapped[
        [
            "MRN",
            "CSN",
            "mapped_name_features",
        ]
    ]
    .explode(
        "mapped_name_features"
    )
    .rename(
        columns={
            "mapped_name_features":
                "feature_name"
        }
    )
)


mcmed_medication_name_positive_long = (
    mcmed_medication_name_positive_long.loc[
        mcmed_medication_name_positive_long[
            "feature_name"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)


mcmed_medication_class_positive_long = (
    mcmed_active_medications_mapped[
        [
            "MRN",
            "CSN",
            "mapped_class_features",
        ]
    ]
    .explode(
        "mapped_class_features"
    )
    .rename(
        columns={
            "mapped_class_features":
                "feature_name"
        }
    )
)


mcmed_medication_class_positive_long = (
    mcmed_medication_class_positive_long.loc[
        mcmed_medication_class_positive_long[
            "feature_name"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mcmed_medication_name_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mcmed_medication_name_positive_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated[
                "CSN"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


mcmed_medication_class_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mcmed_medication_class_positive_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated[
                "CSN"
            ]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
    )
)


mcmed_s1_medication_features_provisional = (
    mcmed_s0_validated[
        [
            "MRN",
            "CSN",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mcmed_medication_name_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mcmed_medication_class_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


MCMED_FROZEN_MEDICATION_FEATURES = (
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


for feature_name in (
    MCMED_FROZEN_MEDICATION_FEATURES
):
    mcmed_s1_medication_features_provisional[
        feature_name
    ] = (
        mcmed_s1_medication_features_provisional[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )



assert len(
    mcmed_s1_medication_features_provisional
) == 6707

assert mcmed_s1_medication_features_provisional[
    "CSN"
].is_unique

observed_mcmed_medication_columns = [
    column
    for column
    in (
        mcmed_s1_medication_features_provisional
        .columns
    )
    if (
        column.startswith(
            "pmh_med_name__"
        )
        or
        column.startswith(
            "pmh_med_class__"
        )
    )
]

assert (
    observed_mcmed_medication_columns
    ==
    MCMED_FROZEN_MEDICATION_FEATURES
)

assert len(
    observed_mcmed_medication_columns
) == 1176



mcmed_name_mapping_method_summary = (
    mcmed_medication_mapping_audit[
        "name_match_method"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "name_match_method"
    )
    .reset_index(
        name="n_unique_medication_records"
    )
)


mcmed_class_mapping_method_summary = (
    mcmed_medication_mapping_audit[
        "class_match_method"
    ]
    .value_counts(dropna=False)
    .rename_axis(
        "class_match_method"
    )
    .reset_index(
        name="n_unique_medication_records"
    )
)


mcmed_medication_mapping_audit_summary = pd.DataFrame(
    [
        {
            "measure": (
                "Strict active-at-arrival "
                "medication rows"
            ),
            "value": len(
                mcmed_active_medications_strict
            ),
        },
        {
            "measure": (
                "Encounters with >=1 strict "
                "active medication"
            ),
            "value": (
                mcmed_active_medications_strict[
                    "CSN"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique medication records audited"
            ),
            "value": len(
                mcmed_medication_mapping_audit
            ),
        },
        {
            "measure": (
                "Unique records with >=1 mapped "
                "medication name"
            ),
            "value": int(
                mcmed_medication_mapping_audit[
                    "n_name_features"
                ].gt(0).sum()
            ),
        },
        {
            "measure": (
                "Unique records with >=1 mapped "
                "medication class"
            ),
            "value": int(
                mcmed_medication_mapping_audit[
                    "n_class_features"
                ].gt(0).sum()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "medication name"
            ),
            "value": (
                mcmed_medication_name_positive_long[
                    "CSN"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped "
                "medication class"
            ),
            "value": (
                mcmed_medication_class_positive_long[
                    "CSN"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Frozen medication-name columns "
                "with >=1 external positive"
            ),
            "value": int(
                (
                    mcmed_s1_medication_features_provisional[
                        FROZEN_PMH_MEDICATION_NAME_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
        {
            "measure": (
                "Frozen medication-class columns "
                "with >=1 external positive"
            ),
            "value": int(
                (
                    mcmed_s1_medication_features_provisional[
                        FROZEN_PMH_MEDICATION_CLASS_FEATURES
                    ].sum(axis=0)
                    > 0
                ).sum()
            ),
        },
    ]
)



mcmed_medication_mapping_usage = (
    mcmed_active_medications_mapped
    .groupby(
        "medication_mapping_key"
    )
    .agg(
        n_active_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
    )
    .reset_index()
)


mcmed_medication_unmatched_summary = (
    mcmed_medication_mapping_audit.merge(
        mcmed_medication_mapping_usage,
        on="medication_mapping_key",
        how="left",
        validate="one_to_one",
    )
)


mcmed_medication_unmatched_summary = (
    mcmed_medication_unmatched_summary.loc[
        mcmed_medication_unmatched_summary[
            "n_name_features"
        ].eq(0)
        |
        mcmed_medication_unmatched_summary[
            "n_class_features"
        ].eq(0)
    ]
    .sort_values(
        [
            "n_encounters",
            "n_active_rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "MC-MED medication mapping audit:"
)

display(
    mcmed_medication_mapping_audit_summary
)


print(
    "\nMedication-name matching methods:"
)

display(
    mcmed_name_mapping_method_summary
)


print(
    "\nMedication-class matching methods:"
)

display(
    mcmed_class_mapping_method_summary
)


print(
    "\nHighest-frequency unmatched or "
    "ambiguous medication records:"
)

display(
    mcmed_medication_unmatched_summary[
        [
            "Name",
            "Generic_name",
            "NDC",
            "Med_class",
            "Med_subclass",
            "name_match_method",
            "class_match_method",
            "n_encounters",
        ]
    ].head(40)
)


print(
    "\nProvisional MC-MED medication matrix shape:",
    mcmed_s1_medication_features_provisional.shape,
)

print(
    "Aligned frozen medication columns:",
    len(
        observed_mcmed_medication_columns
    ),
)

print(
    "\nThis matrix has not yet been frozen or saved."
)


#### MC-MED medication-mapping comparison

The provisional high-confidence medication mapper prioritises exact names, unique canonical names, NDC crosswalks and exact medication subclasses. This provides strong mapping precision but may miss valid equivalents that differ only by dose, formulation, route, release type or brand name.

This section reproduces the stricter legacy-compatible mapping used in the original external-validation analysis and applies it to the same unique MC-MED medication records used by the provisional reconstruction.

For medication names, the legacy-compatible method:

- removes dose, route and formulation terms;
- generates one- to six-token candidate phrases;
- performs exact candidate-to-frozen-feature matching;
- performs multi-token target-subset matching;
- prevents vague single-token mappings involving terms such as vitamin, calcium, oral, tablet or acid.

For medication classes, the method:

- attempts exact class and subclass matching;
- calculates token overlap and character similarity when exact matching fails;
- accepts the best class match only when the combined score is at least 0.80 and token overlap is at least 0.55.

The high-confidence and legacy-compatible methods are compared at the unique-record, linked-row, encounter and frozen-feature levels. Potentially broad medication-name mappings and all accepted fuzzy medication-class mappings are displayed for review.

This is a comparison and audit step only. No final MC-MED medication matrix is saved or frozen in this section.


In [ ]:



required_objects = [
    "mcmed_unique_medication_records",
    "mcmed_medication_mapping_audit",
    "mcmed_active_medications_mapped",
    "mcmed_s1_medication_features_provisional",
    "FROZEN_PMH_MEDICATION_NAME_FEATURES",
    "FROZEN_PMH_MEDICATION_CLASS_FEATURES",
    "mcmed_s0_validated",
    "build_binary_feature_matrix",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run Cells 78–81 before this comparison. "
        f"Missing objects: {missing_objects}"
    )


required_unique_columns = {
    "medication_mapping_key",
    "Name",
    "Generic_name",
    "Med_class",
    "Med_subclass",
}

missing_unique_columns = (
    required_unique_columns
    - set(
        mcmed_unique_medication_records.columns
    )
)

if missing_unique_columns:
    raise ValueError(
        "mcmed_unique_medication_records is missing: "
        f"{sorted(missing_unique_columns)}"
    )


assert (
    mcmed_unique_medication_records[
        "medication_mapping_key"
    ].is_unique
)

assert (
    len(mcmed_unique_medication_records)
    ==
    len(mcmed_medication_mapping_audit)
)


print(
    "Unique MC-MED medication records to compare:",
    f"{len(mcmed_unique_medication_records):,}",
)



LEGACY_AMBIGUOUS_SINGLE_TOKENS = {
    "vitamin",
    "vitamins",
    "mineral",
    "minerals",
    "acid",
    "oral",
    "tablet",
    "tablets",
    "capsule",
    "capsules",
    "solution",
    "injection",
    "mg",
    "mcg",
    "unit",
    "units",
    "d",
    "c",
    "b",
    "e",
    "no",
    "and",
    "with",
    "plus",
    "calcium",
}


LEGACY_CLASS_GENERIC_TOKENS = {
    "agents",
    "agent",
    "preparations",
    "preparation",
    "combinations",
    "combination",
    "products",
    "product",
    "misc",
    "other",
    "and",
    "with",
    "plus",
    "therapy",
    "therapies",
}


def legacy_normalise_mcmed_medication_slug(
    value,
) -> str:
    """
    Reproduce the stricter medication normalisation used in
    the original MC-MED external-validation workflow.
    """

    if pd.isna(value):
        return ""

    text = str(value).lower().strip()

    text = text.replace(
        "&",
        " and ",
    )

    text = text.replace(
        "+",
        " plus ",
    )

    text = re.sub(
        r"\bvit\b",
        "vitamin",
        text,
    )

    text = re.sub(
        r"\bvitamin\s*d\s*3\b",
        "vitamin d3",
        text,
    )

    text = re.sub(
        r"\bvitamin\s*d\s*2\b",
        "vitamin d2",
        text,
    )

    text = re.sub(
        r"\bvitamin\s*b\s*12\b",
        "vitamin b12",
        text,
    )

    text = re.sub(
        r"\bvitamin\s*b\s*1\b",
        "vitamin b1",
        text,
    )

    text = re.sub(
        r"\bvitamin\s*b\s*6\b",
        "vitamin b6",
        text,
    )

    text = re.sub(
        r"\([^)]*\)",
        " ",
        text,
    )

    text = re.sub(
        (
            r"\b\d+(?:\.\d+)?\s*"
            r"(?:mg|mcg|g|gram|grams|ml|unit|units|"
            r"iu|meq|%)\b"
        ),
        " ",
        text,
    )

    text = re.sub(
        r"\b\d+(?:\.\d+)?\b",
        " ",
        text,
    )

    removable_terms = [
        "po",
        "iv",
        "im",
        "sc",
        "oral",
        "intravenous",
        "subcutaneous",
        "tablet",
        "tablets",
        "tab",
        "tabs",
        "capsule",
        "capsules",
        "cap",
        "caps",
        "solution",
        "suspension",
        "syrup",
        "injection",
        "injectable",
        "cream",
        "ointment",
        "gel",
        "patch",
        "powder",
        "spray",
        "delayed",
        "release",
        "extended",
        "enteric",
        "coated",
        "er",
        "xr",
        "sr",
        "ec",
        "dr",
        "tbec",
        "daily",
        "bid",
        "tid",
        "qid",
        "qhs",
        "prn",
    ]

    removable_pattern = (
        r"\b("
        +
        "|".join(
            re.escape(term)
            for term in removable_terms
        )
        +
        r")\b"
    )

    text = re.sub(
        removable_pattern,
        " ",
        text,
    )

    text = re.sub(
        r"[^a-z0-9]+",
        "_",
        text,
    )

    text = re.sub(
        r"_+",
        "_",
        text,
    ).strip("_")

    return text


def legacy_slug_tokens(
    slug,
) -> list[str]:
    """Split one normalised medication slug into tokens."""

    return [
        token
        for token in str(slug).split("_")
        if token
    ]



legacy_name_slug_to_feature = {
    feature_name.replace(
        "pmh_med_name__",
        "",
        1,
    ): feature_name
    for feature_name
    in FROZEN_PMH_MEDICATION_NAME_FEATURES
}


legacy_class_slug_to_feature = {
    feature_name.replace(
        "pmh_med_class__",
        "",
        1,
    ): feature_name
    for feature_name
    in FROZEN_PMH_MEDICATION_CLASS_FEATURES
}


assert (
    len(legacy_name_slug_to_feature)
    ==
    len(FROZEN_PMH_MEDICATION_NAME_FEATURES)
)

assert (
    len(legacy_class_slug_to_feature)
    ==
    len(FROZEN_PMH_MEDICATION_CLASS_FEATURES)
)


legacy_name_reference_slugs = list(
    legacy_name_slug_to_feature
)

legacy_class_reference_slugs = list(
    legacy_class_slug_to_feature
)



def make_legacy_medication_name_candidates(
    name,
    generic_name,
) -> list[str]:
    """
    Generate the one- to six-token medication candidates used
    by the stricter original mapping.
    """

    raw_texts = []

    for raw_value in [
        generic_name,
        name,
    ]:
        if pd.isna(raw_value):
            continue

        raw_text = str(raw_value)

        raw_texts.append(
            raw_text
        )

        for component in re.split(
            r"[,;/]",
            raw_text,
        ):
            raw_texts.append(
                component
            )

    candidates = []

    for raw_text in raw_texts:
        full_slug = (
            legacy_normalise_mcmed_medication_slug(
                raw_text
            )
        )

        if not full_slug:
            continue

        candidates.append(
            full_slug
        )

        tokens = legacy_slug_tokens(
            full_slug
        )

        for start_position in range(
            len(tokens)
        ):
            maximum_length = min(
                6,
                len(tokens) - start_position,
            )

            for n_tokens in range(
                1,
                maximum_length + 1,
            ):
                candidate = "_".join(
                    tokens[
                        start_position:
                        start_position + n_tokens
                    ]
                )

                if (
                    n_tokens == 1
                    and candidate
                    in LEGACY_AMBIGUOUS_SINGLE_TOKENS
                ):
                    continue

                candidates.append(
                    candidate
                )

    return list(
        dict.fromkeys(
            candidate
            for candidate in candidates
            if candidate
        )
    )


def legacy_match_medication_names(
    name,
    generic_name,
) -> dict:
    """
    Reproduce exact n-gram and target-subset medication-name
    matching while retaining audit evidence.
    """

    candidates = (
        make_legacy_medication_name_candidates(
            name=name,
            generic_name=generic_name,
        )
    )

    candidate_set = set(
        candidates
    )

    combined_full_slug = (
        legacy_normalise_mcmed_medication_slug(
            " ".join(
                [
                    (
                        str(generic_name)
                        if pd.notna(
                            generic_name
                        )
                        else ""
                    ),
                    (
                        str(name)
                        if pd.notna(name)
                        else ""
                    ),
                ]
            )
        )
    )

    combined_full_tokens = set(
        legacy_slug_tokens(
            combined_full_slug
        )
    )

    exact_features = set()
    subset_features = set()

    for candidate in candidate_set:
        if candidate in (
            legacy_name_slug_to_feature
        ):
            exact_features.add(
                legacy_name_slug_to_feature[
                    candidate
                ]
            )

    for (
        target_slug,
        target_feature,
    ) in legacy_name_slug_to_feature.items():

        target_tokens = set(
            legacy_slug_tokens(
                target_slug
            )
        )

        if len(target_tokens) < 2:
            continue

        distinctive_tokens = (
            target_tokens
            -
            LEGACY_AMBIGUOUS_SINGLE_TOKENS
        )

        if not distinctive_tokens:
            continue

        if target_tokens.issubset(
            combined_full_tokens
        ):
            subset_features.add(
                target_feature
            )

    matched_features = sorted(
        exact_features
        |
        subset_features
    )

    if exact_features and subset_features:
        match_method = (
            "EXACT_NGRAM_AND_TARGET_SUBSET"
        )

    elif exact_features:
        match_method = (
            "EXACT_NGRAM"
        )

    elif subset_features:
        match_method = (
            "TARGET_TOKEN_SUBSET"
        )

    else:
        match_method = "UNMATCHED"

    return {
        "legacy_name_features": (
            matched_features
        ),
        "legacy_name_exact_features": (
            sorted(exact_features)
        ),
        "legacy_name_subset_features": (
            sorted(subset_features)
        ),
        "legacy_name_match_method": (
            match_method
        ),
        "legacy_n_name_matches": len(
            matched_features
        ),
        "legacy_n_exact_name_matches": len(
            exact_features
        ),
        "legacy_n_subset_name_matches": len(
            subset_features
        ),
        "legacy_name_candidates": (
            candidates
        ),
    }



def legacy_class_token_overlap(
    first_slug,
    second_slug,
) -> float:
    """
    Calculate class-token overlap after removing generic class
    terminology.
    """

    first_tokens = (
        set(
            legacy_slug_tokens(
                first_slug
            )
        )
        -
        LEGACY_CLASS_GENERIC_TOKENS
    )

    second_tokens = (
        set(
            legacy_slug_tokens(
                second_slug
            )
        )
        -
        LEGACY_CLASS_GENERIC_TOKENS
    )

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(
            first_tokens
            & second_tokens
        )
        /
        max(
            len(first_tokens),
            len(second_tokens),
        )
    )


def legacy_string_similarity(
    first_slug,
    second_slug,
) -> float:
    """Calculate character-level similarity."""

    return SequenceMatcher(
        None,
        str(first_slug),
        str(second_slug),
    ).ratio()


def legacy_match_medication_classes(
    med_class,
    med_subclass,
) -> dict:
    """
    Reproduce exact and conservative fuzzy class matching.

    Fuzzy acceptance thresholds:
    - combined score >= 0.80;
    - token overlap >= 0.55.
    """

    candidates = []

    for raw_value in [
        med_subclass,
        med_class,
    ]:
        candidate_slug = (
            legacy_normalise_mcmed_medication_slug(
                raw_value
            )
        )

        if candidate_slug:
            candidates.append(
                candidate_slug
            )

    candidates = list(
        dict.fromkeys(
            candidates
        )
    )

    exact_features = set()
    fuzzy_features = set()
    fuzzy_evidence = []

    for candidate in candidates:
        if candidate in (
            legacy_class_slug_to_feature
        ):
            exact_features.add(
                legacy_class_slug_to_feature[
                    candidate
                ]
            )

    for candidate in candidates:
        best_target_slug = None
        best_score = 0.0
        best_overlap = 0.0
        best_similarity = 0.0

        for target_slug in (
            legacy_class_reference_slugs
        ):
            token_overlap = (
                legacy_class_token_overlap(
                    candidate,
                    target_slug,
                )
            )

            string_similarity = (
                legacy_string_similarity(
                    candidate,
                    target_slug,
                )
            )

            combined_score = (
                0.85 * token_overlap
                +
                0.15 * string_similarity
            )

            if combined_score > best_score:
                best_target_slug = (
                    target_slug
                )
                best_score = (
                    combined_score
                )
                best_overlap = (
                    token_overlap
                )
                best_similarity = (
                    string_similarity
                )

        accepted = (
            best_target_slug is not None
            and best_score >= 0.80
            and best_overlap >= 0.55
        )

        target_feature = (
            legacy_class_slug_to_feature[
                best_target_slug
            ]
            if accepted
            else pd.NA
        )

        if accepted:
            fuzzy_features.add(
                target_feature
            )

        fuzzy_evidence.append(
            {
                "source_class_candidate": (
                    candidate
                ),
                "best_target_slug": (
                    best_target_slug
                ),
                "best_target_feature": (
                    target_feature
                ),
                "combined_score": (
                    best_score
                ),
                "token_overlap": (
                    best_overlap
                ),
                "string_similarity": (
                    best_similarity
                ),
                "fuzzy_accepted": (
                    accepted
                ),
            }
        )

    matched_features = sorted(
        exact_features
        |
        fuzzy_features
    )

    if exact_features and fuzzy_features:
        match_method = (
            "EXACT_AND_LEGACY_FUZZY"
        )

    elif exact_features:
        match_method = "EXACT_CLASS"

    elif fuzzy_features:
        match_method = (
            "LEGACY_FUZZY_CLASS"
        )

    else:
        match_method = "UNMATCHED"

    return {
        "legacy_class_features": (
            matched_features
        ),
        "legacy_class_exact_features": (
            sorted(exact_features)
        ),
        "legacy_class_fuzzy_features": (
            sorted(fuzzy_features)
        ),
        "legacy_class_match_method": (
            match_method
        ),
        "legacy_n_class_matches": len(
            matched_features
        ),
        "legacy_n_exact_class_matches": len(
            exact_features
        ),
        "legacy_n_fuzzy_class_matches": len(
            fuzzy_features
        ),
        "legacy_class_fuzzy_evidence": (
            fuzzy_evidence
        ),
    }



legacy_medication_mapping_records = []
legacy_class_fuzzy_evidence_records = []


for _, medication_row in (
    mcmed_unique_medication_records.iterrows()
):
    medication_mapping_key = (
        medication_row[
            "medication_mapping_key"
        ]
    )

    name_result = (
        legacy_match_medication_names(
            name=medication_row["Name"],
            generic_name=(
                medication_row[
                    "Generic_name"
                ]
            ),
        )
    )

    class_result = (
        legacy_match_medication_classes(
            med_class=(
                medication_row[
                    "Med_class"
                ]
            ),
            med_subclass=(
                medication_row[
                    "Med_subclass"
                ]
            ),
        )
    )

    total_matches = (
        name_result[
            "legacy_n_name_matches"
        ]
        +
        class_result[
            "legacy_n_class_matches"
        ]
    )

    legacy_medication_mapping_records.append(
        {
            "medication_mapping_key": (
                medication_mapping_key
            ),
            "Name": medication_row["Name"],
            "Generic_name": (
                medication_row[
                    "Generic_name"
                ]
            ),
            "Med_class": (
                medication_row[
                    "Med_class"
                ]
            ),
            "Med_subclass": (
                medication_row[
                    "Med_subclass"
                ]
            ),
            **name_result,
            **class_result,
            "legacy_n_total_matches": (
                total_matches
            ),
            "legacy_many_name_matches_flag": (
                name_result[
                    "legacy_n_name_matches"
                ]
                > 10
            ),
            "legacy_many_total_matches_flag": (
                total_matches > 12
            ),
        }
    )

    for evidence_record in (
        class_result[
            "legacy_class_fuzzy_evidence"
        ]
    ):
        legacy_class_fuzzy_evidence_records.append(
            {
                "medication_mapping_key": (
                    medication_mapping_key
                ),
                "Name": medication_row["Name"],
                "Generic_name": (
                    medication_row[
                        "Generic_name"
                    ]
                ),
                "Med_class": (
                    medication_row[
                        "Med_class"
                    ]
                ),
                "Med_subclass": (
                    medication_row[
                        "Med_subclass"
                    ]
                ),
                **evidence_record,
            }
        )


mcmed_legacy_medication_mapping_audit = (
    pd.DataFrame(
        legacy_medication_mapping_records
    )
)


mcmed_legacy_class_fuzzy_evidence = (
    pd.DataFrame(
        legacy_class_fuzzy_evidence_records
    )
)


assert (
    len(
        mcmed_legacy_medication_mapping_audit
    )
    ==
    len(
        mcmed_unique_medication_records
    )
)

assert (
    mcmed_legacy_medication_mapping_audit[
        "medication_mapping_key"
    ].is_unique
)



legacy_mapping_columns = [
    "medication_mapping_key",
    "legacy_name_features",
    "legacy_name_match_method",
    "legacy_n_name_matches",
    "legacy_class_features",
    "legacy_class_match_method",
    "legacy_n_class_matches",
    "legacy_n_total_matches",
]


mcmed_active_medications_legacy_mapped = (
    mcmed_active_medications_mapped.merge(
        mcmed_legacy_medication_mapping_audit[
            legacy_mapping_columns
        ],
        on="medication_mapping_key",
        how="left",
        validate="many_to_one",
    )
)


assert (
    len(
        mcmed_active_medications_legacy_mapped
    )
    ==
    len(
        mcmed_active_medications_mapped
    )
)



mcmed_legacy_medication_name_positive_long = (
    mcmed_active_medications_legacy_mapped[
        [
            "MRN",
            "CSN",
            "legacy_name_features",
        ]
    ]
    .explode(
        "legacy_name_features"
    )
    .rename(
        columns={
            "legacy_name_features":
                "feature_name"
        }
    )
)


mcmed_legacy_medication_name_positive_long = (
    mcmed_legacy_medication_name_positive_long.loc[
        mcmed_legacy_medication_name_positive_long[
            "feature_name"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)


mcmed_legacy_medication_class_positive_long = (
    mcmed_active_medications_legacy_mapped[
        [
            "MRN",
            "CSN",
            "legacy_class_features",
        ]
    ]
    .explode(
        "legacy_class_features"
    )
    .rename(
        columns={
            "legacy_class_features":
                "feature_name"
        }
    )
)


mcmed_legacy_medication_class_positive_long = (
    mcmed_legacy_medication_class_positive_long.loc[
        mcmed_legacy_medication_class_positive_long[
            "feature_name"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)



mcmed_legacy_medication_name_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mcmed_legacy_medication_name_positive_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated["CSN"]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


mcmed_legacy_medication_class_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            mcmed_legacy_medication_class_positive_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated["CSN"]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
    )
)


mcmed_s1_medication_features_legacy_comparison = (
    mcmed_s0_validated[
        [
            "MRN",
            "CSN",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mcmed_legacy_medication_name_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mcmed_legacy_medication_class_wide,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


ALL_FROZEN_MEDICATION_FEATURES = (
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


for feature_name in (
    ALL_FROZEN_MEDICATION_FEATURES
):
    mcmed_s1_medication_features_legacy_comparison[
        feature_name
    ] = (
        mcmed_s1_medication_features_legacy_comparison[
            feature_name
        ]
        .fillna(0)
        .astype("int8")
    )


assert (
    len(
        mcmed_s1_medication_features_legacy_comparison
    )
    ==
    len(mcmed_s0_validated)
)

assert (
    mcmed_s1_medication_features_legacy_comparison[
        "CSN"
    ].is_unique
)

assert (
    [
        column
        for column
        in (
            mcmed_s1_medication_features_legacy_comparison
            .columns
        )
        if (
            column.startswith(
                "pmh_med_name__"
            )
            or
            column.startswith(
                "pmh_med_class__"
            )
        )
    ]
    ==
    ALL_FROZEN_MEDICATION_FEATURES
)



mcmed_medication_record_usage = (
    mcmed_active_medications_mapped
    .groupby(
        "medication_mapping_key"
    )
    .agg(
        n_active_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
    )
    .reset_index()
)



current_record_mapping = (
    mcmed_medication_mapping_audit[
        [
            "medication_mapping_key",
            "Name",
            "Generic_name",
            "Med_class",
            "Med_subclass",
            "name_match_method",
            "class_match_method",
            "n_name_features",
            "n_class_features",
        ]
    ]
    .copy()
)


current_record_mapping[
    "current_n_total_matches"
] = (
    current_record_mapping[
        "n_name_features"
    ]
    +
    current_record_mapping[
        "n_class_features"
    ]
)


legacy_record_mapping = (
    mcmed_legacy_medication_mapping_audit[
        [
            "medication_mapping_key",
            "legacy_name_match_method",
            "legacy_class_match_method",
            "legacy_n_name_matches",
            "legacy_n_class_matches",
            "legacy_n_total_matches",
            "legacy_name_features",
            "legacy_class_features",
            "legacy_many_name_matches_flag",
            "legacy_many_total_matches_flag",
        ]
    ]
    .copy()
)


mcmed_medication_mapping_method_comparison = (
    current_record_mapping.merge(
        legacy_record_mapping,
        on="medication_mapping_key",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        mcmed_medication_record_usage,
        on="medication_mapping_key",
        how="left",
        validate="one_to_one",
    )
)


mcmed_medication_mapping_method_comparison[
    "legacy_recovers_currently_unmapped_record"
] = (
    mcmed_medication_mapping_method_comparison[
        "current_n_total_matches"
    ].eq(0)
    &
    mcmed_medication_mapping_method_comparison[
        "legacy_n_total_matches"
    ].gt(0)
)


mcmed_medication_mapping_method_comparison[
    "legacy_adds_name_mapping"
] = (
    mcmed_medication_mapping_method_comparison[
        "n_name_features"
    ].eq(0)
    &
    mcmed_medication_mapping_method_comparison[
        "legacy_n_name_matches"
    ].gt(0)
)


mcmed_medication_mapping_method_comparison[
    "legacy_adds_class_mapping"
] = (
    mcmed_medication_mapping_method_comparison[
        "n_class_features"
    ].eq(0)
    &
    mcmed_medication_mapping_method_comparison[
        "legacy_n_class_matches"
    ].gt(0)
)



current_name_encounters = set(
    mcmed_medication_name_positive_long[
        "CSN"
    ]
)

current_class_encounters = set(
    mcmed_medication_class_positive_long[
        "CSN"
    ]
)

current_any_encounters = (
    current_name_encounters
    |
    current_class_encounters
)


legacy_name_encounters = set(
    mcmed_legacy_medication_name_positive_long[
        "CSN"
    ]
)

legacy_class_encounters = set(
    mcmed_legacy_medication_class_positive_long[
        "CSN"
    ]
)

legacy_any_encounters = (
    legacy_name_encounters
    |
    legacy_class_encounters
)


n_active_medication_encounters = (
    mcmed_active_medications_mapped[
        "CSN"
    ].nunique()
)


current_name_positive_columns = int(
    (
        mcmed_s1_medication_features_provisional[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


current_class_positive_columns = int(
    (
        mcmed_s1_medication_features_provisional[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


legacy_name_positive_columns = int(
    (
        mcmed_s1_medication_features_legacy_comparison[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


legacy_class_positive_columns = int(
    (
        mcmed_s1_medication_features_legacy_comparison[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


current_mapped_record_mask = (
    mcmed_medication_mapping_method_comparison[
        "current_n_total_matches"
    ].gt(0)
)

legacy_mapped_record_mask = (
    mcmed_medication_mapping_method_comparison[
        "legacy_n_total_matches"
    ].gt(0)
)


current_mapped_active_rows = int(
    mcmed_medication_mapping_method_comparison.loc[
        current_mapped_record_mask,
        "n_active_rows",
    ].sum()
)


legacy_mapped_active_rows = int(
    mcmed_medication_mapping_method_comparison.loc[
        legacy_mapped_record_mask,
        "n_active_rows",
    ].sum()
)


mcmed_medication_legacy_comparison_summary = (
    pd.DataFrame(
        [
            {
                "mapping_measure": (
                    "Unique medication records "
                    "mapped to >=1 feature"
                ),
                "high_confidence": int(
                    current_mapped_record_mask.sum()
                ),
                "legacy_compatible": int(
                    legacy_mapped_record_mask.sum()
                ),
                "denominator": len(
                    mcmed_medication_mapping_method_comparison
                ),
            },
            {
                "mapping_measure": (
                    "Strict active medication rows "
                    "mapped to >=1 feature"
                ),
                "high_confidence": (
                    current_mapped_active_rows
                ),
                "legacy_compatible": (
                    legacy_mapped_active_rows
                ),
                "denominator": len(
                    mcmed_active_medications_mapped
                ),
            },
            {
                "mapping_measure": (
                    "Active-medication encounters "
                    "with >=1 mapped feature"
                ),
                "high_confidence": len(
                    current_any_encounters
                ),
                "legacy_compatible": len(
                    legacy_any_encounters
                ),
                "denominator": (
                    n_active_medication_encounters
                ),
            },
            {
                "mapping_measure": (
                    "Active-medication encounters "
                    "with mapped medication name"
                ),
                "high_confidence": len(
                    current_name_encounters
                ),
                "legacy_compatible": len(
                    legacy_name_encounters
                ),
                "denominator": (
                    n_active_medication_encounters
                ),
            },
            {
                "mapping_measure": (
                    "Active-medication encounters "
                    "with mapped medication class"
                ),
                "high_confidence": len(
                    current_class_encounters
                ),
                "legacy_compatible": len(
                    legacy_class_encounters
                ),
                "denominator": (
                    n_active_medication_encounters
                ),
            },
            {
                "mapping_measure": (
                    "Frozen medication-name columns "
                    "with >=1 external positive"
                ),
                "high_confidence": (
                    current_name_positive_columns
                ),
                "legacy_compatible": (
                    legacy_name_positive_columns
                ),
                "denominator": len(
                    FROZEN_PMH_MEDICATION_NAME_FEATURES
                ),
            },
            {
                "mapping_measure": (
                    "Frozen medication-class columns "
                    "with >=1 external positive"
                ),
                "high_confidence": (
                    current_class_positive_columns
                ),
                "legacy_compatible": (
                    legacy_class_positive_columns
                ),
                "denominator": len(
                    FROZEN_PMH_MEDICATION_CLASS_FEATURES
                ),
            },
        ]
    )
)


mcmed_medication_legacy_comparison_summary[
    "high_confidence_pct"
] = (
    mcmed_medication_legacy_comparison_summary[
        "high_confidence"
    ]
    /
    mcmed_medication_legacy_comparison_summary[
        "denominator"
    ]
    * 100
)


mcmed_medication_legacy_comparison_summary[
    "legacy_compatible_pct"
] = (
    mcmed_medication_legacy_comparison_summary[
        "legacy_compatible"
    ]
    /
    mcmed_medication_legacy_comparison_summary[
        "denominator"
    ]
    * 100
)


mcmed_medication_legacy_comparison_summary[
    "absolute_gain"
] = (
    mcmed_medication_legacy_comparison_summary[
        "legacy_compatible"
    ]
    -
    mcmed_medication_legacy_comparison_summary[
        "high_confidence"
    ]
)


mcmed_medication_legacy_comparison_summary[
    "percentage_point_gain"
] = (
    mcmed_medication_legacy_comparison_summary[
        "legacy_compatible_pct"
    ]
    -
    mcmed_medication_legacy_comparison_summary[
        "high_confidence_pct"
    ]
)


print(
    "\nHigh-confidence versus legacy-compatible mapping:"
)

display(
    mcmed_medication_legacy_comparison_summary.round(
        {
            "high_confidence_pct": 2,
            "legacy_compatible_pct": 2,
            "percentage_point_gain": 2,
        }
    )
)



current_feature_support = (
    mcmed_s1_medication_features_provisional[
        ALL_FROZEN_MEDICATION_FEATURES
    ]
    .sum(axis=0)
    .astype("int64")
)


legacy_feature_support = (
    mcmed_s1_medication_features_legacy_comparison[
        ALL_FROZEN_MEDICATION_FEATURES
    ]
    .sum(axis=0)
    .astype("int64")
)


mcmed_medication_feature_support_comparison = (
    pd.DataFrame(
        {
            "feature_name": (
                ALL_FROZEN_MEDICATION_FEATURES
            ),
            "current_positive_encounters": (
                current_feature_support.reindex(
                    ALL_FROZEN_MEDICATION_FEATURES
                ).to_numpy()
            ),
            "legacy_positive_encounters": (
                legacy_feature_support.reindex(
                    ALL_FROZEN_MEDICATION_FEATURES
                ).to_numpy()
            ),
        }
    )
)


mcmed_medication_feature_support_comparison[
    "feature_family"
] = np.where(
    mcmed_medication_feature_support_comparison[
        "feature_name"
    ].str.startswith(
        "pmh_med_name__"
    ),
    "MEDICATION_NAME",
    "MEDICATION_CLASS",
)


mcmed_medication_feature_support_comparison[
    "legacy_minus_current"
] = (
    mcmed_medication_feature_support_comparison[
        "legacy_positive_encounters"
    ]
    -
    mcmed_medication_feature_support_comparison[
        "current_positive_encounters"
    ]
)


mcmed_medication_feature_support_comparison[
    "support_status"
] = np.select(
    [
        (
            mcmed_medication_feature_support_comparison[
                "current_positive_encounters"
            ].eq(0)
            &
            mcmed_medication_feature_support_comparison[
                "legacy_positive_encounters"
            ].gt(0)
        ),
        (
            mcmed_medication_feature_support_comparison[
                "current_positive_encounters"
            ].gt(0)
            &
            mcmed_medication_feature_support_comparison[
                "legacy_positive_encounters"
            ].eq(0)
        ),
        (
            mcmed_medication_feature_support_comparison[
                "legacy_positive_encounters"
            ]
            >
            mcmed_medication_feature_support_comparison[
                "current_positive_encounters"
            ]
        ),
        (
            mcmed_medication_feature_support_comparison[
                "legacy_positive_encounters"
            ]
            <
            mcmed_medication_feature_support_comparison[
                "current_positive_encounters"
            ]
        ),
    ],
    [
        "RECOVERED_ONLY_BY_LEGACY",
        "CURRENT_ONLY",
        "HIGHER_WITH_LEGACY",
        "HIGHER_WITH_CURRENT",
    ],
    default="SAME_SUPPORT",
)


feature_support_status_summary = (
    mcmed_medication_feature_support_comparison[
        "support_status"
    ]
    .value_counts()
    .rename_axis(
        "support_status"
    )
    .reset_index(
        name="n_features"
    )
)


print(
    "\nFeature-support comparison:"
)

display(
    feature_support_status_summary
)


print(
    "\nFrozen features recovered only by "
    "legacy-compatible mapping:"
)

display(
    mcmed_medication_feature_support_comparison.loc[
        mcmed_medication_feature_support_comparison[
            "support_status"
        ].eq(
            "RECOVERED_ONLY_BY_LEGACY"
        )
    ]
    .sort_values(
        "legacy_positive_encounters",
        ascending=False,
    )
    .head(40)
)



mcmed_medication_legacy_record_recovery_summary = (
    mcmed_medication_mapping_method_comparison.loc[
        mcmed_medication_mapping_method_comparison[
            "legacy_recovers_currently_unmapped_record"
        ]
        |
        mcmed_medication_mapping_method_comparison[
            "legacy_adds_name_mapping"
        ]
        |
        mcmed_medication_mapping_method_comparison[
            "legacy_adds_class_mapping"
        ]
    ]
    .copy()
    .sort_values(
        [
            "n_encounters",
            "n_active_rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nHighest-frequency records for which legacy logic "
    "adds a mapping:"
)

display(
    mcmed_medication_legacy_record_recovery_summary[
        [
            "Name",
            "Generic_name",
            "Med_class",
            "Med_subclass",
            "name_match_method",
            "legacy_name_match_method",
            "n_name_features",
            "legacy_n_name_matches",
            "class_match_method",
            "legacy_class_match_method",
            "n_class_features",
            "legacy_n_class_matches",
            "n_encounters",
        ]
    ].head(50)
)



mcmed_legacy_broad_medication_name_audit = (
    mcmed_medication_mapping_method_comparison.loc[
        mcmed_medication_mapping_method_comparison[
            "legacy_n_name_matches"
        ].gt(1)
    ]
    .copy()
    .sort_values(
        [
            "legacy_n_name_matches",
            "n_encounters",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nLegacy medication-name records mapping to "
    "multiple frozen name features:"
)

display(
    mcmed_legacy_broad_medication_name_audit[
        [
            "Name",
            "Generic_name",
            "legacy_name_match_method",
            "legacy_n_name_matches",
            "legacy_name_features",
            "n_encounters",
        ]
    ].head(50)
)


print(
    "\nLegacy potentially excessive mappings "
    "(old notebook warning thresholds):"
)

legacy_excessive_mapping_audit = (
    mcmed_medication_mapping_method_comparison.loc[
        mcmed_medication_mapping_method_comparison[
            "legacy_many_name_matches_flag"
        ]
        |
        mcmed_medication_mapping_method_comparison[
            "legacy_many_total_matches_flag"
        ]
    ]
    .copy()
    .sort_values(
        "legacy_n_total_matches",
        ascending=False,
    )
)


display(
    legacy_excessive_mapping_audit[
        [
            "Name",
            "Generic_name",
            "Med_class",
            "Med_subclass",
            "legacy_n_name_matches",
            "legacy_n_class_matches",
            "legacy_n_total_matches",
            "legacy_name_features",
            "legacy_class_features",
            "n_encounters",
        ]
    ].head(30)
)



if not mcmed_legacy_class_fuzzy_evidence.empty:
    mcmed_legacy_class_fuzzy_evidence = (
        mcmed_legacy_class_fuzzy_evidence.merge(
            mcmed_medication_record_usage,
            on="medication_mapping_key",
            how="left",
            validate="many_to_one",
        )
    )


accepted_legacy_class_fuzzy_mappings = (
    mcmed_legacy_class_fuzzy_evidence.loc[
        mcmed_legacy_class_fuzzy_evidence[
            "fuzzy_accepted"
        ]
    ]
    .copy()
    .sort_values(
        [
            "n_encounters",
            "combined_score",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


print(
    "\nAccepted legacy fuzzy medication-class mappings:"
)

display(
    accepted_legacy_class_fuzzy_mappings[
        [
            "Name",
            "Generic_name",
            "Med_class",
            "Med_subclass",
            "source_class_candidate",
            "best_target_feature",
            "combined_score",
            "token_overlap",
            "string_similarity",
            "n_encounters",
        ]
    ]
    .head(60)
    .round(
        {
            "combined_score": 3,
            "token_overlap": 3,
            "string_similarity": 3,
        }
    )
)



mcmed_legacy_mapping_safety_summary = (
    pd.DataFrame(
        [
            {
                "measure": (
                    "Unique medication records audited"
                ),
                "count": len(
                    mcmed_legacy_medication_mapping_audit
                ),
            },
            {
                "measure": (
                    "Records mapped by current method"
                ),
                "count": int(
                    current_mapped_record_mask.sum()
                ),
            },
            {
                "measure": (
                    "Records mapped by legacy-compatible method"
                ),
                "count": int(
                    legacy_mapped_record_mask.sum()
                ),
            },
            {
                "measure": (
                    "Currently unmapped records recovered "
                    "by legacy logic"
                ),
                "count": int(
                    mcmed_medication_mapping_method_comparison[
                        "legacy_recovers_currently_unmapped_record"
                    ].sum()
                ),
            },
            {
                "measure": (
                    "Records with >1 legacy medication-name match"
                ),
                "count": int(
                    mcmed_medication_mapping_method_comparison[
                        "legacy_n_name_matches"
                    ].gt(1).sum()
                ),
            },
            {
                "measure": (
                    "Records with >10 legacy name matches"
                ),
                "count": int(
                    mcmed_medication_mapping_method_comparison[
                        "legacy_many_name_matches_flag"
                    ].sum()
                ),
            },
            {
                "measure": (
                    "Records with >12 total legacy matches"
                ),
                "count": int(
                    mcmed_medication_mapping_method_comparison[
                        "legacy_many_total_matches_flag"
                    ].sum()
                ),
            },
            {
                "measure": (
                    "Accepted fuzzy class-mapping evidence rows"
                ),
                "count": len(
                    accepted_legacy_class_fuzzy_mappings
                ),
            },
        ]
    )
)


print(
    "\nLegacy-compatible mapping safety summary:"
)

display(
    mcmed_legacy_mapping_safety_summary
)



assert (
    len(
        ALL_FROZEN_MEDICATION_FEATURES
    )
    == 1176
)

assert set(
    np.unique(
        mcmed_s1_medication_features_legacy_comparison[
            ALL_FROZEN_MEDICATION_FEATURES
        ].to_numpy()
    ).tolist()
).issubset(
    {
        0,
        1,
    }
)


print(
    "\nLEGACY-COMPATIBLE MEDICATION MAPPING "
    "COMPARISON PASSED"
)

print(
    "Temporary legacy comparison matrix shape:",
    mcmed_s1_medication_features_legacy_comparison.shape,
)

print(
    "Aligned medication columns:",
    len(
        ALL_FROZEN_MEDICATION_FEATURES
    ),
)

print(
    "\nNo medication matrix has been saved or frozen."
)


In [ ]:



required_objects = [
    "mcmed_medication_mapping_method_comparison",
    "mcmed_active_medications_legacy_mapped",
    "mcmed_s1_medication_features_provisional",
    "mcmed_s1_medication_features_legacy_comparison",
    "mcmed_active_medications_mapped",
    "mcmed_s0_validated",
    "FROZEN_PMH_MEDICATION_NAME_FEATURES",
    "FROZEN_PMH_MEDICATION_CLASS_FEATURES",
    "build_binary_feature_matrix",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the current and legacy medication-mapping cells first. "
        f"Missing objects: {missing_objects}"
    )


assert len(
    FROZEN_PMH_MEDICATION_NAME_FEATURES
) == 812

assert len(
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
) == 364


ALL_FROZEN_MEDICATION_FEATURES = (
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)



SAFE_LEGACY_NAME_METHODS = {
    "EXACT_NGRAM",
    "EXACT_NGRAM_AND_TARGET_SUBSET",
}


safe_legacy_name_recovery_records = (
    mcmed_medication_mapping_method_comparison.loc[
        (
            mcmed_medication_mapping_method_comparison[
                "n_name_features"
            ].eq(0)
        )
        &
        (
            mcmed_medication_mapping_method_comparison[
                "legacy_n_name_matches"
            ].eq(1)
        )
        &
        (
            mcmed_medication_mapping_method_comparison[
                "legacy_name_match_method"
            ].isin(
                SAFE_LEGACY_NAME_METHODS
            )
        ),
        [
            "medication_mapping_key",
            "Name",
            "Generic_name",
            "legacy_name_match_method",
            "legacy_name_features",
            "n_active_rows",
            "n_encounters",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)


assert (
    safe_legacy_name_recovery_records[
        "medication_mapping_key"
    ].is_unique
)

assert (
    safe_legacy_name_recovery_records[
        "legacy_name_features"
    ].map(len).eq(1).all()
)



safe_recovery_active_rows = (
    mcmed_active_medications_legacy_mapped[
        [
            "MRN",
            "CSN",
            "medication_mapping_key",
        ]
    ]
    .merge(
        safe_legacy_name_recovery_records[
            [
                "medication_mapping_key",
                "legacy_name_features",
                "legacy_name_match_method",
            ]
        ],
        on="medication_mapping_key",
        how="inner",
        validate="many_to_one",
    )
)


safe_recovery_name_long = (
    safe_recovery_active_rows[
        [
            "MRN",
            "CSN",
            "medication_mapping_key",
            "legacy_name_match_method",
            "legacy_name_features",
        ]
    ]
    .explode(
        "legacy_name_features"
    )
    .rename(
        columns={
            "legacy_name_features": "feature_name",
        }
    )
)


safe_recovery_name_long = (
    safe_recovery_name_long.loc[
        safe_recovery_name_long[
            "feature_name"
        ].notna()
    ]
    .drop_duplicates(
        subset=[
            "CSN",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)


assert set(
    safe_recovery_name_long[
        "feature_name"
    ].unique()
).issubset(
    set(
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    )
)



safe_recovery_name_wide = (
    build_binary_feature_matrix(
        mapped_long_dataframe=(
            safe_recovery_name_long
        ),
        identifier_column="CSN",
        feature_column="feature_name",
        all_identifiers=(
            mcmed_s0_validated["CSN"]
        ),
        ordered_feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


assert len(
    safe_recovery_name_wide
) == len(
    mcmed_s0_validated
)

assert safe_recovery_name_wide[
    "CSN"
].is_unique



current_candidate_indexed = (
    mcmed_s1_medication_features_provisional
    .set_index("CSN")
    .copy()
)


safe_recovery_indexed = (
    safe_recovery_name_wide
    .set_index("CSN")
    .reindex(
        current_candidate_indexed.index
    )
    .fillna(0)
)


current_name_values = (
    current_candidate_indexed[
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ]
    .astype("int8")
)


recovered_name_values = (
    safe_recovery_indexed[
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ]
    .astype("int8")
)


current_candidate_indexed[
    FROZEN_PMH_MEDICATION_NAME_FEATURES
] = np.maximum(
    current_name_values.to_numpy(),
    recovered_name_values.to_numpy(),
).astype("int8")


mcmed_s1_medication_features_hybrid_candidate = (
    current_candidate_indexed
    .reset_index()
)


mcmed_s1_medication_features_hybrid_candidate = (
    mcmed_s1_medication_features_hybrid_candidate[
        mcmed_s1_medication_features_provisional.columns
    ]
    .copy()
)



assert np.array_equal(
    mcmed_s1_medication_features_hybrid_candidate[
        FROZEN_PMH_MEDICATION_CLASS_FEATURES
    ].to_numpy(),
    mcmed_s1_medication_features_provisional[
        FROZEN_PMH_MEDICATION_CLASS_FEATURES
    ].to_numpy(),
), (
    "Medication-class values changed unexpectedly."
)



assert (
    len(
        mcmed_s1_medication_features_hybrid_candidate
    )
    ==
    len(mcmed_s0_validated)
)

assert (
    mcmed_s1_medication_features_hybrid_candidate[
        "CSN"
    ].is_unique
)


assert set(
    np.unique(
        mcmed_s1_medication_features_hybrid_candidate[
            ALL_FROZEN_MEDICATION_FEATURES
        ].to_numpy()
    ).tolist()
).issubset(
    {
        0,
        1,
    }
)


hybrid_feature_columns = [
    column
    for column
    in (
        mcmed_s1_medication_features_hybrid_candidate
        .columns
    )
    if (
        column.startswith(
            "pmh_med_name__"
        )
        or
        column.startswith(
            "pmh_med_class__"
        )
    )
]


assert (
    hybrid_feature_columns
    ==
    ALL_FROZEN_MEDICATION_FEATURES
)



active_medication_csns = set(
    mcmed_active_medications_mapped[
        "CSN"
    ].unique()
)


def calculate_active_encounter_coverage(
    dataframe,
    feature_columns,
):
    active_rows = (
        dataframe.loc[
            dataframe[
                "CSN"
            ].isin(
                active_medication_csns
            )
        ]
        .copy()
    )

    positive_mask = (
        active_rows[
            feature_columns
        ]
        .gt(0)
        .any(axis=1)
    )

    return int(
        positive_mask.sum()
    )


current_name_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_provisional
        ),
        feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


hybrid_name_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_hybrid_candidate
        ),
        feature_columns=(
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ),
    )
)


current_class_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_provisional
        ),
        feature_columns=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
    )
)


hybrid_class_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_hybrid_candidate
        ),
        feature_columns=(
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ),
    )
)


current_any_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_provisional
        ),
        feature_columns=(
            ALL_FROZEN_MEDICATION_FEATURES
        ),
    )
)


hybrid_any_encounters = (
    calculate_active_encounter_coverage(
        dataframe=(
            mcmed_s1_medication_features_hybrid_candidate
        ),
        feature_columns=(
            ALL_FROZEN_MEDICATION_FEATURES
        ),
    )
)



current_name_positive_columns = int(
    (
        mcmed_s1_medication_features_provisional[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


hybrid_name_positive_columns = int(
    (
        mcmed_s1_medication_features_hybrid_candidate[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


current_class_positive_columns = int(
    (
        mcmed_s1_medication_features_provisional[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)


hybrid_class_positive_columns = int(
    (
        mcmed_s1_medication_features_hybrid_candidate[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)



current_name_presence = (
    mcmed_s1_medication_features_provisional[
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ]
    .gt(0)
    .any(axis=1)
)


hybrid_name_presence = (
    mcmed_s1_medication_features_hybrid_candidate[
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ]
    .gt(0)
    .any(axis=1)
)


newly_recovered_name_encounter_mask = (
    ~current_name_presence
    &
    hybrid_name_presence
    &
    mcmed_s1_medication_features_hybrid_candidate[
        "CSN"
    ].isin(
        active_medication_csns
    )
)


newly_recovered_name_encounters = (
    mcmed_s1_medication_features_hybrid_candidate.loc[
        newly_recovered_name_encounter_mask,
        "CSN",
    ]
    .nunique()
)



excluded_multi_target_legacy_records = (
    mcmed_medication_mapping_method_comparison.loc[
        (
            mcmed_medication_mapping_method_comparison[
                "n_name_features"
            ].eq(0)
        )
        &
        (
            mcmed_medication_mapping_method_comparison[
                "legacy_n_name_matches"
            ].gt(1)
        )
    ]
    .copy()
    .sort_values(
        [
            "n_encounters",
            "legacy_n_name_matches",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)



n_active_medication_encounters = len(
    active_medication_csns
)


hybrid_mapping_summary = pd.DataFrame(
    [
        {
            "measure": (
                "Safe single-target legacy records accepted"
            ),
            "current": 0,
            "hybrid": len(
                safe_legacy_name_recovery_records
            ),
            "denominator": len(
                mcmed_medication_mapping_method_comparison
            ),
        },
        {
            "measure": (
                "Strict active rows receiving safe recovery"
            ),
            "current": 0,
            "hybrid": len(
                safe_recovery_active_rows
            ),
            "denominator": len(
                mcmed_active_medications_mapped
            ),
        },
        {
            "measure": (
                "Active encounters with mapped medication name"
            ),
            "current": current_name_encounters,
            "hybrid": hybrid_name_encounters,
            "denominator": n_active_medication_encounters,
        },
        {
            "measure": (
                "Active encounters with mapped medication class"
            ),
            "current": current_class_encounters,
            "hybrid": hybrid_class_encounters,
            "denominator": n_active_medication_encounters,
        },
        {
            "measure": (
                "Active encounters with any medication feature"
            ),
            "current": current_any_encounters,
            "hybrid": hybrid_any_encounters,
            "denominator": n_active_medication_encounters,
        },
        {
            "measure": (
                "Medication-name columns with external support"
            ),
            "current": current_name_positive_columns,
            "hybrid": hybrid_name_positive_columns,
            "denominator": len(
                FROZEN_PMH_MEDICATION_NAME_FEATURES
            ),
        },
        {
            "measure": (
                "Medication-class columns with external support"
            ),
            "current": current_class_positive_columns,
            "hybrid": hybrid_class_positive_columns,
            "denominator": len(
                FROZEN_PMH_MEDICATION_CLASS_FEATURES
            ),
        },
    ]
)


hybrid_mapping_summary[
    "current_pct"
] = (
    hybrid_mapping_summary[
        "current"
    ]
    /
    hybrid_mapping_summary[
        "denominator"
    ]
    * 100
)


hybrid_mapping_summary[
    "hybrid_pct"
] = (
    hybrid_mapping_summary[
        "hybrid"
    ]
    /
    hybrid_mapping_summary[
        "denominator"
    ]
    * 100
)


hybrid_mapping_summary[
    "absolute_gain"
] = (
    hybrid_mapping_summary[
        "hybrid"
    ]
    -
    hybrid_mapping_summary[
        "current"
    ]
)


hybrid_mapping_summary[
    "percentage_point_gain"
] = (
    hybrid_mapping_summary[
        "hybrid_pct"
    ]
    -
    hybrid_mapping_summary[
        "current_pct"
    ]
)


print(
    "Precision-preserving hybrid mapping summary:"
)

display(
    hybrid_mapping_summary.round(
        {
            "current_pct": 2,
            "hybrid_pct": 2,
            "percentage_point_gain": 2,
        }
    )
)


print(
    "\nHighest-frequency safe medication-name recoveries:"
)

display(
    safe_legacy_name_recovery_records[
        [
            "Name",
            "Generic_name",
            "legacy_name_match_method",
            "legacy_name_features",
            "n_active_rows",
            "n_encounters",
        ]
    ]
    .sort_values(
        "n_encounters",
        ascending=False,
    )
    .head(40)
)


print(
    "\nHighest-frequency multi-target legacy records excluded "
    "from automatic recovery:"
)

display(
    excluded_multi_target_legacy_records[
        [
            "Name",
            "Generic_name",
            "legacy_name_match_method",
            "legacy_n_name_matches",
            "legacy_name_features",
            "n_encounters",
        ]
    ]
    .head(30)
)


print(
    "\nNewly recovered active-medication encounters:",
    newly_recovered_name_encounters,
)


print(
    "\nHYBRID MEDICATION CANDIDATE CONSTRUCTION PASSED"
)

print(
    "Candidate matrix shape:",
    mcmed_s1_medication_features_hybrid_candidate.shape,
)

print(
    "Medication-name features:",
    len(
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ),
)

print(
    "Medication-class features:",
    len(
        FROZEN_PMH_MEDICATION_CLASS_FEATURES
    ),
)

print(
    "\nThe hybrid matrix has not yet been saved or frozen."
)


#### Freeze the final MC-MED medication block

The final MC-MED previous-medication representation uses the precision-preserving hybrid mapping. All high-confidence medication-name and medication-class mappings are retained. A legacy medication-name mapping is added only when the primary mapper found no medication-name feature and an exact n-gram-based rule identified one unique frozen target.

Multi-target medication-name mappings and unrestricted fuzzy medication-class mappings are excluded because they can activate overlapping brand, generic, ingredient and combination-product features from the same medication record.

The frozen feature inventory remains unchanged at 812 medication-name features and 364 medication-class features. No feature has been selected or removed using outcome values, model performance or SHAP results.

This cell freezes the medication block in memory. It is not saved separately because the diagnosis and medication blocks will next be combined into the complete S1 previous-history feature matrix.MC-MED medication records were mapped to the frozen MIMIC-IV-derived medication vocabulary using exact canonical, medication-class and identifier-based mappings. Unmatched medication names were additionally recovered only when an exact n-gram-based rule identified a single frozen target. Multi-target legacy mappings and unrestricted fuzzy class mappings were excluded to reduce synonymous and component-level feature inflation.


In [ ]:



required_objects = [
    "mcmed_s1_medication_features_hybrid_candidate",
    "mcmed_s0_validated",
    "FROZEN_PMH_MEDICATION_NAME_FEATURES",
    "FROZEN_PMH_MEDICATION_CLASS_FEATURES",
    "hybrid_mapping_summary",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Required objects are missing. Run the hybrid medication "
        f"construction cell first: {missing_objects}"
    )



FROZEN_PMH_MEDICATION_FEATURES = (
    list(FROZEN_PMH_MEDICATION_NAME_FEATURES)
    + list(FROZEN_PMH_MEDICATION_CLASS_FEATURES)
)

assert len(FROZEN_PMH_MEDICATION_NAME_FEATURES) == 812
assert len(FROZEN_PMH_MEDICATION_CLASS_FEATURES) == 364
assert len(FROZEN_PMH_MEDICATION_FEATURES) == 1176
assert len(set(FROZEN_PMH_MEDICATION_FEATURES)) == 1176



mcmed_s1_medication_features_final = (
    mcmed_s1_medication_features_hybrid_candidate[
        ["MRN", "CSN", "outcome", "outcome_binary"]
        + FROZEN_PMH_MEDICATION_FEATURES
    ]
    .copy()
    .reset_index(drop=True)
)


mcmed_s1_medication_features_final[
    FROZEN_PMH_MEDICATION_FEATURES
] = (
    mcmed_s1_medication_features_final[
        FROZEN_PMH_MEDICATION_FEATURES
    ]
    .fillna(0)
    .astype("int8")
)



assert (
    len(mcmed_s1_medication_features_final)
    == len(mcmed_s0_validated)
    == 6707
)

assert mcmed_s1_medication_features_final["CSN"].is_unique
assert mcmed_s1_medication_features_final["CSN"].notna().all()
assert mcmed_s1_medication_features_final["MRN"].notna().all()

assert (
    mcmed_s1_medication_features_final[
        FROZEN_PMH_MEDICATION_FEATURES
    ].columns.tolist()
    == FROZEN_PMH_MEDICATION_FEATURES
)

observed_binary_values = set(
    np.unique(
        mcmed_s1_medication_features_final[
            FROZEN_PMH_MEDICATION_FEATURES
        ].to_numpy()
    ).tolist()
)

assert observed_binary_values.issubset({0, 1})



name_positive_columns = int(
    (
        mcmed_s1_medication_features_final[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)

class_positive_columns = int(
    (
        mcmed_s1_medication_features_final[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ].sum(axis=0)
        > 0
    ).sum()
)

encounters_with_name = int(
    mcmed_s1_medication_features_final[
        FROZEN_PMH_MEDICATION_NAME_FEATURES
    ]
    .gt(0)
    .any(axis=1)
    .sum()
)

encounters_with_class = int(
    mcmed_s1_medication_features_final[
        FROZEN_PMH_MEDICATION_CLASS_FEATURES
    ]
    .gt(0)
    .any(axis=1)
    .sum()
)

encounters_with_any_medication = int(
    mcmed_s1_medication_features_final[
        FROZEN_PMH_MEDICATION_FEATURES
    ]
    .gt(0)
    .any(axis=1)
    .sum()
)



feature_order_text = "\n".join(
    FROZEN_PMH_MEDICATION_FEATURES
)

feature_order_sha256 = hashlib.sha256(
    feature_order_text.encode("utf-8")
).hexdigest()

matrix_values_sha256 = hashlib.sha256(
    pd.util.hash_pandas_object(
        mcmed_s1_medication_features_final[
            ["CSN"] + FROZEN_PMH_MEDICATION_FEATURES
        ],
        index=False,
    )
    .values
    .tobytes()
).hexdigest()


mcmed_s1_medication_freeze_metadata = {
    "stage": "S1",
    "dataset": "MC-MED",
    "feature_block": "previous_medications",
    "mapping_strategy": "precision_preserving_hybrid",
    "n_encounters": int(
        len(mcmed_s1_medication_features_final)
    ),
    "n_medication_name_features": int(
        len(FROZEN_PMH_MEDICATION_NAME_FEATURES)
    ),
    "n_medication_class_features": int(
        len(FROZEN_PMH_MEDICATION_CLASS_FEATURES)
    ),
    "n_total_medication_features": int(
        len(FROZEN_PMH_MEDICATION_FEATURES)
    ),
    "n_name_features_with_external_support": (
        name_positive_columns
    ),
    "n_class_features_with_external_support": (
        class_positive_columns
    ),
    "n_encounters_with_mapped_name": encounters_with_name,
    "n_encounters_with_mapped_class": encounters_with_class,
    "n_encounters_with_any_medication_feature": (
        encounters_with_any_medication
    ),
    "feature_order_sha256": feature_order_sha256,
    "matrix_values_sha256": matrix_values_sha256,
    "saved_to_disk": False,
}



mcmed_s1_medication_freeze_audit = pd.DataFrame(
    [
        {
            "measure": "Frozen cohort encounters",
            "value": len(
                mcmed_s1_medication_features_final
            ),
        },
        {
            "measure": "Medication-name features",
            "value": len(
                FROZEN_PMH_MEDICATION_NAME_FEATURES
            ),
        },
        {
            "measure": "Medication-class features",
            "value": len(
                FROZEN_PMH_MEDICATION_CLASS_FEATURES
            ),
        },
        {
            "measure": "Total medication features",
            "value": len(
                FROZEN_PMH_MEDICATION_FEATURES
            ),
        },
        {
            "measure": (
                "Medication-name features with external support"
            ),
            "value": name_positive_columns,
        },
        {
            "measure": (
                "Medication-class features with external support"
            ),
            "value": class_positive_columns,
        },
        {
            "measure": "Encounters with mapped medication name",
            "value": encounters_with_name,
        },
        {
            "measure": "Encounters with mapped medication class",
            "value": encounters_with_class,
        },
        {
            "measure": "Encounters with any medication feature",
            "value": encounters_with_any_medication,
        },
    ]
)


display(mcmed_s1_medication_freeze_audit)

print(
    "\nFeature-order SHA-256:",
    feature_order_sha256,
)

print(
    "Matrix-values SHA-256:",
    matrix_values_sha256,
)

print(
    "\nMC-MED S1 MEDICATION BLOCK FROZEN IN MEMORY"
)

print(
    "Shape:",
    mcmed_s1_medication_features_final.shape,
)

print(
    "No file has been written yet."
)

print(
    "Next step: combine this block with the frozen "
    "879-feature previous-diagnosis matrix."
)


#### Complete S1 previous-history feature block

The S1 previous-history block combines the frozen previous-diagnosis and previous-medication representations.

For each dataset, the final S1 feature space contains:

- 879 previous-diagnosis features;
- 812 previous-medication-name features;
- 364 previous-medication-class features;
- 2,055 previous-history features in total.

The feature inventory and feature order originate from the original MIMIC-IV development analysis. MC-MED is aligned to this same frozen vocabulary and does not introduce additional external features.

The diagnosis and medication matrices are joined using the unique ED encounter identifier. The patient identifier, approximate temporal split and binary outcome are taken from the validated S0 cohort rather than from individual feature blocks. This prevents duplicated or inconsistent metadata from entering the combined matrix.

All history variables remain binary. A value of 1 indicates that the corresponding historical diagnosis or medication concept was identified under the predefined temporal and harmonisation rules. A value of 0 indicates that the concept was not identified in the available data; it does not prove the clinical absence of that condition or medication.

Features with no positive values in the MIMIC-IV training split remain in the structural frozen inventory for reproducibility and cross-dataset alignment. They will be identified in an audit and can be removed inside the later model-training pipeline using training data only.


In [ ]:



required_objects = [
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "mimic_s1_diagnosis_features",
    "mcmed_s1_diagnosis_features",
    "mimic_s1_medication_features",
    "mcmed_s1_medication_features_final",
    "FROZEN_PMH_DIAGNOSIS_FEATURES",
    "FROZEN_PMH_MEDICATION_NAME_FEATURES",
    "FROZEN_PMH_MEDICATION_CLASS_FEATURES",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the diagnosis and medication reconstruction cells "
        f"before combining S1. Missing objects: {missing_objects}"
    )



FROZEN_PMH_DIAGNOSIS_FEATURES = list(
    FROZEN_PMH_DIAGNOSIS_FEATURES
)

FROZEN_PMH_MEDICATION_NAME_FEATURES = list(
    FROZEN_PMH_MEDICATION_NAME_FEATURES
)

FROZEN_PMH_MEDICATION_CLASS_FEATURES = list(
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


FROZEN_PMH_MEDICATION_FEATURES = (
    FROZEN_PMH_MEDICATION_NAME_FEATURES
    +
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
)


FROZEN_S1_HISTORY_FEATURES = (
    FROZEN_PMH_DIAGNOSIS_FEATURES
    +
    FROZEN_PMH_MEDICATION_FEATURES
)


assert len(
    FROZEN_PMH_DIAGNOSIS_FEATURES
) == 879

assert len(
    FROZEN_PMH_MEDICATION_NAME_FEATURES
) == 812

assert len(
    FROZEN_PMH_MEDICATION_CLASS_FEATURES
) == 364

assert len(
    FROZEN_PMH_MEDICATION_FEATURES
) == 1176

assert len(
    FROZEN_S1_HISTORY_FEATURES
) == 2055

assert len(
    set(FROZEN_S1_HISTORY_FEATURES)
) == 2055


diagnosis_medication_overlap = (
    set(FROZEN_PMH_DIAGNOSIS_FEATURES)
    &
    set(FROZEN_PMH_MEDICATION_FEATURES)
)

assert not diagnosis_medication_overlap, (
    "Diagnosis and medication feature names overlap: "
    f"{sorted(diagnosis_medication_overlap)[:20]}"
)



def validate_binary_feature_block(
    dataframe: pd.DataFrame,
    identifier_column: str,
    expected_identifiers: pd.Series,
    feature_columns: list[str],
    block_name: str,
) -> None:
    """
    Validate encounter coverage, feature order and binary values
    before combining one reconstructed feature block.
    """

    if identifier_column not in dataframe.columns:
        raise ValueError(
            f"{block_name} is missing identifier "
            f"{identifier_column}."
        )

    if not dataframe[
        identifier_column
    ].is_unique:
        raise ValueError(
            f"{block_name} contains duplicated "
            f"{identifier_column} values."
        )

    missing_feature_columns = [
        column
        for column in feature_columns
        if column not in dataframe.columns
    ]

    if missing_feature_columns:
        raise ValueError(
            f"{block_name} is missing "
            f"{len(missing_feature_columns)} expected features. "
            f"Examples: {missing_feature_columns[:20]}"
        )

    expected_identifier_set = set(
        expected_identifiers.tolist()
    )

    observed_identifier_set = set(
        dataframe[
            identifier_column
        ].tolist()
    )

    if (
        expected_identifier_set
        != observed_identifier_set
    ):
        missing_identifiers = (
            expected_identifier_set
            -
            observed_identifier_set
        )

        unexpected_identifiers = (
            observed_identifier_set
            -
            expected_identifier_set
        )

        raise ValueError(
            f"{block_name} encounter mismatch. "
            f"Missing encounters: {len(missing_identifiers)}; "
            f"unexpected encounters: "
            f"{len(unexpected_identifiers)}."
        )

    observed_values = set(
        np.unique(
            dataframe[
                feature_columns
            ]
            .fillna(0)
            .to_numpy()
        ).tolist()
    )

    if not observed_values.issubset(
        {
            0,
            1,
        }
    ):
        raise ValueError(
            f"{block_name} contains non-binary feature values: "
            f"{sorted(observed_values)[:20]}"
        )



validate_binary_feature_block(
    dataframe=mimic_s1_diagnosis_features,
    identifier_column="stay_id",
    expected_identifiers=(
        mimic_s0_validated["stay_id"]
    ),
    feature_columns=(
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ),
    block_name=(
        "MIMIC-IV S1 diagnosis block"
    ),
)


validate_binary_feature_block(
    dataframe=mimic_s1_medication_features,
    identifier_column="stay_id",
    expected_identifiers=(
        mimic_s0_validated["stay_id"]
    ),
    feature_columns=(
        FROZEN_PMH_MEDICATION_FEATURES
    ),
    block_name=(
        "MIMIC-IV S1 medication block"
    ),
)


validate_binary_feature_block(
    dataframe=mcmed_s1_diagnosis_features,
    identifier_column="CSN",
    expected_identifiers=(
        mcmed_s0_validated["CSN"]
    ),
    feature_columns=(
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ),
    block_name=(
        "MC-MED S1 diagnosis block"
    ),
)


validate_binary_feature_block(
    dataframe=(
        mcmed_s1_medication_features_final
    ),
    identifier_column="CSN",
    expected_identifiers=(
        mcmed_s0_validated["CSN"]
    ),
    feature_columns=(
        FROZEN_PMH_MEDICATION_FEATURES
    ),
    block_name=(
        "MC-MED S1 medication block"
    ),
)



mimic_diagnosis_block = (
    mimic_s1_diagnosis_features[
        ["stay_id"]
        +
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ]
    .copy()
)


mimic_medication_block = (
    mimic_s1_medication_features[
        ["stay_id"]
        +
        FROZEN_PMH_MEDICATION_FEATURES
    ]
    .copy()
)


mcmed_diagnosis_block = (
    mcmed_s1_diagnosis_features[
        ["CSN"]
        +
        FROZEN_PMH_DIAGNOSIS_FEATURES
    ]
    .copy()
)


mcmed_medication_block = (
    mcmed_s1_medication_features_final[
        ["CSN"]
        +
        FROZEN_PMH_MEDICATION_FEATURES
    ]
    .copy()
)



mimic_metadata_columns = [
    column
    for column in [
        "subject_id",
        "stay_id",
        "split",
        "outcome",
        "outcome_binary",
    ]
    if column in mimic_s0_validated.columns
]


mcmed_metadata_columns = [
    column
    for column in [
        "MRN",
        "CSN",
        "split",
        "outcome",
        "outcome_binary",
    ]
    if column in mcmed_s0_validated.columns
]


required_mimic_metadata = {
    "subject_id",
    "stay_id",
    "split",
    "outcome",
    "outcome_binary",
}

required_mcmed_metadata = {
    "MRN",
    "CSN",
    "split",
    "outcome",
    "outcome_binary",
}


missing_mimic_metadata = (
    required_mimic_metadata
    -
    set(mimic_metadata_columns)
)

missing_mcmed_metadata = (
    required_mcmed_metadata
    -
    set(mcmed_metadata_columns)
)


if missing_mimic_metadata:
    raise ValueError(
        "MIMIC S0 is missing required metadata: "
        f"{sorted(missing_mimic_metadata)}"
    )

if missing_mcmed_metadata:
    raise ValueError(
        "MC-MED S0 is missing required metadata: "
        f"{sorted(missing_mcmed_metadata)}"
    )



mimic_s1_history_features_final = (
    mimic_s0_validated[
        mimic_metadata_columns
    ]
    .merge(
        mimic_diagnosis_block,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mimic_medication_block,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
)


mcmed_s1_history_features_final = (
    mcmed_s0_validated[
        mcmed_metadata_columns
    ]
    .merge(
        mcmed_diagnosis_block,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mcmed_medication_block,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)



mimic_s1_history_features_final[
    FROZEN_S1_HISTORY_FEATURES
] = (
    mimic_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ]
    .fillna(0)
    .astype("int8")
)


mcmed_s1_history_features_final[
    FROZEN_S1_HISTORY_FEATURES
] = (
    mcmed_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ]
    .fillna(0)
    .astype("int8")
)



mimic_s1_history_features_final = (
    mimic_s1_history_features_final[
        mimic_metadata_columns
        +
        FROZEN_S1_HISTORY_FEATURES
    ]
    .copy()
)


mcmed_s1_history_features_final = (
    mcmed_s1_history_features_final[
        mcmed_metadata_columns
        +
        FROZEN_S1_HISTORY_FEATURES
    ]
    .copy()
)



assert len(
    mimic_s1_history_features_final
) == 8550

assert len(
    mcmed_s1_history_features_final
) == 6707

assert mimic_s1_history_features_final[
    "stay_id"
].is_unique

assert mcmed_s1_history_features_final[
    "CSN"
].is_unique


assert (
    mimic_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ].columns.tolist()
    ==
    FROZEN_S1_HISTORY_FEATURES
)


assert (
    mcmed_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ].columns.tolist()
    ==
    FROZEN_S1_HISTORY_FEATURES
)


for dataset_name, dataframe in [
    (
        "MIMIC-IV",
        mimic_s1_history_features_final,
    ),
    (
        "MC-MED",
        mcmed_s1_history_features_final,
    ),
]:
    observed_values = set(
        np.unique(
            dataframe[
                FROZEN_S1_HISTORY_FEATURES
            ].to_numpy()
        ).tolist()
    )

    assert observed_values.issubset(
        {
            0,
            1,
        }
    ), (
        f"{dataset_name} S1 matrix contains "
        f"non-binary values: {observed_values}"
    )



mimic_outcome_check = (
    mimic_s1_history_features_final[
        [
            "stay_id",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mimic_s0_validated[
            [
                "stay_id",
                "outcome",
                "outcome_binary",
            ]
        ],
        on="stay_id",
        how="inner",
        suffixes=(
            "_s1",
            "_s0",
        ),
        validate="one_to_one",
    )
)


mcmed_outcome_check = (
    mcmed_s1_history_features_final[
        [
            "CSN",
            "outcome",
            "outcome_binary",
        ]
    ]
    .merge(
        mcmed_s0_validated[
            [
                "CSN",
                "outcome",
                "outcome_binary",
            ]
        ],
        on="CSN",
        how="inner",
        suffixes=(
            "_s1",
            "_s0",
        ),
        validate="one_to_one",
    )
)


assert mimic_outcome_check[
    "outcome_s1"
].equals(
    mimic_outcome_check[
        "outcome_s0"
    ]
)

assert mimic_outcome_check[
    "outcome_binary_s1"
].equals(
    mimic_outcome_check[
        "outcome_binary_s0"
    ]
)


assert mcmed_outcome_check[
    "outcome_s1"
].equals(
    mcmed_outcome_check[
        "outcome_s0"
    ]
)

assert mcmed_outcome_check[
    "outcome_binary_s1"
].equals(
    mcmed_outcome_check[
        "outcome_binary_s0"
    ]
)



mimic_train_mask = (
    mimic_s1_history_features_final[
        "split"
    ].eq("train")
)


mimic_train_feature_support = (
    mimic_s1_history_features_final.loc[
        mimic_train_mask,
        FROZEN_S1_HISTORY_FEATURES,
    ]
    .sum(axis=0)
    .astype("int64")
)


mimic_all_feature_support = (
    mimic_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ]
    .sum(axis=0)
    .astype("int64")
)


mcmed_external_feature_support = (
    mcmed_s1_history_features_final[
        FROZEN_S1_HISTORY_FEATURES
    ]
    .sum(axis=0)
    .astype("int64")
)


s1_feature_support_audit = pd.DataFrame(
    {
        "feature_name": (
            FROZEN_S1_HISTORY_FEATURES
        ),
        "feature_family": (
            ["PREVIOUS_DIAGNOSIS"]
            *
            len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            )
            +
            ["PREVIOUS_MEDICATION_NAME"]
            *
            len(
                FROZEN_PMH_MEDICATION_NAME_FEATURES
            )
            +
            ["PREVIOUS_MEDICATION_CLASS"]
            *
            len(
                FROZEN_PMH_MEDICATION_CLASS_FEATURES
            )
        ),
        "mimic_train_positive_count": (
            mimic_train_feature_support.reindex(
                FROZEN_S1_HISTORY_FEATURES
            ).to_numpy()
        ),
        "mimic_all_positive_count": (
            mimic_all_feature_support.reindex(
                FROZEN_S1_HISTORY_FEATURES
            ).to_numpy()
        ),
        "mcmed_external_positive_count": (
            mcmed_external_feature_support.reindex(
                FROZEN_S1_HISTORY_FEATURES
            ).to_numpy()
        ),
    }
)


s1_feature_support_audit[
    "zero_in_mimic_training"
] = (
    s1_feature_support_audit[
        "mimic_train_positive_count"
    ].eq(0)
)


s1_feature_support_audit[
    "zero_in_all_mimic"
] = (
    s1_feature_support_audit[
        "mimic_all_positive_count"
    ].eq(0)
)


s1_feature_support_audit[
    "present_in_mcmed"
] = (
    s1_feature_support_audit[
        "mcmed_external_positive_count"
    ].gt(0)
)


MIMIC_S1_TRAIN_NONZERO_FEATURES = (
    s1_feature_support_audit.loc[
        ~s1_feature_support_audit[
            "zero_in_mimic_training"
        ],
        "feature_name",
    ]
    .tolist()
)



def encounter_history_summary(
    dataframe: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:
    """Summarise encounter-level S1 history availability."""

    diagnosis_present = (
        dataframe[
            FROZEN_PMH_DIAGNOSIS_FEATURES
        ]
        .gt(0)
        .any(axis=1)
    )

    medication_name_present = (
        dataframe[
            FROZEN_PMH_MEDICATION_NAME_FEATURES
        ]
        .gt(0)
        .any(axis=1)
    )

    medication_class_present = (
        dataframe[
            FROZEN_PMH_MEDICATION_CLASS_FEATURES
        ]
        .gt(0)
        .any(axis=1)
    )

    medication_any_present = (
        medication_name_present
        |
        medication_class_present
    )

    history_any_present = (
        diagnosis_present
        |
        medication_any_present
    )

    return pd.DataFrame(
        [
            {
                "dataset": dataset_name,
                "measure": (
                    "Total cohort encounters"
                ),
                "count": len(dataframe),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with >=1 mapped "
                    "previous diagnosis"
                ),
                "count": int(
                    diagnosis_present.sum()
                ),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with >=1 mapped "
                    "medication name"
                ),
                "count": int(
                    medication_name_present.sum()
                ),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with >=1 mapped "
                    "medication class"
                ),
                "count": int(
                    medication_class_present.sum()
                ),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with >=1 mapped "
                    "medication feature"
                ),
                "count": int(
                    medication_any_present.sum()
                ),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with >=1 mapped "
                    "S1 history feature"
                ),
                "count": int(
                    history_any_present.sum()
                ),
            },
            {
                "dataset": dataset_name,
                "measure": (
                    "Encounters with no mapped "
                    "S1 history feature"
                ),
                "count": int(
                    (~history_any_present).sum()
                ),
            },
        ]
    )


s1_encounter_history_audit = pd.concat(
    [
        encounter_history_summary(
            dataframe=(
                mimic_s1_history_features_final
            ),
            dataset_name="MIMIC-IV",
        ),
        encounter_history_summary(
            dataframe=(
                mcmed_s1_history_features_final
            ),
            dataset_name="MC-MED",
        ),
    ],
    ignore_index=True,
)


s1_encounter_history_audit[
    "percentage"
] = (
    s1_encounter_history_audit.groupby(
        "dataset"
    )["count"].transform(
        lambda values: (
            values
            /
            values.iloc[0]
            *
            100
        )
    )
)



s1_feature_family_audit = (
    s1_feature_support_audit
    .groupby(
        "feature_family",
        sort=False,
    )
    .agg(
        frozen_features=(
            "feature_name",
            "size",
        ),
        positive_in_mimic_train=(
            "mimic_train_positive_count",
            lambda series: int(
                series.gt(0).sum()
            ),
        ),
        positive_in_all_mimic=(
            "mimic_all_positive_count",
            lambda series: int(
                series.gt(0).sum()
            ),
        ),
        positive_in_mcmed=(
            "mcmed_external_positive_count",
            lambda series: int(
                series.gt(0).sum()
            ),
        ),
        zero_in_mimic_training=(
            "zero_in_mimic_training",
            "sum",
        ),
    )
    .reset_index()
)



s1_feature_order_text = "\n".join(
    FROZEN_S1_HISTORY_FEATURES
)


s1_feature_order_sha256 = hashlib.sha256(
    s1_feature_order_text.encode(
        "utf-8"
    )
).hexdigest()


mimic_s1_matrix_sha256 = hashlib.sha256(
    pd.util.hash_pandas_object(
        mimic_s1_history_features_final[
            ["stay_id"]
            +
            FROZEN_S1_HISTORY_FEATURES
        ],
        index=False,
    )
    .values
    .tobytes()
).hexdigest()


mcmed_s1_matrix_sha256 = hashlib.sha256(
    pd.util.hash_pandas_object(
        mcmed_s1_history_features_final[
            ["CSN"]
            +
            FROZEN_S1_HISTORY_FEATURES
        ],
        index=False,
    )
    .values
    .tobytes()
).hexdigest()


s1_freeze_metadata = {
    "stage": "S1",
    "feature_block": "previous_history",
    "n_diagnosis_features": 879,
    "n_medication_name_features": 812,
    "n_medication_class_features": 364,
    "n_total_features": 2055,
    "n_mimic_encounters": int(
        len(
            mimic_s1_history_features_final
        )
    ),
    "n_mcmed_encounters": int(
        len(
            mcmed_s1_history_features_final
        )
    ),
    "n_mimic_training_nonzero_features": int(
        len(
            MIMIC_S1_TRAIN_NONZERO_FEATURES
        )
    ),
    "feature_order_sha256": (
        s1_feature_order_sha256
    ),
    "mimic_matrix_sha256": (
        mimic_s1_matrix_sha256
    ),
    "mcmed_matrix_sha256": (
        mcmed_s1_matrix_sha256
    ),
    "saved_to_disk": False,
}



print(
    "Complete S1 feature-family audit:"
)

display(
    s1_feature_family_audit
)


print(
    "\nS1 encounter-level history availability:"
)

display(
    s1_encounter_history_audit.round(
        {
            "percentage": 2,
        }
    )
)


print(
    "\nS1 matrix summary:"
)

display(
    pd.DataFrame(
        [
            {
                "dataset": "MIMIC-IV",
                "n_rows": len(
                    mimic_s1_history_features_final
                ),
                "n_metadata_columns": len(
                    mimic_metadata_columns
                ),
                "n_history_features": len(
                    FROZEN_S1_HISTORY_FEATURES
                ),
                "total_columns": (
                    mimic_s1_history_features_final
                    .shape[1]
                ),
            },
            {
                "dataset": "MC-MED",
                "n_rows": len(
                    mcmed_s1_history_features_final
                ),
                "n_metadata_columns": len(
                    mcmed_metadata_columns
                ),
                "n_history_features": len(
                    FROZEN_S1_HISTORY_FEATURES
                ),
                "total_columns": (
                    mcmed_s1_history_features_final
                    .shape[1]
                ),
            },
        ]
    )
)


print(
    "\nFrozen S1 feature order SHA-256:",
    s1_feature_order_sha256,
)

print(
    "MIMIC-IV S1 matrix SHA-256:",
    mimic_s1_matrix_sha256,
)

print(
    "MC-MED S1 matrix SHA-256:",
    mcmed_s1_matrix_sha256,
)


print(
    "\nCOMPLETE S1 PREVIOUS-HISTORY CONSTRUCTION PASSED"
)

print(
    "Structural frozen S1 features:",
    len(
        FROZEN_S1_HISTORY_FEATURES
    ),
)

print(
    "MIMIC training non-zero S1 features:",
    len(
        MIMIC_S1_TRAIN_NONZERO_FEATURES
    ),
)

print(
    "\nNo S1 file has been written yet."
)


In [ ]:



if "PROJECT_ROOT" not in globals():
    raise NameError(
        "PROJECT_ROOT is not defined. Run the project-path "
        "configuration cell first."
    )


PROCESSED_DATA_DIR = (
    Path(PROJECT_ROOT)
    / "data"
    / "processed"
)

S1_OUTPUT_DIR = (
    PROCESSED_DATA_DIR
    / "s1_previous_history"
)

S1_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



MIMIC_S1_HISTORY_PATH = (
    S1_OUTPUT_DIR
    / "mimiciv_s1_previous_history.parquet"
)

MCMED_S1_HISTORY_PATH = (
    S1_OUTPUT_DIR
    / "mcmed_s1_previous_history.parquet"
)

S1_FEATURE_MANIFEST_PATH = (
    S1_OUTPUT_DIR
    / "s1_previous_history_feature_manifest.csv"
)

S1_FEATURE_SUPPORT_PATH = (
    S1_OUTPUT_DIR
    / "s1_previous_history_feature_support.csv"
)

S1_ENCOUNTER_AUDIT_PATH = (
    S1_OUTPUT_DIR
    / "s1_previous_history_encounter_audit.csv"
)

S1_METADATA_PATH = (
    S1_OUTPUT_DIR
    / "s1_previous_history_metadata.json"
)



mimic_s1_history_features_final.to_parquet(
    MIMIC_S1_HISTORY_PATH,
    index=False,
)

mcmed_s1_history_features_final.to_parquet(
    MCMED_S1_HISTORY_PATH,
    index=False,
)



s1_feature_manifest = pd.DataFrame(
    {
        "feature_position": range(
            1,
            len(
                FROZEN_S1_HISTORY_FEATURES
            ) + 1,
        ),
        "feature_name": (
            FROZEN_S1_HISTORY_FEATURES
        ),
        "stage": "S1",
        "feature_family": (
            ["PREVIOUS_DIAGNOSIS"]
            *
            len(
                FROZEN_PMH_DIAGNOSIS_FEATURES
            )
            +
            ["PREVIOUS_MEDICATION_NAME"]
            *
            len(
                FROZEN_PMH_MEDICATION_NAME_FEATURES
            )
            +
            ["PREVIOUS_MEDICATION_CLASS"]
            *
            len(
                FROZEN_PMH_MEDICATION_CLASS_FEATURES
            )
        ),
        "data_type": "binary_int8",
        "structurally_frozen": True,
        "primary_model_candidate": True,
    }
)


s1_feature_manifest.to_csv(
    S1_FEATURE_MANIFEST_PATH,
    index=False,
)



s1_feature_support_audit.to_csv(
    S1_FEATURE_SUPPORT_PATH,
    index=False,
)

s1_encounter_history_audit.to_csv(
    S1_ENCOUNTER_AUDIT_PATH,
    index=False,
)


s1_freeze_metadata_saved = (
    s1_freeze_metadata.copy()
)

s1_freeze_metadata_saved[
    "saved_to_disk"
] = True

s1_freeze_metadata_saved[
    "mimic_s1_history_path"
] = str(
    MIMIC_S1_HISTORY_PATH
)

s1_freeze_metadata_saved[
    "mcmed_s1_history_path"
] = str(
    MCMED_S1_HISTORY_PATH
)

s1_freeze_metadata_saved[
    "feature_manifest_path"
] = str(
    S1_FEATURE_MANIFEST_PATH
)

s1_freeze_metadata_saved[
    "feature_support_path"
] = str(
    S1_FEATURE_SUPPORT_PATH
)


with open(
    S1_METADATA_PATH,
    "w",
    encoding="utf-8",
) as metadata_file:
    json.dump(
        s1_freeze_metadata_saved,
        metadata_file,
        indent=2,
    )



mimic_s1_reload = pd.read_parquet(
    MIMIC_S1_HISTORY_PATH
)

mcmed_s1_reload = pd.read_parquet(
    MCMED_S1_HISTORY_PATH
)

manifest_reload = pd.read_csv(
    S1_FEATURE_MANIFEST_PATH
)


assert mimic_s1_reload.shape == (
    mimic_s1_history_features_final.shape
)

assert mcmed_s1_reload.shape == (
    mcmed_s1_history_features_final.shape
)

assert manifest_reload[
    "feature_name"
].tolist() == FROZEN_S1_HISTORY_FEATURES

assert len(
    manifest_reload
) == 2055



s1_saved_file_audit = pd.DataFrame(
    [
        {
            "output": (
                "MIMIC-IV S1 previous-history matrix"
            ),
            "exists": (
                MIMIC_S1_HISTORY_PATH.exists()
            ),
            "size_mb": round(
                MIMIC_S1_HISTORY_PATH.stat().st_size
                / 1024**2,
                2,
            ),
            "path": str(
                MIMIC_S1_HISTORY_PATH
            ),
        },
        {
            "output": (
                "MC-MED S1 previous-history matrix"
            ),
            "exists": (
                MCMED_S1_HISTORY_PATH.exists()
            ),
            "size_mb": round(
                MCMED_S1_HISTORY_PATH.stat().st_size
                / 1024**2,
                2,
            ),
            "path": str(
                MCMED_S1_HISTORY_PATH
            ),
        },
        {
            "output": (
                "S1 feature manifest"
            ),
            "exists": (
                S1_FEATURE_MANIFEST_PATH.exists()
            ),
            "size_mb": round(
                S1_FEATURE_MANIFEST_PATH.stat().st_size
                / 1024**2,
                2,
            ),
            "path": str(
                S1_FEATURE_MANIFEST_PATH
            ),
        },
        {
            "output": (
                "S1 feature-support audit"
            ),
            "exists": (
                S1_FEATURE_SUPPORT_PATH.exists()
            ),
            "size_mb": round(
                S1_FEATURE_SUPPORT_PATH.stat().st_size
                / 1024**2,
                2,
            ),
            "path": str(
                S1_FEATURE_SUPPORT_PATH
            ),
        },
        {
            "output": (
                "S1 metadata"
            ),
            "exists": (
                S1_METADATA_PATH.exists()
            ),
            "size_mb": round(
                S1_METADATA_PATH.stat().st_size
                / 1024**2,
                4,
            ),
            "path": str(
                S1_METADATA_PATH
            ),
        },
    ]
)


display(
    s1_saved_file_audit
)


print(
    "\nCOMPLETE S1 PREVIOUS-HISTORY BLOCK SAVED"
)

print(
    "MIMIC-IV matrix:",
    mimic_s1_reload.shape,
)

print(
    "MC-MED matrix:",
    mcmed_s1_reload.shape,
)

print(
    "Frozen feature manifest:",
    len(
        manifest_reload
    ),
)

print(
    "\nThese files can now be loaded independently "
    "by Notebook 2."
)


#### S2 objective work-up inventory and source audit

The S2 feature block represents objective investigation findings available during the current emergency-department work-up.

The frozen S2 inventory contains 431 binary features:

- 351 abnormal laboratory-result features;
- 80 abnormal imaging/radiology-result features.

These features were defined during the original MIMIC-IV development analysis and subsequently used during external validation and clinician comparison. Their names and order are therefore preserved exactly. MC-MED is aligned to the same frozen MIMIC-IV-derived feature space and does not introduce new external features.

A value of 1 indicates that the corresponding investigation produced an explicitly abnormal result under the predefined mapping and abnormality rules. A value of 0 indicates that no mapped abnormal result was identified. It may represent a normal result, an investigation that was not performed, an unavailable result, or an externally unmapped result. These situations will be audited separately rather than interpreted as clinically equivalent.

Laboratory and imaging features are reconstructed separately. Laboratory abnormality is primarily derived from structured abnormal-result flags. Imaging abnormality requires exam-to-report alignment followed by rule-based interpretation of radiology impression text.

This initial section loads the exact frozen S2 vocabulary, confirms its feature order and audits the required source-table schemas. It does not yet create the S2 feature matrices.


In [ ]:



if "PROJECT_ROOT" not in globals():
    raise NameError(
        "PROJECT_ROOT is not defined. Run the project-path "
        "configuration cell before this section."
    )


PROJECT_ROOT = Path(PROJECT_ROOT).resolve()



def find_unique_project_file(
    filename: str,
) -> Path:
    """
    Find one exact filename within the publication project.

    An error is raised when the file is absent or when multiple
    copies exist, because silently choosing one copy could make
    the reconstruction non-reproducible.
    """

    matches = [
        path.resolve()
        for path in PROJECT_ROOT.rglob(filename)
        if ".git" not in path.parts
    ]

    unique_matches = sorted(
        set(matches)
    )

    if len(unique_matches) == 0:
        raise FileNotFoundError(
            f"{filename} was not found under:\n"
            f"{PROJECT_ROOT}"
        )

    if len(unique_matches) > 1:
        display(
            pd.DataFrame(
                {
                    "duplicate_candidate_path": [
                        str(path)
                        for path in unique_matches
                    ]
                }
            )
        )

        raise ValueError(
            f"Multiple copies of {filename} were found. "
            "Keep one authoritative copy or provide its exact path."
        )

    return unique_matches[0]



required_s2_filenames = [
    "frozen_s1_s2_feature_manifest.csv",
    "legacy_path5_feature_order.csv",
    "s2_lab_result_vocabulary.csv",
    "s2_lab_feature_plan.csv",
    "s2_lab_feature_audit.csv",
    "s2_imaging_result_vocabulary.csv",
    "s2_imaging_feature_summary.csv",
    "s2_imaging_overall_summary.csv",
    "frozen_feature_metadata.json",
]


S2_FROZEN_PATHS = {
    filename: find_unique_project_file(
        filename
    )
    for filename in required_s2_filenames
}



frozen_s1_s2_manifest = pd.read_csv(
    S2_FROZEN_PATHS[
        "frozen_s1_s2_feature_manifest.csv"
    ],
    low_memory=False,
)

legacy_path5_feature_order = pd.read_csv(
    S2_FROZEN_PATHS[
        "legacy_path5_feature_order.csv"
    ],
    low_memory=False,
)

s2_lab_result_vocabulary = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_lab_result_vocabulary.csv"
    ],
    low_memory=False,
)

s2_lab_feature_plan = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_lab_feature_plan.csv"
    ],
    low_memory=False,
)

s2_lab_feature_audit_original = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_lab_feature_audit.csv"
    ],
    low_memory=False,
)

s2_imaging_result_vocabulary = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_imaging_result_vocabulary.csv"
    ],
    low_memory=False,
)

s2_imaging_feature_summary = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_imaging_feature_summary.csv"
    ],
    low_memory=False,
)

s2_imaging_overall_summary = pd.read_csv(
    S2_FROZEN_PATHS[
        "s2_imaging_overall_summary.csv"
    ],
    low_memory=False,
)


with open(
    S2_FROZEN_PATHS[
        "frozen_feature_metadata.json"
    ],
    "r",
    encoding="utf-8",
) as metadata_file:
    frozen_feature_metadata = json.load(
        metadata_file
    )



def resolve_first_column(
    dataframe: pd.DataFrame,
    candidates: list[str],
    object_name: str,
) -> str:
    """Return the first existing candidate column."""

    resolved_column = next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )

    if resolved_column is None:
        raise ValueError(
            f"No recognised column was found in {object_name}. "
            f"Expected one of {candidates}. "
            f"Available columns: {dataframe.columns.tolist()}"
        )

    return resolved_column


manifest_feature_column = resolve_first_column(
    dataframe=frozen_s1_s2_manifest,
    candidates=[
        "final_feature_name",
        "feature_name",
        "model_feature_name",
        "column_name",
    ],
    object_name="frozen_s1_s2_manifest",
)


manifest_working = (
    frozen_s1_s2_manifest.copy()
)

manifest_working[
    "_resolved_feature_name"
] = (
    manifest_working[
        manifest_feature_column
    ]
    .astype("string")
    .str.strip()
)


manifest_working = (
    manifest_working.loc[
        manifest_working[
            "_resolved_feature_name"
        ].notna()
        &
        manifest_working[
            "_resolved_feature_name"
        ].ne("")
    ]
    .drop_duplicates(
        subset="_resolved_feature_name",
        keep="first",
    )
    .reset_index(drop=True)
)



FROZEN_S2_LAB_FEATURES = (
    manifest_working.loc[
        manifest_working[
            "_resolved_feature_name"
        ].str.startswith(
            "poe_result__lab__",
            na=False,
        ),
        "_resolved_feature_name",
    ]
    .astype(str)
    .tolist()
)


FROZEN_S2_IMAGING_FEATURES = (
    manifest_working.loc[
        manifest_working[
            "_resolved_feature_name"
        ].str.startswith(
            "poe_result__imaging_radiology",
            na=False,
        ),
        "_resolved_feature_name",
    ]
    .astype(str)
    .tolist()
)


FROZEN_S2_FEATURES = (
    FROZEN_S2_LAB_FEATURES
    +
    FROZEN_S2_IMAGING_FEATURES
)



assert len(
    FROZEN_S2_LAB_FEATURES
) == 351, (
    "Expected 351 laboratory-result features, but recovered "
    f"{len(FROZEN_S2_LAB_FEATURES)}."
)

assert len(
    FROZEN_S2_IMAGING_FEATURES
) == 80, (
    "Expected 80 imaging-result features, but recovered "
    f"{len(FROZEN_S2_IMAGING_FEATURES)}."
)

assert len(
    FROZEN_S2_FEATURES
) == 431

assert len(
    set(FROZEN_S2_FEATURES)
) == 431


lab_imaging_overlap = (
    set(FROZEN_S2_LAB_FEATURES)
    &
    set(FROZEN_S2_IMAGING_FEATURES)
)

assert not lab_imaging_overlap



s2_frozen_file_audit_rows = []

for filename, path in (
    S2_FROZEN_PATHS.items()
):
    if path.suffix.lower() == ".csv":
        dataframe = pd.read_csv(
            path,
            nrows=5,
            low_memory=False,
        )

        n_columns = len(
            dataframe.columns
        )

        columns_preview = " | ".join(
            dataframe.columns.tolist()
        )

    else:
        n_columns = pd.NA
        columns_preview = (
            "JSON metadata"
        )

    s2_frozen_file_audit_rows.append(
        {
            "filename": filename,
            "exists": path.exists(),
            "size_mb": round(
                path.stat().st_size
                / 1024**2,
                4,
            ),
            "n_columns": n_columns,
            "columns": columns_preview,
            "path": str(path),
        }
    )


s2_frozen_file_audit = pd.DataFrame(
    s2_frozen_file_audit_rows
)


s2_frozen_inventory_audit = pd.DataFrame(
    [
        {
            "stage": "S2",
            "feature_family": (
                "Abnormal laboratory results"
            ),
            "n_features": len(
                FROZEN_S2_LAB_FEATURES
            ),
            "expected": 351,
        },
        {
            "stage": "S2",
            "feature_family": (
                "Abnormal imaging results"
            ),
            "n_features": len(
                FROZEN_S2_IMAGING_FEATURES
            ),
            "expected": 80,
        },
        {
            "stage": "S2",
            "feature_family": (
                "Complete objective investigation block"
            ),
            "n_features": len(
                FROZEN_S2_FEATURES
            ),
            "expected": 431,
        },
    ]
)


print(
    "Frozen S2 inventory:"
)

display(
    s2_frozen_inventory_audit
)


print(
    "\nFrozen S2 file audit:"
)

display(
    s2_frozen_file_audit
)


print(
    "\nFirst 15 frozen laboratory-result features:"
)

display(
    pd.DataFrame(
        {
            "feature_position": range(
                1,
                min(
                    15,
                    len(
                        FROZEN_S2_LAB_FEATURES
                    ),
                ) + 1,
            ),
            "feature_name": (
                FROZEN_S2_LAB_FEATURES[:15]
            ),
        }
    )
)


print(
    "\nFirst 15 frozen imaging-result features:"
)

display(
    pd.DataFrame(
        {
            "feature_position": range(
                1,
                min(
                    15,
                    len(
                        FROZEN_S2_IMAGING_FEATURES
                    ),
                ) + 1,
            ),
            "feature_name": (
                FROZEN_S2_IMAGING_FEATURES[:15]
            ),
        }
    )
)


print(
    "\nS2 LAB FEATURE-PLAN COLUMNS:"
)

print(
    s2_lab_feature_plan.columns.tolist()
)

display(
    s2_lab_feature_plan.head(10)
)


print(
    "\nS2 LAB VOCABULARY COLUMNS:"
)

print(
    s2_lab_result_vocabulary.columns.tolist()
)

display(
    s2_lab_result_vocabulary.head(10)
)


print(
    "\nS2 IMAGING FEATURE-SUMMARY COLUMNS:"
)

print(
    s2_imaging_feature_summary.columns.tolist()
)

display(
    s2_imaging_feature_summary.head(10)
)


print(
    "\nS2 IMAGING VOCABULARY COLUMNS:"
)

print(
    s2_imaging_result_vocabulary.columns.tolist()
)

display(
    s2_imaging_result_vocabulary.head(10)
)


print(
    "\nFROZEN S2 INVENTORY LOAD PASSED"
)

print(
    "Laboratory-result features:",
    len(
        FROZEN_S2_LAB_FEATURES
    ),
)

print(
    "Imaging-result features:",
    len(
        FROZEN_S2_IMAGING_FEATURES
    ),
)

print(
    "Total S2 features:",
    len(
        FROZEN_S2_FEATURES
    ),
)

print(
    "\nNo S2 matrix has been created or saved."
)


In [ ]:



required_objects = [
    "SOURCE_PATHS",
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "FROZEN_S2_LAB_FEATURES",
    "FROZEN_S2_IMAGING_FEATURES",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding cohort and frozen-inventory cells. "
        f"Missing objects: {missing_objects}"
    )



required_s2_source_keys = {
    "MIMIC-IV laboratory events":
        "mimic_labevents",

    "MIMIC-IV laboratory dictionary":
        "mimic_d_labitems",

    "MC-MED laboratory results":
        "mcmed_labs",

    "MIMIC-IV radiology reports":
        "mimic_radiology_notes",

    "MIMIC-IV radiology detail":
        "mimic_radiology_detail",

    "MC-MED radiology results":
        "mcmed_rads",
}


missing_source_keys = [
    source_key
    for source_key
    in required_s2_source_keys.values()
    if source_key not in SOURCE_PATHS
]

if missing_source_keys:
    raise KeyError(
        "SOURCE_PATHS is missing required S2 entries: "
        f"{missing_source_keys}"
    )



s2_source_schema_rows = []
S2_SOURCE_HEADERS = {}


for (
    source_description,
    source_key,
) in required_s2_source_keys.items():

    source_path = Path(
        SOURCE_PATHS[source_key]
    ).resolve()

    if not source_path.exists():
        raise FileNotFoundError(
            f"{source_description} was not found:\n"
            f"{source_path}"
        )

    source_header = pd.read_csv(
        source_path,
        nrows=0,
    )

    source_columns = (
        source_header.columns.tolist()
    )

    S2_SOURCE_HEADERS[
        source_key
    ] = source_columns

    s2_source_schema_rows.append(
        {
            "source": source_description,
            "source_key": source_key,
            "size_mb": round(
                source_path.stat().st_size
                / 1024**2,
                2,
            ),
            "n_columns": len(
                source_columns
            ),
            "columns": " | ".join(
                source_columns
            ),
            "path": str(
                source_path
            ),
        }
    )


s2_source_schema_audit = pd.DataFrame(
    s2_source_schema_rows
)



source_role_candidates = {
    "mimic_labevents": {
        "patient_identifier": [
            "subject_id",
        ],
        "hospital_identifier": [
            "hadm_id",
        ],
        "lab_item_identifier": [
            "itemid",
        ],
        "specimen_time": [
            "charttime",
        ],
        "result_available_time": [
            "storetime",
        ],
        "result_value": [
            "value",
            "valuenum",
        ],
        "result_unit": [
            "valueuom",
        ],
        "abnormal_flag": [
            "flag",
        ],
        "lower_reference": [
            "ref_range_lower",
        ],
        "upper_reference": [
            "ref_range_upper",
        ],
    },

    "mimic_d_labitems": {
        "lab_item_identifier": [
            "itemid",
        ],
        "lab_name": [
            "label",
        ],
        "fluid": [
            "fluid",
        ],
        "category": [
            "category",
        ],
    },

    "mcmed_labs": {
        "encounter_identifier": [
            "CSN",
        ],
        "order_time": [
            "Order_time",
        ],
        "result_available_time": [
            "Result_time",
        ],
        "order_name": [
            "Display_name",
        ],
        "component_name": [
            "Component_name",
        ],
        "component_result": [
            "Component_result",
            "Component_value",
        ],
        "component_unit": [
            "Component_units",
        ],
        "component_abnormal_flag": [
            "Component_abnormal",
        ],
        "order_abnormal_flag": [
            "Abnormal",
        ],
        "lower_reference": [
            "Component_nml_low",
        ],
        "upper_reference": [
            "Component_nml_high",
        ],
    },

    "mimic_radiology_notes": {
        "patient_identifier": [
            "subject_id",
        ],
        "hospital_identifier": [
            "hadm_id",
        ],
        "note_identifier": [
            "note_id",
        ],
        "exam_time": [
            "charttime",
        ],
        "report_available_time": [
            "storetime",
        ],
        "report_text": [
            "text",
        ],
    },

    "mimic_radiology_detail": {
        "note_identifier": [
            "note_id",
        ],
        "exam_name_or_metadata": [
            "field_name",
            "field_value",
        ],
    },

    "mcmed_rads": {
        "encounter_identifier": [
            "CSN",
        ],
        "patient_identifier": [
            "MRN",
        ],
        "order_time": [
            "Order_time",
            "order_time",
        ],
        "result_available_time": [
            "Result_time",
            "result_time",
        ],
        "exam_name": [
            "Display_name",
            "Exam_name",
            "Procedure_name",
            "Study",
        ],
        "report_text": [
            "Result",
            "Report",
            "Impression",
            "Narrative",
            "Text",
        ],
    },
}


detected_role_rows = []


for (
    source_key,
    role_definitions,
) in source_role_candidates.items():

    source_columns = (
        S2_SOURCE_HEADERS[
            source_key
        ]
    )

    lower_to_original = {
        str(column).lower():
            column
        for column in source_columns
    }

    for (
        role,
        candidate_columns,
    ) in role_definitions.items():

        detected_columns = []

        for candidate in candidate_columns:
            exact_match = (
                lower_to_original.get(
                    candidate.lower()
                )
            )

            if exact_match is not None:
                detected_columns.append(
                    exact_match
                )

        detected_role_rows.append(
            {
                "source_key": source_key,
                "role": role,
                "detected_columns": (
                    " | ".join(
                        detected_columns
                    )
                ),
                "role_available": (
                    len(
                        detected_columns
                    )
                    > 0
                ),
            }
        )


s2_detected_source_roles = pd.DataFrame(
    detected_role_rows
)



mimic_timing_candidates = [
    "subject_id",
    "stay_id",
    "hadm_id",
    "intime",
    "outtime",
    "split",
    "outcome",
    "outcome_binary",
]


mcmed_timing_candidates = [
    "MRN",
    "CSN",
    "Arrival_time",
    "current_arrival_time",
    "Departure_time",
    "Dispo_time",
    "split",
    "outcome",
    "outcome_binary",
]


cohort_timing_rows = []


for (
    dataset_name,
    dataframe,
    candidates,
) in [
    (
        "MIMIC-IV",
        mimic_s0_validated,
        mimic_timing_candidates,
    ),
    (
        "MC-MED",
        mcmed_s0_validated,
        mcmed_timing_candidates,
    ),
]:

    for column in candidates:
        cohort_timing_rows.append(
            {
                "dataset": dataset_name,
                "object": (
                    "mimic_s0_validated"
                    if dataset_name == "MIMIC-IV"
                    else "mcmed_s0_validated"
                ),
                "column": column,
                "present": (
                    column
                    in dataframe.columns
                ),
                "dtype": (
                    str(
                        dataframe[
                            column
                        ].dtype
                    )
                    if column
                    in dataframe.columns
                    else None
                ),
                "missing_n": (
                    int(
                        dataframe[
                            column
                        ].isna().sum()
                    )
                    if column
                    in dataframe.columns
                    else pd.NA
                ),
            }
        )


if (
    "mcmed_primary_history_source"
    in globals()
    and isinstance(
        mcmed_primary_history_source,
        pd.DataFrame,
    )
):
    for column in [
        "MRN",
        "CSN",
        "current_arrival_time",
    ]:
        cohort_timing_rows.append(
            {
                "dataset": "MC-MED",
                "object": (
                    "mcmed_primary_history_source"
                ),
                "column": column,
                "present": (
                    column
                    in mcmed_primary_history_source.columns
                ),
                "dtype": (
                    str(
                        mcmed_primary_history_source[
                            column
                        ].dtype
                    )
                    if column
                    in mcmed_primary_history_source.columns
                    else None
                ),
                "missing_n": (
                    int(
                        mcmed_primary_history_source[
                            column
                        ].isna().sum()
                    )
                    if column
                    in mcmed_primary_history_source.columns
                    else pd.NA
                ),
            }
        )


s2_cohort_timing_audit = pd.DataFrame(
    cohort_timing_rows
)



print(
    "S2 source-table schema audit:"
)

display(
    s2_source_schema_audit
)


print(
    "\nDetected source-column roles:"
)

display(
    s2_detected_source_roles
)


print(
    "\nCohort identifier and timing audit:"
)

display(
    s2_cohort_timing_audit
)


print(
    "\nS2 source audit completed."
)

print(
    "Frozen laboratory features:",
    len(
        FROZEN_S2_LAB_FEATURES
    ),
)

print(
    "Frozen imaging features:",
    len(
        FROZEN_S2_IMAGING_FEATURES
    ),
)

print(
    "\nNo source table has been fully loaded and "
    "no S2 feature matrix has been created."
)


#### Frozen abnormal laboratory-result feature block

The laboratory-result block contains 351 frozen binary features derived from the original MIMIC-IV development inventory. The feature names and their order remain unchanged so that the reconstructed matrices remain compatible with the previous modelling, SHAP and clinician-review work.

The original laboratory-result implementation matched normalised POE laboratory names to normalised MIMIC-IV laboratory labels and marked a result feature as positive when a corresponding laboratory event had an abnormal flag during the current emergency-department encounter.

The clean reconstruction retains the same frozen vocabulary, exact label-normalisation method and abnormal-flag definition. However, the corresponding POE order indicator is not required because POE orders are clinician-dependent workflow variables and are excluded from the objective-result model. A recorded laboratory result itself provides evidence that the investigation occurred.

Two temporal definitions are audited:

- legacy-compatible timing: laboratory chart time occurred between ED arrival and ED departure;
- availability-strict timing: laboratory chart time occurred during the ED visit and the stored result was available before ED departure.

The availability-strict representation is the provisional primary candidate because it reduces the risk of using results that became available only after the encounter ended. The legacy-compatible representation is retained temporarily to quantify the effect of this timing refinement before the laboratory block is frozen.


In [ ]:



required_objects = [
    "SOURCE_PATHS",
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "mcmed_primary_history_source",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the cohort and previous-history preparation cells "
        f"first. Missing objects: {missing_objects}"
    )



def parse_datetime_naive(
    series: pd.Series,
) -> pd.Series:
    """
    Parse shifted clinical timestamps consistently.

    UTC parsing is used only to standardise mixed representations.
    The timezone is then removed because both datasets contain
    internally shifted/de-identified timestamps.
    """

    parsed = pd.to_datetime(
        series,
        errors="coerce",
        utc=True,
    )

    return parsed.dt.tz_convert(None)



mimic_edstays_path = Path(
    SOURCE_PATHS["mimic_edstays"]
).resolve()

if not mimic_edstays_path.exists():
    raise FileNotFoundError(
        f"MIMIC-IV edstays file not found:\n"
        f"{mimic_edstays_path}"
    )


mimic_ed_window_source = pd.read_csv(
    mimic_edstays_path,
    usecols=[
        "subject_id",
        "stay_id",
        "hadm_id",
        "intime",
        "outtime",
    ],
    low_memory=False,
)


mimic_ed_window_source[
    "source_intime"
] = parse_datetime_naive(
    mimic_ed_window_source["intime"]
)

mimic_ed_window_source[
    "source_outtime"
] = parse_datetime_naive(
    mimic_ed_window_source["outtime"]
)


mimic_ed_window_source = (
    mimic_ed_window_source[
        [
            "stay_id",
            "subject_id",
            "hadm_id",
            "source_intime",
            "source_outtime",
        ]
    ]
    .rename(
        columns={
            "subject_id": "source_subject_id",
        }
    )
)


mimic_s2_window = (
    mimic_s0_validated.copy()
    .merge(
        mimic_ed_window_source,
        on="stay_id",
        how="left",
        validate="one_to_one",
    )
)


mimic_s2_window[
    "intime"
] = parse_datetime_naive(
    mimic_s2_window["intime"]
)


mimic_subject_mismatch = (
    mimic_s2_window[
        "source_subject_id"
    ].notna()
    &
    mimic_s2_window[
        "subject_id"
    ].ne(
        mimic_s2_window[
            "source_subject_id"
        ]
    )
)


assert not mimic_subject_mismatch.any(), (
    "At least one MIMIC stay_id linked to a different subject_id."
)


mimic_intime_difference_seconds = (
    mimic_s2_window[
        "intime"
    ]
    -
    mimic_s2_window[
        "source_intime"
    ]
).dt.total_seconds().abs()


assert (
    mimic_intime_difference_seconds
    .fillna(0)
    .le(1)
    .all()
), (
    "The S0 MIMIC intime differs from the authoritative "
    "edstays intime."
)


mimic_s2_window[
    "current_arrival_time"
] = mimic_s2_window[
    "source_intime"
]


mimic_s2_window[
    "current_ed_end_time"
] = mimic_s2_window[
    "source_outtime"
]



mcmed_visits_path = Path(
    SOURCE_PATHS["mcmed_visits"]
).resolve()

if not mcmed_visits_path.exists():
    raise FileNotFoundError(
        f"MC-MED visits file not found:\n"
        f"{mcmed_visits_path}"
    )


mcmed_visits_columns = pd.read_csv(
    mcmed_visits_path,
    nrows=0,
).columns.tolist()


required_mcmed_visit_columns = [
    "CSN",
    "MRN",
]

missing_required_mcmed_columns = [
    column
    for column in required_mcmed_visit_columns
    if column not in mcmed_visits_columns
]

if missing_required_mcmed_columns:
    raise ValueError(
        "MC-MED visits.csv is missing required columns: "
        f"{missing_required_mcmed_columns}"
    )


mcmed_time_candidates = [
    "Arrival_time",
    "Dispo_time",
    "Departure_time",
]


mcmed_available_time_columns = [
    column
    for column in mcmed_time_candidates
    if column in mcmed_visits_columns
]


if "Dispo_time" not in mcmed_available_time_columns:
    raise ValueError(
        "MC-MED Dispo_time was not found. It is required for "
        "the primary result-availability window."
    )


mcmed_visit_window_source = pd.read_csv(
    mcmed_visits_path,
    usecols=(
        required_mcmed_visit_columns
        +
        mcmed_available_time_columns
    ),
    low_memory=False,
)


for column in mcmed_available_time_columns:
    mcmed_visit_window_source[
        column
    ] = parse_datetime_naive(
        mcmed_visit_window_source[
            column
        ]
    )


mcmed_visit_window_source = (
    mcmed_visit_window_source
    .rename(
        columns={
            "MRN": "source_MRN",
            "Arrival_time": "source_arrival_time",
            "Dispo_time": "source_dispo_time",
            "Departure_time": "source_departure_time",
        }
    )
)


mcmed_arrival_source = (
    mcmed_primary_history_source[
        [
            "MRN",
            "CSN",
            "current_arrival_time",
        ]
    ]
    .drop_duplicates(
        subset="CSN"
    )
    .copy()
)


mcmed_arrival_source[
    "current_arrival_time"
] = parse_datetime_naive(
    mcmed_arrival_source[
        "current_arrival_time"
    ]
)


mcmed_s2_window = (
    mcmed_s0_validated.copy()
    .merge(
        mcmed_arrival_source[
            [
                "CSN",
                "current_arrival_time",
            ]
        ],
        on="CSN",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mcmed_visit_window_source,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


mcmed_mrn_mismatch = (
    mcmed_s2_window[
        "source_MRN"
    ].notna()
    &
    mcmed_s2_window[
        "MRN"
    ].ne(
        mcmed_s2_window[
            "source_MRN"
        ]
    )
)


assert not mcmed_mrn_mismatch.any(), (
    "At least one MC-MED CSN linked to a different MRN."
)


if "source_departure_time" not in mcmed_s2_window.columns:
    mcmed_s2_window[
        "source_departure_time"
    ] = pd.NaT


mcmed_s2_window[
    "current_ed_end_time"
] = (
    mcmed_s2_window[
        "source_dispo_time"
    ]
    .fillna(
        mcmed_s2_window[
            "source_departure_time"
        ]
    )
)


mcmed_s2_window[
    "ed_end_time_source"
] = np.select(
    [
        mcmed_s2_window[
            "source_dispo_time"
        ].notna(),

        mcmed_s2_window[
            "source_dispo_time"
        ].isna()
        &
        mcmed_s2_window[
            "source_departure_time"
        ].notna(),
    ],
    [
        "DISPOSITION_TIME",
        "DEPARTURE_TIME_FALLBACK",
    ],
    default="MISSING",
)



assert len(
    mimic_s2_window
) == len(
    mimic_s0_validated
) == 8550

assert len(
    mcmed_s2_window
) == len(
    mcmed_s0_validated
) == 6707


assert mimic_s2_window[
    "stay_id"
].is_unique

assert mcmed_s2_window[
    "CSN"
].is_unique


invalid_mimic_window = (
    mimic_s2_window[
        "current_ed_end_time"
    ]
    <
    mimic_s2_window[
        "current_arrival_time"
    ]
)


invalid_mcmed_window = (
    mcmed_s2_window[
        "current_ed_end_time"
    ]
    <
    mcmed_s2_window[
        "current_arrival_time"
    ]
)


assert not invalid_mimic_window.fillna(
    False
).any(), (
    "At least one MIMIC ED outtime precedes intime."
)

assert not invalid_mcmed_window.fillna(
    False
).any(), (
    "At least one MC-MED ED endpoint precedes arrival."
)


assert mimic_s2_window[
    "subject_id"
].is_unique, (
    "MIMIC subject_id is not unique in the frozen cohort."
)

assert mcmed_s2_window[
    "MRN"
].is_unique, (
    "MC-MED MRN is not unique in the frozen cohort."
)



s2_window_recovery_audit = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "measure": "Frozen cohort encounters",
            "value": len(
                mimic_s2_window
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "measure": "Missing arrival time",
            "value": int(
                mimic_s2_window[
                    "current_arrival_time"
                ].isna().sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "measure": "Missing ED end time",
            "value": int(
                mimic_s2_window[
                    "current_ed_end_time"
                ].isna().sum()
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "measure": "Encounters with hadm_id",
            "value": int(
                mimic_s2_window[
                    "hadm_id"
                ].notna().sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "measure": "Frozen cohort encounters",
            "value": len(
                mcmed_s2_window
            ),
        },
        {
            "dataset": "MC-MED",
            "measure": "Missing arrival time",
            "value": int(
                mcmed_s2_window[
                    "current_arrival_time"
                ].isna().sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "measure": "Missing ED end time",
            "value": int(
                mcmed_s2_window[
                    "current_ed_end_time"
                ].isna().sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "measure": "Disposition-time endpoints",
            "value": int(
                mcmed_s2_window[
                    "ed_end_time_source"
                ].eq(
                    "DISPOSITION_TIME"
                ).sum()
            ),
        },
        {
            "dataset": "MC-MED",
            "measure": "Departure-time fallbacks",
            "value": int(
                mcmed_s2_window[
                    "ed_end_time_source"
                ].eq(
                    "DEPARTURE_TIME_FALLBACK"
                ).sum()
            ),
        },
    ]
)


display(
    s2_window_recovery_audit
)


print(
    "\nS2 ED WINDOW RECOVERY PASSED"
)

print(
    "MIMIC-IV window rows:",
    len(
        mimic_s2_window
    ),
)

print(
    "MC-MED window rows:",
    len(
        mcmed_s2_window
    ),
)


In [ ]:



required_objects = [
    "SOURCE_PATHS",
    "s2_lab_feature_plan",
    "FROZEN_S2_LAB_FEATURES",
    "mimic_s2_window",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the frozen inventory and S2 window cells first. "
        f"Missing objects: {missing_objects}"
    )



def normalise_lab_name(
    value,
) -> str:
    """
    Replicate the laboratory-name normalisation used in the
    previous development notebook.
    """

    if pd.isna(value):
        return ""

    value = str(
        value
    ).strip().lower()

    value = value.replace(
        "&",
        " and ",
    )

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value



lab_plan_working = (
    s2_lab_feature_plan.copy()
)


if (
    "selected_for_binary_feature"
    in lab_plan_working.columns
):
    selected_mask = (
        lab_plan_working[
            "selected_for_binary_feature"
        ]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin(
            {
                "true",
                "1",
                "yes",
                "y",
            }
        )
    )

    lab_plan_working = (
        lab_plan_working.loc[
            selected_mask
        ]
        .copy()
    )


required_plan_columns = [
    "poe_result_feature_name",
    "lab_order_name",
]

missing_plan_columns = [
    column
    for column in required_plan_columns
    if column not in lab_plan_working.columns
]

if missing_plan_columns:
    raise ValueError(
        "s2_lab_feature_plan is missing required columns: "
        f"{missing_plan_columns}"
    )


lab_plan_working[
    "lab_order_name_norm_rebuilt"
] = (
    lab_plan_working[
        "lab_order_name"
    ]
    .map(
        normalise_lab_name
    )
)


if "lab_order_name_norm" not in (
    lab_plan_working.columns
):
    lab_plan_working[
        "lab_order_name_norm"
    ] = lab_plan_working[
        "lab_order_name_norm_rebuilt"
    ]
else:
    lab_plan_working[
        "lab_order_name_norm"
    ] = (
        lab_plan_working[
            "lab_order_name_norm"
        ]
        .fillna(
            lab_plan_working[
                "lab_order_name_norm_rebuilt"
            ]
        )
        .astype("string")
        .str.strip()
    )


normalisation_difference = (
    lab_plan_working[
        "lab_order_name_norm"
    ]
    .astype(str)
    .ne(
        lab_plan_working[
            "lab_order_name_norm_rebuilt"
        ]
        .astype(str)
    )
)


lab_plan_working[
    "normalisation_matches_rebuilt_rule"
] = (
    ~normalisation_difference
)


lab_plan_working = (
    lab_plan_working
    .drop_duplicates(
        subset="poe_result_feature_name",
        keep="first",
    )
    .copy()
)



frozen_lab_position = {
    feature_name: position
    for position, feature_name
    in enumerate(
        FROZEN_S2_LAB_FEATURES
    )
}


lab_plan_working[
    "_frozen_position"
] = (
    lab_plan_working[
        "poe_result_feature_name"
    ]
    .map(
        frozen_lab_position
    )
)


unexpected_plan_features = (
    lab_plan_working.loc[
        lab_plan_working[
            "_frozen_position"
        ].isna(),
        "poe_result_feature_name",
    ]
    .tolist()
)


if unexpected_plan_features:
    raise ValueError(
        "The laboratory plan contains features outside the "
        "frozen inventory. Examples: "
        f"{unexpected_plan_features[:20]}"
    )


s2_lab_reconstruction_plan = (
    lab_plan_working
    .sort_values(
        "_frozen_position"
    )
    .reset_index(drop=True)
)


assert (
    s2_lab_reconstruction_plan[
        "poe_result_feature_name"
    ].tolist()
    ==
    list(
        FROZEN_S2_LAB_FEATURES
    )
), (
    "The laboratory feature plan does not exactly match the "
    "351-feature frozen order."
)



mimic_d_labitems = pd.read_csv(
    SOURCE_PATHS[
        "mimic_d_labitems"
    ],
    usecols=[
        "itemid",
        "label",
        "fluid",
        "category",
    ],
    low_memory=False,
)


mimic_d_labitems[
    "itemid"
] = pd.to_numeric(
    mimic_d_labitems[
        "itemid"
    ],
    errors="coerce",
).astype("Int64")


mimic_d_labitems[
    "lab_label_norm"
] = (
    mimic_d_labitems[
        "label"
    ]
    .map(
        normalise_lab_name
    )
)


s2_lab_item_feature_map = (
    mimic_d_labitems
    .merge(
        s2_lab_reconstruction_plan[
            [
                "lab_order_name",
                "lab_order_name_norm",
                "poe_result_feature_name",
            ]
        ],
        left_on="lab_label_norm",
        right_on="lab_order_name_norm",
        how="inner",
        validate="many_to_many",
    )
    [
        [
            "itemid",
            "label",
            "fluid",
            "category",
            "lab_label_norm",
            "lab_order_name",
            "poe_result_feature_name",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)



mapped_frozen_features = set(
    s2_lab_item_feature_map[
        "poe_result_feature_name"
    ]
)


unmapped_frozen_lab_features = [
    feature_name
    for feature_name
    in FROZEN_S2_LAB_FEATURES
    if feature_name not in mapped_frozen_features
]


feature_mapping_counts = (
    s2_lab_item_feature_map
    .groupby(
        "poe_result_feature_name"
    )
    .agg(
        n_itemids=(
            "itemid",
            "nunique",
        ),
        n_labels=(
            "label",
            "nunique",
        ),
        n_fluids=(
            "fluid",
            "nunique",
        ),
        labels=(
            "label",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            ),
        ),
        fluids=(
            "fluid",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            ),
        ),
    )
    .reset_index()
)


lab_norm_multiplicity = (
    s2_lab_reconstruction_plan
    .groupby(
        "lab_order_name_norm"
    )
    .agg(
        n_frozen_features=(
            "poe_result_feature_name",
            "nunique",
        ),
        frozen_features=(
            "poe_result_feature_name",
            lambda values: " | ".join(
                values.astype(str)
                .tolist()
            ),
        ),
    )
    .reset_index()
)


s2_lab_mapping_audit = pd.DataFrame(
    [
        {
            "measure": "Frozen laboratory features",
            "value": len(
                FROZEN_S2_LAB_FEATURES
            ),
        },
        {
            "measure": (
                "Features matched to >=1 MIMIC itemid"
            ),
            "value": len(
                mapped_frozen_features
            ),
        },
        {
            "measure": (
                "Features without exact dictionary match"
            ),
            "value": len(
                unmapped_frozen_lab_features
            ),
        },
        {
            "measure": (
                "Unique matched MIMIC itemids"
            ),
            "value": int(
                s2_lab_item_feature_map[
                    "itemid"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Plan rows matching rebuilt normalisation"
            ),
            "value": int(
                s2_lab_reconstruction_plan[
                    "normalisation_matches_rebuilt_rule"
                ].sum()
            ),
        },
        {
            "measure": (
                "Normalised names linked to multiple "
                "frozen features"
            ),
            "value": int(
                lab_norm_multiplicity[
                    "n_frozen_features"
                ].gt(1).sum()
            ),
        },
    ]
)


display(
    s2_lab_mapping_audit
)


print(
    "\nFrozen laboratory features without an exact "
    "d_labitems label match:"
)

display(
    pd.DataFrame(
        {
            "feature_name": (
                unmapped_frozen_lab_features
            )
        }
    )
)


print(
    "\nLaboratory mappings involving multiple itemids "
    "or fluids:"
)

display(
    feature_mapping_counts.loc[
        feature_mapping_counts[
            "n_itemids"
        ].gt(1)
        |
        feature_mapping_counts[
            "n_fluids"
        ].gt(1)
    ]
    .sort_values(
        [
            "n_fluids",
            "n_itemids",
        ],
        ascending=False,
    )
    .head(50)
)


print(
    "\nS2 LABORATORY MAPPING PLAN PASSED"
)

print(
    "Frozen features:",
    len(
        s2_lab_reconstruction_plan
    ),
)

print(
    "Mapped itemids:",
    s2_lab_item_feature_map[
        "itemid"
    ].nunique(),
)


In [ ]:



required_objects = [
    "SOURCE_PATHS",
    "mimic_s2_window",
    "s2_lab_item_feature_map",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the S2 window and laboratory mapping cells first. "
        f"Missing objects: {missing_objects}"
    )


LABEVENTS_CHUNKSIZE = 1_000_000



mimic_lab_linkage_window = (
    mimic_s2_window[
        [
            "subject_id",
            "stay_id",
            "current_arrival_time",
            "current_ed_end_time",
        ]
    ]
    .copy()
)


assert mimic_lab_linkage_window[
    "subject_id"
].is_unique


cohort_subject_ids = set(
    mimic_lab_linkage_window[
        "subject_id"
    ]
    .dropna()
    .astype(int)
    .tolist()
)


mapped_lab_itemids = set(
    s2_lab_item_feature_map[
        "itemid"
    ]
    .dropna()
    .astype(int)
    .tolist()
)


global_min_arrival = (
    mimic_lab_linkage_window[
        "current_arrival_time"
    ].min()
)

global_max_ed_end = (
    mimic_lab_linkage_window[
        "current_ed_end_time"
    ].max()
)



mimic_labevents_path = Path(
    SOURCE_PATHS[
        "mimic_labevents"
    ]
).resolve()


labevents_header = pd.read_csv(
    mimic_labevents_path,
    nrows=0,
).columns.tolist()


preferred_labevent_columns = [
    "labevent_id",
    "subject_id",
    "hadm_id",
    "itemid",
    "charttime",
    "storetime",
    "value",
    "valuenum",
    "valueuom",
    "flag",
    "ref_range_lower",
    "ref_range_upper",
]


available_labevent_columns = [
    column
    for column in preferred_labevent_columns
    if column in labevents_header
]


required_labevent_columns = {
    "subject_id",
    "itemid",
    "charttime",
    "flag",
}


missing_labevent_columns = (
    required_labevent_columns
    -
    set(
        available_labevent_columns
    )
)


if missing_labevent_columns:
    raise ValueError(
        "MIMIC labevents is missing required columns: "
        f"{sorted(missing_labevent_columns)}"
    )



mimic_lab_candidate_chunks = []

n_source_rows_processed = 0
n_rows_after_subject_filter = 0
n_rows_after_item_filter = 0
n_rows_in_legacy_ed_window = 0
n_rows_in_strict_available_window = 0


for chunk_number, chunk in enumerate(
    pd.read_csv(
        mimic_labevents_path,
        usecols=available_labevent_columns,
        chunksize=LABEVENTS_CHUNKSIZE,
        low_memory=False,
    ),
    start=1,
):
    n_source_rows_processed += len(
        chunk
    )

    chunk[
        "subject_id"
    ] = pd.to_numeric(
        chunk[
            "subject_id"
        ],
        errors="coerce",
    ).astype("Int64")

    chunk[
        "itemid"
    ] = pd.to_numeric(
        chunk[
            "itemid"
        ],
        errors="coerce",
    ).astype("Int64")

    chunk = chunk.loc[
        chunk[
            "subject_id"
        ].isin(
            cohort_subject_ids
        )
    ].copy()

    n_rows_after_subject_filter += len(
        chunk
    )

    if chunk.empty:
        if chunk_number % 10 == 0:
            print(
                "Processed chunks:",
                chunk_number,
            )
        continue

    chunk = chunk.loc[
        chunk[
            "itemid"
        ].isin(
            mapped_lab_itemids
        )
    ].copy()

    n_rows_after_item_filter += len(
        chunk
    )

    if chunk.empty:
        continue

    chunk[
        "charttime"
    ] = parse_datetime_naive(
        chunk[
            "charttime"
        ]
    )

    if "storetime" in chunk.columns:
        chunk[
            "storetime"
        ] = parse_datetime_naive(
            chunk[
                "storetime"
            ]
        )
    else:
        chunk[
            "storetime"
        ] = pd.NaT

    chunk = chunk.loc[
        chunk[
            "charttime"
        ].between(
            global_min_arrival,
            global_max_ed_end,
            inclusive="both",
        )
    ].copy()

    if chunk.empty:
        continue

    chunk = chunk.merge(
        mimic_lab_linkage_window,
        on="subject_id",
        how="inner",
        validate="many_to_one",
    )

    chunk[
        "legacy_charttime_in_ed"
    ] = (
        chunk[
            "charttime"
        ].ge(
            chunk[
                "current_arrival_time"
            ]
        )
        &
        chunk[
            "charttime"
        ].le(
            chunk[
                "current_ed_end_time"
            ]
        )
    )

    chunk[
        "result_available_time"
    ] = (
        chunk[
            "storetime"
        ]
        .fillna(
            chunk[
                "charttime"
            ]
        )
    )

    chunk[
        "availability_strict_in_ed"
    ] = (
        chunk[
            "legacy_charttime_in_ed"
        ]
        &
        chunk[
            "result_available_time"
        ].ge(
            chunk[
                "current_arrival_time"
            ]
        )
        &
        chunk[
            "result_available_time"
        ].le(
            chunk[
                "current_ed_end_time"
            ]
        )
    )

    chunk = chunk.loc[
        chunk[
            "legacy_charttime_in_ed"
        ]
    ].copy()

    n_rows_in_legacy_ed_window += len(
        chunk
    )

    n_rows_in_strict_available_window += int(
        chunk[
            "availability_strict_in_ed"
        ].sum()
    )

    if chunk.empty:
        continue

    flag_text = (
        chunk[
            "flag"
        ]
        .astype("string")
        .str.strip()
    )

    flag_lower = (
        flag_text
        .str.lower()
    )

    chunk[
        "flag_clean"
    ] = flag_text

    chunk[
        "flag_available"
    ] = (
        flag_text.notna()
        &
        ~flag_lower.isin(
            {
                "",
                "nan",
                "none",
                "null",
                "missing",
            }
        )
    ).fillna(False)

    chunk[
        "abnormal_by_flag"
    ] = (
        chunk[
            "flag_available"
        ]
        &
        ~flag_lower.isin(
            {
                "normal",
                "n",
            }
        )
    ).fillna(False)

    compact_columns = [
        column
        for column in [
            "labevent_id",
            "subject_id",
            "stay_id",
            "hadm_id",
            "itemid",
            "charttime",
            "storetime",
            "result_available_time",
            "flag",
            "flag_clean",
            "flag_available",
            "abnormal_by_flag",
            "legacy_charttime_in_ed",
            "availability_strict_in_ed",
            "value",
            "valuenum",
            "valueuom",
            "ref_range_lower",
            "ref_range_upper",
        ]
        if column in chunk.columns
    ]

    mimic_lab_candidate_chunks.append(
        chunk[
            compact_columns
        ].copy()
    )

    if chunk_number % 10 == 0:
        print(
            "Processed chunks:",
            chunk_number,
            "| retained legacy-window rows:",
            n_rows_in_legacy_ed_window,
        )



if not mimic_lab_candidate_chunks:
    raise ValueError(
        "No MIMIC laboratory rows matched the frozen cohort, "
        "laboratory itemids and ED windows."
    )


mimic_s2_lab_rows = pd.concat(
    mimic_lab_candidate_chunks,
    ignore_index=True,
)


del mimic_lab_candidate_chunks



mimic_s2_lab_extraction_audit = pd.DataFrame(
    [
        {
            "measure": "Source labevents rows processed",
            "value": n_source_rows_processed,
        },
        {
            "measure": (
                "Rows after cohort-subject filtering"
            ),
            "value": n_rows_after_subject_filter,
        },
        {
            "measure": (
                "Rows after frozen-itemid filtering"
            ),
            "value": n_rows_after_item_filter,
        },
        {
            "measure": (
                "Rows with charttime in ED window"
            ),
            "value": n_rows_in_legacy_ed_window,
        },
        {
            "measure": (
                "Rows available before ED end"
            ),
            "value": n_rows_in_strict_available_window,
        },
        {
            "measure": (
                "Charttime-window rows excluded because "
                "result was stored after ED end"
            ),
            "value": (
                n_rows_in_legacy_ed_window
                -
                n_rows_in_strict_available_window
            ),
        },
        {
            "measure": (
                "Unique frozen cohort encounters with "
                "matched laboratory rows"
            ),
            "value": int(
                mimic_s2_lab_rows[
                    "stay_id"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Rows with available abnormality flag"
            ),
            "value": int(
                mimic_s2_lab_rows[
                    "flag_available"
                ].sum()
            ),
        },
        {
            "measure": (
                "Rows abnormal by flag"
            ),
            "value": int(
                mimic_s2_lab_rows[
                    "abnormal_by_flag"
                ].sum()
            ),
        },
    ]
)


mimic_s2_lab_flag_value_audit = (
    mimic_s2_lab_rows[
        "flag_clean"
    ]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "flag_value"
    )
    .reset_index(
        name="n_rows"
    )
)


mimic_s2_lab_flag_value_audit[
    "percentage"
] = (
    mimic_s2_lab_flag_value_audit[
        "n_rows"
    ]
    /
    len(
        mimic_s2_lab_rows
    )
    * 100
)


display(
    mimic_s2_lab_extraction_audit
)


print(
    "\nMIMIC laboratory flag values:"
)

display(
    mimic_s2_lab_flag_value_audit.round(
        {
            "percentage": 3,
        }
    )
)


print(
    "\nMIMIC S2 LABORATORY EXTRACTION PASSED"
)

print(
    "Retained rows:",
    len(
        mimic_s2_lab_rows
    ),
)

print(
    "Unique stays:",
    mimic_s2_lab_rows[
        "stay_id"
    ].nunique(),
)


In [ ]:



required_objects = [
    "mimic_s2_window",
    "mimic_s2_lab_rows",
    "s2_lab_item_feature_map",
    "FROZEN_S2_LAB_FEATURES",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the MIMIC laboratory extraction cell first. "
        f"Missing objects: {missing_objects}"
    )



mimic_s2_lab_rows_mapped = (
    mimic_s2_lab_rows
    .merge(
        s2_lab_item_feature_map[
            [
                "itemid",
                "label",
                "fluid",
                "category",
                "poe_result_feature_name",
            ]
        ],
        on="itemid",
        how="inner",
        validate="many_to_many",
    )
)



mimic_lab_positive_legacy = (
    mimic_s2_lab_rows_mapped.loc[
        mimic_s2_lab_rows_mapped[
            "abnormal_by_flag"
        ],
        [
            "stay_id",
            "poe_result_feature_name",
        ],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


mimic_lab_positive_strict = (
    mimic_s2_lab_rows_mapped.loc[
        mimic_s2_lab_rows_mapped[
            "abnormal_by_flag"
        ]
        &
        mimic_s2_lab_rows_mapped[
            "availability_strict_in_ed"
        ],
        [
            "stay_id",
            "poe_result_feature_name",
        ],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)



def build_frozen_lab_matrix(
    cohort_dataframe: pd.DataFrame,
    positive_pairs: pd.DataFrame,
) -> pd.DataFrame:
    """
    Construct one row per frozen MIMIC stay with the exact
    351-feature frozen laboratory order.
    """

    metadata_columns = [
        column
        for column in [
            "subject_id",
            "stay_id",
            "split",
            "outcome",
            "outcome_binary",
        ]
        if column in cohort_dataframe.columns
    ]

    cohort_metadata = (
        cohort_dataframe[
            metadata_columns
        ]
        .copy()
        .reset_index(drop=True)
    )

    if positive_pairs.empty:
        wide = pd.DataFrame(
            0,
            index=cohort_metadata[
                "stay_id"
            ],
            columns=(
                FROZEN_S2_LAB_FEATURES
            ),
            dtype="int8",
        )

    else:
        positive_pairs_working = (
            positive_pairs.copy()
        )

        positive_pairs_working[
            "value"
        ] = 1

        wide = (
            positive_pairs_working
            .pivot_table(
                index="stay_id",
                columns=(
                    "poe_result_feature_name"
                ),
                values="value",
                aggfunc="max",
                fill_value=0,
            )
            .reindex(
                index=cohort_metadata[
                    "stay_id"
                ].tolist(),
                columns=(
                    FROZEN_S2_LAB_FEATURES
                ),
                fill_value=0,
            )
            .astype("int8")
        )

    wide.index.name = "stay_id"

    wide = (
        wide
        .reset_index()
    )

    result = (
        cohort_metadata
        .merge(
            wide,
            on="stay_id",
            how="left",
            validate="one_to_one",
        )
    )

    result[
        FROZEN_S2_LAB_FEATURES
    ] = (
        result[
            FROZEN_S2_LAB_FEATURES
        ]
        .fillna(0)
        .astype("int8")
    )

    return result



mimic_s2_lab_features_legacy_audit = (
    build_frozen_lab_matrix(
        cohort_dataframe=(
            mimic_s2_window
        ),
        positive_pairs=(
            mimic_lab_positive_legacy
        ),
    )
)


mimic_s2_lab_features_candidate = (
    build_frozen_lab_matrix(
        cohort_dataframe=(
            mimic_s2_window
        ),
        positive_pairs=(
            mimic_lab_positive_strict
        ),
    )
)



for matrix_name, matrix in [
    (
        "legacy-compatible",
        mimic_s2_lab_features_legacy_audit,
    ),
    (
        "availability-strict candidate",
        mimic_s2_lab_features_candidate,
    ),
]:
    assert len(
        matrix
    ) == 8550

    assert matrix[
        "stay_id"
    ].is_unique

    assert (
        matrix[
            FROZEN_S2_LAB_FEATURES
        ].columns.tolist()
        ==
        list(
            FROZEN_S2_LAB_FEATURES
        )
    )

    observed_values = set(
        np.unique(
            matrix[
                FROZEN_S2_LAB_FEATURES
            ].to_numpy()
        ).tolist()
    )

    assert observed_values.issubset(
        {
            0,
            1,
        }
    ), (
        f"{matrix_name} matrix contains non-binary values: "
        f"{observed_values}"
    )



legacy_support = (
    mimic_s2_lab_features_legacy_audit[
        FROZEN_S2_LAB_FEATURES
    ]
    .sum(axis=0)
    .astype(int)
)


strict_support = (
    mimic_s2_lab_features_candidate[
        FROZEN_S2_LAB_FEATURES
    ]
    .sum(axis=0)
    .astype(int)
)


mimic_s2_lab_feature_support_comparison = pd.DataFrame(
    {
        "feature_name": (
            FROZEN_S2_LAB_FEATURES
        ),
        "legacy_charttime_positive_stays": (
            legacy_support.reindex(
                FROZEN_S2_LAB_FEATURES
            ).to_numpy()
        ),
        "strict_available_positive_stays": (
            strict_support.reindex(
                FROZEN_S2_LAB_FEATURES
            ).to_numpy()
        ),
    }
)


mimic_s2_lab_feature_support_comparison[
    "positive_stays_removed_by_availability_rule"
] = (
    mimic_s2_lab_feature_support_comparison[
        "legacy_charttime_positive_stays"
    ]
    -
    mimic_s2_lab_feature_support_comparison[
        "strict_available_positive_stays"
    ]
)


mimic_s2_lab_feature_support_comparison[
    "retained_pct"
] = np.where(
    mimic_s2_lab_feature_support_comparison[
        "legacy_charttime_positive_stays"
    ].gt(0),

    (
        mimic_s2_lab_feature_support_comparison[
            "strict_available_positive_stays"
        ]
        /
        mimic_s2_lab_feature_support_comparison[
            "legacy_charttime_positive_stays"
        ]
        * 100
    ),

    np.nan,
)



legacy_any_abnormal = (
    mimic_s2_lab_features_legacy_audit[
        FROZEN_S2_LAB_FEATURES
    ]
    .gt(0)
    .any(axis=1)
)


strict_any_abnormal = (
    mimic_s2_lab_features_candidate[
        FROZEN_S2_LAB_FEATURES
    ]
    .gt(0)
    .any(axis=1)
)


mimic_s2_lab_candidate_summary = pd.DataFrame(
    [
        {
            "definition": (
                "Legacy-compatible charttime window"
            ),
            "n_cohort_encounters": len(
                mimic_s2_lab_features_legacy_audit
            ),
            "n_encounters_with_any_abnormal_lab": int(
                legacy_any_abnormal.sum()
            ),
            "n_features_with_at_least_one_positive": int(
                legacy_support.gt(0).sum()
            ),
            "total_positive_feature_values": int(
                legacy_support.sum()
            ),
        },
        {
            "definition": (
                "Availability-strict candidate"
            ),
            "n_cohort_encounters": len(
                mimic_s2_lab_features_candidate
            ),
            "n_encounters_with_any_abnormal_lab": int(
                strict_any_abnormal.sum()
            ),
            "n_features_with_at_least_one_positive": int(
                strict_support.gt(0).sum()
            ),
            "total_positive_feature_values": int(
                strict_support.sum()
            ),
        },
    ]
)



split_summary_rows = []


for split_name in [
    "train",
    "validation",
    "test",
]:
    split_rows = (
        mimic_s2_lab_features_candidate[
            "split"
        ].eq(
            split_name
        )
    )

    split_matrix = (
        mimic_s2_lab_features_candidate.loc[
            split_rows,
            FROZEN_S2_LAB_FEATURES,
        ]
    )

    split_summary_rows.append(
        {
            "split": split_name,
            "n_rows": int(
                split_rows.sum()
            ),
            "encounters_with_any_abnormal_lab": int(
                split_matrix
                .gt(0)
                .any(axis=1)
                .sum()
            ),
            "features_with_at_least_one_positive": int(
                split_matrix
                .sum(axis=0)
                .gt(0)
                .sum()
            ),
            "total_positive_feature_values": int(
                split_matrix
                .sum()
                .sum()
            ),
        }
    )


mimic_s2_lab_split_summary = pd.DataFrame(
    split_summary_rows
)



print(
    "MIMIC laboratory candidate summary:"
)

display(
    mimic_s2_lab_candidate_summary
)


print(
    "\nAvailability-strict split summary:"
)

display(
    mimic_s2_lab_split_summary
)


print(
    "\nFeatures most affected by requiring result availability "
    "before ED departure:"
)

display(
    mimic_s2_lab_feature_support_comparison
    .sort_values(
        [
            "positive_stays_removed_by_availability_rule",
            "legacy_charttime_positive_stays",
        ],
        ascending=False,
    )
    .head(40)
    .round(
        {
            "retained_pct": 2,
        }
    )
)


print(
    "\nTop availability-strict abnormal laboratory features:"
)

display(
    mimic_s2_lab_feature_support_comparison
    .sort_values(
        "strict_available_positive_stays",
        ascending=False,
    )
    .head(40)
)


print(
    "\nMIMIC 351-FEATURE LABORATORY CANDIDATE PASSED"
)

print(
    "Candidate matrix shape:",
    mimic_s2_lab_features_candidate.shape,
)

print(
    "Frozen laboratory features:",
    len(
        FROZEN_S2_LAB_FEATURES
    ),
)

print(
    "\nThe candidate has not yet been saved or frozen."
)


In [ ]:



required_objects = [
    "FROZEN_S2_LAB_FEATURES",
    "s2_lab_feature_audit_original",
    "s2_lab_item_feature_map",
    "mimic_s2_lab_feature_support_comparison",
    "mimic_s2_lab_features_legacy_audit",
    "mimic_s2_lab_features_candidate",
    "mcmed_s2_window",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding S2 laboratory cells first. "
        f"Missing objects: {missing_objects}"
    )



def resolve_first_existing_column(
    dataframe,
    candidates,
    object_name,
    required=True,
):
    """Return the first available column from a candidate list."""

    selected = next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )

    if selected is None and required:
        raise ValueError(
            f"No recognised column was found in {object_name}. "
            f"Expected one of {candidates}. "
            f"Available columns: {dataframe.columns.tolist()}"
        )

    return selected


original_feature_column = resolve_first_existing_column(
    dataframe=s2_lab_feature_audit_original,
    candidates=[
        "poe_result_feature_name",
        "feature_name",
        "final_feature_name",
    ],
    object_name="s2_lab_feature_audit_original",
)


original_support_column = resolve_first_existing_column(
    dataframe=s2_lab_feature_audit_original,
    candidates=[
        "n_abnormal_flag_stays",
        "n_result_abnormal_positive_stays",
        "n_abnormal_positive_stays",
        "n_positive_stays",
    ],
    object_name="s2_lab_feature_audit_original",
)


original_order_support_column = resolve_first_existing_column(
    dataframe=s2_lab_feature_audit_original,
    candidates=[
        "n_ordered_stays",
        "n_order_positive_stays",
    ],
    object_name="s2_lab_feature_audit_original",
    required=False,
)



original_audit_columns = [
    original_feature_column,
    original_support_column,
]

if original_order_support_column is not None:
    original_audit_columns.append(
        original_order_support_column
    )


original_lab_support = (
    s2_lab_feature_audit_original[
        original_audit_columns
    ]
    .copy()
    .rename(
        columns={
            original_feature_column:
                "feature_name",
            original_support_column:
                "original_abnormal_positive_stays",
            original_order_support_column:
                "original_order_positive_stays",
        }
        if original_order_support_column is not None
        else {
            original_feature_column:
                "feature_name",
            original_support_column:
                "original_abnormal_positive_stays",
        }
    )
)


original_lab_support[
    "original_abnormal_positive_stays"
] = pd.to_numeric(
    original_lab_support[
        "original_abnormal_positive_stays"
    ],
    errors="coerce",
).fillna(0).astype(int)


original_lab_support = (
    original_lab_support
    .drop_duplicates(
        subset="feature_name",
        keep="first",
    )
)



current_mapping_summary = (
    s2_lab_item_feature_map
    .groupby(
        "poe_result_feature_name",
        as_index=False,
    )
    .agg(
        n_mapped_itemids=(
            "itemid",
            "nunique",
        ),
        n_mapped_labels=(
            "label",
            "nunique",
        ),
        n_mapped_fluids=(
            "fluid",
            "nunique",
        ),
        mapped_labels=(
            "label",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            ),
        ),
        mapped_fluids=(
            "fluid",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )
            ),
        ),
    )
    .rename(
        columns={
            "poe_result_feature_name":
                "feature_name",
        }
    )
)



s2_lab_final_compatibility_audit = (
    pd.DataFrame(
        {
            "feature_position": range(
                1,
                len(FROZEN_S2_LAB_FEATURES) + 1,
            ),
            "feature_name": FROZEN_S2_LAB_FEATURES,
        }
    )
    .merge(
        current_mapping_summary,
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
    .merge(
        mimic_s2_lab_feature_support_comparison,
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
    .merge(
        original_lab_support,
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
)


numeric_fill_columns = [
    "n_mapped_itemids",
    "n_mapped_labels",
    "n_mapped_fluids",
    "legacy_charttime_positive_stays",
    "strict_available_positive_stays",
    "positive_stays_removed_by_availability_rule",
    "original_abnormal_positive_stays",
]


for column in numeric_fill_columns:
    if column in s2_lab_final_compatibility_audit.columns:
        s2_lab_final_compatibility_audit[
            column
        ] = pd.to_numeric(
            s2_lab_final_compatibility_audit[
                column
            ],
            errors="coerce",
        ).fillna(0).astype(int)


s2_lab_final_compatibility_audit[
    "mapping_class"
] = np.select(
    [
        s2_lab_final_compatibility_audit[
            "n_mapped_itemids"
        ].eq(0),

        s2_lab_final_compatibility_audit[
            "n_mapped_fluids"
        ].gt(1),

        s2_lab_final_compatibility_audit[
            "n_mapped_itemids"
        ].gt(1),
    ],
    [
        "UNMAPPED_TO_D_LABITEMS",
        "MULTIPLE_SPECIMEN_FLUIDS",
        "MULTIPLE_ITEMIDS_SINGLE_FLUID",
    ],
    default="SINGLE_FLUID_MAPPING",
)


s2_lab_final_compatibility_audit[
    "originally_active"
] = (
    s2_lab_final_compatibility_audit[
        "original_abnormal_positive_stays"
    ].gt(0)
)


s2_lab_final_compatibility_audit[
    "active_in_current_encounter_window"
] = (
    s2_lab_final_compatibility_audit[
        "legacy_charttime_positive_stays"
    ].gt(0)
)


s2_lab_final_compatibility_audit[
    "active_in_strict_availability_window"
] = (
    s2_lab_final_compatibility_audit[
        "strict_available_positive_stays"
    ].gt(0)
)


s2_lab_final_compatibility_audit[
    "compatibility_status"
] = np.select(
    [
        (
            s2_lab_final_compatibility_audit[
                "mapping_class"
            ].eq(
                "UNMAPPED_TO_D_LABITEMS"
            )
            &
            s2_lab_final_compatibility_audit[
                "originally_active"
            ]
        ),

        (
            s2_lab_final_compatibility_audit[
                "originally_active"
            ]
            &
            ~s2_lab_final_compatibility_audit[
                "active_in_current_encounter_window"
            ]
        ),

        (
            ~s2_lab_final_compatibility_audit[
                "originally_active"
            ]
            &
            s2_lab_final_compatibility_audit[
                "active_in_current_encounter_window"
            ]
        ),
    ],
    [
        "REQUIRES_LEGACY_MAPPING_REVIEW",
        "ORIGINAL_ACTIVE_CURRENT_ZERO",
        "CURRENT_ACTIVE_ORIGINAL_ZERO",
    ],
    default="COMPATIBLE_OR_COHORT_DIFFERENCE",
)



mapping_summary = (
    s2_lab_final_compatibility_audit
    .groupby(
        "mapping_class",
        as_index=False,
    )
    .agg(
        n_features=(
            "feature_name",
            "size",
        ),
        n_originally_active=(
            "originally_active",
            "sum",
        ),
        n_currently_active=(
            "active_in_current_encounter_window",
            "sum",
        ),
        n_strictly_active=(
            "active_in_strict_availability_window",
            "sum",
        ),
    )
)


mapping_summary[
    "percentage_of_351"
] = (
    mapping_summary[
        "n_features"
    ]
    /
    len(FROZEN_S2_LAB_FEATURES)
    * 100
)


compatibility_summary = (
    s2_lab_final_compatibility_audit
    .groupby(
        "compatibility_status",
        as_index=False,
    )
    .agg(
        n_features=(
            "feature_name",
            "size",
        )
    )
)



unmapped_originally_active = (
    s2_lab_final_compatibility_audit.loc[
        (
            s2_lab_final_compatibility_audit[
                "mapping_class"
            ].eq(
                "UNMAPPED_TO_D_LABITEMS"
            )
            &
            s2_lab_final_compatibility_audit[
                "originally_active"
            ]
        )
    ]
    .sort_values(
        "original_abnormal_positive_stays",
        ascending=False,
    )
)


multi_fluid_active_features = (
    s2_lab_final_compatibility_audit.loc[
        (
            s2_lab_final_compatibility_audit[
                "mapping_class"
            ].eq(
                "MULTIPLE_SPECIMEN_FLUIDS"
            )
            &
            s2_lab_final_compatibility_audit[
                "active_in_current_encounter_window"
            ]
        )
    ]
    .sort_values(
        "legacy_charttime_positive_stays",
        ascending=False,
    )
)



mcmed_endpoint_completeness_by_outcome = (
    mcmed_s2_window
    .assign(
        ed_endpoint_available=(
            mcmed_s2_window[
                "current_ed_end_time"
            ].notna()
        )
    )
    .groupby(
        "outcome",
        dropna=False,
        as_index=False,
    )
    .agg(
        n_encounters=(
            "CSN",
            "size",
        ),
        n_with_ed_endpoint=(
            "ed_endpoint_available",
            "sum",
        ),
    )
)


mcmed_endpoint_completeness_by_outcome[
    "n_missing_ed_endpoint"
] = (
    mcmed_endpoint_completeness_by_outcome[
        "n_encounters"
    ]
    -
    mcmed_endpoint_completeness_by_outcome[
        "n_with_ed_endpoint"
    ]
)


mcmed_endpoint_completeness_by_outcome[
    "endpoint_available_pct"
] = (
    mcmed_endpoint_completeness_by_outcome[
        "n_with_ed_endpoint"
    ]
    /
    mcmed_endpoint_completeness_by_outcome[
        "n_encounters"
    ]
    * 100
)



novel_lab_features = [
    "poe_result__lab__c_reactive_protein",
    "poe_result__lab__lactate",
    "poe_result__lab__iron",
    "poe_result__lab__ferritin",
    "poe_result__lab__transferrin",
    "poe_result__lab__iron_binding_capacity_total",
]


novel_feature_timing_audit = (
    s2_lab_final_compatibility_audit.loc[
        s2_lab_final_compatibility_audit[
            "feature_name"
        ].isin(
            novel_lab_features
        ),
        [
            "feature_name",
            "mapping_class",
            "original_abnormal_positive_stays",
            "legacy_charttime_positive_stays",
            "strict_available_positive_stays",
            "positive_stays_removed_by_availability_rule",
            "retained_pct",
            "mapped_fluids",
        ],
    ]
    .copy()
)



print(
    "Frozen laboratory mapping summary:"
)

display(
    mapping_summary.round(
        {
            "percentage_of_351": 2,
        }
    )
)


print(
    "\nCompatibility-status summary:"
)

display(
    compatibility_summary
)


print(
    "\nUnmapped features that were positive in the "
    "original laboratory audit:"
)

display(
    unmapped_originally_active[
        [
            "feature_name",
            "original_abnormal_positive_stays",
            "legacy_charttime_positive_stays",
            "compatibility_status",
        ]
    ]
)


print(
    "\nActive features aggregating multiple specimen fluids:"
)

display(
    multi_fluid_active_features[
        [
            "feature_name",
            "n_mapped_itemids",
            "n_mapped_fluids",
            "mapped_fluids",
            "legacy_charttime_positive_stays",
            "strict_available_positive_stays",
        ]
    ].head(50)
)


print(
    "\nMC-MED ED endpoint completeness by outcome:"
)

display(
    mcmed_endpoint_completeness_by_outcome.round(
        {
            "endpoint_available_pct": 2,
        }
    )
)


print(
    "\nNovel laboratory-feature timing audit:"
)

display(
    novel_feature_timing_audit.round(
        {
            "retained_pct": 2,
        }
    )
)



mimic_s2_lab_features_primary_candidate = (
    mimic_s2_lab_features_legacy_audit.copy()
)


assert (
    mimic_s2_lab_features_primary_candidate[
        FROZEN_S2_LAB_FEATURES
    ].columns.tolist()
    ==
    list(FROZEN_S2_LAB_FEATURES)
)


n_unresolved_originally_active = len(
    unmapped_originally_active
)


print(
    "\nFINAL LABORATORY COMPATIBILITY AUDIT COMPLETED"
)

print(
    "Proposed primary definition: "
    "abnormal result with charttime inside the ED encounter."
)

print(
    "Unmapped features with historical positive support:",
    n_unresolved_originally_active,
)


if n_unresolved_originally_active == 0:
    print(
        "\nNo historically positive feature is lost solely because "
        "of missing d_labitems mapping."
    )
    print(
        "The encounter-window matrix is ready for final freezing."
    )
else:
    print(
        "\nDo not freeze yet."
    )
    print(
        "The displayed historically positive but currently "
        "unmapped features require recovery of the original "
        "mapping logic."
    )


#### Freeze the MIMIC-IV laboratory-result block

The MIMIC-IV objective laboratory block contains 351 frozen binary feature columns retained from the original development inventory. Of these, 241 features map to at least one MIMIC-IV laboratory dictionary item and 157 have at least one positive value in the current cohort. The remaining structurally unsupported columns are retained as zeros to preserve the original feature order used for modelling, SHAP analysis and clinician review.

The primary laboratory definition marks a feature as present when an abnormal result with a matching laboratory label has a chart time within the current ED encounter. A stricter result-availability definition based on store time is retained only as a temporal-quality audit because it disproportionately removes slower specialised assays and cannot be applied consistently to MC-MED.

Some laboratory labels correspond to multiple item identifiers or specimen fluids. This matching behaviour is preserved to reproduce the established development representation. Consequently, these features represent generic abnormal laboratory-label evidence and are not interpreted as specimen-specific measurements unless the specimen type is explicit in the feature name.

This cell creates the authoritative in-memory MIMIC-IV laboratory matrix. It does not yet write the matrix to disk.


In [ ]:



required_objects = [
    "FROZEN_S2_LAB_FEATURES",
    "mimic_s2_lab_features_legacy_audit",
    "mimic_s2_lab_features_candidate",
    "s2_lab_final_compatibility_audit",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding S2 laboratory reconstruction and "
        f"compatibility-audit cells first: {missing_objects}"
    )



mimic_s2_lab_support_exceptions = (
    s2_lab_final_compatibility_audit.loc[
        s2_lab_final_compatibility_audit[
            "compatibility_status"
        ].ne(
            "COMPATIBLE_OR_COHORT_DIFFERENCE"
        ),
        [
            "feature_position",
            "feature_name",
            "mapping_class",
            "original_abnormal_positive_stays",
            "legacy_charttime_positive_stays",
            "strict_available_positive_stays",
            "compatibility_status",
        ],
    ]
    .sort_values(
        [
            "compatibility_status",
            "feature_position",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Features with changed support relative to the "
    "original development audit:"
)

display(
    mimic_s2_lab_support_exceptions
)



mimic_s2_lab_structural_zero_features = (
    s2_lab_final_compatibility_audit.loc[
        s2_lab_final_compatibility_audit[
            "original_abnormal_positive_stays"
        ].eq(0)
        &
        s2_lab_final_compatibility_audit[
            "legacy_charttime_positive_stays"
        ].eq(0),
        "feature_name",
    ]
    .tolist()
)


historically_positive_unmapped = (
    s2_lab_final_compatibility_audit[
        "mapping_class"
    ].eq(
        "UNMAPPED_TO_D_LABITEMS"
    )
    &
    s2_lab_final_compatibility_audit[
        "original_abnormal_positive_stays"
    ].gt(0)
)


assert not historically_positive_unmapped.any(), (
    "At least one historically positive frozen feature is "
    "currently unmapped. Do not freeze the matrix."
)



mimic_lab_metadata_columns = [
    column
    for column in [
        "subject_id",
        "stay_id",
        "split",
        "outcome",
        "outcome_binary",
    ]
    if column in mimic_s2_lab_features_legacy_audit.columns
]


mimic_s2_lab_features_primary = (
    mimic_s2_lab_features_legacy_audit[
        mimic_lab_metadata_columns
        +
        list(
            FROZEN_S2_LAB_FEATURES
        )
    ]
    .copy()
    .reset_index(drop=True)
)


mimic_s2_lab_features_availability_audit = (
    mimic_s2_lab_features_candidate[
        mimic_lab_metadata_columns
        +
        list(
            FROZEN_S2_LAB_FEATURES
        )
    ]
    .copy()
    .reset_index(drop=True)
)


MIMIC_S2_LAB_PRIMARY_DEFINITION = (
    "Binary abnormal laboratory-label result with charttime "
    "between current ED arrival and ED departure."
)

MIMIC_S2_LAB_AVAILABILITY_AUDIT_DEFINITION = (
    "Primary encounter-window definition plus result storetime "
    "on or before ED departure."
)



assert len(
    mimic_s2_lab_features_primary
) == 8550

assert mimic_s2_lab_features_primary[
    "stay_id"
].is_unique

assert (
    mimic_s2_lab_features_primary[
        FROZEN_S2_LAB_FEATURES
    ].columns.tolist()
    ==
    list(
        FROZEN_S2_LAB_FEATURES
    )
)


observed_values = set(
    np.unique(
        mimic_s2_lab_features_primary[
            FROZEN_S2_LAB_FEATURES
        ].to_numpy()
    ).tolist()
)


assert observed_values.issubset(
    {
        0,
        1,
    }
), (
    "The frozen MIMIC laboratory matrix contains values other "
    f"than 0 and 1: {observed_values}"
)


assert not mimic_s2_lab_features_primary[
    FROZEN_S2_LAB_FEATURES
].isna().any().any(), (
    "The frozen laboratory feature block contains missing values."
)


structural_zero_support = (
    mimic_s2_lab_features_primary[
        mimic_s2_lab_structural_zero_features
    ]
    .sum(axis=0)
)


assert structural_zero_support.eq(0).all(), (
    "At least one feature classified as a structural zero "
    "contains a positive value."
)



mimic_s2_lab_feature_order_checksum = (
    hashlib.sha256(
        "\n".join(
            FROZEN_S2_LAB_FEATURES
        ).encode(
            "utf-8"
        )
    )
    .hexdigest()
)


matrix_hash_values = pd.util.hash_pandas_object(
    mimic_s2_lab_features_primary[
        [
            "stay_id",
            *
            FROZEN_S2_LAB_FEATURES,
        ]
    ],
    index=False,
).to_numpy()


mimic_s2_lab_matrix_checksum = (
    hashlib.sha256(
        matrix_hash_values.tobytes()
    )
    .hexdigest()
)



primary_feature_support = (
    mimic_s2_lab_features_primary[
        FROZEN_S2_LAB_FEATURES
    ]
    .sum(axis=0)
)


strict_feature_support = (
    mimic_s2_lab_features_availability_audit[
        FROZEN_S2_LAB_FEATURES
    ]
    .sum(axis=0)
)


mimic_s2_lab_freeze_audit = pd.DataFrame(
    [
        {
            "measure": "Frozen cohort encounters",
            "value": len(
                mimic_s2_lab_features_primary
            ),
        },
        {
            "measure": "Frozen laboratory columns",
            "value": len(
                FROZEN_S2_LAB_FEATURES
            ),
        },
        {
            "measure": (
                "Primary columns with >=1 positive value"
            ),
            "value": int(
                primary_feature_support.gt(0).sum()
            ),
        },
        {
            "measure": (
                "Availability-audit columns with >=1 "
                "positive value"
            ),
            "value": int(
                strict_feature_support.gt(0).sum()
            ),
        },
        {
            "measure": "Structural-zero columns retained",
            "value": len(
                mimic_s2_lab_structural_zero_features
            ),
        },
        {
            "measure": (
                "Originally active but currently zero"
            ),
            "value": int(
                (
                    s2_lab_final_compatibility_audit[
                        "compatibility_status"
                    ]
                    .eq(
                        "ORIGINAL_ACTIVE_CURRENT_ZERO"
                    )
                ).sum()
            ),
        },
        {
            "measure": (
                "Currently active but originally zero"
            ),
            "value": int(
                (
                    s2_lab_final_compatibility_audit[
                        "compatibility_status"
                    ]
                    .eq(
                        "CURRENT_ACTIVE_ORIGINAL_ZERO"
                    )
                ).sum()
            ),
        },
        {
            "measure": (
                "Historically positive but currently unmapped"
            ),
            "value": int(
                historically_positive_unmapped.sum()
            ),
        },
        {
            "measure": "Feature-order SHA256",
            "value": (
                mimic_s2_lab_feature_order_checksum
            ),
        },
        {
            "measure": "Primary matrix SHA256",
            "value": (
                mimic_s2_lab_matrix_checksum
            ),
        },
    ]
)


display(
    mimic_s2_lab_freeze_audit
)


print(
    "\nMIMIC-IV S2 LABORATORY BLOCK FROZEN IN MEMORY"
)

print(
    "Primary object: mimic_s2_lab_features_primary"
)

print(
    "Timing-audit object: "
    "mimic_s2_lab_features_availability_audit"
)

print(
    "Primary matrix shape:",
    mimic_s2_lab_features_primary.shape,
)

print(
    "No matrix has yet been written to disk."
)


#### Prepare MC-MED laboratory results

The MC-MED laboratory source is restricted to the encounters in the frozen external-validation cohort. Laboratory rows are linked directly through the ED encounter identifier, CSN.

Component-level abnormality is preferred because an order may contain several components and an order-level abnormal flag may be caused by a different component. The order-level flag is therefore used only when the component-level abnormality field is unavailable.

Order and result timestamps are audited against the available ED encounter window but are not used as mandatory exclusion criteria. A substantial proportion of MC-MED encounters do not contain a recorded disposition or departure endpoint, and applying an endpoint-based rule would remove encounters selectively according to timestamp availability rather than clinical eligibility.

This cell extracts and validates the relevant source rows. It does not yet map laboratory names to the frozen 351-feature vocabulary.


In [ ]:



required_objects = [
    "SOURCE_PATHS",
    "mcmed_s2_window",
    "FROZEN_S2_LAB_FEATURES",
    "parse_datetime_naive",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding cohort, timing and laboratory cells "
        f"first. Missing objects: {missing_objects}"
    )


MCMED_LAB_CHUNKSIZE = 500_000



mcmed_labs_path = Path(
    SOURCE_PATHS["mcmed_labs"]
).resolve()


if not mcmed_labs_path.exists():
    raise FileNotFoundError(
        f"MC-MED laboratory file not found:\n{mcmed_labs_path}"
    )


mcmed_labs_header = pd.read_csv(
    mcmed_labs_path,
    nrows=0,
).columns.tolist()


preferred_columns = [
    "CSN",
    "Order_time",
    "Result_time",
    "Display_name",
    "Abnormal",
    "Component_name",
    "Component_result",
    "Component_value",
    "Component_units",
    "Component_abnormal",
    "Component_nml_low",
    "Component_nml_high",
]


available_columns = [
    column
    for column in preferred_columns
    if column in mcmed_labs_header
]


required_columns = {
    "CSN",
    "Display_name",
    "Component_name",
    "Component_abnormal",
}


missing_required_columns = (
    required_columns
    -
    set(
        available_columns
    )
)


if missing_required_columns:
    raise ValueError(
        "MC-MED labs.csv is missing required columns: "
        f"{sorted(missing_required_columns)}"
    )



cohort_csns = set(
    mcmed_s2_window[
        "CSN"
    ]
    .dropna()
    .astype("int64")
    .tolist()
)


mcmed_lab_chunks = []

n_source_rows_processed = 0
n_rows_matching_frozen_csns = 0


for chunk_number, chunk in enumerate(
    pd.read_csv(
        mcmed_labs_path,
        usecols=available_columns,
        chunksize=MCMED_LAB_CHUNKSIZE,
        low_memory=False,
    ),
    start=1,
):
    n_source_rows_processed += len(
        chunk
    )

    chunk[
        "CSN"
    ] = pd.to_numeric(
        chunk[
            "CSN"
        ],
        errors="coerce",
    ).astype("Int64")

    chunk = chunk.loc[
        chunk[
            "CSN"
        ].isin(
            cohort_csns
        )
    ].copy()

    n_rows_matching_frozen_csns += len(
        chunk
    )

    if not chunk.empty:
        mcmed_lab_chunks.append(
            chunk
        )

    if chunk_number % 10 == 0:
        print(
            "Processed chunks:",
            chunk_number,
            "| retained rows:",
            n_rows_matching_frozen_csns,
        )


if not mcmed_lab_chunks:
    raise ValueError(
        "No MC-MED laboratory rows matched the frozen cohort."
    )


mcmed_s2_lab_rows = pd.concat(
    mcmed_lab_chunks,
    ignore_index=True,
)


del mcmed_lab_chunks



for column in preferred_columns:
    if column not in mcmed_s2_lab_rows.columns:
        mcmed_s2_lab_rows[
            column
        ] = pd.NA


for column in [
    "Order_time",
    "Result_time",
]:
    mcmed_s2_lab_rows[
        column
    ] = parse_datetime_naive(
        mcmed_s2_lab_rows[
            column
        ]
    )



def clean_mcmed_lab_flag(
    value,
) -> str:
    """Normalise a component- or order-level flag."""

    if pd.isna(value):
        return ""

    text = str(
        value
    ).strip().lower()

    if text in {
        "",
        "nan",
        "none",
        "null",
        "na",
        "n/a",
        "not available",
    }:
        return ""

    return text


def classify_mcmed_lab_flag(
    value,
) -> int:
    """
    Return 1 only for explicit abnormal/high/low/critical flags.

    Ambiguous or missing values are returned as zero.
    """

    text = clean_mcmed_lab_flag(
        value
    )

    if not text:
        return 0

    abnormal_exact = {
        "1",
        "true",
        "yes",
        "y",
        "abnormal",
        "critical",
        "panic",
        "high",
        "low",
        "very high",
        "very low",
        "h",
        "l",
        "hh",
        "ll",
    }

    normal_exact = {
        "0",
        "false",
        "no",
        "n",
        "normal",
        "within normal limits",
        "within range",
        "negative",
    }

    if text in abnormal_exact:
        return 1

    if text in normal_exact:
        return 0

    abnormal_pattern = (
        r"\b(abnormal|critical|panic|high|low)\b"
    )

    if re.search(
        abnormal_pattern,
        text,
    ):
        return 1

    return 0


def derive_mcmed_component_abnormality(
    row,
):
    """
    Prefer Component_abnormal.

    Use the order-level Abnormal field only when the component
    field is empty.
    """

    component_flag = clean_mcmed_lab_flag(
        row[
            "Component_abnormal"
        ]
    )

    order_flag = clean_mcmed_lab_flag(
        row[
            "Abnormal"
        ]
    )

    if component_flag:
        return pd.Series(
            {
                "lab_abnormal_binary":
                    classify_mcmed_lab_flag(
                        component_flag
                    ),
                "lab_abnormal_source":
                    "COMPONENT_ABNORMAL",
                "lab_abnormal_source_value":
                    component_flag,
            }
        )

    if order_flag:
        return pd.Series(
            {
                "lab_abnormal_binary":
                    classify_mcmed_lab_flag(
                        order_flag
                    ),
                "lab_abnormal_source":
                    "ORDER_ABNORMAL_FALLBACK",
                "lab_abnormal_source_value":
                    order_flag,
            }
        )

    return pd.Series(
        {
            "lab_abnormal_binary": 0,
            "lab_abnormal_source":
                "NO_EXPLICIT_FLAG",
            "lab_abnormal_source_value": "",
        }
    )


mcmed_s2_lab_rows[
    [
        "lab_abnormal_binary",
        "lab_abnormal_source",
        "lab_abnormal_source_value",
    ]
] = mcmed_s2_lab_rows.apply(
    derive_mcmed_component_abnormality,
    axis=1,
)


mcmed_s2_lab_rows[
    "lab_abnormal_binary"
] = mcmed_s2_lab_rows[
    "lab_abnormal_binary"
].astype("int8")



mcmed_lab_time_window = (
    mcmed_s2_window[
        [
            "CSN",
            "current_arrival_time",
            "current_ed_end_time",
            "outcome",
        ]
    ]
    .copy()
)


mcmed_s2_lab_rows = (
    mcmed_s2_lab_rows
    .merge(
        mcmed_lab_time_window,
        on="CSN",
        how="left",
        validate="many_to_one",
    )
)


mcmed_s2_lab_rows[
    "order_on_or_after_arrival"
] = (
    mcmed_s2_lab_rows[
        "Order_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "current_arrival_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "Order_time"
    ].ge(
        mcmed_s2_lab_rows[
            "current_arrival_time"
        ]
    )
)


mcmed_s2_lab_rows[
    "result_on_or_after_arrival"
] = (
    mcmed_s2_lab_rows[
        "Result_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "current_arrival_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "Result_time"
    ].ge(
        mcmed_s2_lab_rows[
            "current_arrival_time"
        ]
    )
)


mcmed_s2_lab_rows[
    "order_within_known_ed_window"
] = (
    mcmed_s2_lab_rows[
        "order_on_or_after_arrival"
    ]
    &
    mcmed_s2_lab_rows[
        "current_ed_end_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "Order_time"
    ].le(
        mcmed_s2_lab_rows[
            "current_ed_end_time"
        ]
    )
)


mcmed_s2_lab_rows[
    "result_within_known_ed_window"
] = (
    mcmed_s2_lab_rows[
        "result_on_or_after_arrival"
    ]
    &
    mcmed_s2_lab_rows[
        "current_ed_end_time"
    ].notna()
    &
    mcmed_s2_lab_rows[
        "Result_time"
    ].le(
        mcmed_s2_lab_rows[
            "current_ed_end_time"
        ]
    )
)



encounters_with_labs = set(
    mcmed_s2_lab_rows[
        "CSN"
    ].dropna().tolist()
)


encounters_with_abnormal_labs = set(
    mcmed_s2_lab_rows.loc[
        mcmed_s2_lab_rows[
            "lab_abnormal_binary"
        ].eq(1),
        "CSN",
    ]
    .dropna()
    .tolist()
)


mcmed_s2_lab_source_audit = pd.DataFrame(
    [
        {
            "measure": "Source laboratory rows processed",
            "value": n_source_rows_processed,
        },
        {
            "measure": (
                "Rows linked to frozen cohort CSNs"
            ),
            "value": len(
                mcmed_s2_lab_rows
            ),
        },
        {
            "measure": (
                "Frozen cohort encounters"
            ),
            "value": len(
                mcmed_s2_window
            ),
        },
        {
            "measure": (
                "Encounters with >=1 laboratory row"
            ),
            "value": len(
                encounters_with_labs
            ),
        },
        {
            "measure": (
                "Encounters with >=1 explicitly abnormal row"
            ),
            "value": len(
                encounters_with_abnormal_labs
            ),
        },
        {
            "measure": (
                "Explicitly abnormal laboratory rows"
            ),
            "value": int(
                mcmed_s2_lab_rows[
                    "lab_abnormal_binary"
                ].sum()
            ),
        },
        {
            "measure": (
                "Rows with known ED endpoint"
            ),
            "value": int(
                mcmed_s2_lab_rows[
                    "current_ed_end_time"
                ].notna().sum()
            ),
        },
        {
            "measure": (
                "Orders inside known ED window"
            ),
            "value": int(
                mcmed_s2_lab_rows[
                    "order_within_known_ed_window"
                ].sum()
            ),
        },
        {
            "measure": (
                "Results inside known ED window"
            ),
            "value": int(
                mcmed_s2_lab_rows[
                    "result_within_known_ed_window"
                ].sum()
            ),
        },
    ]
)


mcmed_s2_lab_flag_source_audit = (
    mcmed_s2_lab_rows
    .groupby(
        [
            "lab_abnormal_source",
            "lab_abnormal_source_value",
            "lab_abnormal_binary",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="n_rows"
    )
    .sort_values(
        "n_rows",
        ascending=False,
    )
)


mcmed_s2_lab_encounter_coverage = pd.DataFrame(
    {
        "CSN": mcmed_s2_window[
            "CSN"
        ]
    }
)


mcmed_s2_lab_encounter_coverage[
    "has_any_lab_row"
] = (
    mcmed_s2_lab_encounter_coverage[
        "CSN"
    ].isin(
        encounters_with_labs
    )
)


mcmed_s2_lab_encounter_coverage[
    "has_any_abnormal_lab_row"
] = (
    mcmed_s2_lab_encounter_coverage[
        "CSN"
    ].isin(
        encounters_with_abnormal_labs
    )
)


mcmed_s2_lab_encounter_coverage = (
    mcmed_s2_lab_encounter_coverage
    .merge(
        mcmed_s2_window[
            [
                "CSN",
                "outcome",
            ]
        ],
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)


mcmed_s2_lab_coverage_by_outcome = (
    mcmed_s2_lab_encounter_coverage
    .groupby(
        "outcome",
        as_index=False,
    )
    .agg(
        n_encounters=(
            "CSN",
            "size",
        ),
        n_with_any_lab_row=(
            "has_any_lab_row",
            "sum",
        ),
        n_with_any_abnormal_lab_row=(
            "has_any_abnormal_lab_row",
            "sum",
        ),
    )
)


mcmed_s2_lab_coverage_by_outcome[
    "pct_with_any_lab_row"
] = (
    mcmed_s2_lab_coverage_by_outcome[
        "n_with_any_lab_row"
    ]
    /
    mcmed_s2_lab_coverage_by_outcome[
        "n_encounters"
    ]
    * 100
)


mcmed_s2_lab_coverage_by_outcome[
    "pct_with_any_abnormal_lab_row"
] = (
    mcmed_s2_lab_coverage_by_outcome[
        "n_with_any_abnormal_lab_row"
    ]
    /
    mcmed_s2_lab_coverage_by_outcome[
        "n_encounters"
    ]
    * 100
)



print(
    "MC-MED laboratory source audit:"
)

display(
    mcmed_s2_lab_source_audit
)


print(
    "\nMC-MED abnormal-flag audit:"
)

display(
    mcmed_s2_lab_flag_source_audit.head(40)
)


print(
    "\nMC-MED laboratory coverage by outcome:"
)

display(
    mcmed_s2_lab_coverage_by_outcome.round(
        {
            "pct_with_any_lab_row": 2,
            "pct_with_any_abnormal_lab_row": 2,
        }
    )
)


print(
    "\nMC-MED LABORATORY SOURCE PREPARATION PASSED"
)

print(
    "Retained laboratory rows:",
    len(
        mcmed_s2_lab_rows
    ),
)

print(
    "Unique frozen-cohort CSNs with laboratory data:",
    mcmed_s2_lab_rows[
        "CSN"
    ].nunique(),
)

print(
    "\nNo frozen-feature mapping or matrix has yet been created."
)


In [ ]:



required_objects = [
    "mcmed_s2_lab_rows",
    "mcmed_s2_window",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run S2-LAB-08 first. Missing objects: "
        f"{missing_objects}"
    )



mcmed_lab_fallback_audit_rows = (
    mcmed_s2_lab_rows.copy()
    .reset_index(drop=True)
)


mcmed_lab_fallback_audit_rows[
    "CSN"
] = pd.to_numeric(
    mcmed_lab_fallback_audit_rows["CSN"],
    errors="coerce",
).astype("Int64")


mcmed_lab_fallback_audit_rows[
    "lab_abnormal_source"
] = (
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_source"
    ]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)


mcmed_lab_fallback_audit_rows[
    "lab_abnormal_binary"
] = pd.to_numeric(
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_binary"
    ],
    errors="coerce",
).fillna(0).astype("int8")



mcmed_lab_fallback_audit_rows[
    "order_instance_time"
] = (
    mcmed_lab_fallback_audit_rows[
        "Order_time"
    ]
    .where(
        mcmed_lab_fallback_audit_rows[
            "Order_time"
        ].notna(),
        mcmed_lab_fallback_audit_rows[
            "Result_time"
        ],
    )
)


mcmed_lab_fallback_audit_rows[
    "display_name_key"
] = (
    mcmed_lab_fallback_audit_rows[
        "Display_name"
    ]
    .astype("string")
    .fillna("<MISSING_DISPLAY_NAME>")
    .str.strip()
    .str.upper()
)


mcmed_lab_fallback_audit_rows[
    "component_name_key"
] = (
    mcmed_lab_fallback_audit_rows[
        "Component_name"
    ]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
    .replace(
        "",
        pd.NA,
    )
)


mcmed_lab_fallback_audit_rows[
    "order_instance_time_key"
] = (
    mcmed_lab_fallback_audit_rows[
        "order_instance_time"
    ]
    .astype("string")
    .fillna("<MISSING_ORDER_AND_RESULT_TIME>")
)


mcmed_lab_fallback_audit_rows[
    "order_instance_key"
] = (
    mcmed_lab_fallback_audit_rows[
        "CSN"
    ].astype("string")
    +
    "||"
    +
    mcmed_lab_fallback_audit_rows[
        "display_name_key"
    ]
    +
    "||"
    +
    mcmed_lab_fallback_audit_rows[
        "order_instance_time_key"
    ]
)



mcmed_lab_order_component_counts = (
    mcmed_lab_fallback_audit_rows
    .groupby(
        "order_instance_key",
        as_index=False,
        dropna=False,
    )
    .agg(
        CSN=(
            "CSN",
            "first",
        ),
        Display_name=(
            "Display_name",
            "first",
        ),
        order_instance_time=(
            "order_instance_time",
            "first",
        ),
        n_source_rows=(
            "order_instance_key",
            "size",
        ),
        n_unique_components=(
            "component_name_key",
            lambda values: int(
                values.dropna().nunique()
            ),
        ),
    )
)


mcmed_lab_fallback_audit_rows = (
    mcmed_lab_fallback_audit_rows
    .merge(
        mcmed_lab_order_component_counts[
            [
                "order_instance_key",
                "n_source_rows",
                "n_unique_components",
            ]
        ],
        on="order_instance_key",
        how="left",
        validate="many_to_one",
    )
)



component_positive_mask = (
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_binary"
    ].eq(1)
    &
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_source"
    ].eq(
        "COMPONENT_ABNORMAL"
    )
)


order_fallback_positive_mask = (
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_binary"
    ].eq(1)
    &
    mcmed_lab_fallback_audit_rows[
        "lab_abnormal_source"
    ].eq(
        "ORDER_ABNORMAL_FALLBACK"
    )
)


single_component_fallback_mask = (
    order_fallback_positive_mask
    &
    mcmed_lab_fallback_audit_rows[
        "n_unique_components"
    ].eq(1)
)


multi_component_fallback_mask = (
    order_fallback_positive_mask
    &
    mcmed_lab_fallback_audit_rows[
        "n_unique_components"
    ].gt(1)
)


unresolved_component_count_fallback_mask = (
    order_fallback_positive_mask
    &
    mcmed_lab_fallback_audit_rows[
        "n_unique_components"
    ].eq(0)
)



candidate_rule_masks = {
    "01_COMPONENT_FLAG_ONLY":
        component_positive_mask,

    "02_COMPONENT_PLUS_SINGLE_COMPONENT_FALLBACK":
        (
            component_positive_mask
            |
            single_component_fallback_mask
        ),

    "03_COMPONENT_PLUS_ALL_ORDER_FALLBACK":
        (
            component_positive_mask
            |
            order_fallback_positive_mask
        ),
}



rule_summary_rows = []


for rule_name, rule_mask in candidate_rule_masks.items():

    positive_rows = (
        mcmed_lab_fallback_audit_rows.loc[
            rule_mask
        ]
    )

    rule_summary_rows.append(
        {
            "rule": rule_name,
            "n_positive_source_rows": int(
                rule_mask.sum()
            ),
            "n_encounters_with_any_positive_row": int(
                positive_rows[
                    "CSN"
                ].nunique()
            ),
            "percentage_of_6707_encounters": (
                positive_rows[
                    "CSN"
                ].nunique()
                /
                mcmed_s2_window[
                    "CSN"
                ].nunique()
                * 100
            ),
            "n_order_instances_with_positive_row": int(
                positive_rows[
                    "order_instance_key"
                ].nunique()
            ),
        }
    )


mcmed_lab_abnormality_rule_summary = (
    pd.DataFrame(
        rule_summary_rows
    )
)


print(
    "Comparison of candidate MC-MED abnormality rules:"
)

display(
    mcmed_lab_abnormality_rule_summary.round(
        {
            "percentage_of_6707_encounters": 2,
        }
    )
)



mcmed_lab_order_fallback_summary = pd.DataFrame(
    [
        {
            "measure": (
                "All explicitly abnormal source rows"
            ),
            "count": int(
                mcmed_lab_fallback_audit_rows[
                    "lab_abnormal_binary"
                ].eq(1).sum()
            ),
        },
        {
            "measure": (
                "Component-level abnormal rows"
            ),
            "count": int(
                component_positive_mask.sum()
            ),
        },
        {
            "measure": (
                "Order-fallback abnormal rows"
            ),
            "count": int(
                order_fallback_positive_mask.sum()
            ),
        },
        {
            "measure": (
                "Order-fallback rows from "
                "single-component orders"
            ),
            "count": int(
                single_component_fallback_mask.sum()
            ),
        },
        {
            "measure": (
                "Order-fallback rows from "
                "multi-component orders"
            ),
            "count": int(
                multi_component_fallback_mask.sum()
            ),
        },
        {
            "measure": (
                "Order-fallback rows with no identifiable "
                "component"
            ),
            "count": int(
                unresolved_component_count_fallback_mask.sum()
            ),
        },
        {
            "measure": (
                "Encounters receiving any order-level fallback"
            ),
            "count": int(
                mcmed_lab_fallback_audit_rows.loc[
                    order_fallback_positive_mask,
                    "CSN",
                ].nunique()
            ),
        },
        {
            "measure": (
                "Encounters receiving multi-component fallback"
            ),
            "count": int(
                mcmed_lab_fallback_audit_rows.loc[
                    multi_component_fallback_mask,
                    "CSN",
                ].nunique()
            ),
        },
    ]
)


print(
    "\nOrder-level fallback summary:"
)

display(
    mcmed_lab_order_fallback_summary
)



outcome_rule_rows = []


cohort_outcome_reference = (
    mcmed_s2_window[
        [
            "CSN",
            "outcome",
        ]
    ]
    .drop_duplicates(
        subset="CSN"
    )
    .copy()
)


for rule_name, rule_mask in candidate_rule_masks.items():

    positive_csns = set(
        mcmed_lab_fallback_audit_rows.loc[
            rule_mask,
            "CSN",
        ]
        .dropna()
        .astype("int64")
        .tolist()
    )

    temporary_outcome_table = (
        cohort_outcome_reference.copy()
    )

    temporary_outcome_table[
        "has_positive_lab"
    ] = (
        temporary_outcome_table[
            "CSN"
        ]
        .astype("Int64")
        .isin(
            positive_csns
        )
    )

    outcome_summary = (
        temporary_outcome_table
        .groupby(
            "outcome",
            as_index=False,
            dropna=False,
        )
        .agg(
            n_encounters=(
                "CSN",
                "size",
            ),
            n_with_positive_lab=(
                "has_positive_lab",
                "sum",
            ),
        )
    )

    outcome_summary[
        "percentage_with_positive_lab"
    ] = (
        outcome_summary[
            "n_with_positive_lab"
        ]
        /
        outcome_summary[
            "n_encounters"
        ]
        * 100
    )

    outcome_summary[
        "rule"
    ] = rule_name

    outcome_rule_rows.append(
        outcome_summary
    )


mcmed_lab_rule_comparison_by_outcome = (
    pd.concat(
        outcome_rule_rows,
        ignore_index=True,
    )
    [
        [
            "rule",
            "outcome",
            "n_encounters",
            "n_with_positive_lab",
            "percentage_with_positive_lab",
        ]
    ]
)


print(
    "\nCandidate abnormality rules by outcome:"
)

display(
    mcmed_lab_rule_comparison_by_outcome.round(
        {
            "percentage_with_positive_lab": 2,
        }
    )
)



mcmed_multicomponent_fallback_rows = (
    mcmed_lab_fallback_audit_rows.loc[
        multi_component_fallback_mask
    ]
    .copy()
)


mcmed_multicomponent_fallback_top_orders = (
    mcmed_multicomponent_fallback_rows
    .groupby(
        "display_name_key",
        as_index=False,
    )
    .agg(
        n_fallback_rows=(
            "order_instance_key",
            "size",
        ),
        n_order_instances=(
            "order_instance_key",
            "nunique",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        median_components_per_order=(
            "n_unique_components",
            "median",
        ),
        maximum_components_per_order=(
            "n_unique_components",
            "max",
        ),
        example_components=(
            "component_name_key",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:12]
            ),
        ),
    )
    .sort_values(
        [
            "n_fallback_rows",
            "n_encounters",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nTop multi-component orders receiving "
    "order-level abnormal fallback:"
)

display(
    mcmed_multicomponent_fallback_top_orders.head(
        40
    )
)



fallback_order_instance_distribution = (
    mcmed_lab_fallback_audit_rows.loc[
        order_fallback_positive_mask,
        [
            "order_instance_key",
            "n_unique_components",
        ],
    ]
    .drop_duplicates(
        subset="order_instance_key"
    )
    .groupby(
        "n_unique_components",
        as_index=False,
    )
    .agg(
        n_order_instances=(
            "order_instance_key",
            "size",
        )
    )
    .sort_values(
        "n_unique_components"
    )
)


fallback_order_instance_distribution[
    "percentage_of_fallback_orders"
] = (
    fallback_order_instance_distribution[
        "n_order_instances"
    ]
    /
    fallback_order_instance_distribution[
        "n_order_instances"
    ].sum()
    * 100
)


print(
    "\nComponent-count distribution among orders "
    "using abnormal fallback:"
)

display(
    fallback_order_instance_distribution.round(
        {
            "percentage_of_fallback_orders": 2,
        }
    )
)



mcmed_s2_lab_rows_component_only = (
    mcmed_lab_fallback_audit_rows.loc[
        candidate_rule_masks[
            "01_COMPONENT_FLAG_ONLY"
        ]
    ]
    .copy()
)


mcmed_s2_lab_rows_conservative_candidate = (
    mcmed_lab_fallback_audit_rows.loc[
        candidate_rule_masks[
            "02_COMPONENT_PLUS_SINGLE_COMPONENT_FALLBACK"
        ]
    ]
    .copy()
)


mcmed_s2_lab_rows_legacy_intent_candidate = (
    mcmed_lab_fallback_audit_rows.loc[
        candidate_rule_masks[
            "03_COMPONENT_PLUS_ALL_ORDER_FALLBACK"
        ]
    ]
    .copy()
)


print(
    "\nMC-MED ORDER-FALLBACK AUDIT COMPLETED"
)

print(
    "No laboratory-name mapping or feature matrix "
    "has been created."
)

print(
    "Recommended provisional candidate: "
    "mcmed_s2_lab_rows_conservative_candidate"
)


#### MC-MED exact laboratory mapping audit

The authoritative MC-MED laboratory candidate contains only rows with an explicit component-level abnormal or critical flag. Order-level abnormality is excluded because all fallback-positive rows originated from multi-component panels and did not identify any additional encounters or abnormal order instances.

This section maps MC-MED component names to the frozen 351-feature laboratory vocabulary used during model development and clinician review. Exact normalised component-name matching is attempted first. Display-name matching is permitted only for reconstructed orders containing one identifiable component, because a panel-level display name cannot safely be assigned to each component of a multi-component order.

The purpose of this cell is to measure exact cross-dataset compatibility and identify unresolved high-frequency laboratory labels. It does not use fuzzy matching, create the final binary matrix or write any output to disk.In MC-MED, laboratory abnormality was determined using component-level abnormality indicators. Order-level abnormality indicators were not propagated to individual components because multi-component panels may be classified as abnormal when only one component lies outside its reference range. An audit showed that order-level fallback added 47,447 component-positive records without identifying any additional encounters or laboratory order instances containing abnormal results. Consequently, only explicitly abnormal or critical component-level results were used for feature reconstruction.


In [ ]:



required_objects = [
    "mcmed_s2_lab_rows_component_only",
    "mcmed_s2_window",
    "s2_lab_feature_plan",
    "FROZEN_S2_LAB_FEATURES",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the preceding S2 laboratory cells first. "
        f"Missing objects: {missing_objects}"
    )



def normalise_lab_label(value) -> str:
    """
    Convert a laboratory label into a conservative comparison key.

    The function standardises case, Unicode characters,
    punctuation and whitespace. It does not remove clinically
    meaningful words or expand abbreviations.
    """

    if pd.isna(value):
        return ""

    text = unicodedata.normalize(
        "NFKD",
        str(value),
    )

    text = (
        text.encode(
            "ascii",
            errors="ignore",
        )
        .decode("ascii")
        .lower()
        .strip()
    )

    text = text.replace(
        "&",
        " and ",
    )

    text = text.replace(
        "%",
        " percent ",
    )

    text = re.sub(
        r"[_/\\|:;,()\[\]{}\-]+",
        " ",
        text,
    )

    text = re.sub(
        r"[^a-z0-9\s]+",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text



feature_plan = (
    s2_lab_feature_plan.copy()
    .reset_index(drop=True)
)


feature_name_candidates = [
    "poe_result_feature_name",
    "feature_name",
]


feature_name_column = next(
    (
        column
        for column in feature_name_candidates
        if column in feature_plan.columns
    ),
    None,
)


if feature_name_column is None:
    raise ValueError(
        "No laboratory result feature-name column was found in "
        "s2_lab_feature_plan."
    )


feature_plan[
    "frozen_feature_name"
] = (
    feature_plan[
        feature_name_column
    ]
    .astype("string")
    .str.strip()
)


feature_plan = feature_plan.loc[
    feature_plan[
        "frozen_feature_name"
    ].isin(
        FROZEN_S2_LAB_FEATURES
    )
].copy()


feature_plan = (
    feature_plan
    .drop_duplicates(
        subset="frozen_feature_name",
        keep="first",
    )
    .reset_index(drop=True)
)


missing_from_plan = sorted(
    set(
        FROZEN_S2_LAB_FEATURES
    )
    -
    set(
        feature_plan[
            "frozen_feature_name"
        ]
    )
)


if missing_from_plan:
    raise ValueError(
        "Some frozen laboratory features are absent from the "
        "feature plan. Example missing features:\n"
        f"{missing_from_plan[:20]}"
    )


assert len(
    feature_plan
) == 351

assert feature_plan[
    "frozen_feature_name"
].is_unique



plan_alias_columns = [
    column
    for column in [
        "lab_order_name_norm",
        "lab_order_name",
        "poe_item",
        "display_name",
    ]
    if column in feature_plan.columns
]


alias_records = []


for _, row in feature_plan.iterrows():

    feature_name = row[
        "frozen_feature_name"
    ]

    for alias_column in plan_alias_columns:

        alias_value = row[
            alias_column
        ]

        alias_key = normalise_lab_label(
            alias_value
        )

        if alias_key:
            alias_records.append(
                {
                    "feature_name": feature_name,
                    "alias_key": alias_key,
                    "alias_source": alias_column,
                    "alias_original": str(
                        alias_value
                    ),
                }
            )

    feature_token = feature_name.replace(
        "poe_result__lab__",
        "",
        1,
    ).replace(
        "_",
        " ",
    )

    feature_token_key = normalise_lab_label(
        feature_token
    )

    if feature_token_key:
        alias_records.append(
            {
                "feature_name": feature_name,
                "alias_key": feature_token_key,
                "alias_source": "frozen_feature_token",
                "alias_original": feature_token,
            }
        )


frozen_lab_alias_table = (
    pd.DataFrame(
        alias_records
    )
    .drop_duplicates(
        subset=[
            "feature_name",
            "alias_key",
        ]
    )
    .reset_index(drop=True)
)


alias_feature_counts = (
    frozen_lab_alias_table
    .groupby(
        "alias_key",
        as_index=False,
    )
    .agg(
        n_frozen_features=(
            "feature_name",
            "nunique",
        ),
        mapped_features=(
            "feature_name",
            lambda values: " | ".join(
                sorted(
                    values.unique()
                )
            ),
        ),
    )
)


frozen_lab_alias_table = (
    frozen_lab_alias_table
    .merge(
        alias_feature_counts,
        on="alias_key",
        how="left",
        validate="many_to_one",
    )
)


unique_frozen_lab_aliases = (
    frozen_lab_alias_table.loc[
        frozen_lab_alias_table[
            "n_frozen_features"
        ].eq(1)
    ]
    [
        [
            "alias_key",
            "feature_name",
        ]
    ]
    .drop_duplicates(
        subset="alias_key"
    )
)


ambiguous_frozen_lab_aliases = (
    alias_feature_counts.loc[
        alias_feature_counts[
            "n_frozen_features"
        ].gt(1)
    ]
    .sort_values(
        [
            "n_frozen_features",
            "alias_key",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


exact_alias_to_feature = dict(
    zip(
        unique_frozen_lab_aliases[
            "alias_key"
        ],
        unique_frozen_lab_aliases[
            "feature_name"
        ],
    )
)



mcmed_s2_lab_mapping_audit_rows = (
    mcmed_s2_lab_rows_component_only
    .copy()
    .reset_index(drop=True)
)


assert mcmed_s2_lab_mapping_audit_rows[
    "lab_abnormal_binary"
].eq(1).all()


assert mcmed_s2_lab_mapping_audit_rows[
    "lab_abnormal_source"
].astype("string").str.upper().eq(
    "COMPONENT_ABNORMAL"
).all()


mcmed_s2_lab_mapping_audit_rows[
    "component_name_key"
] = (
    mcmed_s2_lab_mapping_audit_rows[
        "Component_name"
    ]
    .map(
        normalise_lab_label
    )
)


mcmed_s2_lab_mapping_audit_rows[
    "display_name_key"
] = (
    mcmed_s2_lab_mapping_audit_rows[
        "Display_name"
    ]
    .map(
        normalise_lab_label
    )
)



mcmed_s2_lab_mapping_audit_rows[
    "component_exact_feature"
] = (
    mcmed_s2_lab_mapping_audit_rows[
        "component_name_key"
    ]
    .map(
        exact_alias_to_feature
    )
)



single_component_order_mask = (
    pd.to_numeric(
        mcmed_s2_lab_mapping_audit_rows[
            "n_unique_components"
        ],
        errors="coerce",
    )
    .eq(1)
)


display_fallback_feature = (
    mcmed_s2_lab_mapping_audit_rows[
        "display_name_key"
    ]
    .map(
        exact_alias_to_feature
    )
)


display_fallback_eligible = (
    mcmed_s2_lab_mapping_audit_rows[
        "component_exact_feature"
    ].isna()
    &
    single_component_order_mask
    &
    display_fallback_feature.notna()
)


mcmed_s2_lab_mapping_audit_rows[
    "mapped_feature_name"
] = (
    mcmed_s2_lab_mapping_audit_rows[
        "component_exact_feature"
    ]
)


mcmed_s2_lab_mapping_audit_rows.loc[
    display_fallback_eligible,
    "mapped_feature_name",
] = display_fallback_feature.loc[
    display_fallback_eligible
]


mcmed_s2_lab_mapping_audit_rows[
    "mapping_method"
] = np.select(
    [
        mcmed_s2_lab_mapping_audit_rows[
            "component_exact_feature"
        ].notna(),

        display_fallback_eligible,
    ],
    [
        "EXACT_COMPONENT_NAME",
        "EXACT_SINGLE_COMPONENT_DISPLAY_NAME",
    ],
    default="UNMAPPED_EXACT",
)



mapped_feature_values = set(
    mcmed_s2_lab_mapping_audit_rows[
        "mapped_feature_name"
    ]
    .dropna()
    .tolist()
)


unexpected_mapped_features = sorted(
    mapped_feature_values
    -
    set(
        FROZEN_S2_LAB_FEATURES
    )
)


if unexpected_mapped_features:
    raise AssertionError(
        "Unexpected mapped laboratory features were produced:\n"
        f"{unexpected_mapped_features[:20]}"
    )



mapped_rows_mask = (
    mcmed_s2_lab_mapping_audit_rows[
        "mapped_feature_name"
    ].notna()
)


mapped_encounters = (
    mcmed_s2_lab_mapping_audit_rows.loc[
        mapped_rows_mask,
        "CSN",
    ]
    .dropna()
    .nunique()
)


mcmed_s2_lab_exact_mapping_summary = pd.DataFrame(
    [
        {
            "measure": (
                "Component-confirmed abnormal source rows"
            ),
            "value": len(
                mcmed_s2_lab_mapping_audit_rows
            ),
        },
        {
            "measure": (
                "Rows with a usable component name"
            ),
            "value": int(
                mcmed_s2_lab_mapping_audit_rows[
                    "component_name_key"
                ].ne("").sum()
            ),
        },
        {
            "measure": (
                "Rows mapped by exact component name"
            ),
            "value": int(
                mcmed_s2_lab_mapping_audit_rows[
                    "mapping_method"
                ].eq(
                    "EXACT_COMPONENT_NAME"
                ).sum()
            ),
        },
        {
            "measure": (
                "Rows mapped by single-component display name"
            ),
            "value": int(
                mcmed_s2_lab_mapping_audit_rows[
                    "mapping_method"
                ].eq(
                    "EXACT_SINGLE_COMPONENT_DISPLAY_NAME"
                ).sum()
            ),
        },
        {
            "measure": (
                "Rows remaining unmatched after exact mapping"
            ),
            "value": int(
                (
                    ~mapped_rows_mask
                ).sum()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 exactly mapped abnormal lab"
            ),
            "value": int(
                mapped_encounters
            ),
        },
        {
            "measure": (
                "Frozen features with >=1 exact external positive"
            ),
            "value": int(
                mcmed_s2_lab_mapping_audit_rows.loc[
                    mapped_rows_mask,
                    "mapped_feature_name",
                ].nunique()
            ),
        },
        {
            "measure": (
                "Unique conservative aliases in frozen lookup"
            ),
            "value": len(
                unique_frozen_lab_aliases
            ),
        },
        {
            "measure": (
                "Ambiguous aliases excluded from automatic mapping"
            ),
            "value": len(
                ambiguous_frozen_lab_aliases
            ),
        },
    ]
)


print(
    "MC-MED exact laboratory mapping summary:"
)

display(
    mcmed_s2_lab_exact_mapping_summary
)



mcmed_s2_lab_mapping_method_summary = (
    mcmed_s2_lab_mapping_audit_rows
    .groupby(
        "mapping_method",
        as_index=False,
        dropna=False,
    )
    .agg(
        n_source_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        n_unique_component_labels=(
            "component_name_key",
            "nunique",
        ),
        n_mapped_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .sort_values(
        "n_source_rows",
        ascending=False,
    )
    .reset_index(drop=True)
)


mcmed_s2_lab_mapping_method_summary[
    "percentage_of_component_abnormal_rows"
] = (
    mcmed_s2_lab_mapping_method_summary[
        "n_source_rows"
    ]
    /
    len(
        mcmed_s2_lab_mapping_audit_rows
    )
    * 100
)


print(
    "\nExact mapping methods:"
)

display(
    mcmed_s2_lab_mapping_method_summary.round(
        {
            "percentage_of_component_abnormal_rows": 2,
        }
    )
)



mcmed_s2_lab_exact_feature_support = (
    mcmed_s2_lab_mapping_audit_rows.loc[
        mapped_rows_mask
    ]
    .groupby(
        "mapped_feature_name",
        as_index=False,
    )
    .agg(
        external_positive_rows=(
            "CSN",
            "size",
        ),
        external_positive_encounters=(
            "CSN",
            "nunique",
        ),
        n_unique_component_labels=(
            "component_name_key",
            "nunique",
        ),
        component_label_examples=(
            "Component_name",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:10]
            ),
        ),
    )
    .rename(
        columns={
            "mapped_feature_name":
                "feature_name",
        }
    )
)


mcmed_s2_lab_exact_feature_support = (
    pd.DataFrame(
        {
            "feature_name":
                FROZEN_S2_LAB_FEATURES
        }
    )
    .merge(
        mcmed_s2_lab_exact_feature_support,
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
)


for count_column in [
    "external_positive_rows",
    "external_positive_encounters",
    "n_unique_component_labels",
]:
    mcmed_s2_lab_exact_feature_support[
        count_column
    ] = (
        mcmed_s2_lab_exact_feature_support[
            count_column
        ]
        .fillna(0)
        .astype("int64")
    )


mcmed_s2_lab_exact_feature_support[
    "has_external_support"
] = (
    mcmed_s2_lab_exact_feature_support[
        "external_positive_encounters"
    ].gt(0)
)


print(
    "\nTop frozen laboratory features supported by exact mapping:"
)

display(
    mcmed_s2_lab_exact_feature_support.loc[
        mcmed_s2_lab_exact_feature_support[
            "has_external_support"
        ]
    ]
    .sort_values(
        [
            "external_positive_encounters",
            "external_positive_rows",
        ],
        ascending=False,
    )
    .head(40)
)



unmatched_rows = (
    mcmed_s2_lab_mapping_audit_rows.loc[
        ~mapped_rows_mask
    ]
    .copy()
)


mcmed_s2_lab_top_unmatched_components = (
    unmatched_rows
    .groupby(
        [
            "component_name_key",
            "Component_name",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_abnormal_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        n_order_instances=(
            "order_instance_key",
            "nunique",
        ),
        display_name_examples=(
            "Display_name",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:8]
            ),
        ),
    )
    .sort_values(
        [
            "n_encounters",
            "n_abnormal_rows",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nHighest-frequency unmatched abnormal component labels:"
)

display(
    mcmed_s2_lab_top_unmatched_components.head(
        60
    )
)



print(
    "\nAmbiguous frozen aliases excluded from automatic mapping:"
)

display(
    ambiguous_frozen_lab_aliases.head(
        40
    )
)


print(
    "\nMC-MED EXACT LABORATORY MAPPING AUDIT PASSED"
)

print(
    "Authoritative abnormality source: component-level flags only"
)

print(
    "No fuzzy mapping has been applied."
)

print(
    "No 351-column matrix has been created or frozen."
)


In [ ]:



required_objects = [
    "mcmed_s2_lab_mapping_audit_rows",
    "FROZEN_S2_LAB_FEATURES",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        "Run the exact laboratory-mapping cell first. "
        f"Missing objects: {missing_objects}"
    )


frozen_lab_features = [
    str(feature)
    for feature in FROZEN_S2_LAB_FEATURES
]

frozen_lab_set = set(frozen_lab_features)

assert len(frozen_lab_features) == 351
assert len(frozen_lab_set) == 351



def resolve_frozen_lab_target(suffix_candidates):
    """
    Return one unambiguous frozen feature from candidate suffixes.
    """

    candidate_names = [
        f"poe_result__lab__{suffix}"
        for suffix in suffix_candidates
    ]

    matches = [
        feature
        for feature in candidate_names
        if feature in frozen_lab_set
    ]

    matches = list(dict.fromkeys(matches))

    if len(matches) == 1:
        return matches[0], "RESOLVED"

    if len(matches) == 0:
        return pd.NA, "TARGET_NOT_FOUND"

    return pd.NA, "AMBIGUOUS_TARGET"



high_confidence_rules = [
    {
        "component_name_key": "neutrophil absolute auto diff",
        "target_suffixes": ["absolute_neutrophil_count"],
        "reason": "Same analyte; MC-MED auto-differential wording.",
    },
    {
        "component_name_key": "hemoglobin hgb",
        "target_suffixes": ["hemoglobin"],
        "reason": "HGB is a standard abbreviation for haemoglobin.",
    },
    {
        "component_name_key": "white blood cells wbc",
        "target_suffixes": ["white_blood_cells"],
        "reason": "WBC is a standard abbreviation.",
    },
    {
        "component_name_key": "eosinophil absolute auto diff",
        "target_suffixes": ["absolute_eosinophil_count"],
        "reason": "Same absolute eosinophil measurement.",
    },
    {
        "component_name_key": "lymphocyte absolute auto diff",
        "target_suffixes": ["absolute_lymphocyte_count"],
        "reason": "Same absolute lymphocyte measurement.",
    },
    {
        "component_name_key": (
            "mean corpuscular hemoglobin concentration mchc"
        ),
        "target_suffixes": ["mchc"],
        "reason": "Expanded name and abbreviation identify MCHC.",
    },
    {
        "component_name_key": "hematocrit hct",
        "target_suffixes": ["hematocrit"],
        "reason": "HCT is a standard abbreviation for haematocrit.",
    },
    {
        "component_name_key": "alt sgpt",
        "target_suffixes": ["alanine_aminotransferase_alt"],
        "reason": "SGPT is the historical synonym for ALT.",
    },
    {
        "component_name_key": "red blood cells rbc",
        "target_suffixes": ["red_blood_cells"],
        "reason": "CBC blood RBC corresponds to red blood cells.",
    },
    {
        "component_name_key": "red cell distribution width rdw",
        "target_suffixes": ["rdw"],
        "reason": "Expanded name and abbreviation identify RDW.",
    },
    {
        "component_name_key": "ast sgot",
        "target_suffixes": [
            "asparate_aminotransferase_ast",
            "aspartate_aminotransferase_ast",
        ],
        "reason": "SGOT is the historical synonym for AST.",
    },
    {
        "component_name_key": "monocyte absolute auto diff",
        "target_suffixes": ["absolute_monocyte_count"],
        "reason": "Same absolute monocyte measurement.",
    },
    {
        "component_name_key": "mean corpuscular hemoglobin mch",
        "target_suffixes": ["mch"],
        "reason": "Expanded name and abbreviation identify MCH.",
    },
    {
        "component_name_key": "mean corpuscular volume mcv",
        "target_suffixes": ["mcv"],
        "reason": "Expanded name and abbreviation identify MCV.",
    },
    {
        "component_name_key": "platelet count plt",
        "target_suffixes": ["platelet_count"],
        "reason": "PLT is a standard abbreviation for platelet count.",
    },
    {
        "component_name_key": "blood urea nitrogen bun",
        "target_suffixes": ["urea_nitrogen"],
        "reason": "BUN corresponds to blood urea nitrogen.",
    },
    {
        "component_name_key": "specific gravity urine ua",
        "target_suffixes": ["specific_gravity"],
        "reason": "Same urine-specific-gravity measurement.",
    },
    {
        "component_name_key": "calcium",
        "target_suffixes": ["calcium_total"],
        "reason": "Routine metabolic-panel calcium is total calcium.",
    },
    {
        "component_name_key": "prothrombin time",
        "target_suffixes": ["pt"],
        "reason": "PT is the standard abbreviation.",
    },
    {
        "component_name_key": "poc lactate istat",
        "target_suffixes": ["lactate"],
        "reason": "Same lactate analyte measured using a POC platform.",
    },
    {
        "component_name_key": "inr",
        "target_suffixes": ["inr_pt", "inr"],
        "reason": "Same international normalised ratio concept.",
    },
    {
        "component_name_key": "part thromboplastin time",
        "target_suffixes": [
            "ptt",
            "partial_thromboplastin_time",
        ],
        "reason": "Same partial thromboplastin-time measurement.",
    },
    {
        "component_name_key": "poc glucose by meter",
        "target_suffixes": ["glucose"],
        "reason": "Same glucose analyte measured using a POC device.",
    },
    {
        "component_name_key": "bilirubin conjugated",
        "target_suffixes": ["bilirubin_direct"],
        "reason": "Conjugated bilirubin corresponds to direct bilirubin.",
    },
    {
        "component_name_key": "osmolality serum",
        "target_suffixes": [
            "osmolality",
            "osmolality_serum",
        ],
        "reason": "Same serum-osmolality measurement.",
    },
    {
        "component_name_key": "triglyceride",
        "target_suffixes": ["triglycerides"],
        "reason": "Singular and plural naming difference.",
    },
    {
        "component_name_key": "ldh total",
        "target_suffixes": ["lactate_dehydrogenase_ld"],
        "reason": "LDH and lactate dehydrogenase are synonyms.",
    },
    {
        "component_name_key": "esr automated",
        "target_suffixes": ["sedimentation_rate"],
        "reason": "ESR corresponds to erythrocyte sedimentation rate.",
    },
    {
        "component_name_key": "iron total",
        "target_suffixes": ["iron"],
        "reason": "Same serum total-iron analyte.",
    },
    {
        "component_name_key": "tsh",
        "target_suffixes": ["thyroid_stimulating_hormone"],
        "reason": "TSH is the standard abbreviation.",
    },
    {
        "component_name_key": "ammonia plasma",
        "target_suffixes": ["ammonia"],
        "reason": "Same plasma-ammonia analyte.",
    },
    {
        "component_name_key": "transferrin saturation",
        "target_suffixes": ["transferrin_saturation"],
        "reason": "Same calculated iron-status measurement.",
    },
    {
        "component_name_key": "nitrite urine ua",
        "target_suffixes": [
            "nitrite",
            "nitrite_urine",
        ],
        "reason": "Same urine nitrite measurement.",
    },
]



review_only_rules = [
    {
        "component_name_key": "poc po2 v istat",
        "possible_target_suffixes": ["po2"],
        "reason": (
            "Venous POC PO2 should be mapped only if the frozen "
            "PO2 feature intentionally combines blood specimens."
        ),
    },
    {
        "component_name_key": "poc pco2 v istat",
        "possible_target_suffixes": ["pco2"],
        "reason": (
            "Venous POC PCO2 requires confirmation of frozen "
            "specimen semantics."
        ),
    },
    {
        "component_name_key": "poc ph v istat",
        "possible_target_suffixes": ["ph"],
        "reason": (
            "Venous blood pH must not automatically be combined "
            "with urine or other-fluid pH."
        ),
    },
    {
        "component_name_key": "poc hco3 v istat",
        "possible_target_suffixes": ["bicarbonate", "hco3"],
        "reason": (
            "Blood-gas calculated bicarbonate may differ from "
            "serum chemistry bicarbonate."
        ),
    },
    {
        "component_name_key": "poc tco2 v istat",
        "possible_target_suffixes": ["bicarbonate", "total_co2"],
        "reason": (
            "Total CO2 is related to, but not automatically "
            "identical to, bicarbonate."
        ),
    },
    {
        "component_name_key": "co2",
        "possible_target_suffixes": ["bicarbonate", "total_co2"],
        "reason": (
            "Serum total CO2 and bicarbonate require an explicit "
            "cross-dataset definition."
        ),
    },
    {
        "component_name_key": "poc o2 saturation istat venous",
        "possible_target_suffixes": [
            "oxygen_saturation",
            "o2_saturation",
        ],
        "reason": (
            "Laboratory venous oxygen saturation is not equivalent "
            "to triage pulse oximetry."
        ),
    },
    {
        "component_name_key": "egfr refit without race 2021",
        "possible_target_suffixes": ["egfr"],
        "reason": (
            "The calculation equation should be compatible with "
            "the frozen eGFR definition."
        ),
    },
    {
        "component_name_key": "egfr for african american",
        "possible_target_suffixes": ["egfr"],
        "reason": (
            "Race-adjusted historical eGFR should not automatically "
            "be combined with race-free eGFR."
        ),
    },
    {
        "component_name_key": "high sensitivity troponin",
        "possible_target_suffixes": [
            "troponin_i_high_sensitivity",
            "high_sensitivity_troponin",
            "troponin_i",
        ],
        "reason": (
            "The assay type and troponin subtype must match the "
            "frozen target."
        ),
    },
    {
        "component_name_key": "hstni previous value",
        "possible_target_suffixes": [],
        "reason": (
            "A previous-value field is not a current independently "
            "measured troponin result."
        ),
    },
]



high_confidence_rule_table = pd.DataFrame(
    high_confidence_rules
)

resolved_targets = [
    resolve_frozen_lab_target(
        suffixes
    )
    for suffixes in high_confidence_rule_table[
        "target_suffixes"
    ]
]

high_confidence_rule_table[
    "resolved_target"
] = [
    result[0]
    for result in resolved_targets
]

high_confidence_rule_table[
    "resolution_status"
] = [
    result[1]
    for result in resolved_targets
]


source_support = (
    mcmed_s2_lab_mapping_audit_rows
    .groupby(
        "component_name_key",
        as_index=False,
    )
    .agg(
        n_abnormal_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        component_name_examples=(
            "Component_name",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:5]
            ),
        ),
        unit_examples=(
            "Component_units",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:8]
            ),
        ),
    )
)


high_confidence_rule_table = (
    high_confidence_rule_table
    .merge(
        source_support,
        on="component_name_key",
        how="left",
        validate="one_to_one",
    )
)


for column in [
    "n_abnormal_rows",
    "n_encounters",
]:
    high_confidence_rule_table[
        column
    ] = (
        high_confidence_rule_table[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


high_confidence_rule_table[
    "apply_mapping"
] = (
    high_confidence_rule_table[
        "resolution_status"
    ].eq("RESOLVED")
    &
    high_confidence_rule_table[
        "n_abnormal_rows"
    ].gt(0)
)


assert high_confidence_rule_table[
    "component_name_key"
].is_unique


print("High-confidence curated alias audit:")

display(
    high_confidence_rule_table[
        [
            "component_name_key",
            "resolved_target",
            "resolution_status",
            "n_abnormal_rows",
            "n_encounters",
            "unit_examples",
            "reason",
            "apply_mapping",
        ]
    ]
    .sort_values(
        [
            "apply_mapping",
            "n_encounters",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)



curated_alias_dictionary = dict(
    zip(
        high_confidence_rule_table.loc[
            high_confidence_rule_table[
                "apply_mapping"
            ],
            "component_name_key",
        ],
        high_confidence_rule_table.loc[
            high_confidence_rule_table[
                "apply_mapping"
            ],
            "resolved_target",
        ],
    )
)


mcmed_s2_lab_mapping_curated_rows = (
    mcmed_s2_lab_mapping_audit_rows
    .copy()
)


curated_candidate = (
    mcmed_s2_lab_mapping_curated_rows[
        "component_name_key"
    ]
    .map(
        curated_alias_dictionary
    )
)


curated_mapping_mask = (
    mcmed_s2_lab_mapping_curated_rows[
        "mapped_feature_name"
    ].isna()
    &
    curated_candidate.notna()
)


mcmed_s2_lab_mapping_curated_rows.loc[
    curated_mapping_mask,
    "mapped_feature_name",
] = curated_candidate.loc[
    curated_mapping_mask
]


mcmed_s2_lab_mapping_curated_rows.loc[
    curated_mapping_mask,
    "mapping_method",
] = "CURATED_HIGH_CONFIDENCE_ALIAS"



mapped_values = set(
    mcmed_s2_lab_mapping_curated_rows[
        "mapped_feature_name"
    ]
    .dropna()
    .astype(str)
)


unexpected_features = sorted(
    mapped_values
    -
    frozen_lab_set
)


if unexpected_features:
    raise AssertionError(
        "Curated mapping produced unexpected frozen features:\n"
        f"{unexpected_features[:20]}"
    )


source_target_counts = (
    mcmed_s2_lab_mapping_curated_rows
    .dropna(
        subset=[
            "mapped_feature_name",
        ]
    )
    .groupby(
        "component_name_key"
    )[
        "mapped_feature_name"
    ]
    .nunique()
)


assert source_target_counts.le(1).all(), (
    "At least one MC-MED component label maps to multiple "
    "frozen laboratory features."
)



curated_mapping_summary = (
    mcmed_s2_lab_mapping_curated_rows
    .groupby(
        "mapping_method",
        as_index=False,
        dropna=False,
    )
    .agg(
        n_source_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        n_unique_component_labels=(
            "component_name_key",
            "nunique",
        ),
        n_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .sort_values(
        "n_source_rows",
        ascending=False,
    )
    .reset_index(drop=True)
)


curated_mapping_summary[
    "percentage_of_component_abnormal_rows"
] = (
    curated_mapping_summary[
        "n_source_rows"
    ]
    /
    len(
        mcmed_s2_lab_mapping_curated_rows
    )
    * 100
)


print("\nMapping summary after curated aliases:")

display(
    curated_mapping_summary.round(
        {
            "percentage_of_component_abnormal_rows": 2,
        }
    )
)



mapped_curated_rows = (
    mcmed_s2_lab_mapping_curated_rows.loc[
        mcmed_s2_lab_mapping_curated_rows[
            "mapped_feature_name"
        ].notna()
    ]
)


curated_support_observed = (
    mapped_curated_rows
    .groupby(
        "mapped_feature_name",
        as_index=False,
    )
    .agg(
        external_positive_rows=(
            "CSN",
            "size",
        ),
        external_positive_encounters=(
            "CSN",
            "nunique",
        ),
        n_component_labels=(
            "component_name_key",
            "nunique",
        ),
        mapping_methods=(
            "mapping_method",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                )
            ),
        ),
    )
    .rename(
        columns={
            "mapped_feature_name": "feature_name",
        }
    )
)


mcmed_s2_lab_curated_feature_support = (
    pd.DataFrame(
        {
            "feature_name": frozen_lab_features,
        }
    )
    .merge(
        curated_support_observed,
        on="feature_name",
        how="left",
        validate="one_to_one",
    )
)


for column in [
    "external_positive_rows",
    "external_positive_encounters",
    "n_component_labels",
]:
    mcmed_s2_lab_curated_feature_support[
        column
    ] = (
        mcmed_s2_lab_curated_feature_support[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


mcmed_s2_lab_curated_feature_support[
    "has_external_support"
] = (
    mcmed_s2_lab_curated_feature_support[
        "external_positive_encounters"
    ].gt(0)
)


support_summary = pd.DataFrame(
    [
        {
            "measure": "Component-confirmed abnormal rows",
            "value": len(
                mcmed_s2_lab_mapping_curated_rows
            ),
        },
        {
            "measure": "Rows mapped after curated aliases",
            "value": len(
                mapped_curated_rows
            ),
        },
        {
            "measure": "Rows still unmatched",
            "value": int(
                mcmed_s2_lab_mapping_curated_rows[
                    "mapped_feature_name"
                ].isna().sum()
            ),
        },
        {
            "measure": (
                "Encounters with >=1 mapped abnormal feature"
            ),
            "value": int(
                mapped_curated_rows[
                    "CSN"
                ].nunique()
            ),
        },
        {
            "measure": (
                "Frozen features with >=1 external positive"
            ),
            "value": int(
                mcmed_s2_lab_curated_feature_support[
                    "has_external_support"
                ].sum()
            ),
        },
        {
            "measure": (
                "High-confidence curated aliases applied"
            ),
            "value": int(
                high_confidence_rule_table[
                    "apply_mapping"
                ].sum()
            ),
        },
    ]
)


print("\nCurated mapping support summary:")

display(support_summary)


print(
    "\nTop supported frozen features after curated mapping:"
)

display(
    mcmed_s2_lab_curated_feature_support.loc[
        mcmed_s2_lab_curated_feature_support[
            "has_external_support"
        ]
    ]
    .sort_values(
        "external_positive_encounters",
        ascending=False,
    )
    .head(50)
)



review_rule_table = pd.DataFrame(
    review_only_rules
)


review_rule_table = (
    review_rule_table
    .merge(
        source_support,
        on="component_name_key",
        how="left",
        validate="one_to_one",
    )
)


for column in [
    "n_abnormal_rows",
    "n_encounters",
]:
    review_rule_table[
        column
    ] = (
        review_rule_table[
            column
        ]
        .fillna(0)
        .astype("int64")
    )


review_rule_table[
    "possible_existing_targets"
] = (
    review_rule_table[
        "possible_target_suffixes"
    ]
    .map(
        lambda suffixes: " | ".join(
            [
                f"poe_result__lab__{suffix}"
                for suffix in suffixes
                if (
                    f"poe_result__lab__{suffix}"
                    in frozen_lab_set
                )
            ]
        )
    )
)


print(
    "\nPotential mappings deliberately withheld for review:"
)

display(
    review_rule_table[
        [
            "component_name_key",
            "n_abnormal_rows",
            "n_encounters",
            "unit_examples",
            "possible_existing_targets",
            "reason",
        ]
    ]
    .sort_values(
        "n_encounters",
        ascending=False,
    )
    .reset_index(drop=True)
)



remaining_unmatched = (
    mcmed_s2_lab_mapping_curated_rows.loc[
        mcmed_s2_lab_mapping_curated_rows[
            "mapped_feature_name"
        ].isna()
    ]
)


remaining_unmatched_summary = (
    remaining_unmatched
    .groupby(
        [
            "component_name_key",
            "Component_name",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        n_abnormal_rows=(
            "CSN",
            "size",
        ),
        n_encounters=(
            "CSN",
            "nunique",
        ),
        n_order_instances=(
            "order_instance_key",
            "nunique",
        ),
        unit_examples=(
            "Component_units",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:8]
            ),
        ),
        display_name_examples=(
            "Display_name",
            lambda values: " | ".join(
                sorted(
                    values.dropna()
                    .astype(str)
                    .unique()
                    .tolist()
                )[:6]
            ),
        ),
    )
    .sort_values(
        [
            "n_encounters",
            "n_abnormal_rows",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)


print(
    "\nHighest-frequency labels remaining unmatched:"
)

display(
    remaining_unmatched_summary.head(60)
)


print(
    "\nCURATED MC-MED LABORATORY MAPPING AUDIT PASSED"
)

print(
    "Exact and high-confidence deterministic aliases were applied."
)

print(
    "Review-only mappings were not applied."
)

print(
    "No final 351-column matrix has been created or saved."
)


#### Final MC-MED laboratory harmonisation

This section completes reconstruction of the MC-MED S2 laboratory block using the frozen MIMIC-IV development vocabulary.

1. The existing exact and curated high-confidence mappings are preserved without alteration.
2. A final small alias set is applied only to clearly equivalent laboratory analytes, including manual-differential variants, point-of-care measurements and whole-blood measurements.
3. Venous blood-gas measurements, calculated eGFR variants, previous-value fields, assay-specific troponins and other clinically non-equivalent measurements remain unmapped.
4. A source label is accepted only when exactly one corresponding target exists in the frozen 351-feature laboratory vocabulary.
5. The final patient-level matrix contains all 351 frozen columns in their original development order.
6. Features unavailable in MC-MED are retained as structural-zero columns rather than being removed.
7. No fuzzy matching, feature selection, imputation or model fitting is performed in this section.
8. The resulting matrix is frozen in memory only and is not yet written to disk.


In [ ]:



def _first_series(dataframe, column):
    """
    Safely return one Series even if a dataframe accidentally
    contains duplicate column labels.
    """
    selected = dataframe.loc[:, column]

    if isinstance(selected, pd.DataFrame):
        return selected.iloc[:, 0]

    return selected


def _normalise_csn(series):
    """
    Harmonise CSN values without changing the underlying
    encounter identity.
    """
    values = (
        series
        .astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    return values.mask(
        values.str.upper().isin(
            {
                "",
                "CSN",
                "NAN",
                "NONE",
                "<NA>",
            }
        )
    )


def _count_lab_feature_values(series):
    """
    Count values that look like frozen laboratory-feature names.
    """
    return int(
        series
        .astype("string")
        .str.startswith(
            "poe_result__lab__",
            na=False,
        )
        .sum()
    )



preferred_mapping_objects = [
    "mcmed_s2_lab_rows_curated",
    "mcmed_s2_lab_rows_mapped_curated",
    "mcmed_s2_lab_curated_rows",
    "mcmed_s2_lab_component_mapping_curated",
    "mcmed_s2_lab_rows_exact",
    "mcmed_s2_lab_rows_conservative_candidate",
    "mcmed_s2_lab_rows",
]

mapping_object_candidates = []

for object_name, object_value in list(globals().items()):

    if not isinstance(object_value, pd.DataFrame):
        continue

    if "CSN" not in object_value.columns:
        continue

    if "component_name_key" not in object_value.columns:
        continue

    possible_feature_columns = [
        column
        for column in object_value.columns
        if str(column).lower() in {
            "resolved_target",
            "mapped_feature",
            "mapped_feature_name",
            "feature_name",
            "final_feature_name",
            "laboratory_feature_name",
        }
        or "resolved_target" in str(column).lower()
        or "mapped_feature" in str(column).lower()
    ]

    best_feature_column = None
    best_mapped_count = -1

    for column in possible_feature_columns:

        mapped_count = _count_lab_feature_values(
            _first_series(
                object_value,
                column,
            )
        )

        if mapped_count > best_mapped_count:
            best_mapped_count = mapped_count
            best_feature_column = column

    if best_feature_column is not None:

        preferred_rank = (
            preferred_mapping_objects.index(object_name)
            if object_name in preferred_mapping_objects
            else len(preferred_mapping_objects)
        )

        mapping_object_candidates.append(
            {
                "object_name": object_name,
                "dataframe": object_value,
                "feature_column": best_feature_column,
                "mapped_count": best_mapped_count,
                "preferred_rank": preferred_rank,
                "n_rows": len(object_value),
            }
        )


if not mapping_object_candidates:
    raise ValueError(
        "No current MC-MED laboratory mapping dataframe was found.\n"
        "Run the exact and curated-alias laboratory mapping cells first."
    )


mapping_object_candidates = sorted(
    mapping_object_candidates,
    key=lambda item: (
        item["mapped_count"],
        -item["preferred_rank"],
        item["n_rows"],
    ),
    reverse=True,
)

selected_mapping_object = mapping_object_candidates[0]

mcmed_lab_mapping_source_name = selected_mapping_object[
    "object_name"
]

mcmed_lab_existing_feature_column = selected_mapping_object[
    "feature_column"
]

mcmed_s2_lab_rows_final = selected_mapping_object[
    "dataframe"
].copy()


print(
    "Selected current laboratory mapping object:",
    mcmed_lab_mapping_source_name,
)

print(
    "Selected existing mapped-feature column:",
    mcmed_lab_existing_feature_column,
)



vocabulary_candidates = []

for object_name, object_value in list(globals().items()):

    if not isinstance(object_value, pd.DataFrame):
        continue

    if "feature_name" not in object_value.columns:
        continue

    feature_series = _first_series(
        object_value,
        "feature_name",
    )

    feature_values = (
        feature_series
        .astype("string")
        .dropna()
        .drop_duplicates()
    )

    laboratory_values = feature_values[
        feature_values.str.startswith(
            "poe_result__lab__",
            na=False,
        )
    ].tolist()

    if laboratory_values:

        vocabulary_candidates.append(
            {
                "object_name": object_name,
                "dataframe": object_value,
                "features": laboratory_values,
                "n_features": len(laboratory_values),
            }
        )


exact_vocabulary_candidates = [
    candidate
    for candidate in vocabulary_candidates
    if candidate["n_features"] == 351
]


if exact_vocabulary_candidates:

    selected_vocabulary = exact_vocabulary_candidates[0]

    frozen_lab_feature_order = selected_vocabulary[
        "features"
    ]

    frozen_lab_vocabulary_source = selected_vocabulary[
        "object_name"
    ]

else:

    candidate_paths = []

    if "S2_LAB_RESULT_VOCABULARY_PATH" in globals():
        candidate_paths.append(
            Path(S2_LAB_RESULT_VOCABULARY_PATH)
        )

    if "FEATURE_VOCABULARY_DIR" in globals():
        candidate_paths.append(
            Path(FEATURE_VOCABULARY_DIR)
            / "s2_lab_result_vocabulary.csv"
        )

    if "PROJECT_ROOT" in globals():
        candidate_paths.append(
            Path(PROJECT_ROOT)
            / "config"
            / "feature_vocabularies"
            / "s2_lab_result_vocabulary.csv"
        )

    vocabulary_path = next(
        (
            path
            for path in candidate_paths
            if path.exists()
        ),
        None,
    )

    if vocabulary_path is None:
        raise FileNotFoundError(
            "The frozen S2 laboratory vocabulary could not "
            "be recovered from memory or disk."
        )

    frozen_lab_vocabulary_df = pd.read_csv(
        vocabulary_path
    )

    frozen_lab_feature_order = (
        frozen_lab_vocabulary_df["feature_name"]
        .astype("string")
        .dropna()
        .loc[
            lambda values:
            values.str.startswith(
                "poe_result__lab__",
                na=False,
            )
        ]
        .drop_duplicates()
        .tolist()
    )

    frozen_lab_vocabulary_source = str(
        vocabulary_path
    )


if len(frozen_lab_feature_order) != 351:
    raise AssertionError(
        "Expected exactly 351 frozen laboratory features, "
        f"but recovered {len(frozen_lab_feature_order)}."
    )


if len(set(frozen_lab_feature_order)) != 351:
    raise AssertionError(
        "The frozen laboratory vocabulary contains "
        "duplicate feature names."
    )


frozen_lab_feature_set = set(
    frozen_lab_feature_order
)


print(
    "Frozen laboratory vocabulary source:",
    frozen_lab_vocabulary_source,
)

print(
    "Frozen laboratory features recovered:",
    len(frozen_lab_feature_order),
)



mcmed_s2_lab_rows_final["CSN"] = _normalise_csn(
    mcmed_s2_lab_rows_final["CSN"]
)

mcmed_s2_lab_rows_final[
    "_final_lab_feature_name"
] = (
    _first_series(
        mcmed_s2_lab_rows_final,
        mcmed_lab_existing_feature_column,
    )
    .astype("string")
    .str.strip()
)


possible_method_columns = [
    "mapping_method",
    "lab_mapping_method",
    "match_method",
    "resolution_method",
]

existing_method_column = next(
    (
        column
        for column in possible_method_columns
        if column in mcmed_s2_lab_rows_final.columns
    ),
    None,
)


if existing_method_column is not None:

    mcmed_s2_lab_rows_final[
        "_final_lab_mapping_method"
    ] = (
        _first_series(
            mcmed_s2_lab_rows_final,
            existing_method_column,
        )
        .astype("string")
    )

else:

    mcmed_s2_lab_rows_final[
        "_final_lab_mapping_method"
    ] = pd.Series(
        pd.NA,
        index=mcmed_s2_lab_rows_final.index,
        dtype="string",
    )


component_key = (
    mcmed_s2_lab_rows_final[
        "component_name_key"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
)



final_conservative_alias_candidates = {

    "lymphocytes percent manual diff": [
        "poe_result__lab__lymphocytes",
    ],

    "lymphocytes absolute manual diff": [
        "poe_result__lab__absolute_lymphocyte_count",
    ],

    "neutrophils absolute manual diff segs bands": [
        "poe_result__lab__absolute_neutrophil_count",
    ],

    "neutrophils segmented percent manual diff": [
        "poe_result__lab__neutrophils",
    ],

    "monocytes absolute manual diff": [
        "poe_result__lab__absolute_monocyte_count",
    ],

    "monocytes percent manual diff": [
        "poe_result__lab__monocytes",
    ],

    "eosinophils absolute manual diff": [
        "poe_result__lab__absolute_eosinophil_count",
    ],

    "eosinophils percent manual diff": [
        "poe_result__lab__eosinophils",
    ],

    "neutrophils bands percent manual diff": [
        "poe_result__lab__bands",
    ],

    "metamyelocytes percent manual diff": [
        "poe_result__lab__metamyelocytes",
    ],

    "myelocytes percent manual diff": [
        "poe_result__lab__myelocytes",
    ],

    "immature granulocyte absolute auto diff wam": [
        "poe_result__lab__immature_granulocyte_absolute",
        "poe_result__lab__immature_granulocytes_absolute",
        "poe_result__lab__immature_granulocyte_count",
        "poe_result__lab__immature_granulocytes_abs",
    ],

    "immature granulocyte percent autodiff wam": [
        "poe_result__lab__immature_granulocyte_percent",
        "poe_result__lab__immature_granulocytes_percent",
        "poe_result__lab__immature_granulocytes",
    ],

    "poc creatinine istat": [
        "poe_result__lab__creatinine",
    ],

    "poc hgb calc istat": [
        "poe_result__lab__hemoglobin",
    ],

    "poc hct istat": [
        "poe_result__lab__hematocrit",
    ],

    "poc sodium istat": [
        "poe_result__lab__sodium",
    ],

    "poc potassium istat": [
        "poe_result__lab__potassium",
    ],

    "poc glucose istat": [
        "poe_result__lab__glucose",
    ],

    "glucose whole bld": [
        "poe_result__lab__glucose",
    ],

    "lactate whole bld": [
        "poe_result__lab__lactate",
    ],

    "d dimer elisa": [
        "poe_result__lab__d_dimer",
    ],

    "phosphorus": [
        "poe_result__lab__phosphate",
        "poe_result__lab__phosphorus",
    ],

    "tibc": [
        "poe_result__lab__iron_binding_capacity_total",
        "poe_result__lab__tibc",
    ],

    "beta hcg": [
        "poe_result__lab__beta_hcg",
        "poe_result__lab__hcg_beta",
        "poe_result__lab__hcg",
    ],

    "poc bhcg istat quantitative": [
        "poe_result__lab__beta_hcg",
        "poe_result__lab__hcg_beta",
        "poe_result__lab__hcg",
    ],
}



alias_resolution_rows = []
resolved_final_aliases = {}

for source_key, target_candidates in (
    final_conservative_alias_candidates.items()
):

    existing_targets = [
        target
        for target in target_candidates
        if target in frozen_lab_feature_set
    ]

    if len(existing_targets) == 1:

        status = "RESOLVED"
        resolved_target = existing_targets[0]

        resolved_final_aliases[
            source_key
        ] = resolved_target

    elif len(existing_targets) == 0:

        status = "TARGET_NOT_IN_FROZEN_VOCABULARY"
        resolved_target = pd.NA

    else:

        status = "AMBIGUOUS_MULTIPLE_TARGETS"
        resolved_target = " | ".join(
            existing_targets
        )

    alias_resolution_rows.append(
        {
            "component_name_key": source_key,
            "resolved_target": resolved_target,
            "resolution_status": status,
            "candidate_targets": " | ".join(
                target_candidates
            ),
        }
    )


mcmed_final_lab_alias_resolution_audit = pd.DataFrame(
    alias_resolution_rows
)



existing_valid_mapping = (
    mcmed_s2_lab_rows_final[
        "_final_lab_feature_name"
    ]
    .isin(
        frozen_lab_feature_set
    )
)

resolved_target_by_row = component_key.map(
    resolved_final_aliases
)

new_alias_mask = (
    ~existing_valid_mapping
    &
    resolved_target_by_row.notna()
)


mcmed_s2_lab_rows_final.loc[
    new_alias_mask,
    "_final_lab_feature_name",
] = resolved_target_by_row.loc[
    new_alias_mask
]


mcmed_s2_lab_rows_final.loc[
    new_alias_mask,
    "_final_lab_mapping_method",
] = "FINAL_CONSERVATIVE_ALIAS"



possible_abnormal_columns = [
    "lab_abnormal_binary",
    "component_abnormal_binary",
    "abnormal_binary",
    "is_abnormal",
]

abnormal_column = next(
    (
        column
        for column in possible_abnormal_columns
        if column in mcmed_s2_lab_rows_final.columns
    ),
    None,
)


if abnormal_column is None:

    abnormal_row_mask = pd.Series(
        True,
        index=mcmed_s2_lab_rows_final.index,
    )

    abnormal_rule_description = (
        "No separate binary column found; all retained rows "
        "were treated as the previously prepared "
        "component-confirmed abnormal source rows."
    )

else:

    abnormal_values = _first_series(
        mcmed_s2_lab_rows_final,
        abnormal_column,
    )

    if pd.api.types.is_bool_dtype(
        abnormal_values
    ):

        abnormal_row_mask = (
            abnormal_values
            .fillna(False)
            .astype(bool)
        )

    else:

        abnormal_row_mask = (
            pd.to_numeric(
                abnormal_values,
                errors="coerce",
            )
            .fillna(0)
            .eq(1)
        )

    abnormal_rule_description = (
        f"Authoritative abnormality column: "
        f"{abnormal_column}"
    )



before_feature_values = (
    _first_series(
        selected_mapping_object["dataframe"],
        mcmed_lab_existing_feature_column,
    )
    .astype("string")
)

before_valid_mapping = before_feature_values.isin(
    frozen_lab_feature_set
)

after_valid_mapping = (
    mcmed_s2_lab_rows_final[
        "_final_lab_feature_name"
    ]
    .isin(
        frozen_lab_feature_set
    )
)


def _mapping_metrics(valid_mask):

    eligible = (
        abnormal_row_mask
        &
        mcmed_s2_lab_rows_final["CSN"].notna()
    )

    mapped = eligible & valid_mask

    return {
        "mapped_abnormal_source_rows": int(
            mapped.sum()
        ),
        "encounters_with_at_least_one_mapped_abnormal_lab": int(
            mcmed_s2_lab_rows_final.loc[
                mapped,
                "CSN",
            ].nunique()
        ),
        "frozen_features_with_external_support": int(
            mcmed_s2_lab_rows_final.loc[
                mapped,
                "_final_lab_feature_name",
            ].nunique()
        ),
    }


before_metrics = _mapping_metrics(
    before_valid_mapping
)

after_metrics = _mapping_metrics(
    after_valid_mapping
)


mcmed_final_lab_mapping_comparison = pd.DataFrame(
    [
        {
            "definition": "Before final conservative pass",
            **before_metrics,
        },
        {
            "definition": "After final conservative pass",
            **after_metrics,
        },
    ]
)


newly_mapped_abnormal_mask = (
    abnormal_row_mask
    &
    new_alias_mask
    &
    mcmed_s2_lab_rows_final["CSN"].notna()
)


if newly_mapped_abnormal_mask.any():

    mcmed_final_lab_alias_application_audit = (
        mcmed_s2_lab_rows_final.loc[
            newly_mapped_abnormal_mask,
            [
                "component_name_key",
                "_final_lab_feature_name",
                "CSN",
            ],
        ]
        .groupby(
            [
                "component_name_key",
                "_final_lab_feature_name",
            ],
            dropna=False,
        )
        .agg(
            n_abnormal_source_rows=(
                "CSN",
                "size",
            ),
            n_encounters=(
                "CSN",
                "nunique",
            ),
        )
        .reset_index()
        .rename(
            columns={
                "_final_lab_feature_name":
                "resolved_target",
            }
        )
        .sort_values(
            [
                "n_encounters",
                "n_abnormal_source_rows",
            ],
            ascending=False,
        )
        .reset_index(drop=True)
    )

else:

    mcmed_final_lab_alias_application_audit = pd.DataFrame(
        columns=[
            "component_name_key",
            "resolved_target",
            "n_abnormal_source_rows",
            "n_encounters",
        ]
    )


print(
    "\nFinal conservative alias-resolution audit:"
)

display(
    mcmed_final_lab_alias_resolution_audit
)


print(
    "\nAliases actually applied to abnormal source rows:"
)

display(
    mcmed_final_lab_alias_application_audit
)


print(
    "\nFinal MC-MED laboratory mapping comparison:"
)

display(
    mcmed_final_lab_mapping_comparison
)


print(
    "\nAbnormal-row rule:"
)

print(
    abnormal_rule_description
)


print(
    "\nFINAL CONSERVATIVE LABORATORY ALIAS PASS COMPLETED"
)

print(
    "No fuzzy matching was used."
)

print(
    "The row-level final mapping object is:"
)

print(
    "mcmed_s2_lab_rows_final"
)


In [ ]:



required_objects = [
    "mcmed_s2_lab_rows_final",
    "frozen_lab_feature_order",
    "abnormal_row_mask",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the final conservative alias cell first. "
        f"Missing objects: {missing_objects}"
    )



preferred_cohort_objects = [
    "mcmed_s0_validated",
    "mcmed_primary_binary",
    "mcmed_primary_history_source",
]

mcmed_cohort_base_name = None
mcmed_cohort_base = None


for object_name in preferred_cohort_objects:

    if object_name not in globals():
        continue

    object_value = globals()[object_name]

    if not isinstance(
        object_value,
        pd.DataFrame,
    ):
        continue

    if "CSN" not in object_value.columns:
        continue

    if len(object_value) != 6707:
        continue

    mcmed_cohort_base_name = object_name
    mcmed_cohort_base = object_value.copy()
    break


if mcmed_cohort_base is None:

    fallback_candidates = []

    for object_name, object_value in list(
        globals().items()
    ):

        if not isinstance(
            object_value,
            pd.DataFrame,
        ):
            continue

        if "CSN" not in object_value.columns:
            continue

        if len(object_value) != 6707:
            continue

        fallback_candidates.append(
            (
                object_name,
                object_value,
            )
        )

    if not fallback_candidates:
        raise ValueError(
            "No 6,707-row frozen MC-MED cohort dataframe "
            "with a CSN column was found."
        )

    mcmed_cohort_base_name = (
        fallback_candidates[0][0]
    )

    mcmed_cohort_base = (
        fallback_candidates[0][1].copy()
    )


print(
    "Selected frozen MC-MED cohort object:",
    mcmed_cohort_base_name,
)



mcmed_cohort_base["CSN"] = _normalise_csn(
    mcmed_cohort_base["CSN"]
)

if mcmed_cohort_base["CSN"].isna().any():
    raise AssertionError(
        "The frozen MC-MED cohort contains missing CSNs."
    )


if mcmed_cohort_base["CSN"].duplicated().any():
    duplicate_examples = (
        mcmed_cohort_base.loc[
            mcmed_cohort_base["CSN"].duplicated(
                keep=False
            ),
            "CSN",
        ]
        .head(10)
        .tolist()
    )

    raise AssertionError(
        "The frozen MC-MED cohort contains duplicate "
        f"CSNs. Examples: {duplicate_examples}"
    )



valid_feature_mask = (
    mcmed_s2_lab_rows_final[
        "_final_lab_feature_name"
    ]
    .isin(
        frozen_lab_feature_order
    )
)

valid_csn_mask = (
    mcmed_s2_lab_rows_final["CSN"]
    .notna()
)

final_pair_mask = (
    abnormal_row_mask
    &
    valid_feature_mask
    &
    valid_csn_mask
)


mcmed_s2_lab_positive_pairs = (
    mcmed_s2_lab_rows_final.loc[
        final_pair_mask,
        [
            "CSN",
            "_final_lab_feature_name",
        ],
    ]
    .drop_duplicates()
    .rename(
        columns={
            "_final_lab_feature_name":
            "feature_name",
        }
    )
    .reset_index(drop=True)
)


frozen_csn_set = set(
    mcmed_cohort_base["CSN"]
)

mcmed_s2_lab_positive_pairs = (
    mcmed_s2_lab_positive_pairs.loc[
        mcmed_s2_lab_positive_pairs[
            "CSN"
        ].isin(
            frozen_csn_set
        )
    ]
    .copy()
)



if mcmed_s2_lab_positive_pairs.empty:

    mcmed_s2_lab_binary_wide = pd.DataFrame(
        0,
        index=mcmed_cohort_base["CSN"],
        columns=frozen_lab_feature_order,
        dtype="int8",
    )

else:

    mcmed_s2_lab_binary_wide = pd.crosstab(
        mcmed_s2_lab_positive_pairs["CSN"],
        mcmed_s2_lab_positive_pairs[
            "feature_name"
        ],
    )

    mcmed_s2_lab_binary_wide = (
        mcmed_s2_lab_binary_wide
        .gt(0)
        .astype("int8")
        .reindex(
            index=mcmed_cohort_base["CSN"],
            columns=frozen_lab_feature_order,
            fill_value=0,
        )
    )


mcmed_s2_lab_binary_wide.index.name = "CSN"



mcmed_s2_lab_features_primary = (
    mcmed_cohort_base
    .set_index("CSN", drop=False)
    .join(
        mcmed_s2_lab_binary_wide,
        how="left",
    )
    .reset_index(drop=True)
)


mcmed_s2_lab_features_primary[
    frozen_lab_feature_order
] = (
    mcmed_s2_lab_features_primary[
        frozen_lab_feature_order
    ]
    .fillna(0)
    .astype("int8")
)



mcmed_s2_lab_feature_order = list(
    frozen_lab_feature_order
)

mcmed_s2_lab_feature_order_sha256 = (
    hashlib.sha256(
        "\n".join(
            mcmed_s2_lab_feature_order
        ).encode("utf-8")
    )
    .hexdigest()
)


mcmed_s2_lab_matrix_sha256 = (
    hashlib.sha256(
        pd.util.hash_pandas_object(
            mcmed_s2_lab_features_primary[
                ["CSN"]
                +
                mcmed_s2_lab_feature_order
            ],
            index=False,
        )
        .values
        .tobytes()
    )
    .hexdigest()
)



assert len(
    mcmed_s2_lab_features_primary
) == 6707, (
    "The final MC-MED laboratory matrix must contain "
    "exactly 6,707 encounters."
)

assert len(
    mcmed_s2_lab_feature_order
) == 351, (
    "The final matrix must contain exactly 351 frozen "
    "laboratory features."
)

assert (
    mcmed_s2_lab_features_primary[
        "CSN"
    ]
    .nunique()
    == 6707
), (
    "Each frozen MC-MED encounter must appear exactly once."
)

assert not (
    mcmed_s2_lab_features_primary[
        "CSN"
    ]
    .duplicated()
    .any()
), (
    "Duplicate CSNs were found in the final matrix."
)

actual_lab_column_order = [
    column
    for column in (
        mcmed_s2_lab_features_primary.columns
    )
    if column.startswith(
        "poe_result__lab__"
    )
]

assert actual_lab_column_order == (
    mcmed_s2_lab_feature_order
), (
    "The laboratory columns do not match the frozen "
    "development feature order."
)

matrix_unique_values = set(
    np.unique(
        mcmed_s2_lab_features_primary[
            mcmed_s2_lab_feature_order
        ].to_numpy()
    )
    .tolist()
)

assert matrix_unique_values.issubset(
    {
        0,
        1,
    }
), (
    "The final laboratory matrix contains values other "
    "than binary 0 and 1."
)



feature_positive_counts = (
    mcmed_s2_lab_features_primary[
        mcmed_s2_lab_feature_order
    ]
    .sum(axis=0)
    .astype(int)
)

active_feature_count = int(
    feature_positive_counts.gt(0).sum()
)

structural_zero_count = int(
    feature_positive_counts.eq(0).sum()
)

encounter_has_any_positive = (
    mcmed_s2_lab_features_primary[
        mcmed_s2_lab_feature_order
    ]
    .sum(axis=1)
    .gt(0)
)


mcmed_s2_lab_freeze_summary = pd.DataFrame(
    {
        "measure": [
            "Frozen MC-MED cohort encounters",
            "Unique frozen CSNs",
            "Frozen laboratory columns",
            "Laboratory columns with >=1 positive",
            "Structural-zero laboratory columns retained",
            "Encounters with >=1 mapped abnormal lab",
            "Encounters without a mapped abnormal lab",
            "Positive encounter-feature pairs",
            "Duplicate CSNs",
            "Non-binary values detected",
            "Feature-order SHA256",
            "Matrix SHA256",
        ],
        "value": [
            len(
                mcmed_s2_lab_features_primary
            ),
            mcmed_s2_lab_features_primary[
                "CSN"
            ].nunique(),
            len(
                mcmed_s2_lab_feature_order
            ),
            active_feature_count,
            structural_zero_count,
            int(
                encounter_has_any_positive.sum()
            ),
            int(
                (
                    ~encounter_has_any_positive
                ).sum()
            ),
            len(
                mcmed_s2_lab_positive_pairs
            ),
            int(
                mcmed_s2_lab_features_primary[
                    "CSN"
                ]
                .duplicated()
                .sum()
            ),
            int(
                not matrix_unique_values.issubset(
                    {
                        0,
                        1,
                    }
                )
            ),
            mcmed_s2_lab_feature_order_sha256,
            mcmed_s2_lab_matrix_sha256,
        ],
    }
)


mcmed_s2_lab_feature_support = (
    feature_positive_counts
    .rename(
        "positive_encounters"
    )
    .reset_index()
    .rename(
        columns={
            "index": "feature_name",
        }
    )
)

mcmed_s2_lab_feature_support[
    "percentage_of_6707_encounters"
] = (
    mcmed_s2_lab_feature_support[
        "positive_encounters"
    ]
    / len(
        mcmed_s2_lab_features_primary
    )
    * 100
).round(2)

mcmed_s2_lab_feature_support = (
    mcmed_s2_lab_feature_support
    .sort_values(
        [
            "positive_encounters",
            "feature_name",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)



outcome_column = next(
    (
        column
        for column in [
            "outcome",
            "outcome_primary_strict",
        ]
        if column
        in mcmed_s2_lab_features_primary.columns
    ),
    None,
)


if outcome_column is not None:

    mcmed_s2_lab_outcome_coverage = (
        mcmed_s2_lab_features_primary[
            [
                outcome_column,
            ]
        ]
        .assign(
            has_any_mapped_abnormal_lab=(
                encounter_has_any_positive
                .to_numpy()
            )
        )
        .groupby(
            outcome_column,
            dropna=False,
        )
        .agg(
            n_encounters=(
                "has_any_mapped_abnormal_lab",
                "size",
            ),
            n_with_any_mapped_abnormal_lab=(
                "has_any_mapped_abnormal_lab",
                "sum",
            ),
        )
        .reset_index()
        .rename(
            columns={
                outcome_column:
                "outcome",
            }
        )
    )

    mcmed_s2_lab_outcome_coverage[
        "percentage_with_any_mapped_abnormal_lab"
    ] = (
        mcmed_s2_lab_outcome_coverage[
            "n_with_any_mapped_abnormal_lab"
        ]
        /
        mcmed_s2_lab_outcome_coverage[
            "n_encounters"
        ]
        * 100
    ).round(2)

else:

    mcmed_s2_lab_outcome_coverage = pd.DataFrame(
        {
            "message": [
                "No outcome column was found in the "
                "selected cohort base."
            ]
        }
    )



print(
    "\nFinal MC-MED S2 laboratory freeze summary:"
)

display(
    mcmed_s2_lab_freeze_summary
)


print(
    "\nLaboratory coverage by primary outcome:"
)

display(
    mcmed_s2_lab_outcome_coverage
)


print(
    "\nTop 40 frozen laboratory features in MC-MED:"
)

display(
    mcmed_s2_lab_feature_support.head(40)
)


print(
    "\nMC-MED S2 LABORATORY BLOCK FROZEN IN MEMORY"
)

print(
    "Primary object: "
    "mcmed_s2_lab_features_primary"
)

print(
    "Feature-order object: "
    "mcmed_s2_lab_feature_order"
)

print(
    "Positive-pair object: "
    "mcmed_s2_lab_positive_pairs"
)

print(
    "Final row-level mapping object: "
    "mcmed_s2_lab_rows_final"
)

print(
    "\nNo dataframe has been written to disk."
)


#### S2 abnormal imaging-result reconstruction

This section reconstructs the 80 frozen abnormal imaging-result features for the current MIMIC-IV development cohort and the MC-MED external cohort.

The reconstruction is performed at report level rather than assigning one stay-level abnormality flag to every imaging feature. MIMIC-IV radiology reports are linked to the frozen encounter through the patient identifier and restricted to examinations whose chart time falls inside the current ED encounter. The corresponding examination name is recovered from `radiology_detail.csv.gz` using `field_name == "exam_name"`.

MC-MED radiology rows are linked directly through the encounter identifier `CSN`. The `Study` field provides the examination name and `Impression` provides the report text.

This first step audits source availability, report multiplicity, examination names and temporal completeness. It does not yet map examination names to the frozen vocabulary, apply abnormality rules or construct the final binary feature matrices. Those steps will be performed only after reviewing the audit results.


In [ ]:
PATHS = {
    "mimic_edstays": SOURCE_PATHS["mimic_edstays"],
    "mimic_radiology": SOURCE_PATHS["mimic_radiology_notes"],
    "mimic_radiology_detail": SOURCE_PATHS["mimic_radiology_detail"],
    "mcmed_visits": SOURCE_PATHS["mcmed_visits"],
    "mcmed_rads": SOURCE_PATHS["mcmed_rads"],
    "imaging_vocab": (
        PROJECT_ROOT
        / "config"
        / "feature_vocabularies"
        / "s2_imaging_result_vocabulary.csv"
    ),
}


for path in PATHS.values():

    if not path.exists():

        raise FileNotFoundError(
            f"Required source file was not found:\n{path}"
        )


for object_name in [
    "mimic_s0_validated",
    "mcmed_s0_validated",
]:

    if object_name not in globals():

        raise ValueError(
            f"{object_name} is not available in memory. "
            "Run the frozen S0 cohort cells first."
        )


def to_datetime(series):
    """
    Parse mixed-format datetime fields without changing rows.
    """

    try:

        return pd.to_datetime(
            series,
            errors="coerce",
            format="mixed",
        )

    except (TypeError, ValueError):

        return pd.to_datetime(
            series,
            errors="coerce",
        )


def join_unique(series):
    """
    Join unique non-empty text values in their source order.
    """

    values = (
        series
        .dropna()
        .astype(str)
        .str.replace(
            r"\s+",
            " ",
            regex=True,
        )
        .str.strip()
    )

    values = values[
        ~values
        .str.lower()
        .isin(
            [
                "",
                "nan",
                "none",
                "null",
            ]
        )
    ]

    return " | ".join(
        values.drop_duplicates()
    )



s2_imaging_result_vocabulary = pd.read_csv(
    PATHS["imaging_vocab"],
    low_memory=False,
)


if "feature_name" not in (
    s2_imaging_result_vocabulary.columns
):

    raise ValueError(
        "The frozen imaging vocabulary does not contain "
        "the feature_name column."
    )


s2_imaging_feature_order = (
    s2_imaging_result_vocabulary[
        "feature_name"
    ]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)


assert len(s2_imaging_feature_order) == 80, (
    "Expected 80 frozen imaging features, "
    f"but recovered {len(s2_imaging_feature_order)}."
)



mimic_windows = pd.read_csv(
    PATHS["mimic_edstays"],
    usecols=[
        "subject_id",
        "stay_id",
        "hadm_id",
        "intime",
        "outtime",
    ],
    low_memory=False,
)


mimic_windows["intime"] = to_datetime(
    mimic_windows["intime"]
)

mimic_windows["outtime"] = to_datetime(
    mimic_windows["outtime"]
)


mimic_imaging_cohort = (
    mimic_s0_validated[
        [
            "subject_id",
            "stay_id",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .drop_duplicates(
        "stay_id"
    )
    .merge(
        mimic_windows,
        on=[
            "subject_id",
            "stay_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    mimic_imaging_cohort[
        [
            "intime",
            "outtime",
        ]
    ]
    .notna()
    .all()
    .all()
), (
    "At least one frozen MIMIC-IV encounter "
    "is missing its ED time window."
)


assert (
    mimic_imaging_cohort[
        "stay_id"
    ].nunique()
    ==
    len(mimic_imaging_cohort)
), (
    "Duplicate MIMIC-IV stay_id values were detected."
)



mimic_subject_ids = set(
    mimic_imaging_cohort[
        "subject_id"
    ]
    .astype(int)
)


mimic_chunks = []

mimic_rows_processed = 0

mimic_rows_after_subject_filter = 0


for chunk in pd.read_csv(
    PATHS["mimic_radiology"],
    usecols=[
        "note_id",
        "subject_id",
        "hadm_id",
        "charttime",
        "storetime",
        "text",
    ],
    chunksize=200_000,
    low_memory=False,
):

    mimic_rows_processed += len(chunk)

    chunk = chunk[
        chunk[
            "subject_id"
        ].isin(
            mimic_subject_ids
        )
    ].copy()

    mimic_rows_after_subject_filter += len(
        chunk
    )

    if chunk.empty:

        continue

    chunk["charttime"] = to_datetime(
        chunk["charttime"]
    )

    chunk["storetime"] = to_datetime(
        chunk["storetime"]
    )

    chunk = chunk.merge(
        mimic_imaging_cohort,
        on="subject_id",
        how="inner",
        suffixes=(
            "_report",
            "_cohort",
        ),
        validate="many_to_one",
    )

    chunk = chunk[
        chunk[
            "charttime"
        ].between(
            chunk["intime"],
            chunk["outtime"],
        )
    ].copy()

    if not chunk.empty:

        mimic_chunks.append(
            chunk
        )


if not mimic_chunks:

    raise AssertionError(
        "No MIMIC-IV radiology reports were found "
        "inside the frozen ED encounter windows."
    )


mimic_s2_imaging_rows_source = pd.concat(
    mimic_chunks,
    ignore_index=True,
)



mimic_s2_imaging_rows_source[
    "note_id"
] = (
    mimic_s2_imaging_rows_source[
        "note_id"
    ]
    .astype(str)
)


mimic_note_ids = set(
    mimic_s2_imaging_rows_source[
        "note_id"
    ]
)


mimic_detail = pd.read_csv(
    PATHS["mimic_radiology_detail"],
    usecols=[
        "note_id",
        "field_name",
        "field_value",
    ],
    low_memory=False,
)


mimic_detail["note_id"] = (
    mimic_detail[
        "note_id"
    ]
    .astype(str)
)


mimic_detail = mimic_detail[
    mimic_detail[
        "note_id"
    ].isin(
        mimic_note_ids
    )
].copy()


mimic_detail = mimic_detail[
    mimic_detail[
        "field_name"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .eq(
        "exam_name"
    )
].copy()


mimic_exam_names = (
    mimic_detail
    .groupby(
        "note_id",
        as_index=False,
    )
    .agg(
        exam_name=(
            "field_value",
            join_unique,
        )
    )
)


mimic_s2_imaging_rows_source = (
    mimic_s2_imaging_rows_source
    .merge(
        mimic_exam_names,
        on="note_id",
        how="left",
        validate="one_to_one",
    )
)


mimic_s2_imaging_rows_source[
    "exam_name"
] = (
    mimic_s2_imaging_rows_source[
        "exam_name"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)


mimic_s2_imaging_rows_source[
    "has_exam_name"
] = (
    mimic_s2_imaging_rows_source[
        "exam_name"
    ]
    .ne("")
)


mimic_s2_imaging_rows_source[
    "has_report_text"
] = (
    mimic_s2_imaging_rows_source[
        "text"
    ]
    .notna()
    &
    mimic_s2_imaging_rows_source[
        "text"
    ]
    .astype(str)
    .str.strip()
    .ne("")
)


mimic_s2_imaging_rows_source[
    "report_available_by_ed_end"
] = (
    mimic_s2_imaging_rows_source[
        "storetime"
    ]
    .notna()
    &
    mimic_s2_imaging_rows_source[
        "storetime"
    ]
    .le(
        mimic_s2_imaging_rows_source[
            "outtime"
        ]
    )
)


print(
    "MIMIC-IV imaging source preparation passed."
)

print(
    "Frozen MIMIC-IV cohort encounters:",
    len(mimic_imaging_cohort),
)

print(
    "Reports with charttime inside the ED window:",
    len(mimic_s2_imaging_rows_source),
)

print(
    "Encounters with at least one ED-window report:",
    mimic_s2_imaging_rows_source[
        "stay_id"
    ].nunique(),
)


#### MC-MED imaging source audit


In [ ]:

mcmed_windows = pd.read_csv(
    PATHS["mcmed_visits"],
    usecols=[
        "MRN",
        "CSN",
        "Arrival_time",
        "Dispo_time",
        "Departure_time",
    ],
    low_memory=False,
)



for column in [
    "MRN",
    "CSN",
]:

    mcmed_windows[column] = pd.to_numeric(
        mcmed_windows[column],
        errors="coerce",
    ).astype("Int64")


mcmed_s0_validated["MRN"] = pd.to_numeric(
    mcmed_s0_validated["MRN"],
    errors="coerce",
).astype("Int64")


mcmed_s0_validated["CSN"] = pd.to_numeric(
    mcmed_s0_validated["CSN"],
    errors="coerce",
).astype("Int64")



for column in [
    "Arrival_time",
    "Dispo_time",
    "Departure_time",
]:

    mcmed_windows[column] = to_datetime(
        mcmed_windows[column]
    )


assert (
    mcmed_windows["CSN"].notna().all()
), (
    "At least one visits.csv row has a missing or "
    "non-numeric CSN."
)


assert (
    mcmed_windows["CSN"].is_unique
), (
    "Duplicate CSN values were detected in visits.csv. "
    "Inspect before imaging reconstruction."
)



mcmed_window_lookup = (
    mcmed_windows[
        [
            "CSN",
            "MRN",
            "Arrival_time",
            "Dispo_time",
            "Departure_time",
        ]
    ]
    .rename(
        columns={
            "MRN": "MRN_visits",
        }
    )
    .copy()
)


mcmed_window_lookup[
    "current_arrival_time"
] = (
    mcmed_window_lookup[
        "Arrival_time"
    ]
)


mcmed_window_lookup[
    "ed_end_time"
] = (
    mcmed_window_lookup[
        "Dispo_time"
    ]
    .combine_first(
        mcmed_window_lookup[
            "Departure_time"
        ]
    )
)


mcmed_imaging_cohort = (
    mcmed_s0_validated[
        [
            "MRN",
            "CSN",
            "split",
            "outcome",
            "outcome_binary",
        ]
    ]
    .drop_duplicates(
        "CSN"
    )
    .merge(
        mcmed_window_lookup,
        on="CSN",
        how="left",
        validate="one_to_one",
    )
)



if (
    "mcmed_primary_history_source" in globals()
    and
    {
        "CSN",
        "current_arrival_time",
    }.issubset(
        mcmed_primary_history_source.columns
    )
):

    validated_arrival_lookup = (
        mcmed_primary_history_source[
            [
                "CSN",
                "current_arrival_time",
            ]
        ]
        .copy()
    )

    validated_arrival_lookup["CSN"] = pd.to_numeric(
        validated_arrival_lookup["CSN"],
        errors="coerce",
    ).astype("Int64")

    validated_arrival_lookup[
        "validated_current_arrival_time"
    ] = to_datetime(
        validated_arrival_lookup[
            "current_arrival_time"
        ]
    )

    validated_arrival_lookup = (
        validated_arrival_lookup[
            [
                "CSN",
                "validated_current_arrival_time",
            ]
        ]
        .drop_duplicates(
            "CSN"
        )
    )

    mcmed_imaging_cohort = (
        mcmed_imaging_cohort
        .merge(
            validated_arrival_lookup,
            on="CSN",
            how="left",
            validate="one_to_one",
        )
    )

    mcmed_imaging_cohort[
        "current_arrival_time"
    ] = (
        mcmed_imaging_cohort[
            "current_arrival_time"
        ]
        .combine_first(
            mcmed_imaging_cohort[
                "validated_current_arrival_time"
            ]
        )
    )

    mcmed_imaging_cohort = (
        mcmed_imaging_cohort
        .drop(
            columns=[
                "validated_current_arrival_time",
            ]
        )
    )



mrn_comparable = (
    mcmed_imaging_cohort["MRN"].notna()
    &
    mcmed_imaging_cohort["MRN_visits"].notna()
)


mrn_mismatch_n = int(
    (
        mrn_comparable
        &
        (
            mcmed_imaging_cohort["MRN"]
            !=
            mcmed_imaging_cohort["MRN_visits"]
        )
    ).sum()
)


mcmed_imaging_window_audit = pd.DataFrame(
    [
        [
            "Frozen MC-MED cohort encounters",
            len(mcmed_imaging_cohort),
        ],
        [
            "CSNs matched to visits.csv",
            int(
                mcmed_imaging_cohort[
                    "Arrival_time"
                ]
                .notna()
                .sum()
            ),
        ],
        [
            "Encounters with recovered arrival time",
            int(
                mcmed_imaging_cohort[
                    "current_arrival_time"
                ]
                .notna()
                .sum()
            ),
        ],
        [
            "Encounters with disposition-time endpoint",
            int(
                mcmed_imaging_cohort[
                    "Dispo_time"
                ]
                .notna()
                .sum()
            ),
        ],
        [
            "Encounters using departure-time fallback",
            int(
                (
                    mcmed_imaging_cohort[
                        "Dispo_time"
                    ]
                    .isna()
                    &
                    mcmed_imaging_cohort[
                        "Departure_time"
                    ]
                    .notna()
                ).sum()
            ),
        ],
        [
            "Encounters without a known ED endpoint",
            int(
                mcmed_imaging_cohort[
                    "ed_end_time"
                ]
                .isna()
                .sum()
            ),
        ],
        [
            "MRN disagreements after successful CSN linkage",
            mrn_mismatch_n,
        ],
    ],
    columns=[
        "measure",
        "value",
    ],
)


print(
    "MC-MED imaging encounter-window recovery:"
)

display(
    mcmed_imaging_window_audit
)



assert (
    mcmed_imaging_cohort["CSN"].nunique()
    ==
    len(mcmed_imaging_cohort)
), (
    "Duplicate MC-MED CSN values were detected "
    "after encounter-window recovery."
)


assert (
    mcmed_imaging_cohort[
        "current_arrival_time"
    ]
    .notna()
    .all()
), (
    "At least one frozen MC-MED encounter still lacks "
    "an arrival time after CSN-based recovery."
)


assert len(mcmed_imaging_cohort) == 6707, (
    "The recovered MC-MED imaging cohort does not contain "
    "the expected 6,707 frozen encounters."
)


mcmed_radiology = pd.read_csv(
    PATHS["mcmed_rads"],
    usecols=[
        "CSN",
        "Order_time",
        "Result_time",
        "Study",
        "Impression",
    ],
    low_memory=False,
)


mcmed_rows_processed = len(
    mcmed_radiology
)


mcmed_radiology = mcmed_radiology[
    mcmed_radiology[
        "CSN"
    ].isin(
        set(
            mcmed_imaging_cohort[
                "CSN"
            ]
        )
    )
].copy()


for column in [
    "Order_time",
    "Result_time",
]:

    mcmed_radiology[column] = to_datetime(
        mcmed_radiology[column]
    )


mcmed_s2_imaging_rows_source = (
    mcmed_radiology
    .merge(
        mcmed_imaging_cohort,
        on="CSN",
        how="inner",
        validate="many_to_one",
    )
)


mcmed_s2_imaging_rows_source[
    "exam_name"
] = (
    mcmed_s2_imaging_rows_source[
        "Study"
    ]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)


mcmed_s2_imaging_rows_source[
    "has_exam_name"
] = (
    mcmed_s2_imaging_rows_source[
        "exam_name"
    ]
    .ne("")
)


mcmed_s2_imaging_rows_source[
    "has_report_text"
] = (
    mcmed_s2_imaging_rows_source[
        "Impression"
    ]
    .notna()
    &
    mcmed_s2_imaging_rows_source[
        "Impression"
    ]
    .astype(str)
    .str.strip()
    .ne("")
)


mcmed_s2_imaging_rows_source[
    "order_after_arrival"
] = (
    mcmed_s2_imaging_rows_source[
        "Order_time"
    ]
    .isna()
    |
    mcmed_s2_imaging_rows_source[
        "Order_time"
    ]
    .ge(
        mcmed_s2_imaging_rows_source[
            "current_arrival_time"
        ]
    )
)


mcmed_s2_imaging_rows_source[
    "result_after_arrival"
] = (
    mcmed_s2_imaging_rows_source[
        "Result_time"
    ]
    .isna()
    |
    mcmed_s2_imaging_rows_source[
        "Result_time"
    ]
    .ge(
        mcmed_s2_imaging_rows_source[
            "current_arrival_time"
        ]
    )
)


mcmed_s2_imaging_rows_source[
    "result_before_known_ed_end"
] = (
    mcmed_s2_imaging_rows_source[
        "ed_end_time"
    ]
    .notna()
    &
    mcmed_s2_imaging_rows_source[
        "Result_time"
    ]
    .notna()
    &
    mcmed_s2_imaging_rows_source[
        "Result_time"
    ]
    .le(
        mcmed_s2_imaging_rows_source[
            "ed_end_time"
        ]
    )
)



def multiplicity_table(
    dataframe,
    identifier_column,
    count_column_name,
):

    return (
        dataframe
        .groupby(
            identifier_column
        )
        .size()
        .value_counts()
        .sort_index()
        .rename_axis(
            count_column_name
        )
        .reset_index(
            name="n_encounters"
        )
    )


def top_exam_names(
    dataframe,
    count_column_name,
):

    return (
        dataframe
        .loc[
            dataframe[
                "has_exam_name"
            ],
            "exam_name",
        ]
        .value_counts()
        .head(40)
        .rename_axis(
            "exam_name"
        )
        .reset_index(
            name=count_column_name
        )
    )


mimic_imaging_report_multiplicity = (
    multiplicity_table(
        mimic_s2_imaging_rows_source,
        "stay_id",
        "n_reports",
    )
)


mcmed_imaging_report_multiplicity = (
    multiplicity_table(
        mcmed_s2_imaging_rows_source,
        "CSN",
        "n_rows",
    )
)


mimic_top_imaging_exam_names = (
    top_exam_names(
        mimic_s2_imaging_rows_source,
        "n_reports",
    )
)


mcmed_top_imaging_exam_names = (
    top_exam_names(
        mcmed_s2_imaging_rows_source,
        "n_rows",
    )
)



s2_imaging_source_timing_audit = pd.DataFrame(
    [
        [
            "MIMIC-IV",
            "Frozen cohort encounters",
            len(mimic_imaging_cohort),
        ],
        [
            "MIMIC-IV",
            "Radiology source rows processed",
            mimic_rows_processed,
        ],
        [
            "MIMIC-IV",
            "Rows after cohort-subject filtering",
            mimic_rows_after_subject_filter,
        ],
        [
            "MIMIC-IV",
            "Reports with charttime inside ED window",
            len(mimic_s2_imaging_rows_source),
        ],
        [
            "MIMIC-IV",
            "Encounters with >=1 ED-window report",
            mimic_s2_imaging_rows_source[
                "stay_id"
            ].nunique(),
        ],
        [
            "MIMIC-IV",
            "Reports with exam_name",
            int(
                mimic_s2_imaging_rows_source[
                    "has_exam_name"
                ].sum()
            ),
        ],
        [
            "MIMIC-IV",
            "Reports with non-empty text",
            int(
                mimic_s2_imaging_rows_source[
                    "has_report_text"
                ].sum()
            ),
        ],
        [
            "MIMIC-IV",
            "Reports stored by ED end",
            int(
                mimic_s2_imaging_rows_source[
                    "report_available_by_ed_end"
                ].sum()
            ),
        ],
        [
            "MC-MED",
            "Frozen cohort encounters",
            len(mcmed_imaging_cohort),
        ],
        [
            "MC-MED",
            "Radiology source rows processed",
            mcmed_rows_processed,
        ],
        [
            "MC-MED",
            "Rows linked to frozen cohort CSNs",
            len(mcmed_s2_imaging_rows_source),
        ],
        [
            "MC-MED",
            "Encounters with >=1 radiology row",
            mcmed_s2_imaging_rows_source[
                "CSN"
            ].nunique(),
        ],
        [
            "MC-MED",
            "Rows with Study",
            int(
                mcmed_s2_imaging_rows_source[
                    "has_exam_name"
                ].sum()
            ),
        ],
        [
            "MC-MED",
            "Rows with non-empty Impression",
            int(
                mcmed_s2_imaging_rows_source[
                    "has_report_text"
                ].sum()
            ),
        ],
        [
            "MC-MED",
            "Rows ordered on or after ED arrival",
            int(
                mcmed_s2_imaging_rows_source[
                    "order_after_arrival"
                ].sum()
            ),
        ],
        [
            "MC-MED",
            "Rows resulted on or after ED arrival",
            int(
                mcmed_s2_imaging_rows_source[
                    "result_after_arrival"
                ].sum()
            ),
        ],
        [
            "MC-MED",
            "Rows with a known ED endpoint",
            int(
                mcmed_s2_imaging_rows_source[
                    "ed_end_time"
                ]
                .notna()
                .sum()
            ),
        ],
        [
            "MC-MED",
            "Results available by known ED endpoint",
            int(
                mcmed_s2_imaging_rows_source[
                    "result_before_known_ed_end"
                ].sum()
            ),
        ],
    ],
    columns=[
        "dataset",
        "measure",
        "value",
    ],
)


print(
    "Frozen imaging features:",
    len(s2_imaging_feature_order),
)

print(
    "No imaging feature matrix has yet been "
    "created or frozen."
)


print(
    "\nS2 imaging source and timing audit:"
)

display(
    s2_imaging_source_timing_audit
)


print(
    "\nMIMIC-IV report multiplicity:"
)

display(
    mimic_imaging_report_multiplicity.head(20)
)


print(
    "\nMC-MED radiology-row multiplicity:"
)

display(
    mcmed_imaging_report_multiplicity.head(20)
)


print(
    "\nTop MIMIC-IV examination names:"
)

display(
    mimic_top_imaging_exam_names
)


print(
    "\nTop MC-MED Study values:"
)

display(
    mcmed_top_imaging_exam_names
)


print(
    "\nFirst 15 frozen imaging features:"
)

display(
    s2_imaging_result_vocabulary[
        [
            column
            for column in [
                "family_position",
                "feature_name",
            ]
            if column in (
                s2_imaging_result_vocabulary.columns
            )
        ]
    ]
    .head(15)
)


print(
    "\nCreated source objects:"
)

print(
    "mimic_imaging_cohort"
)

print(
    "mcmed_imaging_cohort"
)

print(
    "mimic_s2_imaging_rows_source"
)

print(
    "mcmed_s2_imaging_rows_source"
)

print(
    "s2_imaging_source_timing_audit"
)

print(
    "mimic_top_imaging_exam_names"
)

print(
    "mcmed_top_imaging_exam_names"
)

print(
    "s2_imaging_result_vocabulary"
)

print(
    "s2_imaging_feature_order"
)


#### Deterministic radiology-exam mapping

This section maps each eligible MIMIC-IV radiology report and MC-MED radiology study to the previously frozen set of 80 imaging-result features.

The mapping is performed at the individual report or study level before analysing the report impression. It uses only exact normalised names and explicitly curated cross-dataset aliases. Unrestricted fuzzy matching is not used because a superficially similar examination may represent a clinically different imaging procedure.

For MIMIC-IV, pipe-delimited combined examination names are retained as complete examination blocks and are also separated into their explicitly named components. This permits a combined pelvic examination, for example, to support both the historical combined feature and its clearly stated component examinations.

For MC-MED, local examination names are mapped only when they are semantically equivalent to a frozen MIMIC-IV examination concept. Composite MC-MED studies can support more than one frozen feature only when those components are explicitly present in the study name.

Generic modality features such as CT, ultrasound, MRI and general X-ray are not mapped directly in this cell. They will be derived later from the corresponding mapped modality-specific abnormal findings.

Legacy non-examination columns are retained in the frozen 80-feature inventory for feature-order compatibility, but they are not treated as radiology examinations.

The outputs audit mapping coverage, mapping multiplicity, unmatched examination names and the number of frozen features with direct source support. No abnormality labels or patient-level imaging matrices are created at this stage.


In [ ]:

PREFIX = "poe_result__imaging_radiology__"

vocab = globals().get(
    "s2_imaging_result_vocabulary"
)

if not isinstance(vocab, pd.DataFrame):

    vocab_path = (
        PROJECT_ROOT
        / "config"
        / "feature_vocabularies"
        / "s2_imaging_result_vocabulary.csv"
    )

    if not vocab_path.exists():

        vocab_path = (
            Path.cwd()
            / "config"
            / "feature_vocabularies"
            / "s2_imaging_result_vocabulary.csv"
        )

    if not vocab_path.exists():
        raise FileNotFoundError(
            "s2_imaging_result_vocabulary.csv "
            "was not found."
        )

    vocab = pd.read_csv(
        vocab_path
    )


feature_col = next(
    (
        column
        for column in [
            "feature_name",
            "final_feature_name",
        ]
        if column in vocab.columns
    ),
    None,
)

if feature_col is None:
    raise ValueError(
        "No feature-name column was found in "
        "the imaging vocabulary.\n"
        f"Available columns: {vocab.columns.tolist()}"
    )


if "family_position" in vocab.columns:

    vocab = (
        vocab
        .sort_values(
            "family_position"
        )
        .reset_index(
            drop=True
        )
    )


frozen_imaging_feature_order = (
    vocab[feature_col]
    .astype("string")
    .str.strip()
    .dropna()
    .tolist()
)


assert (
    len(frozen_imaging_feature_order) == 80
), (
    "Expected 80 frozen imaging features; "
    f"found {len(frozen_imaging_feature_order)}."
)


assert (
    len(
        set(
            frozen_imaging_feature_order
        )
    ) == 80
), (
    "Duplicate imaging features were detected."
)


assert all(
    feature.startswith(
        PREFIX
    )
    for feature in frozen_imaging_feature_order
), (
    "Unexpected imaging feature prefix detected."
)


TOKENS = {
    feature.removeprefix(
        PREFIX
    )
    for feature in frozen_imaging_feature_order
}


DERIVED_LATER = {
    "general_xray",
    "imaging_radiology_orders_ct_scan",
    "imaging_radiology_orders_general_xray",
    "imaging_radiology_orders_interventional_radiology",
    "imaging_radiology_orders_mri",
    "imaging_radiology_orders_nuclear_med",
    "imaging_radiology_orders_ultrasound",
}


NON_EXAM = {
    "frozen_plasma_product_order",
    "imaging_radiology_orders_ett_non_imaging",
    "mod_sedation_first_15_minutes",
    "mod_sedation_first_30_min",
    "unknown",
    "59_distinct_procedural_service",
    "76_by_same_physician",
    "77_by_different_physician",
    "xs_separate_structure",
    "outside_films_read_only",
}


DIRECT_EXCLUDED = (
    DERIVED_LATER
    | NON_EXAM
)



def find_prepared_dataframe(
    preferred_names,
    required_column_groups,
    expected_rows,
):
    """
    Recover the prepared report-level dataframe created
    by the immediately preceding imaging source cell.

    Full raw radiology files are deliberately excluded.
    """

    candidates = []

    for object_name, dataframe in list(
        globals().items()
    ):

        if not isinstance(
            dataframe,
            pd.DataFrame,
        ):
            continue

        if len(dataframe) > 100_000:
            continue

        has_required_columns = all(
            any(
                column in dataframe.columns
                for column in group
            )
            for group in required_column_groups
        )

        if not has_required_columns:
            continue

        preferred_priority = (
            preferred_names.index(
                object_name
            )
            if object_name in preferred_names
            else len(preferred_names)
        )

        object_name_lower = (
            object_name.lower()
        )

        name_bonus = -sum(
            token in object_name_lower
            for token in [
                "s2",
                "imaging",
                "radiology",
                "prepared",
            ]
        )

        row_distance = abs(
            len(dataframe)
            - expected_rows
        )

        candidates.append(
            (
                preferred_priority,
                name_bonus,
                row_distance,
                object_name,
                dataframe,
            )
        )

    if not candidates:
        raise ValueError(
            "A prepared imaging dataframe was not found. "
            "Run the preceding imaging source/timing "
            "cell first."
        )

    candidates = sorted(
        candidates,
        key=lambda value: value[:4],
    )

    (
        _,
        _,
        _,
        selected_name,
        selected_dataframe,
    ) = candidates[0]

    return (
        selected_name,
        selected_dataframe.copy(),
    )


mimic_object_name, mimic_source = (
    find_prepared_dataframe(
        preferred_names=[
            "mimic_s2_imaging_reports_prepared",
            "mimic_s2_imaging_reports_ed_window",
            "mimic_s2_imaging_reports",
            "mimic_s2_radiology_rows",
        ],
        required_column_groups=[
            [
                "stay_id",
            ],
            [
                "exam_name",
                "radiology_exam_name",
                "study_name",
                "field_value",
            ],
            [
                "report_text",
                "text",
                "radiology_text",
                "impression",
            ],
        ],
        expected_rows=8_252,
    )
)


mcmed_object_name, mcmed_source = (
    find_prepared_dataframe(
        preferred_names=[
            "mcmed_s2_imaging_rows_prepared",
            "mcmed_s2_imaging_rows",
            "mcmed_s2_radiology_rows",
            "mcmed_imaging_rows_prepared",
        ],
        required_column_groups=[
            [
                "CSN",
            ],
            [
                "Study",
                "study",
                "exam_name",
            ],
            [
                "Impression",
                "impression",
                "report_text",
            ],
        ],
        expected_rows=7_459,
    )
)


def first_existing_column(
    dataframe,
    candidates,
    required=True,
):
    column = next(
        (
            candidate
            for candidate in candidates
            if candidate in dataframe.columns
        ),
        None,
    )

    if required and column is None:
        raise ValueError(
            f"None of these columns were found: "
            f"{candidates}\n"
            f"Available columns: "
            f"{dataframe.columns.tolist()}"
        )

    return column


def optional_column(
    dataframe,
    column,
):
    if column is None:
        return pd.Series(
            pd.NaT,
            index=dataframe.index,
        )

    return dataframe[column]


mimic_exam_col = first_existing_column(
    mimic_source,
    [
        "exam_name",
        "radiology_exam_name",
        "study_name",
        "field_value",
    ],
)

mimic_text_col = first_existing_column(
    mimic_source,
    [
        "report_text",
        "text",
        "radiology_text",
        "impression",
    ],
)

mimic_exam_time_col = (
    first_existing_column(
        mimic_source,
        [
            "charttime",
            "exam_time",
            "order_time",
        ],
        required=False,
    )
)

mimic_result_time_col = (
    first_existing_column(
        mimic_source,
        [
            "storetime",
            "result_time",
            "report_available_time",
        ],
        required=False,
    )
)


mcmed_exam_col = first_existing_column(
    mcmed_source,
    [
        "Study",
        "study",
        "exam_name",
    ],
)

mcmed_text_col = first_existing_column(
    mcmed_source,
    [
        "Impression",
        "impression",
        "report_text",
    ],
)

mcmed_exam_time_col = (
    first_existing_column(
        mcmed_source,
        [
            "Order_time",
            "order_time",
            "exam_time",
        ],
        required=False,
    )
)

mcmed_result_time_col = (
    first_existing_column(
        mcmed_source,
        [
            "Result_time",
            "result_time",
        ],
        required=False,
    )
)


mimic_s2_imaging_source_rows = pd.DataFrame(
    {
        "dataset": "MIMIC-IV",
        "encounter_id": mimic_source[
            "stay_id"
        ],
        "source_row_number": np.arange(
            len(mimic_source)
        ),
        "source_exam_name": mimic_source[
            mimic_exam_col
        ],
        "report_text": mimic_source[
            mimic_text_col
        ],
        "exam_time": optional_column(
            mimic_source,
            mimic_exam_time_col,
        ),
        "result_available_time": optional_column(
            mimic_source,
            mimic_result_time_col,
        ),
    }
)


mcmed_s2_imaging_source_rows = pd.DataFrame(
    {
        "dataset": "MC-MED",
        "encounter_id": mcmed_source[
            "CSN"
        ],
        "source_row_number": np.arange(
            len(mcmed_source)
        ),
        "source_exam_name": mcmed_source[
            mcmed_exam_col
        ],
        "report_text": mcmed_source[
            mcmed_text_col
        ],
        "exam_time": optional_column(
            mcmed_source,
            mcmed_exam_time_col,
        ),
        "result_available_time": optional_column(
            mcmed_source,
            mcmed_result_time_col,
        ),
    }
)


s2_imaging_source_rows = pd.concat(
    [
        mimic_s2_imaging_source_rows,
        mcmed_s2_imaging_source_rows,
    ],
    ignore_index=True,
)


s2_imaging_source_rows[
    "source_row_key"
] = (
    s2_imaging_source_rows[
        "dataset"
    ].astype(str)
    + "::"
    + s2_imaging_source_rows[
        "source_row_number"
    ].astype(str)
)



def clean_exam_text(value):

    if pd.isna(value):
        return ""

    value = html.unescape(
        str(value)
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    if value.lower() in {
        "",
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return ""

    return value


def exam_key(value):

    value = (
        clean_exam_text(value)
        .lower()
        .replace("&amp;", "&")
        .replace("&", " and ")
    )

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value,
    )

    value = re.sub(
        r"\s+",
        " ",
        value,
    ).strip()

    return value


def canonical_exam_token(value):

    value = (
        clean_exam_text(value)
        .lower()
        .replace("&amp;", "&")
        .replace("u.s.", "u_s")
        .replace("u.s", "u_s")
        .replace("u/s", "u_s")
    )

    value = re.sub(
        r"\bw\s*&\s*w\s*/\s*o\s*c\b",
        " w_and_w_o_c ",
        value,
    )

    value = re.sub(
        r"\bw\s*&\s*w\s*/\s*o\s*contrast\b",
        " w_and_w_o_contrast ",
        value,
    )

    value = re.sub(
        r"\bw\s*/\s*o\b",
        " w_o ",
        value,
    )

    value = re.sub(
        r"\bw\s*/\s*c\b",
        " w_c ",
        value,
    )

    value = re.sub(
        r"\bw\s*/\s*contrast\b",
        " w_contrast ",
        value,
    )

    replacements = [
        (
            "with contrast",
            "with_contrast",
        ),
        (
            "without contrast",
            "w_o_contrast",
        ),
        (
            "non-coronary",
            "non_coronary",
        ),
        (
            "c-spine",
            "c_spine",
        ),
        (
            "<14weeks",
            "14weeks",
        ),
        (
            "< 14weeks",
            "14weeks",
        ),
        (
            "<14 weeks",
            "14weeks",
        ),
        (
            "< 14 weeks",
            "14weeks",
        ),
    ]

    for old_value, new_value in replacements:
        value = value.replace(
            old_value,
            new_value,
        )

    value = value.replace(
        "&",
        " and ",
    )

    value = re.sub(
        r"[^a-z0-9]+",
        "_",
        value,
    )

    value = re.sub(
        r"_+",
        "_",
        value,
    ).strip("_")

    return value


ALIASES_RAW = {

    (
        "PELVIS, NON-OBSTETRIC | "
        "PELVIS U.S., TRANSVAGINAL"
    ): [
        (
            "u52_pelvis_non_obstetric_"
            "pelvis_u_s_transvaginal"
        ),
        "pelvis_non_obstetric",
        "pelvis_u_s_transvaginal",
    ],

    (
        "EARLY OB US <14WEEKS | "
        "TV OB US"
    ): [
        (
            "u431_early_ob_us_14weeks_"
            "tv_ob_us"
        ),
        "early_ob_us_14weeks",
        "tv_ob_us",
    ],

    (
        "DUPLEX DOP ABD/PEL LIMITED | "
        "PELVIS, NON-OBSTETRIC | "
        "PELVIS U.S., TRANSVAGINAL | "
        "-XS SEPARATE STRUCTURE"
    ): [
        (
            "u52_u644_w882_duplex_dop_"
            "abd_pel_limited_pelvis_non_"
            "obstetric_pelvis_u_s_"
            "transvaginal_xs_separate_"
            "structure"
        ),
        "duplex_dop_abd_pel_limited",
        "pelvis_non_obstetric",
        "pelvis_u_s_transvaginal",
    ],

    (
        "-59 DISTINCT PROCEDURAL SERVICE | "
        "DUPLEX DOP ABD/PEL LIMITED | "
        "LIVER OR GALLBLADDER US "
        "(SINGLE ORGAN)"
    ): [
        (
            "u644_w82_59_distinct_"
            "procedural_service_duplex_"
            "dop_abd_pel_limited_liver_"
            "or_gallbladder_us_single_"
            "organ"
        ),
        "duplex_dop_abd_pel_limited",
        (
            "liver_or_gallbladder_"
            "us_single_organ"
        ),
    ],

    (
        "CT ABD & PELVIS WITH CONTRAST | "
        "OUTSIDE FILMS READ ONLY"
    ): [
        (
            "w52_ct_abd_and_pelvis_with_"
            "contrast_outside_films_read_"
            "only"
        ),
        "ct_abd_and_pelvis_with_contrast",
    ],

    (
        "CT ABDOMEN W/O CONTRAST | "
        "OUTSIDE FILMS READ ONLY"
    ): [
        (
            "w52_ct_abdomen_w_o_contrast_"
            "outside_films_read_only"
        ),
        "ct_abdomen_w_o_contrast",
    ],

    (
        "-76 BY SAME PHYSICIAN | "
        "CHEST (PORTABLE AP)"
    ): [
        (
            "w841_76_by_same_physician_"
            "chest_portable_ap"
        ),
        "chest_portable_ap",
    ],

    "CT ABDOMEN PELVIS W IV CONTRAST": [
        "ct_abd_and_pelvis_with_contrast",
    ],

    "CT ABDOMEN PELVIS WO IV CONTRAST": [
        "ct_abd_and_pelvis_w_o_contrast",
    ],

    (
        "CT ABDOMEN PELVIS ANGIOGRAPHY "
        "W AND WO IV CONTRAST"
    ): [
        "cta_abd_and_pelvis",
    ],

    (
        "CT ABDOMEN ANGIOGRAPHY "
        "W AND WO IV CONTRAST"
    ): [
        "cta_abd_and_pelvis",
    ],

    (
        "CT CHEST ANGIOGRAPHY "
        "W IV CONTRAST PULMONARY EMBOLISM"
    ): [
        "cta_chest",
    ],

    "CT CHEST W IV CONTRAST": [
        "ct_chest_w_contrast",
    ],

    "CT CHEST WO IV CONTRAST": [
        "ct_chest_w_o_contrast",
    ],

    "CT HEAD WO IV CONTRAST": [
        "ct_head_w_o_contrast",
    ],

    (
        "CT CHEST ABDOMEN PELVIS "
        "W IV CONTRAST"
    ): [
        "ct_abd_and_pelvis_with_contrast",
        "ct_chest_w_contrast",
    ],

    (
        "CT CHEST (GATED) ABDOMEN PELVIS "
        "ANGIOGRAPHY W AND WO IV CONTRAST"
    ): [
        "cta_abd_and_pelvis",
        "cta_chest",
    ],

    (
        "CT ABDOMEN LIVER PELVIS "
        "W IV CONTRAST TRIPHASIC"
    ): [
        "ct_abd_and_pelvis_with_contrast",
    ],

    "XR CHEST 1 VIEW": [
        "chest_single_view",
    ],

    "XR CHEST 2 VIEWS": [
        "chest_pa_and_lat",
    ],

    "XR ABD 1 VIEW": [
        "abdomen_supine_only",
    ],

    (
        "XR ABD 1 VIEW FOR "
        "FEEDING TUBE PLACEMENT"
    ): [
        "portable_abdomen",
    ],

    (
        "XR ABD SUPINE AND ERECT "
        "OR DECUB"
    ): [
        "abdomen_supine_and_erect",
    ],

    "US ABDOMEN": [
        "abdomen_us_complete_study",
    ],

    "US ABDOMEN LIMITED": [
        "us_abd_limit_single_organ",
    ],

    "US RETROPERITONEUM KIDNEY": [
        "renal_u_s",
    ],

    "US KIDNEY TRANSPLANT DOPPLER": [
        "renal_transplant_u_s",
    ],

    "US SCROTUM": [
        "scrotal_u_s",
    ],

    "US SCROTUM TORSION": [
        "scrotal_u_s",
        "duplex_dop_abd_pel_limited",
    ],

    "US PELVIS TRANSVAGINAL": [
        "pelvis_u_s_transvaginal",
    ],

    "US PELVIS TRANSABDOMINAL": [
        "pelvis_non_obstetric",
    ],

    (
        "US PELVIS TRANSABDOMINAL "
        "TRANSVAGINAL"
    ): [
        "pelvis_non_obstetric",
        "pelvis_u_s_transvaginal",
    ],

    (
        "US PELVIS OVARY TORSION "
        "TRANSVAGINAL"
    ): [
        "pelvis_u_s_transvaginal",
        "duplex_dop_abd_pel_limited",
    ],

    (
        "US PELVIS OVARY TORSION "
        "TRANSABDOMINAL"
    ): [
        "pelvis_non_obstetric",
        "duplex_dop_abd_pel_limited",
    ],

    (
        "US PELVIS OVARY TORSION "
        "TRANSABDOMINAL TRANSVAGINAL"
    ): [
        "pelvis_non_obstetric",
        "pelvis_u_s_transvaginal",
        "duplex_dop_abd_pel_limited",
    ],

    (
        "US PREGNANCY 1ST TRIMESTER "
        "TRANSVAGINAL"
    ): [
        "early_ob_us_14weeks",
        "tv_ob_us",
    ],

    (
        "US PREGNANCY 1ST TRIMESTER "
        "TRANSABDOMINAL TRANSVAGINAL"
    ): [
        "early_ob_us_14weeks",
        "tv_ob_us",
    ],

    (
        "US PREGNANCY 1ST TRIMESTER "
        "AND OVARY TORSION TRANSVAGINAL"
    ): [
        "early_ob_us_14weeks",
        "tv_ob_us",
        "duplex_dop_abd_pel_limited",
    ],

    (
        "US PREGNANCY 1ST TRIMESTER "
        "AND OVARY TORSION "
        "TRANSABDOMINAL TRANSVAGINAL"
    ): [
        "early_ob_us_14weeks",
        "tv_ob_us",
        "duplex_dop_abd_pel_limited",
    ],

    (
        "MR ABDOMEN W AND WO IV "
        "CONTRAST MRCP"
    ): [
        "mrcp_mr_abd_w_and_w_oc",
    ],

    "MR ABDOMEN CHOLANGIOGRAM": [
        "mrcp_mr_abd_w_and_w_oc",
    ],
}


ALIASES = {
    exam_key(source_name): [
        target
        for target in targets
        if target in TOKENS
    ]
    for source_name, targets
    in ALIASES_RAW.items()
}


METHOD_PRIORITY = {
    "EXACT_FULL": 1,
    "CURATED_FULL": 2,
    "EXACT_COMPONENT": 3,
    "CURATED_COMPONENT": 4,
}


def map_exam_piece(
    exam_text,
    mapping_scope,
):
    mappings = []

    canonical_token = (
        canonical_exam_token(
            exam_text
        )
    )

    if (
        canonical_token in TOKENS
        and canonical_token
        not in DIRECT_EXCLUDED
    ):
        mappings.append(
            (
                canonical_token,
                f"EXACT_{mapping_scope}",
                clean_exam_text(
                    exam_text
                ),
            )
        )

    for target_token in ALIASES.get(
        exam_key(
            exam_text
        ),
        [],
    ):

        if target_token in DERIVED_LATER:
            continue

        if target_token in NON_EXAM:
            continue

        mappings.append(
            (
                target_token,
                f"CURATED_{mapping_scope}",
                clean_exam_text(
                    exam_text
                ),
            )
        )

    return mappings


def map_exam_to_frozen_features(
    raw_exam_name,
):
    """
    Map the complete examination name and, where present,
    its explicit pipe-delimited components.

    No fuzzy matching is performed.
    """

    raw_exam_name = clean_exam_text(
        raw_exam_name
    )

    if not raw_exam_name:
        return []

    candidates = map_exam_piece(
        raw_exam_name,
        "FULL",
    )

    components = [
        component.strip()
        for component in re.split(
            r"\s*\|\s*",
            raw_exam_name,
        )
        if component.strip()
    ]

    if len(components) > 1:

        for component in components:

            candidates.extend(
                map_exam_piece(
                    component,
                    "COMPONENT",
                )
            )

    best_mapping_by_target = {}

    for (
        target_token,
        method,
        mapping_input,
    ) in candidates:

        existing = (
            best_mapping_by_target.get(
                target_token
            )
        )

        if (
            existing is None
            or METHOD_PRIORITY[method]
            < METHOD_PRIORITY[
                existing[0]
            ]
        ):
            best_mapping_by_target[
                target_token
            ] = (
                method,
                mapping_input,
            )

    return [
        (
            target_token,
            method,
            mapping_input,
        )
        for target_token, (
            method,
            mapping_input,
        )
        in best_mapping_by_target.items()
    ]



source_audit_rows = []
mapping_pair_rows = []


for row in s2_imaging_source_rows.itertuples(
    index=False
):

    mappings = (
        map_exam_to_frozen_features(
            row.source_exam_name
        )
    )

    mapped_feature_names = [
        PREFIX + target_token
        for (
            target_token,
            _,
            _,
        ) in mappings
    ]

    source_audit_rows.append(
        {
            "dataset": row.dataset,
            "encounter_id": row.encounter_id,
            "source_row_key": row.source_row_key,
            "source_exam_name": (
                row.source_exam_name
            ),
            "n_mapped_features": len(
                mapped_feature_names
            ),
            "mapped_feature_names": (
                " | ".join(
                    mapped_feature_names
                )
            ),
        }
    )

    for (
        target_token,
        mapping_method,
        mapping_input,
    ) in mappings:

        mapping_pair_rows.append(
            {
                "dataset": row.dataset,
                "encounter_id": (
                    row.encounter_id
                ),
                "source_row_key": (
                    row.source_row_key
                ),
                "source_exam_name": (
                    row.source_exam_name
                ),
                "mapping_input_text": (
                    mapping_input
                ),
                "mapped_feature_token": (
                    target_token
                ),
                "mapped_feature_name": (
                    PREFIX
                    + target_token
                ),
                "mapping_method": (
                    mapping_method
                ),
            }
        )


s2_imaging_source_mapping_audit = (
    pd.DataFrame(
        source_audit_rows
    )
)


s2_imaging_mapping_pairs = pd.DataFrame(
    mapping_pair_rows,
    columns=[
        "dataset",
        "encounter_id",
        "source_row_key",
        "source_exam_name",
        "mapping_input_text",
        "mapped_feature_token",
        "mapped_feature_name",
        "mapping_method",
    ],
)


s2_imaging_mapping_pairs = (
    s2_imaging_mapping_pairs
    .drop_duplicates(
        subset=[
            "source_row_key",
            "mapped_feature_name",
        ]
    )
    .reset_index(
        drop=True
    )
)


unknown_targets = (
    set(
        s2_imaging_mapping_pairs[
            "mapped_feature_name"
        ]
    )
    - set(
        frozen_imaging_feature_order
    )
)


assert not unknown_targets, (
    "A mapping outside the frozen vocabulary "
    f"was created: {sorted(unknown_targets)}"
)


assert not (
    s2_imaging_mapping_pairs
    .duplicated(
        [
            "source_row_key",
            "mapped_feature_name",
        ]
    )
    .any()
), (
    "Duplicate source-row/feature mappings "
    "were detected."
)


assert not (
    s2_imaging_mapping_pairs[
        "mapping_method"
    ]
    .astype("string")
    .str.contains(
        "FUZZY",
        case=False,
        na=False,
    )
    .any()
), (
    "A fuzzy mapping was detected."
)


mimic_s2_imaging_mapping_pairs = (
    s2_imaging_mapping_pairs[
        s2_imaging_mapping_pairs[
            "dataset"
        ].eq(
            "MIMIC-IV"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


mcmed_s2_imaging_mapping_pairs = (
    s2_imaging_mapping_pairs[
        s2_imaging_mapping_pairs[
            "dataset"
        ].eq(
            "MC-MED"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)



mapping_summary_rows = []


for (
    dataset_name,
    dataset_rows,
) in s2_imaging_source_mapping_audit.groupby(
    "dataset",
    sort=False,
):

    dataset_pairs = (
        s2_imaging_mapping_pairs[
            s2_imaging_mapping_pairs[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
    )

    mapped_rows = dataset_rows[
        dataset_rows[
            "n_mapped_features"
        ].gt(0)
    ]

    mapping_summary_rows.append(
        {
            "dataset": dataset_name,
            "source_rows": len(
                dataset_rows
            ),
            "unique_encounters": (
                dataset_rows[
                    "encounter_id"
                ].nunique()
            ),
            "rows_with_at_least_one_mapping": (
                len(mapped_rows)
            ),
            "row_mapping_coverage_pct": round(
                (
                    len(mapped_rows)
                    / len(dataset_rows)
                    * 100
                ),
                2,
            ),
            "unique_exam_names": (
                dataset_rows[
                    "source_exam_name"
                ].nunique(
                    dropna=True
                )
            ),
            "unique_exam_names_mapped": (
                mapped_rows[
                    "source_exam_name"
                ].nunique(
                    dropna=True
                )
            ),
            "mapping_pairs": len(
                dataset_pairs
            ),
            "frozen_features_with_direct_support": (
                dataset_pairs[
                    "mapped_feature_name"
                ].nunique()
            ),
            "unmapped_source_rows": int(
                dataset_rows[
                    "n_mapped_features"
                ].eq(0).sum()
            ),
        }
    )


s2_imaging_mapping_summary = (
    pd.DataFrame(
        mapping_summary_rows
    )
)


s2_imaging_mapping_method_summary = (
    s2_imaging_mapping_pairs
    .groupby(
        [
            "dataset",
            "mapping_method",
        ]
    )
    .agg(
        n_mapping_pairs=(
            "mapped_feature_name",
            "size",
        ),
        n_source_rows=(
            "source_row_key",
            "nunique",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_mapping_pairs",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


s2_imaging_mapping_multiplicity_audit = (
    s2_imaging_source_mapping_audit
    .groupby(
        [
            "dataset",
            "n_mapped_features",
        ]
    )
    .agg(
        n_source_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
    )
    .reset_index()
)


s2_imaging_mapping_multiplicity_audit[
    "percentage_of_source_rows"
] = (
    100
    * s2_imaging_mapping_multiplicity_audit[
        "n_source_rows"
    ]
    / s2_imaging_mapping_multiplicity_audit
    .groupby(
        "dataset"
    )[
        "n_source_rows"
    ]
    .transform(
        "sum"
    )
).round(2)



s2_imaging_unmatched_exam_audit = (
    s2_imaging_source_mapping_audit[
        s2_imaging_source_mapping_audit[
            "n_mapped_features"
        ].eq(0)
    ]
    .assign(
        source_exam_name=lambda dataframe: (
            dataframe[
                "source_exam_name"
            ]
            .astype("string")
            .fillna(
                "<MISSING_EXAM_NAME>"
            )
            .replace(
                "",
                "<MISSING_EXAM_NAME>",
            )
        )
    )
    .groupby(
        [
            "dataset",
            "source_exam_name",
        ]
    )
    .agg(
        n_source_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_source_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
)



feature_policy = pd.DataFrame(
    {
        "mapped_feature_name": (
            frozen_imaging_feature_order
        )
    }
)


feature_policy[
    "feature_token"
] = (
    feature_policy[
        "mapped_feature_name"
    ]
    .str.removeprefix(
        PREFIX
    )
)


feature_policy[
    "feature_role"
] = np.select(
    [
        feature_policy[
            "feature_token"
        ].isin(
            DERIVED_LATER
        ),

        feature_policy[
            "feature_token"
        ].isin(
            NON_EXAM
        ),
    ],
    [
        "DERIVED_GENERIC_LATER",
        "LEGACY_NON_EXAM_STRUCTURAL_ZERO",
    ],
    default=(
        "DIRECT_EXAM_OR_COMPOSITE"
    ),
)


feature_support_parts = []


for dataset_name in [
    "MIMIC-IV",
    "MC-MED",
]:

    dataset_support = (
        s2_imaging_mapping_pairs[
            s2_imaging_mapping_pairs[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
        .groupby(
            "mapped_feature_name"
        )
        .agg(
            n_source_rows=(
                "source_row_key",
                "nunique",
            ),
            n_encounters=(
                "encounter_id",
                "nunique",
            ),
        )
        .reset_index()
    )

    dataset_feature_support = (
        feature_policy
        .merge(
            dataset_support,
            on="mapped_feature_name",
            how="left",
            validate="one_to_one",
        )
    )

    dataset_feature_support[
        [
            "n_source_rows",
            "n_encounters",
        ]
    ] = (
        dataset_feature_support[
            [
                "n_source_rows",
                "n_encounters",
            ]
        ]
        .fillna(0)
        .astype(int)
    )

    dataset_feature_support.insert(
        0,
        "dataset",
        dataset_name,
    )

    dataset_feature_support[
        "has_direct_source_support"
    ] = (
        dataset_feature_support[
            "n_source_rows"
        ].gt(0)
    )

    feature_support_parts.append(
        dataset_feature_support
    )


s2_imaging_feature_source_support = (
    pd.concat(
        feature_support_parts,
        ignore_index=True,
    )
)



print(
    "Selected MIMIC-IV source object:",
    mimic_object_name,
    mimic_source.shape,
)

print(
    "Selected MC-MED source object:",
    mcmed_object_name,
    mcmed_source.shape,
)

print(
    "Frozen imaging-result features:",
    len(
        frozen_imaging_feature_order
    ),
)


print(
    "\nDeterministic mapping coverage summary:"
)

display(
    s2_imaging_mapping_summary
)


print(
    "\nMapping methods used:"
)

display(
    s2_imaging_mapping_method_summary
)


print(
    "\nSource-row mapping multiplicity:"
)

display(
    s2_imaging_mapping_multiplicity_audit
)


print(
    "\nTop unmatched examination names:"
)

display(
    s2_imaging_unmatched_exam_audit
    .groupby(
        "dataset",
        group_keys=False,
    )
    .head(40)
    .reset_index(
        drop=True
    )
)


print(
    "\nFrozen-feature source-support summary:"
)

display(
    s2_imaging_feature_source_support
    .groupby(
        [
            "dataset",
            "feature_role",
            "has_direct_source_support",
        ]
    )
    .size()
    .rename(
        "n_frozen_features"
    )
    .reset_index()
)


print(
    "\nTop directly supported frozen features:"
)

display(
    s2_imaging_feature_source_support[
        s2_imaging_feature_source_support[
            "has_direct_source_support"
        ]
    ]
    .sort_values(
        [
            "dataset",
            "n_encounters",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "dataset",
        group_keys=False,
    )
    .head(30)
    .reset_index(
        drop=True
    )
)


print(
    "\nDETERMINISTIC S2 IMAGING "
    "MAPPING AUDIT COMPLETED"
)

print(
    "No fuzzy mapping was used."
)

print(
    "No abnormality classifier was applied."
)

print(
    "No 80-column imaging matrices "
    "were created or frozen."
)

print(
    "\nMain created objects:"
)

print(
    "  s2_imaging_source_rows"
)

print(
    "  s2_imaging_mapping_pairs"
)

print(
    "  mimic_s2_imaging_mapping_pairs"
)

print(
    "  mcmed_s2_imaging_mapping_pairs"
)

print(
    "  s2_imaging_mapping_summary"
)

print(
    "  s2_imaging_unmatched_exam_audit"
)

print(
    "  s2_imaging_feature_source_support"
)


In [ ]:


SAFE_IMAGING_ALIASES_RAW = {


    "US PREGNANCY 1ST TRIMESTER TRANSABDOMINAL": [
        "early_ob_us_14weeks",
    ],

    (
        "US PREGNANCY 1ST TRIMESTER "
        "AND OVARY TORSION TRANSABDOMINAL"
    ): [
        "early_ob_us_14weeks",
        "duplex_dop_abd_pel_limited",
    ],


    "CT ABDOMEN LIVER W IV CONTRAST TRIPHASIC": [
        "ct_abdomen_w_contrast",
    ],

    (
        "CT ABDOMEN PANCREAS W IV CONTRAST "
        "PELVIS W IV CONTRAST WITH 3D"
    ): [
        "ct_abd_and_pelvis_with_contrast",
    ],

    (
        "CT ABDOMEN PELVIS ANGIO "
        "W WO IV CONTRAST GI BLEED"
    ): [
        "cta_abd_and_pelvis",
    ],

    (
        "CT ABDOMEN PELVIS ANGIO "
        "W WO IV CONTRAST MESENTERIC ISCHEMIA"
    ): [
        "cta_abd_and_pelvis",
    ],

    (
        "CT ABDOMEN PELVIS "
        "WO LOW DOSE RENAL STONE"
    ): [
        "ct_abd_and_pelvis_w_o_contrast",
    ],

    (
        "CT ABDOMEN PELVIS "
        "W AND WO IV CONTRAST "
        "IVP UROGRAM WITH 3D"
    ): [
        "ctu_abd_pel_w_and_w_o_contrast",
    ],

    (
        "CT ABDOMEN PELVIS "
        "W IV CONTRAST PO CONTRAST"
    ): [
        "ct_abd_and_pelvis_with_contrast",
    ],

    (
        "CT ABDOMEN PELVIS "
        "WO IV CONTRAST PO CONTRAST"
    ): [
        "ct_abd_and_pelvis_w_o_contrast",
    ],

    (
        "CT CHEST ANGIOGRAPHY "
        "W IV CONTRAST TRAUMA"
    ): [
        "cta_chest",
    ],


    "XR ABD TUBE PLACEMENT": [
        "portable_abdomen",
    ],


    (
        "US LOWER EXTREMITY VEINS "
        "DEEP VEIN THROMBOSIS "
        "UNILATERAL RIGHT"
    ): [
        "unilat_lower_ext_veins_right",
    ],

    (
        "US LOWER EXTREMITY VEINS "
        "DEEP VEIN THROMBOSIS "
        "UNILATERAL LEFT"
    ): [
        "unilat_lower_ext_veins_left",
    ],


    "MR ABDOMEN PELVIS WO IV CONTRAST": [
        "mri_abdomen_w_o_contrast",
        "mri_pelvis_w_o_contrast",
    ],


    "ABD (SINGLE VIEW ONLY)": [
        "abdomen_supine_only",
    ],

    "CHEST PORT LINE/TUBE PLCT 1 EXAM": [
        "chest_port_line_placement",
    ],

    (
        "LIVER OR GALLBLADDER US "
        "(SINGLE ORGAN) PORT"
    ): [
        "liver_or_gallbladder_us_single_organ",
    ],

    "MRCP (MR ABD W&W/OC)": [
        "mrcp_mr_abd_w_and_w_oc",
    ],

    "MRI (ABDOMEN & PELVIS) W/O CONTRAST": [
        "mri_abdomen_w_o_contrast",
        "mri_pelvis_w_o_contrast",
    ],

    "ABDOMEN (SUPINE ONLY) PORT": [
        "abdomen_supine_only",
        "portable_abdomen",
    ],

    "CHEST (PA & LAT) PORT": [
        "chest_pa_and_lat",
    ],

    "COMPLETE GU U.S. (BLADDER & RENAL)": [
        "renal_u_s",
    ],
}



alias_patch_audit_rows = []


for source_exam_name, proposed_targets in (
    SAFE_IMAGING_ALIASES_RAW.items()
):

    accepted_targets = [
        target
        for target in proposed_targets
        if (
            target in TOKENS
            and target not in DIRECT_EXCLUDED
        )
    ]

    rejected_targets = [
        target
        for target in proposed_targets
        if target not in accepted_targets
    ]

    if accepted_targets:

        ALIASES[
            exam_key(
                source_exam_name
            )
        ] = accepted_targets

    alias_patch_audit_rows.append(
        {
            "source_exam_name": source_exam_name,
            "proposed_targets": (
                " | ".join(
                    proposed_targets
                )
            ),
            "accepted_targets": (
                " | ".join(
                    accepted_targets
                )
            ),
            "rejected_targets": (
                " | ".join(
                    rejected_targets
                )
            ),
            "alias_applied": bool(
                accepted_targets
            ),
        }
    )


s2_imaging_alias_patch_audit = pd.DataFrame(
    alias_patch_audit_rows
)


assert (
    s2_imaging_alias_patch_audit[
        "alias_applied"
    ].all()
), (
    "At least one proposed alias target was not found "
    "in the frozen imaging vocabulary."
)


mapping_summary_before_alias_patch = (
    s2_imaging_mapping_summary
    .copy()
)



source_audit_rows = []
mapping_pair_rows = []


for row in s2_imaging_source_rows.itertuples(
    index=False
):

    mappings = map_exam_to_frozen_features(
        row.source_exam_name
    )

    mapped_feature_names = [
        PREFIX + target_token
        for (
            target_token,
            _,
            _,
        ) in mappings
    ]

    source_audit_rows.append(
        {
            "dataset": row.dataset,
            "encounter_id": row.encounter_id,
            "source_row_key": row.source_row_key,
            "source_exam_name": row.source_exam_name,
            "n_mapped_features": len(
                mapped_feature_names
            ),
            "mapped_feature_names": (
                " | ".join(
                    mapped_feature_names
                )
            ),
        }
    )

    for (
        target_token,
        mapping_method,
        mapping_input,
    ) in mappings:

        mapping_pair_rows.append(
            {
                "dataset": row.dataset,
                "encounter_id": row.encounter_id,
                "source_row_key": row.source_row_key,
                "source_exam_name": (
                    row.source_exam_name
                ),
                "mapping_input_text": mapping_input,
                "mapped_feature_token": target_token,
                "mapped_feature_name": (
                    PREFIX + target_token
                ),
                "mapping_method": mapping_method,
            }
        )


s2_imaging_source_mapping_audit = pd.DataFrame(
    source_audit_rows
)


s2_imaging_mapping_pairs = (
    pd.DataFrame(
        mapping_pair_rows,
        columns=[
            "dataset",
            "encounter_id",
            "source_row_key",
            "source_exam_name",
            "mapping_input_text",
            "mapped_feature_token",
            "mapped_feature_name",
            "mapping_method",
        ],
    )
    .drop_duplicates(
        subset=[
            "source_row_key",
            "mapped_feature_name",
        ]
    )
    .reset_index(
        drop=True
    )
)



unknown_targets = (
    set(
        s2_imaging_mapping_pairs[
            "mapped_feature_name"
        ]
    )
    -
    set(
        frozen_imaging_feature_order
    )
)


assert not unknown_targets, (
    "A mapping outside the frozen imaging vocabulary "
    f"was created: {sorted(unknown_targets)}"
)


assert not (
    s2_imaging_mapping_pairs
    .duplicated(
        subset=[
            "source_row_key",
            "mapped_feature_name",
        ]
    )
    .any()
), (
    "Duplicate source-row/feature mappings were detected."
)


assert not (
    s2_imaging_mapping_pairs[
        "mapping_method"
    ]
    .astype("string")
    .str.contains(
        "FUZZY",
        case=False,
        na=False,
    )
    .any()
), (
    "A fuzzy mapping was unexpectedly detected."
)


mimic_s2_imaging_mapping_pairs = (
    s2_imaging_mapping_pairs[
        s2_imaging_mapping_pairs[
            "dataset"
        ].eq(
            "MIMIC-IV"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


mcmed_s2_imaging_mapping_pairs = (
    s2_imaging_mapping_pairs[
        s2_imaging_mapping_pairs[
            "dataset"
        ].eq(
            "MC-MED"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)



mapping_summary_rows = []


for dataset_name, dataset_rows in (
    s2_imaging_source_mapping_audit
    .groupby(
        "dataset",
        sort=False,
    )
):

    dataset_pairs = (
        s2_imaging_mapping_pairs[
            s2_imaging_mapping_pairs[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
    )

    mapped_rows = dataset_rows[
        dataset_rows[
            "n_mapped_features"
        ].gt(0)
    ]

    mapping_summary_rows.append(
        {
            "dataset": dataset_name,
            "source_rows": len(
                dataset_rows
            ),
            "unique_encounters": (
                dataset_rows[
                    "encounter_id"
                ].nunique()
            ),
            "rows_with_at_least_one_mapping": (
                len(
                    mapped_rows
                )
            ),
            "row_mapping_coverage_pct": round(
                100
                * len(
                    mapped_rows
                )
                / len(
                    dataset_rows
                ),
                2,
            ),
            "unique_exam_names": (
                dataset_rows[
                    "source_exam_name"
                ].nunique(
                    dropna=True
                )
            ),
            "unique_exam_names_mapped": (
                mapped_rows[
                    "source_exam_name"
                ].nunique(
                    dropna=True
                )
            ),
            "mapping_pairs": len(
                dataset_pairs
            ),
            "frozen_features_with_direct_support": (
                dataset_pairs[
                    "mapped_feature_name"
                ].nunique()
            ),
            "unmapped_source_rows": int(
                dataset_rows[
                    "n_mapped_features"
                ]
                .eq(0)
                .sum()
            ),
        }
    )


s2_imaging_mapping_summary = pd.DataFrame(
    mapping_summary_rows
)



s2_imaging_mapping_method_summary = (
    s2_imaging_mapping_pairs
    .groupby(
        [
            "dataset",
            "mapping_method",
        ]
    )
    .agg(
        n_mapping_pairs=(
            "mapped_feature_name",
            "size",
        ),
        n_source_rows=(
            "source_row_key",
            "nunique",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_mapping_pairs",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



s2_imaging_unmatched_exam_audit = (
    s2_imaging_source_mapping_audit[
        s2_imaging_source_mapping_audit[
            "n_mapped_features"
        ].eq(0)
    ]
    .assign(
        source_exam_name=lambda dataframe: (
            dataframe[
                "source_exam_name"
            ]
            .astype("string")
            .fillna(
                "<MISSING_EXAM_NAME>"
            )
            .replace(
                "",
                "<MISSING_EXAM_NAME>",
            )
        )
    )
    .groupby(
        [
            "dataset",
            "source_exam_name",
        ]
    )
    .agg(
        n_source_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_source_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



feature_support_parts = []


for dataset_name in [
    "MIMIC-IV",
    "MC-MED",
]:

    dataset_support = (
        s2_imaging_mapping_pairs[
            s2_imaging_mapping_pairs[
                "dataset"
            ].eq(
                dataset_name
            )
        ]
        .groupby(
            "mapped_feature_name"
        )
        .agg(
            n_source_rows=(
                "source_row_key",
                "nunique",
            ),
            n_encounters=(
                "encounter_id",
                "nunique",
            ),
        )
        .reset_index()
    )

    dataset_feature_support = (
        feature_policy
        .merge(
            dataset_support,
            on="mapped_feature_name",
            how="left",
            validate="one_to_one",
        )
    )

    dataset_feature_support[
        [
            "n_source_rows",
            "n_encounters",
        ]
    ] = (
        dataset_feature_support[
            [
                "n_source_rows",
                "n_encounters",
            ]
        ]
        .fillna(0)
        .astype(int)
    )

    dataset_feature_support.insert(
        0,
        "dataset",
        dataset_name,
    )

    dataset_feature_support[
        "has_direct_source_support"
    ] = (
        dataset_feature_support[
            "n_source_rows"
        ].gt(0)
    )

    feature_support_parts.append(
        dataset_feature_support
    )


s2_imaging_feature_source_support = pd.concat(
    feature_support_parts,
    ignore_index=True,
)



s2_imaging_alias_patch_comparison = (
    mapping_summary_before_alias_patch[
        [
            "dataset",
            "rows_with_at_least_one_mapping",
            "row_mapping_coverage_pct",
            "mapping_pairs",
            "frozen_features_with_direct_support",
            "unmapped_source_rows",
        ]
    ]
    .rename(
        columns={
            "rows_with_at_least_one_mapping":
                "mapped_rows_before",
            "row_mapping_coverage_pct":
                "coverage_pct_before",
            "mapping_pairs":
                "mapping_pairs_before",
            "frozen_features_with_direct_support":
                "supported_features_before",
            "unmapped_source_rows":
                "unmapped_rows_before",
        }
    )
    .merge(
        s2_imaging_mapping_summary[
            [
                "dataset",
                "rows_with_at_least_one_mapping",
                "row_mapping_coverage_pct",
                "mapping_pairs",
                "frozen_features_with_direct_support",
                "unmapped_source_rows",
            ]
        ]
        .rename(
            columns={
                "rows_with_at_least_one_mapping":
                    "mapped_rows_after",
                "row_mapping_coverage_pct":
                    "coverage_pct_after",
                "mapping_pairs":
                    "mapping_pairs_after",
                "frozen_features_with_direct_support":
                    "supported_features_after",
                "unmapped_source_rows":
                    "unmapped_rows_after",
            }
        ),
        on="dataset",
        how="inner",
        validate="one_to_one",
    )
)


s2_imaging_alias_patch_comparison[
    "additional_rows_mapped"
] = (
    s2_imaging_alias_patch_comparison[
        "mapped_rows_after"
    ]
    -
    s2_imaging_alias_patch_comparison[
        "mapped_rows_before"
    ]
)


s2_imaging_alias_patch_comparison[
    "coverage_percentage_point_gain"
] = (
    s2_imaging_alias_patch_comparison[
        "coverage_pct_after"
    ]
    -
    s2_imaging_alias_patch_comparison[
        "coverage_pct_before"
    ]
).round(2)


s2_imaging_alias_patch_comparison[
    "additional_supported_features"
] = (
    s2_imaging_alias_patch_comparison[
        "supported_features_after"
    ]
    -
    s2_imaging_alias_patch_comparison[
        "supported_features_before"
    ]
)



print(
    "Conservative imaging-alias patch audit:"
)

display(
    s2_imaging_alias_patch_audit
)


print(
    "\nMapping coverage before and after alias patch:"
)

display(
    s2_imaging_alias_patch_comparison
)


print(
    "\nFinal deterministic mapping summary:"
)

display(
    s2_imaging_mapping_summary
)


print(
    "\nHighest-frequency examinations still unmatched:"
)

display(
    s2_imaging_unmatched_exam_audit
    .groupby(
        "dataset",
        group_keys=False,
    )
    .head(25)
    .reset_index(
        drop=True
    )
)


print(
    "\nFINAL CONSERVATIVE IMAGING MAPPING PASSED"
)

print(
    "No fuzzy mappings were used."
)

print(
    "No impression classifier has yet been applied."
)

print(
    "No patient-level imaging matrix has yet been created."
)


#### Imaging-report abnormality classification

This section applies the deterministic radiology impression abnormality rule used during the original development work to every individual MIMIC-IV and MC-MED radiology report.

The examination-to-feature mapping has already been frozen. Therefore, this section does not change the 80-feature imaging vocabulary or create any additional mappings.

For MIMIC-IV, the available impression or conclusion section is extracted from the radiology report where possible. For MC-MED, the `Impression` field is used directly. If an explicit impression section cannot be identified, the available report text is retained so that the report is not silently discarded.

The classifier searches for clinically relevant positive findings and applies local negation handling. For example, `appendicitis` is considered abnormal, while `no appendicitis` is treated as negated. Mixed reports remain abnormal when at least one other non-negated positive finding is present.

Reports are assigned one of four audit labels:

- `abnormal_positive_finding`: at least one non-negated abnormal finding was detected.
- `normal_negative_impression`: an explicit normal or negative expression was detected without another positive finding.
- `no_abnormal_keyword_detected`: report text was available, but neither a positive finding nor a recognised normal phrase was detected.
- `missing_text`: no usable report text was available.

The last two categories are retained as non-abnormal under the legacy-compatible rule, but they are separately audited because they have lower classification confidence.

This cell creates only report-level rule labels and mapping audits. It does not yet create or freeze patient-level imaging-result features.


In [ ]:



required_objects = [
    "s2_imaging_source_rows",
    "s2_imaging_mapping_pairs",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the S2 imaging source-preparation and "
        "deterministic-mapping cells first. "
        f"Missing objects: {missing_objects}"
    )


s2_imaging_report_rule_source = (
    s2_imaging_source_rows
    .copy()
    .reset_index(drop=True)
)


required_source_columns = [
    "dataset",
    "encounter_id",
    "source_row_key",
    "source_exam_name",
]

missing_source_columns = [
    column
    for column in required_source_columns
    if column not in s2_imaging_report_rule_source.columns
]

if missing_source_columns:
    raise ValueError(
        "s2_imaging_source_rows is missing required columns: "
        f"{missing_source_columns}"
    )


assert not (
    s2_imaging_report_rule_source[
        "source_row_key"
    ]
    .duplicated()
    .any()
), (
    "source_row_key must identify one unique radiology report."
)



report_text_candidates = [
    "source_report_text",
    "report_text",
    "radiology_report_text",
    "radiology_result_text",
    "result_text",
    "Impression",
    "impression",
    "text",
]


available_report_text_columns = [
    column
    for column in report_text_candidates
    if column in s2_imaging_report_rule_source.columns
]


if not available_report_text_columns:
    raise ValueError(
        "No radiology report-text column was found.\n"
        "Checked candidate names:\n"
        f"{report_text_candidates}\n\n"
        "Available columns:\n"
        f"{s2_imaging_report_rule_source.columns.tolist()}"
    )


coalesced_report_text = pd.Series(
    pd.NA,
    index=s2_imaging_report_rule_source.index,
    dtype="string",
)

report_text_source_column = pd.Series(
    pd.NA,
    index=s2_imaging_report_rule_source.index,
    dtype="string",
)


for column in available_report_text_columns:

    candidate_text = (
        s2_imaging_report_rule_source[column]
        .astype("string")
        .str.strip()
    )

    candidate_is_usable = (
        candidate_text.notna()
        &
        candidate_text.ne("")
        &
        ~candidate_text.str.lower().isin(
            [
                "nan",
                "none",
                "null",
                "<na>",
            ]
        )
    )

    fill_mask = (
        coalesced_report_text.isna()
        &
        candidate_is_usable
    )

    coalesced_report_text.loc[
        fill_mask
    ] = candidate_text.loc[
        fill_mask
    ]

    report_text_source_column.loc[
        fill_mask
    ] = column


s2_imaging_report_rule_source[
    "source_report_text"
] = coalesced_report_text


s2_imaging_report_rule_source[
    "report_text_source_column"
] = report_text_source_column


s2_imaging_report_rule_source[
    "has_usable_report_text"
] = (
    s2_imaging_report_rule_source[
        "source_report_text"
    ]
    .notna()
)


s2_imaging_report_rule_source[
    "source_report_text_n_characters"
] = (
    s2_imaging_report_rule_source[
        "source_report_text"
    ]
    .fillna("")
    .astype(str)
    .str.len()
)



normal_regex_patterns = {

    "no_acute_abnormality_or_findings": (
        r"\bno\s+(?:acute\s+)?"
        r"(?:intra[- ]abdominal\s+|"
        r"intrathoracic\s+|"
        r"cardiopulmonary\s+)?"
        r"(?:abnormalit(?:y|ies)|"
        r"finding(?:s)?|process|disease)\b"
    ),

    "no_evidence_of": (
        r"\b(?:"
        r"no evidence of|"
        r"without evidence of|"
        r"no sonographic evidence of|"
        r"no radiographic evidence of|"
        r"no ct evidence of"
        r")\b"
    ),

    "negative_for": (
        r"\bnegative for\b"
    ),

    "unremarkable": (
        r"\bunremarkable\b"
    ),

    "normal_study": (
        r"\bnormal "
        r"(?:study|exam|examination)\b"
    ),

    "no_obstruction": (
        r"\bno (?:bowel )?obstruction\b"
    ),

    "no_appendicitis": (
        r"\bno (?:evidence of )?"
        r"appendicitis\b"
    ),

    "no_cholecystitis": (
        r"\bno (?:evidence of )?"
        r"cholecystitis\b"
    ),

    "no_hydronephrosis": (
        r"\bno (?:evidence of )?"
        r"hydronephrosis\b"
    ),

    "no_free_air": (
        r"\bno "
        r"(?:free air|pneumoperitoneum)\b"
    ),

    "no_abscess_or_collection": (
        r"\bno "
        r"(?:abscess|drainable collection|"
        r"fluid collection)\b"
    ),

    "no_fracture": (
        r"\bno (?:acute )?fracture\b"
    ),

    "no_pulmonary_embolism": (
        r"\bno (?:evidence of )?"
        r"(?:pulmonary embolism|pe)\b"
    ),
}


abnormal_phrases = [

    "appendicitis",
    "diverticulitis",
    "cholecystitis",
    "pancreatitis",
    "colitis",
    "enteritis",
    "gastritis",
    "small bowel obstruction",
    "large bowel obstruction",
    "bowel obstruction",
    "obstruction",
    "ileus",
    "perforation",
    "pneumoperitoneum",
    "free air",
    "drainable collection",
    "fluid collection",
    "abscess",
    "collection",
    "inflammatory changes",
    "inflammatory change",
    "fat stranding",
    "stranding",
    "bowel wall thickening",
    "wall thickening",
    "dilatation",
    "dilation",

    "biliary ductal dilatation",
    "biliary ductal dilation",
    "choledocholithiasis",
    "cholelithiasis",
    "hydroureteronephrosis",
    "hydronephrosis",
    "hydroureter",
    "ureteral stone",
    "renal stone",
    "nephrolithiasis",
    "ureterolithiasis",
    "obstructing calculus",
    "obstructing calculi",
    "calculus",
    "calculi",
    "stone",
    "cystitis",
    "pyelonephritis",

    "lymphadenopathy",
    "adenopathy",
    "metastatic",
    "metastasis",
    "malignancy",
    "tumor",
    "nodule",
    "lesion",
    "mass",

    "pulmonary embolism",
    "hemorrhage",
    "haemorrhage",
    "hematoma",
    "bleeding",
    "ischemia",
    "ischemic",
    "infarct",
    "aneurysm",
    "dissection",
    "thrombosis",
    "thrombus",
    "embolism",
    "embolus",

    "ovarian torsion",
    "torsion",
    "ectopic pregnancy",
    "ectopic",
    "hemorrhagic cyst",
    "ovarian cyst",
    "endometrioma",
    "fibroid",
    "cyst",

    "pleural effusion",
    "pulmonary edema",
    "pneumonia",
    "consolidation",
    "atelectasis",
    "fracture",
    "edema",

    "hepatic steatosis",
    "fatty infiltration",
    "fatty liver",
    "splenomegaly",
    "cirrhosis",
    "ascites",

    "epiploic appendagitis",
    "appendagitis",
    "fecal loading",
    "stool burden",
    "hypodensities",
    "hypodensity",
    "dilated loops",
    "distended loops",
    "cecal volvulus",
    "volvulus",
    "salpingitis",
    "dilated fallopian tubes",
    "intussusception",
    "gastroenteritis",
    "hernia",
    "varices",
    "stent",
]


abnormal_phrases = sorted(
    set(abnormal_phrases),
    key=lambda value: (
        -len(value),
        value,
    ),
)



def extract_radiology_impression(
    report_text,
):
    """
    Recover the impression or conclusion section where possible.

    This supports both the old aggregated format:
        [IMPRESSION] ...

    and raw reports containing:
        IMPRESSION:
        CONCLUSION:

    When no explicit section marker exists, the complete available
    report text is returned. This prevents silent report loss.
    """

    if pd.isna(report_text):
        return ""

    text = str(report_text).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "null",
        "<na>",
    }:
        return ""

    text = re.sub(
        (
            r"\b\d{4}-\d{2}-\d{2}"
            r"\s+\d{2}:\d{2}:\d{2}\b"
        ),
        " ",
        text,
    )

    bracket_parts = re.split(
        r"\[IMPRESSION\]",
        text,
        flags=re.IGNORECASE,
    )

    if len(bracket_parts) > 1:

        impression_parts = [
            part.strip()
            for part in bracket_parts[1:]
            if part.strip()
        ]

        text = " ".join(
            impression_parts
        )

    else:

        section_matches = list(
            re.finditer(
                (
                    r"(?im)^\s*"
                    r"(?:impression|conclusion)"
                    r"\s*:\s*"
                ),
                text,
            )
        )

        if section_matches:

            section_start = (
                section_matches[-1].end()
            )

            text = text[
                section_start:
            ]

    text = re.sub(
        (
            r"\[(?:"
            r"findings|"
            r"conclusion|"
            r"recommendation\(s\)|"
            r"recommendations?"
            r")\]"
        ),
        " ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\brecommendation\(s\)\s*:",
        " recommendation: ",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\brecommendations?\s*:",
        " recommendation: ",
        text,
        flags=re.IGNORECASE,
    )

    return text.strip()


def normalise_radiology_text(
    text,
):
    """
    Standardise whitespace and case for deterministic matching.
    """

    text = str(text).lower()

    text = text.replace(
        "\n",
        " ",
    )

    text = text.replace(
        "\r",
        " ",
    )

    text = text.replace(
        ";",
        ". ",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def phrase_to_regex(
    phrase,
):
    """
    Convert a literal clinical phrase into a word-boundary regex.
    """

    escaped_phrase = re.escape(
        phrase.lower()
    )

    escaped_phrase = escaped_phrase.replace(
        r"\ ",
        r"\s+",
    )

    return (
        r"(?<![a-z0-9])"
        +
        escaped_phrase
        +
        r"(?![a-z0-9])"
    )


def find_normal_rule_hits(
    text_normalised,
):
    """
    Return recognised normal or negative expressions.
    """

    hits = []

    for (
        rule_name,
        pattern,
    ) in normal_regex_patterns.items():

        if re.search(
            pattern,
            text_normalised,
            flags=re.IGNORECASE,
        ):
            hits.append(
                rule_name
            )

    return hits


def abnormal_hit_is_negated(
    text_normalised,
    hit_start,
    hit_end,
):
    """
    Detect local negation around an abnormal finding.

    Examples treated as negated:
        no appendicitis
        no evidence of obstruction
        negative for pulmonary embolism
        fracture is not identified

    Expressions such as 'cannot exclude appendicitis' are not
    treated as negated.
    """

    pre_window = text_normalised[
        max(
            0,
            hit_start - 120,
        ):
        hit_start
    ]

    post_window = text_normalised[
        hit_end:
        min(
            len(text_normalised),
            hit_end + 90,
        )
    ]

    complete_window = (
        pre_window
        +
        " "
        +
        post_window
    )

    uncertainty_pattern = (
        r"\b(?:"
        r"cannot exclude|"
        r"can not exclude|"
        r"not excluded|"
        r"not entirely excluded|"
        r"cannot be excluded|"
        r"not be excluded"
        r")\b"
    )

    if re.search(
        uncertainty_pattern,
        complete_window,
        flags=re.IGNORECASE,
    ):
        return False

    negation_before_patterns = [
        r"\bno\b",
        r"\bwithout\b",
        r"\bnegative for\b",
        r"\babsence of\b",
        r"\black of\b",
        r"\bfree of\b",
        r"\bno evidence of\b",
        r"\bwithout evidence of\b",
        r"\bno sonographic evidence of\b",
        r"\bno radiographic evidence of\b",
        r"\bno ct evidence of\b",
        r"\bno definite\b",
        r"\bnot\b",
    ]

    negation_after_patterns = [
        r"\bnot seen\b",
        r"\bnot identified\b",
        r"\bnot visualized\b",
        r"\bnot demonstrated\b",
        r"\babsent\b",
        r"\bexcluded\b",
        r"\bruled out\b",
    ]

    last_boundary = max(
        pre_window.rfind("."),
        pre_window.rfind(";"),
        pre_window.rfind(":"),
    )

    recent_pre_window = (
        pre_window[
            last_boundary + 1:
        ]
        if last_boundary >= 0
        else pre_window
    )

    if any(
        re.search(
            pattern,
            recent_pre_window,
            flags=re.IGNORECASE,
        )
        for pattern in negation_before_patterns
    ):
        return True

    if any(
        re.search(
            pattern,
            post_window,
            flags=re.IGNORECASE,
        )
        for pattern in negation_after_patterns
    ):

        if not re.search(
            r"\bnot excluded\b",
            post_window,
            flags=re.IGNORECASE,
        ):
            return True

    return False


def find_abnormal_rule_hits(
    text_normalised,
):
    """
    Return non-negated and negated abnormal finding matches.
    """

    non_negated_hits = []
    negated_hits = []

    for phrase in abnormal_phrases:

        phrase_pattern = phrase_to_regex(
            phrase
        )

        for match in re.finditer(
            phrase_pattern,
            text_normalised,
            flags=re.IGNORECASE,
        ):

            hit_record = {
                "phrase": phrase,
                "start": int(
                    match.start()
                ),
                "end": int(
                    match.end()
                ),
                "context": text_normalised[
                    max(
                        0,
                        match.start() - 70,
                    ):
                    min(
                        len(text_normalised),
                        match.end() + 90,
                    )
                ],
            }

            if abnormal_hit_is_negated(
                text_normalised,
                match.start(),
                match.end(),
            ):
                negated_hits.append(
                    hit_record
                )
            else:
                non_negated_hits.append(
                    hit_record
                )

    def deduplicate_hits(
        hit_records,
    ):

        seen = set()
        deduplicated = []

        for hit_record in hit_records:

            key = (
                hit_record["phrase"],
                hit_record["context"],
            )

            if key not in seen:

                seen.add(
                    key
                )

                deduplicated.append(
                    hit_record
                )

        return deduplicated

    return (
        deduplicate_hits(
            non_negated_hits
        ),
        deduplicate_hits(
            negated_hits
        ),
    )


def classify_radiology_report(
    report_text,
):
    """
    Apply the deterministic legacy-compatible abnormality rule.
    """

    impression_text = (
        extract_radiology_impression(
            report_text
        )
    )

    normalised_text = (
        normalise_radiology_text(
            impression_text
        )
    )

    if normalised_text == "":

        return {
            "radiology_impression_text": "",
            "radiology_rule_binary": 0,
            "radiology_rule_label": "missing_text",
            "radiology_rule_confidence": "none",
            "normal_rule_hits": "",
            "abnormal_rule_hits": "",
            "negated_abnormal_rule_hits": "",
            "abnormal_hit_contexts": "",
        }

    normal_hits = (
        find_normal_rule_hits(
            normalised_text
        )
    )

    (
        abnormal_hits,
        negated_abnormal_hits,
    ) = find_abnormal_rule_hits(
        normalised_text
    )

    abnormal_phrases_hit = sorted(
        {
            hit["phrase"]
            for hit in abnormal_hits
        }
    )

    negated_abnormal_phrases_hit = sorted(
        {
            hit["phrase"]
            for hit in negated_abnormal_hits
        }
    )

    abnormal_contexts = " || ".join(
        [
            (
                f"{hit['phrase']}: "
                f"{hit['context']}"
            )
            for hit in abnormal_hits[:8]
        ]
    )

    if abnormal_hits:

        binary_value = 1
        label = (
            "abnormal_positive_finding"
        )
        confidence = "rule_high"

    elif normal_hits:

        binary_value = 0
        label = (
            "normal_negative_impression"
        )
        confidence = "rule_high"

    else:

        binary_value = 0
        label = (
            "no_abnormal_keyword_detected"
        )
        confidence = "rule_low"

    return {
        "radiology_impression_text": impression_text,
        "radiology_rule_binary": binary_value,
        "radiology_rule_label": label,
        "radiology_rule_confidence": confidence,
        "normal_rule_hits": " | ".join(
            normal_hits
        ),
        "abnormal_rule_hits": " | ".join(
            abnormal_phrases_hit
        ),
        "negated_abnormal_rule_hits": " | ".join(
            negated_abnormal_phrases_hit
        ),
        "abnormal_hit_contexts": abnormal_contexts,
    }


RADIOLOGY_RULE_VERSION = (
    "LEGACY_COMPATIBLE_REPORT_LEVEL_V1"
)



radiology_rule_outputs = (
    s2_imaging_report_rule_source[
        "source_report_text"
    ]
    .apply(
        classify_radiology_report
    )
    .apply(
        pd.Series
    )
)


s2_imaging_report_rule_labeled = pd.concat(
    [
        s2_imaging_report_rule_source
        .reset_index(drop=True),

        radiology_rule_outputs
        .reset_index(drop=True),
    ],
    axis=1,
)


s2_imaging_report_rule_labeled[
    "radiology_rule_binary"
] = (
    pd.to_numeric(
        s2_imaging_report_rule_labeled[
            "radiology_rule_binary"
        ],
        errors="raise",
    )
    .astype(np.int8)
)


s2_imaging_report_rule_labeled[
    "radiology_rule_version"
] = RADIOLOGY_RULE_VERSION



assert (
    len(
        s2_imaging_report_rule_labeled
    )
    ==
    len(
        s2_imaging_source_rows
    )
), (
    "The number of report-level rows changed during "
    "radiology classification."
)


assert not (
    s2_imaging_report_rule_labeled[
        "source_row_key"
    ]
    .duplicated()
    .any()
), (
    "Duplicate report rows were introduced."
)


assert set(
    s2_imaging_report_rule_labeled[
        "radiology_rule_binary"
    ]
    .dropna()
    .unique()
).issubset(
    {
        0,
        1,
    }
), (
    "A non-binary radiology-rule value was detected."
)


print(
    "REPORT-LEVEL RADIOLOGY CLASSIFICATION COMPLETED"
)

print(
    "Rule version:",
    RADIOLOGY_RULE_VERSION,
)

print(
    "Report rows classified:",
    len(
        s2_imaging_report_rule_labeled
    ),
)

print(
    "No imaging feature matrix has yet been created."
)


#### Imaging-classification alignment audit


In [ ]:



required_objects = [
    "s2_imaging_report_rule_labeled",
    "s2_imaging_mapping_pairs",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the report-level radiology classification cell first. "
        f"Missing objects: {missing_objects}"
    )



s2_imaging_report_text_source_audit = (
    s2_imaging_report_rule_labeled
    .assign(
        report_text_source_column=lambda dataframe: (
            dataframe[
                "report_text_source_column"
            ]
            .fillna(
                "<NO_USABLE_TEXT>"
            )
        )
    )
    .groupby(
        [
            "dataset",
            "report_text_source_column",
        ],
        dropna=False,
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        median_text_characters=(
            "source_report_text_n_characters",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_report_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



s2_imaging_report_binary_summary = (
    s2_imaging_report_rule_labeled
    .groupby(
        [
            "dataset",
            "radiology_rule_binary",
        ]
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_exam_names=(
            "source_exam_name",
            "nunique",
        ),
    )
    .reset_index()
)


dataset_report_totals = (
    s2_imaging_report_rule_labeled
    .groupby(
        "dataset"
    )
    .size()
    .rename(
        "dataset_report_rows"
    )
    .reset_index()
)


s2_imaging_report_binary_summary = (
    s2_imaging_report_binary_summary
    .merge(
        dataset_report_totals,
        on="dataset",
        how="left",
        validate="many_to_one",
    )
)


s2_imaging_report_binary_summary[
    "percentage_of_dataset_reports"
] = (
    100
    *
    s2_imaging_report_binary_summary[
        "n_report_rows"
    ]
    /
    s2_imaging_report_binary_summary[
        "dataset_report_rows"
    ]
).round(2)


s2_imaging_report_binary_summary = (
    s2_imaging_report_binary_summary
    .sort_values(
        [
            "dataset",
            "radiology_rule_binary",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



s2_imaging_report_label_summary = (
    s2_imaging_report_rule_labeled
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
            "radiology_rule_binary",
            "radiology_rule_confidence",
        ],
        dropna=False,
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        median_text_characters=(
            "source_report_text_n_characters",
            "median",
        ),
    )
    .reset_index()
    .merge(
        dataset_report_totals,
        on="dataset",
        how="left",
        validate="many_to_one",
    )
)


s2_imaging_report_label_summary[
    "percentage_of_dataset_reports"
] = (
    100
    *
    s2_imaging_report_label_summary[
        "n_report_rows"
    ]
    /
    s2_imaging_report_label_summary[
        "dataset_report_rows"
    ]
).round(2)


s2_imaging_report_label_summary = (
    s2_imaging_report_label_summary
    .sort_values(
        [
            "dataset",
            "n_report_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



def summarise_pipe_separated_hits(
    dataframe,
    column_name,
    hit_type,
):
    """
    Count the number of unique report rows and encounters
    containing each pipe-separated rule hit.
    """

    temporary = dataframe[
        [
            "dataset",
            "source_row_key",
            "encounter_id",
            column_name,
        ]
    ].copy()

    temporary[column_name] = (
        temporary[column_name]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    temporary = temporary[
        temporary[column_name].ne("")
    ].copy()

    if temporary.empty:

        return pd.DataFrame(
            columns=[
                "dataset",
                "hit_type",
                "keyword",
                "n_report_rows",
                "n_encounters",
            ]
        )

    temporary["keyword"] = (
        temporary[column_name]
        .str.split(
            r"\s*\|\s*"
        )
    )

    temporary = temporary.explode(
        "keyword"
    )

    temporary["keyword"] = (
        temporary["keyword"]
        .astype(str)
        .str.strip()
    )

    temporary = temporary[
        temporary[
            "keyword"
        ].ne("")
    ].copy()

    temporary["hit_type"] = hit_type

    return (
        temporary
        .groupby(
            [
                "dataset",
                "hit_type",
                "keyword",
            ]
        )
        .agg(
            n_report_rows=(
                "source_row_key",
                "nunique",
            ),
            n_encounters=(
                "encounter_id",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "dataset",
                "n_report_rows",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


s2_imaging_report_keyword_hit_audit = pd.concat(
    [
        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "abnormal_rule_hits",
            "UNNEGATED_ABNORMAL",
        ),

        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "negated_abnormal_rule_hits",
            "NEGATED_ABNORMAL",
        ),

        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "normal_rule_hits",
            "NORMAL_OR_NEGATIVE_EXPRESSION",
        ),
    ],
    ignore_index=True,
)



rule_columns_for_mapping = [
    "dataset",
    "source_row_key",
    "radiology_rule_binary",
    "radiology_rule_label",
    "radiology_rule_confidence",
    "radiology_rule_version",
    "normal_rule_hits",
    "abnormal_rule_hits",
    "negated_abnormal_rule_hits",
    "abnormal_hit_contexts",
    "radiology_impression_text",
]


s2_imaging_mapping_pairs_with_report_rule = (
    s2_imaging_mapping_pairs
    .merge(
        s2_imaging_report_rule_labeled[
            rule_columns_for_mapping
        ],
        on=[
            "dataset",
            "source_row_key",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    s2_imaging_mapping_pairs_with_report_rule[
        "radiology_rule_binary"
    ]
    .notna()
    .all()
), (
    "At least one frozen examination mapping pair did not "
    "receive its report-level abnormality label."
)


s2_imaging_mapping_pairs_with_report_rule[
    "radiology_rule_binary"
] = (
    s2_imaging_mapping_pairs_with_report_rule[
        "radiology_rule_binary"
    ]
    .astype(np.int8)
)



s2_imaging_mapped_report_rule_summary = (
    s2_imaging_mapping_pairs_with_report_rule
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
            "radiology_rule_binary",
            "radiology_rule_confidence",
        ]
    )
    .agg(
        n_mapping_pairs=(
            "mapped_feature_name",
            "size",
        ),
        n_source_reports=(
            "source_row_key",
            "nunique",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_source_reports",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



example_columns = [
    "dataset",
    "encounter_id",
    "source_row_key",
    "source_exam_name",
    "radiology_rule_binary",
    "radiology_rule_label",
    "radiology_rule_confidence",
    "normal_rule_hits",
    "abnormal_rule_hits",
    "negated_abnormal_rule_hits",
    "abnormal_hit_contexts",
    "radiology_impression_text",
]


s2_imaging_report_rule_examples = (
    s2_imaging_report_rule_labeled
    .sort_values(
        [
            "dataset",
            "radiology_rule_label",
            "source_row_key",
        ]
    )
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
        ],
        group_keys=False,
    )
    .head(8)
    [
        example_columns
    ]
    .reset_index(
        drop=True
    )
)


s2_imaging_report_low_confidence_examples = (
    s2_imaging_report_rule_labeled[
        s2_imaging_report_rule_labeled[
            "radiology_rule_confidence"
        ].isin(
            [
                "rule_low",
                "none",
            ]
        )
    ]
    .sort_values(
        [
            "dataset",
            "source_row_key",
        ]
    )
    .groupby(
        "dataset",
        group_keys=False,
    )
    .head(30)
    [
        example_columns
    ]
    .reset_index(
        drop=True
    )
)



sanity_terms = [
    "no evidence",
    "appendicitis",
    "obstruction",
    "free air",
    "cholecystitis",
    "hydronephrosis",
    "wall thickening",
    "cannot exclude",
    "not excluded",
    "recommend",
]


sanity_rows = []


for dataset_name, dataset_rows in (
    s2_imaging_report_rule_labeled
    .groupby(
        "dataset"
    )
):

    normalised_impression = (
        dataset_rows[
            "radiology_impression_text"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    for term in sanity_terms:

        contains_term = (
            normalised_impression
            .str.contains(
                term,
                regex=False,
                na=False,
            )
        )

        n_containing_term = int(
            contains_term.sum()
        )

        n_classified_abnormal = int(
            dataset_rows.loc[
                contains_term,
                "radiology_rule_binary",
            ]
            .sum()
        )

        sanity_rows.append(
            {
                "dataset": dataset_name,
                "term": term,
                "n_reports_containing_term": (
                    n_containing_term
                ),
                "n_classified_abnormal": (
                    n_classified_abnormal
                ),
                "percentage_classified_abnormal": (
                    round(
                        100
                        *
                        n_classified_abnormal
                        /
                        n_containing_term,
                        2,
                    )
                    if n_containing_term > 0
                    else np.nan
                ),
            }
        )


s2_imaging_report_term_sanity_audit = pd.DataFrame(
    sanity_rows
)



print(
    "Radiology report-text source audit:"
)

display(
    s2_imaging_report_text_source_audit
)


print(
    "\nReport-level binary abnormality summary:"
)

display(
    s2_imaging_report_binary_summary
)


print(
    "\nDetailed report-level rule-label summary:"
)

display(
    s2_imaging_report_label_summary
)


print(
    "\nRule summary among deterministically mapped reports:"
)

display(
    s2_imaging_mapped_report_rule_summary
)


print(
    "\nMost frequent radiology-rule hits:"
)

display(
    s2_imaging_report_keyword_hit_audit
    .groupby(
        [
            "dataset",
            "hit_type",
        ],
        group_keys=False,
    )
    .head(20)
    .reset_index(
        drop=True
    )
)


print(
    "\nExamples from each report-rule category:"
)

display(
    s2_imaging_report_rule_examples
)


print(
    "\nLow-confidence or missing-text examples:"
)

display(
    s2_imaging_report_low_confidence_examples
)


print(
    "\nMixed-expression sanity audit:"
)

display(
    s2_imaging_report_term_sanity_audit
)


print(
    "\nS2 REPORT-LEVEL IMAGING ABNORMALITY AUDIT PASSED"
)

print(
    "Mapped report-feature pairs:",
    len(
        s2_imaging_mapping_pairs_with_report_rule
    ),
)

print(
    "No patient-level imaging matrix has yet been created or frozen."
)


#### Radiology abnormality-rule refinement

Manual inspection of the report-level audit identified two correctable limitations.

First, several clinically abnormal expressions were not represented in the original deterministic keyword vocabulary, including enterocolitis, biliary dyskinesia, hydrosalpinx, cardiomegaly, vascular congestion, gallbladder sludge, common-bile-duct dilatation and retained endometrial blood products.

Second, the original negation window was occasionally too broad. A negative statement later in the same report could incorrectly negate an earlier positive finding. Negation is therefore restricted to the immediate sentence or clause surrounding each finding.

This refinement does not change the frozen 80-feature imaging vocabulary, examination mappings, cohort membership or feature order. It changes only the report-level abnormal-versus-non-abnormal classification used to populate the frozen imaging features.

The original version-one classifications are retained in memory for comparison. No patient-level matrix is created or written to disk in this cell.


In [ ]:



required_objects = [
    "s2_imaging_report_rule_source",
    "s2_imaging_report_rule_labeled",
    "classify_radiology_report",
    "find_abnormal_rule_hits",
    "find_normal_rule_hits",
    "normal_regex_patterns",
    "abnormal_phrases",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the original report-level radiology "
        "classification cell first.\n"
        f"Missing objects: {missing_objects}"
    )


s2_imaging_report_rule_labeled_v1 = (
    s2_imaging_report_rule_labeled
    .copy()
)



normal_regex_patterns.update(
    {
        "broad_no_acute_process_or_pathology": (
            r"\bno\s+"
            r"(?:(?:acute|ct)\s+){0,2}"
            r"(?:[a-z0-9/-]+\s+){0,6}"
            r"(?:"
            r"abnormalit(?:y|ies)|"
            r"finding(?:s)?|"
            r"process|"
            r"pathology|"
            r"disease"
            r")\b"
        ),

        "no_evidence_of_or_for": (
            r"\b(?:"
            r"no evidence (?:of|for)|"
            r"without evidence of|"
            r"no sonographic evidence (?:of|for)|"
            r"no radiographic evidence (?:of|for)|"
            r"no ct evidence (?:of|for)"
            r")\b"
        ),

        "specific_normal_examination": (
            r"\bnormal\s+(?:"
            r"bowel gas pattern|"
            r"abdominal radiographs?|"
            r"pelvic ultrasound(?: study)?|"
            r"uterus and ovaries|"
            r"chest radiographs?|"
            r"abdominal ultrasound|"
            r"gallbladder and common bile duct|"
            r"ultrasound evaluation of the female pelvis"
            r")\b"
        ),

        "nonobstructive_bowel_gas_pattern": (
            r"\bnon[- ]?obstructive "
            r"bowel gas pattern\b"
        ),

        "no_correlate_for_pain": (
            r"\bno correlate for "
            r"(?:the )?(?:patient'?s )?pain\b"
        ),

        "no_acute_intracranial_finding": (
            r"\bno acute (?:intracranial )?"
            r"(?:hemorrhage|haemorrhage|infarct)\b"
        ),

        "no_focal_lung_abnormality": (
            r"\bno focal "
            r"(?:lung disease|airspace disease|"
            r"consolidation)\b"
        ),

        "no_gallstones_or_cholecystitis": (
            r"\bno "
            r"(?:gallstones|cholelithiasis)"
            r"(?:\s+or\s+"
            r"(?:sonographic )?"
            r"(?:evidence of )?"
            r"(?:acute )?"
            r"cholecystitis)?\b"
        ),
    }
)



additional_abnormal_phrases = {

    "enterocolitis",
    "biliary dyskinesia",
    "gallbladder ejection fraction is low",
    "low gallbladder ejection fraction",
    "fecalization of stool",
    "slowed gi transit",
    "slowed gastrointestinal transit",

    "gallbladder sludge",
    "biliary sludge",
    "sludge",
    "gallbladder polyp",
    "gallbladder polyps",
    "polyps",
    "adherent stones",
    "pericholecystic fluid",
    "pericholecystic fluid collection",
    "distended gallbladder",
    "common bile duct dilatation",
    "common bile duct dilation",
    "dilated common bile duct",
    "bile duct dilatation",
    "bile duct dilation",

    "hydrosalpinx",
    "blood products",
    "retained products",
    "retained products of conception",
    "ovarian hyperstimulation",
    "free pelvic fluid",
    "free fluid",
    "thickened labial folds",
    "labial thickening",

    "cardiomegaly",
    "pulmonary vascular congestion",
    "vascular congestion",

    "compression fracture",
    "compression fractures",

    "cysts",
    "phlegmon",
    "ductal dilatation",
    "ductal dilation",
}


abnormal_phrases = sorted(
    set(abnormal_phrases).union(
        additional_abnormal_phrases
    ),
    key=lambda value: (
        -len(value),
        value,
    ),
)



def _extract_local_clause(
    text,
    hit_start,
    hit_end,
):
    """
    Return only the sentence or clause containing the finding.

    This prevents a later negative statement from negating an
    earlier positive finding elsewhere in the impression.
    """

    boundary_characters = [
        ".",
        ";",
        "\n",
    ]

    left_positions = [
        text.rfind(
            character,
            0,
            hit_start,
        )
        for character in boundary_characters
    ]

    left_boundary = max(
        left_positions
    )

    right_positions = []

    for character in boundary_characters:

        position = text.find(
            character,
            hit_end,
        )

        if position >= 0:
            right_positions.append(
                position
            )

    right_boundary = (
        min(right_positions)
        if right_positions
        else len(text)
    )

    clause_start = (
        left_boundary + 1
        if left_boundary >= 0
        else 0
    )

    clause_text = text[
        clause_start:right_boundary
    ]

    local_hit_start = (
        hit_start - clause_start
    )

    local_hit_end = (
        hit_end - clause_start
    )

    return (
        clause_text,
        local_hit_start,
        local_hit_end,
    )


def abnormal_hit_is_negated(
    text_normalised,
    hit_start,
    hit_end,
):
    """
    Detect negation only within the immediate local clause.

    Examples:
        'no appendicitis' -> negated
        'no evidence of obstruction' -> negated
        'fracture is not identified' -> negated

    But:
        'unchanged compression fractures. No new fracture'
        -> first finding remains positive

        'ovaries not visualized but CT showed enteritis'
        -> enteritis remains positive

        'cannot exclude appendicitis'
        -> remains positive/uncertain rather than negated
    """

    (
        clause_text,
        local_hit_start,
        local_hit_end,
    ) = _extract_local_clause(
        text_normalised,
        hit_start,
        hit_end,
    )

    before_hit = clause_text[
        :local_hit_start
    ]

    after_hit = clause_text[
        local_hit_end:
    ]

    near_before = before_hit[
        -110:
    ]

    near_after = after_hit[
        :70
    ]

    complete_local_window = (
        near_before
        + " "
        + near_after
    )


    uncertainty_pattern = (
        r"\b(?:"
        r"cannot exclude|"
        r"can not exclude|"
        r"cannot be excluded|"
        r"can not be excluded|"
        r"not excluded|"
        r"not entirely excluded|"
        r"difficult to exclude"
        r")\b"
    )

    if re.search(
        uncertainty_pattern,
        complete_local_window,
        flags=re.IGNORECASE,
    ):
        return False


    direct_preceding_negation = (
        r"(?:"
        r"\bno(?:\s+(?:evidence|sign|signs)\s+(?:of|for))?|"
        r"\bwithout(?:\s+evidence\s+of)?|"
        r"\bnegative\s+for|"
        r"\babsence\s+of|"
        r"\black\s+of|"
        r"\bfree\s+of|"
        r"\bno\s+definite"
        r")"
        r"\s+"
        r"(?:[a-z0-9/-]+\s+){0,6}$"
    )

    if re.search(
        direct_preceding_negation,
        near_before,
        flags=re.IGNORECASE,
    ):
        return True


    no_matches = list(
        re.finditer(
            r"\b(?:no|without)\b",
            near_before,
            flags=re.IGNORECASE,
        )
    )

    if no_matches:

        last_no_position = (
            no_matches[-1].start()
        )

        negative_segment = near_before[
            last_no_position:
        ]

        contains_clause_break = re.search(
            r"\b(?:"
            r"but|"
            r"however|"
            r"although|"
            r"except|"
            r"yet|"
            r"whereas|"
            r"with findings of|"
            r"with evidence of"
            r")\b",
            negative_segment,
            flags=re.IGNORECASE,
        )

        if (
            len(negative_segment) <= 90
            and
            contains_clause_break is None
        ):
            return True


    direct_following_negation = (
        r"^\s*"
        r"(?:"
        r"is|are|was|were|"
        r"has been|have been"
        r")?"
        r"\s*"
        r"(?:"
        r"not\s+(?:"
        r"seen|"
        r"identified|"
        r"visualized|"
        r"visualised|"
        r"demonstrated|"
        r"present"
        r")|"
        r"absent|"
        r"ruled out"
        r")\b"
    )

    if re.search(
        direct_following_negation,
        near_after,
        flags=re.IGNORECASE,
    ):
        return True


    return False



radiology_rule_outputs_v2 = (
    s2_imaging_report_rule_source[
        "source_report_text"
    ]
    .apply(
        classify_radiology_report
    )
    .apply(
        pd.Series
    )
)


s2_imaging_report_rule_labeled = pd.concat(
    [
        s2_imaging_report_rule_source
        .reset_index(drop=True),

        radiology_rule_outputs_v2
        .reset_index(drop=True),
    ],
    axis=1,
)


s2_imaging_report_rule_labeled[
    "radiology_rule_binary"
] = (
    pd.to_numeric(
        s2_imaging_report_rule_labeled[
            "radiology_rule_binary"
        ],
        errors="raise",
    )
    .astype(np.int8)
)


RADIOLOGY_RULE_VERSION = (
    "LEGACY_COMPATIBLE_REPORT_LEVEL_V2"
)


s2_imaging_report_rule_labeled[
    "radiology_rule_version"
] = (
    RADIOLOGY_RULE_VERSION
)



comparison_columns = [
    "dataset",
    "source_row_key",
    "source_exam_name",
    "radiology_impression_text",
    "radiology_rule_binary",
    "radiology_rule_label",
]


s2_imaging_rule_v1_v2_comparison = (
    s2_imaging_report_rule_labeled_v1[
        comparison_columns
    ]
    .rename(
        columns={
            "radiology_rule_binary": (
                "radiology_rule_binary_v1"
            ),
            "radiology_rule_label": (
                "radiology_rule_label_v1"
            ),
            "radiology_impression_text": (
                "radiology_impression_text_v1"
            ),
        }
    )
    .merge(
        s2_imaging_report_rule_labeled[
            comparison_columns
        ]
        .rename(
            columns={
                "radiology_rule_binary": (
                    "radiology_rule_binary_v2"
                ),
                "radiology_rule_label": (
                    "radiology_rule_label_v2"
                ),
                "radiology_impression_text": (
                    "radiology_impression_text_v2"
                ),
            }
        ),
        on=[
            "dataset",
            "source_row_key",
            "source_exam_name",
        ],
        how="inner",
        validate="one_to_one",
    )
)


s2_imaging_rule_v1_v2_comparison[
    "binary_changed"
] = (
    s2_imaging_rule_v1_v2_comparison[
        "radiology_rule_binary_v1"
    ]
    !=
    s2_imaging_rule_v1_v2_comparison[
        "radiology_rule_binary_v2"
    ]
)


s2_imaging_rule_v1_v2_comparison[
    "label_changed"
] = (
    s2_imaging_rule_v1_v2_comparison[
        "radiology_rule_label_v1"
    ]
    !=
    s2_imaging_rule_v1_v2_comparison[
        "radiology_rule_label_v2"
    ]
)


s2_imaging_rule_change_summary = (
    s2_imaging_rule_v1_v2_comparison
    .groupby(
        [
            "dataset",
            "radiology_rule_label_v1",
            "radiology_rule_label_v2",
            "radiology_rule_binary_v1",
            "radiology_rule_binary_v2",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "n_report_rows"
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_report_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)



targeted_review_terms = [
    "enterocolitis",
    "biliary dyskinesia",
    "ejection fraction is low",
    "cardiomegaly",
    "vascular congestion",
    "hydrosalpinx",
    "pericholecystic fluid",
    "sludge",
    "adherent stones",
    "common bile duct",
    "blood products",
    "retained products",
    "ovarian hyperstimulation",
    "compression fracture",
    "free fluid",
    "normal pelvic ultrasound",
    "normal uterus and ovaries",
    "normal abdominal radiographs",
    "normal chest radiographs",
    "nonobstructive bowel gas pattern",
]


targeted_review_pattern = "|".join(
    re.escape(term)
    for term in targeted_review_terms
)


s2_imaging_rule_targeted_review = (
    s2_imaging_report_rule_labeled[
        s2_imaging_report_rule_labeled[
            "radiology_impression_text"
        ]
        .fillna("")
        .astype(str)
        .str.contains(
            targeted_review_pattern,
            case=False,
            regex=True,
            na=False,
        )
    ]
    [
        [
            "dataset",
            "encounter_id",
            "source_row_key",
            "source_exam_name",
            "radiology_rule_binary",
            "radiology_rule_label",
            "radiology_rule_confidence",
            "normal_rule_hits",
            "abnormal_rule_hits",
            "negated_abnormal_rule_hits",
            "radiology_impression_text",
        ]
    ]
    .sort_values(
        [
            "dataset",
            "radiology_rule_binary",
            "source_row_key",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)



assert (
    len(
        s2_imaging_report_rule_labeled
    )
    ==
    len(
        s2_imaging_report_rule_labeled_v1
    )
), (
    "Report rows changed during rule refinement."
)


assert not (
    s2_imaging_report_rule_labeled[
        "source_row_key"
    ]
    .duplicated()
    .any()
), (
    "Duplicate report rows were introduced."
)


assert set(
    s2_imaging_report_rule_labeled[
        "radiology_rule_binary"
    ]
    .unique()
).issubset(
    {
        0,
        1,
    }
), (
    "Non-binary radiology values were detected."
)



print(
    "Radiology-rule V1 versus V2 change summary:"
)

display(
    s2_imaging_rule_change_summary
)


print(
    "\nTargeted terminology regression audit:"
)

display(
    s2_imaging_rule_targeted_review
)


print(
    "\nREFINED REPORT-LEVEL RADIOLOGY RULE COMPLETED"
)

print(
    "Rule version:",
    RADIOLOGY_RULE_VERSION,
)

print(
    "Report rows retained:",
    len(
        s2_imaging_report_rule_labeled
    ),
)

print(
    "Reports whose binary classification changed:",
    int(
        s2_imaging_rule_v1_v2_comparison[
            "binary_changed"
        ]
        .sum()
    ),
)

print(
    "Reports whose detailed label changed:",
    int(
        s2_imaging_rule_v1_v2_comparison[
            "label_changed"
        ]
        .sum()
    ),
)

print(
    "\nNo patient-level imaging matrix has yet been created."
)

print(
    "Rerun the previous report-rule audit cell unchanged."
)


In [ ]:



required_objects = [
    "s2_imaging_report_rule_labeled",
    "s2_imaging_mapping_pairs",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise ValueError(
        "Run the report-level radiology classification cell first. "
        f"Missing objects: {missing_objects}"
    )



s2_imaging_report_text_source_audit = (
    s2_imaging_report_rule_labeled
    .assign(
        report_text_source_column=lambda dataframe: (
            dataframe[
                "report_text_source_column"
            ]
            .fillna(
                "<NO_USABLE_TEXT>"
            )
        )
    )
    .groupby(
        [
            "dataset",
            "report_text_source_column",
        ],
        dropna=False,
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        median_text_characters=(
            "source_report_text_n_characters",
            "median",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_report_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



s2_imaging_report_binary_summary = (
    s2_imaging_report_rule_labeled
    .groupby(
        [
            "dataset",
            "radiology_rule_binary",
        ]
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_exam_names=(
            "source_exam_name",
            "nunique",
        ),
    )
    .reset_index()
)


dataset_report_totals = (
    s2_imaging_report_rule_labeled
    .groupby(
        "dataset"
    )
    .size()
    .rename(
        "dataset_report_rows"
    )
    .reset_index()
)


s2_imaging_report_binary_summary = (
    s2_imaging_report_binary_summary
    .merge(
        dataset_report_totals,
        on="dataset",
        how="left",
        validate="many_to_one",
    )
)


s2_imaging_report_binary_summary[
    "percentage_of_dataset_reports"
] = (
    100
    *
    s2_imaging_report_binary_summary[
        "n_report_rows"
    ]
    /
    s2_imaging_report_binary_summary[
        "dataset_report_rows"
    ]
).round(2)


s2_imaging_report_binary_summary = (
    s2_imaging_report_binary_summary
    .sort_values(
        [
            "dataset",
            "radiology_rule_binary",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



s2_imaging_report_label_summary = (
    s2_imaging_report_rule_labeled
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
            "radiology_rule_binary",
            "radiology_rule_confidence",
        ],
        dropna=False,
    )
    .agg(
        n_report_rows=(
            "source_row_key",
            "size",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        median_text_characters=(
            "source_report_text_n_characters",
            "median",
        ),
    )
    .reset_index()
    .merge(
        dataset_report_totals,
        on="dataset",
        how="left",
        validate="many_to_one",
    )
)


s2_imaging_report_label_summary[
    "percentage_of_dataset_reports"
] = (
    100
    *
    s2_imaging_report_label_summary[
        "n_report_rows"
    ]
    /
    s2_imaging_report_label_summary[
        "dataset_report_rows"
    ]
).round(2)


s2_imaging_report_label_summary = (
    s2_imaging_report_label_summary
    .sort_values(
        [
            "dataset",
            "n_report_rows",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



def summarise_pipe_separated_hits(
    dataframe,
    column_name,
    hit_type,
):
    """
    Count the number of unique report rows and encounters
    containing each pipe-separated rule hit.
    """

    temporary = dataframe[
        [
            "dataset",
            "source_row_key",
            "encounter_id",
            column_name,
        ]
    ].copy()

    temporary[column_name] = (
        temporary[column_name]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    temporary = temporary[
        temporary[column_name].ne("")
    ].copy()

    if temporary.empty:

        return pd.DataFrame(
            columns=[
                "dataset",
                "hit_type",
                "keyword",
                "n_report_rows",
                "n_encounters",
            ]
        )

    temporary["keyword"] = (
        temporary[column_name]
        .str.split(
            r"\s*\|\s*"
        )
    )

    temporary = temporary.explode(
        "keyword"
    )

    temporary["keyword"] = (
        temporary["keyword"]
        .astype(str)
        .str.strip()
    )

    temporary = temporary[
        temporary[
            "keyword"
        ].ne("")
    ].copy()

    temporary["hit_type"] = hit_type

    return (
        temporary
        .groupby(
            [
                "dataset",
                "hit_type",
                "keyword",
            ]
        )
        .agg(
            n_report_rows=(
                "source_row_key",
                "nunique",
            ),
            n_encounters=(
                "encounter_id",
                "nunique",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "dataset",
                "n_report_rows",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


s2_imaging_report_keyword_hit_audit = pd.concat(
    [
        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "abnormal_rule_hits",
            "UNNEGATED_ABNORMAL",
        ),

        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "negated_abnormal_rule_hits",
            "NEGATED_ABNORMAL",
        ),

        summarise_pipe_separated_hits(
            s2_imaging_report_rule_labeled,
            "normal_rule_hits",
            "NORMAL_OR_NEGATIVE_EXPRESSION",
        ),
    ],
    ignore_index=True,
)



rule_columns_for_mapping = [
    "dataset",
    "source_row_key",
    "radiology_rule_binary",
    "radiology_rule_label",
    "radiology_rule_confidence",
    "radiology_rule_version",
    "normal_rule_hits",
    "abnormal_rule_hits",
    "negated_abnormal_rule_hits",
    "abnormal_hit_contexts",
    "radiology_impression_text",
]


s2_imaging_mapping_pairs_with_report_rule = (
    s2_imaging_mapping_pairs
    .merge(
        s2_imaging_report_rule_labeled[
            rule_columns_for_mapping
        ],
        on=[
            "dataset",
            "source_row_key",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    s2_imaging_mapping_pairs_with_report_rule[
        "radiology_rule_binary"
    ]
    .notna()
    .all()
), (
    "At least one frozen examination mapping pair did not "
    "receive its report-level abnormality label."
)


s2_imaging_mapping_pairs_with_report_rule[
    "radiology_rule_binary"
] = (
    s2_imaging_mapping_pairs_with_report_rule[
        "radiology_rule_binary"
    ]
    .astype(np.int8)
)



s2_imaging_mapped_report_rule_summary = (
    s2_imaging_mapping_pairs_with_report_rule
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
            "radiology_rule_binary",
            "radiology_rule_confidence",
        ]
    )
    .agg(
        n_mapping_pairs=(
            "mapped_feature_name",
            "size",
        ),
        n_source_reports=(
            "source_row_key",
            "nunique",
        ),
        n_encounters=(
            "encounter_id",
            "nunique",
        ),
        n_frozen_features=(
            "mapped_feature_name",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "dataset",
            "n_source_reports",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)



example_columns = [
    "dataset",
    "encounter_id",
    "source_row_key",
    "source_exam_name",
    "radiology_rule_binary",
    "radiology_rule_label",
    "radiology_rule_confidence",
    "normal_rule_hits",
    "abnormal_rule_hits",
    "negated_abnormal_rule_hits",
    "abnormal_hit_contexts",
    "radiology_impression_text",
]


s2_imaging_report_rule_examples = (
    s2_imaging_report_rule_labeled
    .sort_values(
        [
            "dataset",
            "radiology_rule_label",
            "source_row_key",
        ]
    )
    .groupby(
        [
            "dataset",
            "radiology_rule_label",
        ],
        group_keys=False,
    )
    .head(8)
    [
        example_columns
    ]
    .reset_index(
        drop=True
    )
)


s2_imaging_report_low_confidence_examples = (
    s2_imaging_report_rule_labeled[
        s2_imaging_report_rule_labeled[
            "radiology_rule_confidence"
        ].isin(
            [
                "rule_low",
                "none",
            ]
        )
    ]
    .sort_values(
        [
            "dataset",
            "source_row_key",
        ]
    )
    .groupby(
        "dataset",
        group_keys=False,
    )
    .head(30)
    [
        example_columns
    ]
    .reset_index(
        drop=True
    )
)



sanity_terms = [
    "no evidence",
    "appendicitis",
    "obstruction",
    "free air",
    "cholecystitis",
    "hydronephrosis",
    "wall thickening",
    "cannot exclude",
    "not excluded",
    "recommend",
]


sanity_rows = []


for dataset_name, dataset_rows in (
    s2_imaging_report_rule_labeled
    .groupby(
        "dataset"
    )
):

    normalised_impression = (
        dataset_rows[
            "radiology_impression_text"
        ]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    for term in sanity_terms:

        contains_term = (
            normalised_impression
            .str.contains(
                term,
                regex=False,
                na=False,
            )
        )

        n_containing_term = int(
            contains_term.sum()
        )

        n_classified_abnormal = int(
            dataset_rows.loc[
                contains_term,
                "radiology_rule_binary",
            ]
            .sum()
        )

        sanity_rows.append(
            {
                "dataset": dataset_name,
                "term": term,
                "n_reports_containing_term": (
                    n_containing_term
                ),
                "n_classified_abnormal": (
                    n_classified_abnormal
                ),
                "percentage_classified_abnormal": (
                    round(
                        100
                        *
                        n_classified_abnormal
                        /
                        n_containing_term,
                        2,
                    )
                    if n_containing_term > 0
                    else np.nan
                ),
            }
        )


s2_imaging_report_term_sanity_audit = pd.DataFrame(
    sanity_rows
)



print(
    "Radiology report-text source audit:"
)

display(
    s2_imaging_report_text_source_audit
)


print(
    "\nReport-level binary abnormality summary:"
)

display(
    s2_imaging_report_binary_summary
)


print(
    "\nDetailed report-level rule-label summary:"
)

display(
    s2_imaging_report_label_summary
)


print(
    "\nRule summary among deterministically mapped reports:"
)

display(
    s2_imaging_mapped_report_rule_summary
)


print(
    "\nMost frequent radiology-rule hits:"
)

display(
    s2_imaging_report_keyword_hit_audit
    .groupby(
        [
            "dataset",
            "hit_type",
        ],
        group_keys=False,
    )
    .head(20)
    .reset_index(
        drop=True
    )
)


print(
    "\nExamples from each report-rule category:"
)

display(
    s2_imaging_report_rule_examples
)


print(
    "\nLow-confidence or missing-text examples:"
)

display(
    s2_imaging_report_low_confidence_examples
)


print(
    "\nMixed-expression sanity audit:"
)

display(
    s2_imaging_report_term_sanity_audit
)


print(
    "\nS2 REPORT-LEVEL IMAGING ABNORMALITY AUDIT PASSED"
)

print(
    "Mapped report-feature pairs:",
    len(
        s2_imaging_mapping_pairs_with_report_rule
    ),
)

print(
    "No patient-level imaging matrix has yet been created or frozen."
)


#### Freeze the S2 imaging-result matrices

This cell converts the final report-level imaging classifications into one binary row per ED encounter with the same frozen 80 imaging-result features in MIMIC-IV and MC-MED.

The logic remains the same as the earlier imaging code: each radiology report stays attached to its own examination and mapped frozen feature. CT, MRI, ultrasound and other examination types remain separate columns. `max()` is used only when the same encounter has repeated evidence for the same feature, so one abnormal report makes that exact feature equal to 1.

The cell also derives the seven generic modality features from examination names, retains unsupported frozen columns as structural zeros, checks the expected cohort sizes of 8,550 MIMIC-IV encounters and 6,707 MC-MED encounters, verifies binary values and identical feature order, and keeps all outputs in memory. It does not write any dataframe to disk.

Run this only after the deterministic examination mapping and V2 report-level abnormality classification cells. The resulting matrices are primary conservative rule-based features rather than clinician-validated ground truth; the low-confidence report group should remain available for later sensitivity analysis or local-LLM/clinician review.


In [ ]:


R_PREFIX = "poe_result__imaging_radiology__"
O_PREFIX = "poe_order__imaging_radiology__"


def first_col(df, names):
    return next((c for c in names if c in df.columns), None)


def norm_dataset(x):
    x = str(x).upper().replace("_", "-").strip()
    if "MC-MED" in x or "MCMED" in x:
        return "MC-MED"
    if "MIMIC" in x:
        return "MIMIC-IV"
    return x


def norm_feature(x):
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x.startswith(R_PREFIX):
        return x
    if x.startswith(O_PREFIX):
        return x.replace(O_PREFIX, R_PREFIX, 1)
    return R_PREFIX + x


def binary(series):
    out = pd.to_numeric(series, errors="coerce")
    if out.notna().any():
        return out.fillna(0).gt(0).astype(np.int8)
    return (
        series.astype("string").str.lower().str.strip()
        .isin(["1", "true", "yes", "abnormal", "positive"])
        .astype(np.int8)
    )


def resolve_dataframe(preferred_names, required_groups, label):
    """Return the best dataframe containing one column from every group."""
    candidates = []

    for name, obj in list(globals().items()):
        if not isinstance(obj, pd.DataFrame) or len(obj) == 0:
            continue

        chosen = [first_col(obj, group) for group in required_groups]
        if any(col is None for col in chosen):
            continue

        dataset_values = set(obj[chosen[0]].dropna().map(norm_dataset).unique())
        if not {"MIMIC-IV", "MC-MED"}.issubset(dataset_values):
            continue

        score = 1000 - preferred_names.index(name) if name in preferred_names else 0
        score += 20 if "final" in name.lower() else 0
        score += 10 if "report" in name.lower() else 0
        candidates.append((score, len(obj), name, obj, chosen))

    if not candidates:
        raise ValueError(f"Could not find {label}. Rerun the preceding imaging cells.")

    return max(candidates, key=lambda x: (x[0], x[1]))



feature_order_names = [
    "s2_imaging_feature_order",
    "frozen_imaging_result_feature_order",
    "frozen_imaging_result_features",
    "mcmed_reference_imaging_result_feature_cols"
]

feature_order_candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        values = list(obj.columns)
    elif isinstance(obj, (list, tuple, np.ndarray, pd.Index, pd.Series)):
        values = list(obj)
    else:
        continue

    values = [
        str(v) for v in values
        if str(v).startswith(R_PREFIX) and str(v) != R_PREFIX + "x"
    ]

    if len(values) == 80 and len(set(values)) == 80:
        score = 1000 - feature_order_names.index(name) if name in feature_order_names else 0
        feature_order_candidates.append((score, name, values))

if not feature_order_candidates:
    raise ValueError("Frozen 80-column imaging feature order was not found.")

_, selected_feature_order_object, s2_imaging_feature_order = max(
    feature_order_candidates,
    key=lambda x: x[0]
)
feature_set = set(s2_imaging_feature_order)



pair_info = resolve_dataframe(
    preferred_names=[
        "s2_imaging_report_feature_pairs",
        "s2_imaging_report_feature_pairs_final",
        "s2_imaging_mapped_report_feature_pairs",
        "imaging_report_feature_pairs"
    ],
    required_groups=[
        ["dataset", "source_dataset"],
        ["encounter_id", "stay_id", "CSN"],
        ["source_row_key", "report_row_key"],
        ["source_exam_name", "exam_name", "Study"],
        ["radiology_rule_binary", "radiology_rule_binary_v2"],
        [
            "frozen_feature_name", "mapped_result_feature_name",
            "result_feature_name", "feature_name", "mapped_feature_name"
        ]
    ],
    label="the final report-feature mapping dataframe"
)

(
    _, _, selected_pair_object, pair_df,
    [pair_dataset, pair_encounter, pair_row, pair_exam, pair_binary, pair_feature]
) = pair_info

feature_match_count = pair_df[pair_feature].map(norm_feature).isin(feature_set).sum()
if feature_match_count == 0:
    raise ValueError(
        f"{selected_pair_object} has the required columns but no frozen imaging mappings."
    )



report_info = resolve_dataframe(
    preferred_names=[
        "s2_imaging_report_rows_classified",
        "s2_imaging_reports_classified",
        "s2_imaging_report_level_classified",
        "radiology_reports_classified"
    ],
    required_groups=[
        ["dataset", "source_dataset"],
        ["encounter_id", "stay_id", "CSN"],
        ["source_row_key", "report_row_key"],
        ["source_exam_name", "exam_name", "Study"],
        ["radiology_rule_binary", "radiology_rule_binary_v2"]
    ],
    label="the classified radiology report dataframe"
)

(
    _, _, selected_report_object, report_df,
    [report_dataset, report_encounter, report_row, report_exam, report_binary]
) = report_info



for name in ["mimic_s0_validated", "mcmed_s0_validated"]:
    if name not in globals():
        raise ValueError(f"{name} is not in memory.")

mimic_frozen = mimic_s0_validated.copy().reset_index(drop=True)
mcmed_frozen = mcmed_s0_validated.copy().reset_index(drop=True)

mimic_id = first_col(mimic_frozen, ["stay_id", "encounter_id"])
mcmed_id = first_col(mcmed_frozen, ["CSN", "encounter_id"])

assert len(mimic_frozen) == 8550 and not mimic_frozen[mimic_id].duplicated().any()
assert len(mcmed_frozen) == 6707 and not mcmed_frozen[mcmed_id].duplicated().any()



direct_rows = pd.DataFrame({
    "dataset": pair_df[pair_dataset].map(norm_dataset),
    "encounter_key": pair_df[pair_encounter].astype("string"),
    "source_row_key": pair_df[pair_row].astype("string"),
    "source_exam_name": pair_df[pair_exam].astype("string"),
    "feature_name": pair_df[pair_feature].map(norm_feature),
    "report_binary": binary(pair_df[pair_binary]),
    "feature_source": "DIRECT_EXAM_MAPPING"
})

direct_rows = direct_rows[direct_rows["feature_name"].isin(feature_set)].copy()



generic = {
    "ct": R_PREFIX + "imaging_radiology_orders_ct_scan",
    "xray_a": R_PREFIX + "imaging_radiology_orders_general_xray",
    "xray_b": R_PREFIX + "general_xray",
    "mri": R_PREFIX + "imaging_radiology_orders_mri",
    "us": R_PREFIX + "imaging_radiology_orders_ultrasound",
    "nm": R_PREFIX + "imaging_radiology_orders_nuclear_med",
    "ir": R_PREFIX + "imaging_radiology_orders_interventional_radiology"
}
generic = {k: v for k, v in generic.items() if v in feature_set}


def generic_targets(exam_name):
    text = re.sub(r"\s+", " ", str(exam_name).upper()).strip()

    if re.search(r"(^|[ |])CT[A-Z ]|(^|[ |])CTA([ |]|$)|(^|[ |])CTU([ |]|$)", text):
        return [generic["ct"]] if "ct" in generic else []
    if re.search(r"(^|[ |])MRI?([ |]|$)|MRCP", text):
        return [generic["mri"]] if "mri" in generic else []
    if re.search(
        r"(^|[ |])US([ |]|$)|U\.S\.|ULTRASOUND|DUPLEX|DOPPLER|"
        r"TRANSVAGINAL|TRANSABDOMINAL|SCROTAL|SONOGRAPH",
        text
    ):
        return [generic["us"]] if "us" in generic else []
    if re.search(r"(^|[ |])NM([ |]|$)|NUCLEAR MED|HEPATOBILIARY", text):
        return [generic["nm"]] if "nm" in generic else []
    if re.search(r"(^|[ |])IR([ |]|$)|INTERVENTIONAL RADIOLOGY", text):
        return [generic["ir"]] if "ir" in generic else []
    if re.search(
        r"(^|[ |])XR([ |]|$)|X[- ]?RAY|CHEST \(|ABDOMEN \(|ABD \(|"
        r"PORTABLE AP|SINGLE VIEW|PA & LAT|SUPINE|ERECT|DECUB|KUB|"
        r"UGI|ESOPHAGRAM|SHUNT SERIES|PELVIS AP|RIB|FEMUR|FOREARM",
        text
    ):
        return [generic[k] for k in ["xray_a", "xray_b"] if k in generic]
    return []


report_rows = pd.DataFrame({
    "dataset": report_df[report_dataset].map(norm_dataset),
    "encounter_key": report_df[report_encounter].astype("string"),
    "source_row_key": report_df[report_row].astype("string"),
    "source_exam_name": report_df[report_exam].astype("string"),
    "report_binary": binary(report_df[report_binary])
})

report_rows["feature_name"] = report_rows["source_exam_name"].map(generic_targets)

generic_rows = (
    report_rows.explode("feature_name")
    .dropna(subset=["feature_name"])
    .assign(feature_source="DERIVED_GENERIC_MODALITY")
)



s2_imaging_report_feature_rows_primary = pd.concat(
    [direct_rows, generic_rows],
    ignore_index=True
)

s2_imaging_encounter_feature_max_primary = (
    s2_imaging_report_feature_rows_primary
    .groupby(["dataset", "encounter_key", "feature_name"], as_index=False)
    .agg(
        imaging_result_binary=("report_binary", "max"),
        n_contributing_rows=("source_row_key", "size"),
        n_unique_reports=("source_row_key", "nunique")
    )
)



def build_matrix(dataset, cohort, id_col):
    keys = cohort[id_col].astype("string")
    long_df = s2_imaging_encounter_feature_max_primary.query(
        "dataset == @dataset"
    )[["encounter_key", "feature_name", "imaging_result_binary"]]

    wide = long_df.pivot(
        index="encounter_key",
        columns="feature_name",
        values="imaging_result_binary"
    )

    return (
        wide.reindex(columns=s2_imaging_feature_order, fill_value=0)
        .reindex(index=keys, fill_value=0)
        .fillna(0)
        .astype(np.int8)
        .rename_axis(id_col)
    )


mimic_s2_imaging_matrix_primary = build_matrix(
    "MIMIC-IV", mimic_frozen, mimic_id
)
mcmed_s2_imaging_matrix_primary = build_matrix(
    "MC-MED", mcmed_frozen, mcmed_id
)


def attach_features(cohort, matrix):
    base = cohort.drop(
        columns=[c for c in s2_imaging_feature_order if c in cohort.columns],
        errors="ignore"
    ).reset_index(drop=True)
    return pd.concat([base, matrix.reset_index(drop=True)], axis=1)


mimic_s2_imaging_features_primary = attach_features(
    mimic_frozen, mimic_s2_imaging_matrix_primary
)
mcmed_s2_imaging_features_primary = attach_features(
    mcmed_frozen, mcmed_s2_imaging_matrix_primary
)



for matrix, n_rows in [
    (mimic_s2_imaging_matrix_primary, 8550),
    (mcmed_s2_imaging_matrix_primary, 6707)
]:
    assert matrix.shape == (n_rows, 80)
    assert list(matrix.columns) == s2_imaging_feature_order
    assert not matrix.isna().any().any()
    assert set(np.unique(matrix.to_numpy())).issubset({0, 1})

feature_hash = hashlib.sha256(
    "\n".join(s2_imaging_feature_order).encode("utf-8")
).hexdigest()

summary = []
for dataset, matrix in [
    ("MIMIC-IV", mimic_s2_imaging_matrix_primary),
    ("MC-MED", mcmed_s2_imaging_matrix_primary)
]:
    support = matrix.sum(axis=0)
    summary.append({
        "dataset": dataset,
        "frozen_encounters": len(matrix),
        "frozen_imaging_features": matrix.shape[1],
        "encounters_with_any_abnormal_imaging": int(matrix.any(axis=1).sum()),
        "positive_encounter_feature_pairs": int(matrix.to_numpy().sum()),
        "features_with_at_least_one_positive": int((support > 0).sum()),
        "structural_zero_features_retained": int((support == 0).sum()),
        "feature_order_sha256": feature_hash
    })

s2_imaging_freeze_summary = pd.DataFrame(summary)

print("Selected feature order:", selected_feature_order_object)
print("Selected report-feature pairs:", selected_pair_object)
print("Selected classified reports:", selected_report_object)
print("\nmax() was used only for repeated reports of the SAME encounter + SAME feature.")
print("CT, MRI and other imaging types remain separate columns.\n")

display(s2_imaging_freeze_summary)

print("S2 IMAGING RESULT BLOCK FROZEN IN MEMORY")
print("MIMIC-IV:", mimic_s2_imaging_matrix_primary.shape)
print("MC-MED:", mcmed_s2_imaging_matrix_primary.shape)
print("No dataframe has been written to disk.")


#### Combine and freeze the S2 objective work-up block

This section combines the two previously validated S2 feature families:

- 351 abnormal laboratory-result features.
- 80 abnormal imaging-result features.

The laboratory columns are placed first and the imaging columns second, preserving the original frozen feature order.

Before combining them, the code verifies that:

- MIMIC-IV contains exactly 8,550 encounters.
- MC-MED contains exactly 6,707 encounters.
- Laboratory and imaging rows correspond to the same frozen encounter order.
- MIMIC-IV is aligned using `stay_id`.
- MC-MED is aligned using `CSN`.
- All 431 features are present without duplicate columns.
- Every feature value is binary, either 0 or 1.
- There are no missing feature values.
- Both datasets contain exactly the same columns in exactly the same order.

A SHA-256 hash is created from the complete 431-feature order. Matching hashes indicate that the two datasets use the same modelling schema.

The final matrices are retained only in memory:

- `mimic_s2_objective_workup_matrix`
- `mcmed_s2_objective_workup_matrix`

No dataframe is written to disk in this section.


In [ ]:

N_MIMIC = 8550
N_MCMED = 6707

N_LAB = 351
N_IMAGING = 80
N_S2_TOTAL = N_LAB + N_IMAGING

LAB_PREFIX = "poe_result__lab__"
IMAGING_PREFIX = "poe_result__imaging_radiology__"



def resolve_feature_order(
    preferred_names,
    prefix,
    expected_n,
):
    """
    Locate an existing frozen feature-order object.

    Accepted objects:
    - list
    - tuple
    - pandas Index
    - pandas Series
    - dataframe containing a feature_name column
    """

    def extract_feature_names(obj):

        if isinstance(obj, pd.DataFrame):

            if "feature_name" in obj.columns:
                values = obj["feature_name"].tolist()

            elif obj.shape[1] == 1:
                values = obj.iloc[:, 0].tolist()

            else:
                return None

        elif isinstance(
            obj,
            (
                pd.Series,
                pd.Index,
                list,
                tuple,
                np.ndarray,
            ),
        ):
            values = list(obj)

        else:
            return None

        values = [
            str(value)
            for value in values
            if pd.notna(value)
        ]

        valid = (
            len(values) == expected_n
            and len(set(values)) == expected_n
            and all(
                value.startswith(prefix)
                for value in values
            )
        )

        return values if valid else None


    for object_name in preferred_names:

        if object_name not in globals():
            continue

        feature_names = extract_feature_names(
            globals()[object_name]
        )

        if feature_names is not None:
            return object_name, feature_names


    for object_name, obj in reversed(
        list(globals().items())
    ):

        feature_names = extract_feature_names(obj)

        if feature_names is not None:
            return object_name, feature_names


    for object_name, obj in reversed(
        list(globals().items())
    ):

        if not isinstance(obj, pd.DataFrame):
            continue

        result_columns = [
            str(column)
            for column in obj.columns
            if str(column).startswith(
                "poe_result__"
            )
        ]

        family_columns = [
            column
            for column in result_columns
            if column.startswith(prefix)
        ]

        if (
            len(result_columns) == expected_n
            and len(family_columns) == expected_n
            and len(set(family_columns)) == expected_n
        ):
            return (
                f"{object_name} columns",
                family_columns,
            )


    raise AssertionError(
        f"Could not resolve {expected_n} frozen "
        f"features beginning with {prefix!r}."
    )


lab_order_source, s2_lab_feature_order = (
    resolve_feature_order(
        preferred_names=[
            "s2_lab_feature_order",
            "s2_lab_result_feature_order",
            "s2_lab_result_vocabulary",
            "s2_lab_feature_vocabulary",
        ],
        prefix=LAB_PREFIX,
        expected_n=N_LAB,
    )
)


(
    imaging_order_source,
    s2_imaging_feature_order_resolved,
) = resolve_feature_order(
    preferred_names=[
        "s2_imaging_feature_order",
        "s2_imaging_result_feature_order",
        "s2_imaging_result_vocabulary",
        "s2_imaging_feature_vocabulary",
    ],
    prefix=IMAGING_PREFIX,
    expected_n=N_IMAGING,
)



def select_feature_matrix(
    preferred_names,
    dataset_token,
    family_token,
    expected_rows,
    feature_order,
):
    """
    Select the existing patient-level matrix containing exactly
    the required feature family.

    Combined or unrelated matrices are excluded because their
    poe_result columns will not exactly match feature_order.
    """

    def is_valid_candidate(obj):

        if not isinstance(obj, pd.DataFrame):
            return False

        result_columns = [
            str(column)
            for column in obj.columns
            if str(column).startswith(
                "poe_result__"
            )
        ]

        return (
            len(obj) == expected_rows
            and len(result_columns)
            == len(feature_order)
            and set(result_columns)
            == set(feature_order)
        )


    for object_name in preferred_names:

        if object_name not in globals():
            continue

        obj = globals()[object_name]

        if is_valid_candidate(obj):
            return object_name, obj


    candidates = []

    for creation_position, (
        object_name,
        obj,
    ) in enumerate(globals().items()):

        if not is_valid_candidate(obj):
            continue

        name_lower = object_name.lower()

        score = (
            100 * (dataset_token in name_lower)
            + 50 * (family_token in name_lower)
            + 20 * ("matrix" in name_lower)
            + 10 * ("frozen" in name_lower)
            + 5 * ("s2" in name_lower)
        )

        candidates.append(
            (
                score,
                creation_position,
                object_name,
                obj,
            )
        )


    assert candidates, (
        f"No {dataset_token} {family_token} matrix "
        f"with {expected_rows} rows and "
        f"{len(feature_order)} features was found."
    )


    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
        ),
        reverse=True,
    )

    selected = candidates[0]

    return selected[2], selected[3]


mimic_lab_name, mimic_lab_source = (
    select_feature_matrix(
        preferred_names=[
            "mimic_s2_lab_result_matrix",
            "mimic_s2_lab_matrix",
            "mimic_s2_laboratory_result_matrix",
            "mimic_s2_lab_result_block",
        ],
        dataset_token="mimic",
        family_token="lab",
        expected_rows=N_MIMIC,
        feature_order=s2_lab_feature_order,
    )
)


mcmed_lab_name, mcmed_lab_source = (
    select_feature_matrix(
        preferred_names=[
            "mcmed_s2_lab_result_matrix",
            "mcmed_s2_lab_matrix",
            "mcmed_s2_laboratory_result_matrix",
            "mcmed_s2_lab_result_block",
        ],
        dataset_token="mcmed",
        family_token="lab",
        expected_rows=N_MCMED,
        feature_order=s2_lab_feature_order,
    )
)


mimic_imaging_name, mimic_imaging_source = (
    select_feature_matrix(
        preferred_names=[
            "mimic_s2_imaging_result_matrix",
            "mimic_s2_imaging_matrix",
            "mimic_s2_imaging_result_block",
        ],
        dataset_token="mimic",
        family_token="imaging",
        expected_rows=N_MIMIC,
        feature_order=(
            s2_imaging_feature_order_resolved
        ),
    )
)


mcmed_imaging_name, mcmed_imaging_source = (
    select_feature_matrix(
        preferred_names=[
            "mcmed_s2_imaging_result_matrix",
            "mcmed_s2_imaging_matrix",
            "mcmed_s2_imaging_result_block",
        ],
        dataset_token="mcmed",
        family_token="imaging",
        expected_rows=N_MCMED,
        feature_order=(
            s2_imaging_feature_order_resolved
        ),
    )
)



def align_feature_matrix(
    source,
    feature_order,
    cohort,
    id_column,
    source_name,
):
    """
    Align a patient-level feature matrix to the authoritative
    frozen cohort order using the encounter identifier.

    MIMIC-IV:
        stay_id

    MC-MED:
        CSN

    The source matrix is explicitly reordered when its rows
    contain the correct encounters in a different order.
    """

    assert id_column in cohort.columns, (
        f"{id_column!r} is missing from the frozen cohort."
    )

    assert cohort[id_column].notna().all(), (
        f"The frozen cohort contains missing {id_column} values."
    )

    assert cohort[id_column].is_unique, (
        f"The frozen cohort contains duplicate {id_column} values."
    )



    missing_features = [
        feature_name
        for feature_name in feature_order
        if feature_name not in source.columns
    ]

    assert not missing_features, (
        f"{source_name} is missing "
        f"{len(missing_features)} expected features.\n"
        f"Examples: {missing_features[:10]}"
    )


    matrix = source.loc[
        :,
        feature_order,
    ].copy()


    assert not matrix.columns.duplicated().any(), (
        f"{source_name} contains duplicate feature columns."
    )



    def standardise_identifier_keys(values):
        """
        Standardise numeric identifier representations such as:

            12345
            12345.0
            pandas Int64(12345)

        to the same string key.
        """

        keys = (
            pd.Series(values)
            .astype("string")
            .str.strip()
            .str.replace(
                r"\.0+$",
                "",
                regex=True,
            )
        )

        keys = keys.replace(
            {
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            }
        )

        return keys


    frozen_ids = pd.Index(
        cohort[id_column].copy(),
        name=id_column,
    )

    frozen_keys = standardise_identifier_keys(
        cohort[id_column]
    )


    assert frozen_keys.notna().all(), (
        f"The frozen cohort contains unusable "
        f"{id_column} values after standardisation."
    )

    assert frozen_keys.is_unique, (
        f"The frozen cohort contains duplicate standardised "
        f"{id_column} keys."
    )



    if id_column in source.columns:

        source_keys = standardise_identifier_keys(
            source[id_column]
        )


    elif source.index.equals(cohort.index):

        source_keys = frozen_keys.copy()


    else:

        source_keys = standardise_identifier_keys(
            source.index
        )


    assert source_keys.notna().all(), (
        f"{source_name} contains missing or unusable "
        f"{id_column} values."
    )

    assert source_keys.is_unique, (
        f"{source_name} contains duplicate {id_column} values."
    )



    source_key_set = set(
        source_keys.tolist()
    )

    frozen_key_set = set(
        frozen_keys.tolist()
    )


    missing_from_source = (
        frozen_key_set
        -
        source_key_set
    )

    extra_in_source = (
        source_key_set
        -
        frozen_key_set
    )


    assert not missing_from_source, (
        f"{source_name} is missing "
        f"{len(missing_from_source)} frozen encounters.\n"
        f"Examples: {list(missing_from_source)[:10]}"
    )


    assert not extra_in_source, (
        f"{source_name} contains "
        f"{len(extra_in_source)} encounters outside the "
        f"frozen cohort.\n"
        f"Examples: {list(extra_in_source)[:10]}"
    )


    assert len(source_keys) == len(frozen_keys), (
        f"{source_name} row count does not match the "
        "frozen cohort after identifier validation."
    )



    alignment_index = pd.Index(
        source_keys,
        name="_alignment_key",
    )

    desired_alignment_index = pd.Index(
        frozen_keys,
        name="_alignment_key",
    )


    matrix.index = alignment_index

    matrix = matrix.reindex(
        desired_alignment_index
    )


    assert len(matrix) == len(frozen_ids), (
        f"{source_name} changed row count during alignment."
    )

    assert not matrix.isna().all(axis=1).any(), (
        f"{source_name} produced an entirely missing row "
        "during encounter reindexing."
    )


    matrix.index = frozen_ids



    assert matrix.index.equals(frozen_ids), (
        f"{source_name} could not be aligned to the "
        f"authoritative {id_column} order."
    )

    assert matrix.columns.tolist() == feature_order, (
        f"{source_name} feature order changed during alignment."
    )

    assert not matrix.isna().any().any(), (
        f"{source_name} contains missing feature values "
        "after alignment."
    )


    binary_mask = matrix.isin(
        [
            0,
            1,
            False,
            True,
        ]
    )

    if not binary_mask.all().all():

        invalid_examples = (
            matrix
            .where(
                ~binary_mask
            )
            .stack()
            .drop_duplicates()
            .head(10)
            .tolist()
        )

        raise AssertionError(
            f"{source_name} contains non-binary values: "
            f"{invalid_examples}"
        )


    return matrix.astype("int8")


mimic_s2_lab_aligned = align_feature_matrix(
    source=mimic_lab_source,
    feature_order=s2_lab_feature_order,
    cohort=mimic_s0_validated,
    id_column="stay_id",
    source_name=mimic_lab_name,
)


mcmed_s2_lab_aligned = align_feature_matrix(
    source=mcmed_lab_source,
    feature_order=s2_lab_feature_order,
    cohort=mcmed_s0_validated,
    id_column="CSN",
    source_name=mcmed_lab_name,
)


mimic_s2_imaging_aligned = align_feature_matrix(
    source=mimic_imaging_source,
    feature_order=(
        s2_imaging_feature_order_resolved
    ),
    cohort=mimic_s0_validated,
    id_column="stay_id",
    source_name=mimic_imaging_name,
)


mcmed_s2_imaging_aligned = align_feature_matrix(
    source=mcmed_imaging_source,
    feature_order=(
        s2_imaging_feature_order_resolved
    ),
    cohort=mcmed_s0_validated,
    id_column="CSN",
    source_name=mcmed_imaging_name,
)



assert mimic_s2_lab_aligned.index.equals(
    mimic_s2_imaging_aligned.index
), (
    "MIMIC-IV laboratory and imaging rows "
    "are not identically aligned."
)


assert mcmed_s2_lab_aligned.index.equals(
    mcmed_s2_imaging_aligned.index
), (
    "MC-MED laboratory and imaging rows "
    "are not identically aligned."
)



s2_objective_workup_feature_order = tuple(
    s2_lab_feature_order
    + s2_imaging_feature_order_resolved
)


assert len(
    s2_objective_workup_feature_order
) == N_S2_TOTAL


assert len(
    set(s2_objective_workup_feature_order)
) == N_S2_TOTAL, (
    "Duplicate feature names were found between "
    "the laboratory and imaging blocks."
)


s2_objective_workup_feature_order_sha256 = (
    hashlib.sha256(
        "\n".join(
            s2_objective_workup_feature_order
        ).encode("utf-8")
    ).hexdigest()
)



mimic_s2_objective_workup_matrix = pd.concat(
    [
        mimic_s2_lab_aligned,
        mimic_s2_imaging_aligned,
    ],
    axis=1,
)


mcmed_s2_objective_workup_matrix = pd.concat(
    [
        mcmed_s2_lab_aligned,
        mcmed_s2_imaging_aligned,
    ],
    axis=1,
)



assert (
    mimic_s2_objective_workup_matrix.shape
    == (N_MIMIC, N_S2_TOTAL)
), (
    "Unexpected MIMIC-IV S2 matrix shape: "
    f"{mimic_s2_objective_workup_matrix.shape}"
)


assert (
    mcmed_s2_objective_workup_matrix.shape
    == (N_MCMED, N_S2_TOTAL)
), (
    "Unexpected MC-MED S2 matrix shape: "
    f"{mcmed_s2_objective_workup_matrix.shape}"
)


assert (
    mimic_s2_objective_workup_matrix
    .columns
    .tolist()
    == list(s2_objective_workup_feature_order)
)


assert (
    mcmed_s2_objective_workup_matrix
    .columns
    .tolist()
    == list(s2_objective_workup_feature_order)
)


assert (
    mimic_s2_objective_workup_matrix
    .columns
    .tolist()
    == mcmed_s2_objective_workup_matrix
    .columns
    .tolist()
), (
    "MIMIC-IV and MC-MED do not have the same "
    "S2 feature order."
)


assert not (
    mimic_s2_objective_workup_matrix
    .columns
    .duplicated()
    .any()
)


assert not (
    mcmed_s2_objective_workup_matrix
    .columns
    .duplicated()
    .any()
)



s2_objective_workup_feature_manifest = (
    pd.DataFrame(
        {
            "s2_position": range(
                1,
                N_S2_TOTAL + 1,
            ),
            "family": (
                ["s2_lab_result"] * N_LAB
                + ["s2_imaging_result"]
                * N_IMAGING
            ),
            "family_position": (
                list(range(1, N_LAB + 1))
                + list(
                    range(
                        1,
                        N_IMAGING + 1,
                    )
                )
            ),
            "feature_name": list(
                s2_objective_workup_feature_order
            ),
        }
    )
)



def summarise_s2_matrix(
    dataset_name,
    matrix,
):
    feature_support = matrix.sum(axis=0)

    return {
        "dataset": dataset_name,
        "frozen_encounters": len(matrix),
        "laboratory_features": N_LAB,
        "imaging_features": N_IMAGING,
        "combined_s2_features": matrix.shape[1],
        "encounters_with_any_positive_s2_result": int(
            matrix.any(axis=1).sum()
        ),
        "positive_encounter_feature_pairs": int(
            matrix.to_numpy().sum()
        ),
        "features_with_at_least_one_positive": int(
            feature_support.gt(0).sum()
        ),
        "structural_zero_features_retained": int(
            feature_support.eq(0).sum()
        ),
        "feature_order_sha256": (
            s2_objective_workup_feature_order_sha256
        ),
    }


s2_objective_workup_freeze_audit = (
    pd.DataFrame(
        [
            summarise_s2_matrix(
                dataset_name="MIMIC-IV",
                matrix=(
                    mimic_s2_objective_workup_matrix
                ),
            ),
            summarise_s2_matrix(
                dataset_name="MC-MED",
                matrix=(
                    mcmed_s2_objective_workup_matrix
                ),
            ),
        ]
    )
)


s2_objective_workup_source_selection = (
    pd.DataFrame(
        [
            {
                "dataset": "MIMIC-IV",
                "family": "Laboratory",
                "selected_object": mimic_lab_name,
                "feature_order_source": (
                    lab_order_source
                ),
            },
            {
                "dataset": "MC-MED",
                "family": "Laboratory",
                "selected_object": mcmed_lab_name,
                "feature_order_source": (
                    lab_order_source
                ),
            },
            {
                "dataset": "MIMIC-IV",
                "family": "Imaging",
                "selected_object": (
                    mimic_imaging_name
                ),
                "feature_order_source": (
                    imaging_order_source
                ),
            },
            {
                "dataset": "MC-MED",
                "family": "Imaging",
                "selected_object": (
                    mcmed_imaging_name
                ),
                "feature_order_source": (
                    imaging_order_source
                ),
            },
        ]
    )
)



print(
    "Selected and aligned S2 source objects:"
)

display(
    s2_objective_workup_source_selection
)


print(
    "\nCombined S2 objective-workup freeze audit:"
)

display(
    s2_objective_workup_freeze_audit
)


print(
    "\nS2 OBJECTIVE WORK-UP BLOCK FROZEN IN MEMORY"
)

print(
    "MIMIC-IV:",
    mimic_s2_objective_workup_matrix.shape,
)

print(
    "MC-MED:",
    mcmed_s2_objective_workup_matrix.shape,
)

print(
    "Laboratory features:",
    N_LAB,
)

print(
    "Imaging features:",
    N_IMAGING,
)

print(
    "Combined features:",
    N_S2_TOTAL,
)

print(
    "Combined feature-order SHA-256:",
    s2_objective_workup_feature_order_sha256,
)

print(
    "\nNo dataframe has been written to disk."
)


#### Complete 2,495-feature modelling matrices

This section combines the frozen S0, S1 and S2 feature families into the complete modelling schema.

S0 contains the nine arrival-level demographic, transport and triage-vital features. S1 contains the frozen historical diagnosis and medication features. S2 contains the frozen abnormal laboratory and imaging-result features.

All S1 and S2 binary matrices are explicitly aligned to the authoritative frozen encounter order using `stay_id` for MIMIC-IV and `CSN` for MC-MED. S0 remains in its validated raw form so that categorical encoding and numerical imputation can subsequently be learned only from the internal training split.

The final column order is checked against the preserved legacy Pathway 5 feature order when that manifest is available. This ensures continuity with the previously developed 2,495-feature model while retaining the newly reconstructed and externally aligned feature values.

The outputs retained in memory are:

- `mimic_full_feature_matrix`
- `mcmed_full_feature_matrix`
- `mimic_model_index`
- `mcmed_model_index`
- `full_feature_manifest`
- `full_feature_freeze_audit`

No preprocessing is fitted and no dataframe is written to disk in this section.


In [ ]:

N_MIMIC = 8550
N_MCMED = 6707

N_S0 = 9
N_DIAGNOSIS = 879
N_MEDICATION_NAME = 812
N_MEDICATION_CLASS = 364
N_MEDICATION_TOTAL = (
    N_MEDICATION_NAME
    + N_MEDICATION_CLASS
)
N_S1 = (
    N_DIAGNOSIS
    + N_MEDICATION_TOTAL
)
N_S2 = 431
N_FULL = (
    N_S0
    + N_S1
    + N_S2
)


S0_FEATURE_ORDER = [
    "age",
    "sex_gender",
    "arrival_transport_grouped",
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
]


assert len(S0_FEATURE_ORDER) == N_S0
assert len(set(S0_FEATURE_ORDER)) == N_S0


CONFIG_DIRECTORY = (
    PROJECT_ROOT
    / "config"
    / "feature_vocabularies"
)



required_objects = [
    "mimic_s0_validated",
    "mcmed_s0_validated",
    "mimic_s2_objective_workup_matrix",
    "mcmed_s2_objective_workup_matrix",
    "s2_objective_workup_feature_order",
    "align_feature_matrix",
]


missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]


if missing_objects:
    raise ValueError(
        "Run the previous S0 and S2 freeze cells first.\n"
        f"Missing objects: {missing_objects}"
    )



def load_feature_order(
    filename,
    expected_n,
):
    """
    Load one frozen feature vocabulary while preserving its
    recorded row order.
    """

    path = CONFIG_DIRECTORY / filename

    if not path.exists():

        fallback_path = (
            Path.cwd()
            / "config"
            / "feature_vocabularies"
            / filename
        )

        if fallback_path.exists():
            path = fallback_path


    if not path.exists():
        raise FileNotFoundError(
            f"Frozen vocabulary not found: {filename}"
        )


    vocabulary = pd.read_csv(path)


    feature_column = next(
        (
            column
            for column in [
                "feature_name",
                "final_feature_name",
                "model_feature_name",
                "column_name",
            ]
            if column in vocabulary.columns
        ),
        None,
    )


    if feature_column is None:

        if vocabulary.shape[1] == 1:
            feature_column = vocabulary.columns[0]

        else:
            raise ValueError(
                f"No feature-name column found in {filename}.\n"
                f"Available columns: "
                f"{vocabulary.columns.tolist()}"
            )


    feature_order = (
        vocabulary[feature_column]
        .astype("string")
        .str.strip()
        .dropna()
        .tolist()
    )


    feature_order = [
        feature
        for feature in feature_order
        if feature != ""
    ]


    assert len(feature_order) == expected_n, (
        f"{filename} should contain {expected_n} features; "
        f"found {len(feature_order)}."
    )


    assert len(set(feature_order)) == expected_n, (
        f"Duplicate feature names were found in {filename}."
    )


    return feature_order, vocabulary, path


(
    s1_diagnosis_feature_order,
    s1_diagnosis_vocabulary_loaded,
    s1_diagnosis_vocabulary_path,
) = load_feature_order(
    filename="s1_diagnosis_vocabulary.csv",
    expected_n=N_DIAGNOSIS,
)


(
    s1_medication_name_feature_order,
    s1_medication_name_vocabulary_loaded,
    s1_medication_name_vocabulary_path,
) = load_feature_order(
    filename="s1_medication_name_vocabulary.csv",
    expected_n=N_MEDICATION_NAME,
)


(
    s1_medication_class_feature_order,
    s1_medication_class_vocabulary_loaded,
    s1_medication_class_vocabulary_path,
) = load_feature_order(
    filename="s1_medication_class_vocabulary.csv",
    expected_n=N_MEDICATION_CLASS,
)


s1_medication_feature_order = (
    s1_medication_name_feature_order
    + s1_medication_class_feature_order
)


s1_feature_order = (
    s1_diagnosis_feature_order
    + s1_medication_feature_order
)


assert len(s1_medication_feature_order) == N_MEDICATION_TOTAL
assert len(set(s1_medication_feature_order)) == N_MEDICATION_TOTAL

assert len(s1_feature_order) == N_S1
assert len(set(s1_feature_order)) == N_S1



def select_binary_feature_source(
    preferred_names,
    required_feature_order,
    expected_rows,
    dataset_tokens,
    family_tokens,
):
    """
    Select an existing patient-level dataframe containing every
    required frozen feature.

    Extra metadata columns such as subject_id, stay_id, MRN,
    CSN, split and outcome are allowed.
    """

    required_features = set(
        required_feature_order
    )


    def is_valid_candidate(obj):

        if not isinstance(obj, pd.DataFrame):
            return False

        if len(obj) != expected_rows:
            return False

        return required_features.issubset(
            set(
                obj.columns
            )
        )


    for object_name in preferred_names:

        if object_name not in globals():
            continue

        candidate = globals()[object_name]

        if is_valid_candidate(candidate):
            return object_name, candidate


    candidates = []


    for creation_position, (
        object_name,
        obj,
    ) in enumerate(
        globals().items()
    ):

        if not is_valid_candidate(obj):
            continue


        name_lower = object_name.lower()


        score = 0

        score += 100 * sum(
            token in name_lower
            for token in dataset_tokens
        )

        score += 60 * sum(
            token in name_lower
            for token in family_tokens
        )

        score += 20 * (
            "primary" in name_lower
        )

        score += 15 * (
            "feature" in name_lower
        )

        score += 10 * (
            "matrix" in name_lower
        )

        score += 5 * (
            "s1" in name_lower
        )


        extra_columns = (
            len(obj.columns)
            - len(required_feature_order)
        )


        candidates.append(
            (
                score,
                -extra_columns,
                creation_position,
                object_name,
                obj,
            )
        )


    if not candidates:
        raise AssertionError(
            "No patient-level source matrix was found for "
            f"{len(required_feature_order)} required features "
            f"and {expected_rows} encounters."
        )


    candidates.sort(
        key=lambda item: (
            item[0],
            item[1],
            item[2],
        ),
        reverse=True,
    )


    selected = candidates[0]

    return selected[3], selected[4]


mimic_diagnosis_name, mimic_diagnosis_source = (
    select_binary_feature_source(
        preferred_names=[
            "mimic_s1_diagnosis_features_primary",
            "mimic_s1_diagnosis_matrix",
            "mimic_s1_pmh_diagnosis_features",
            "mimic_diagnosis_feature_matrix",
        ],
        required_feature_order=(
            s1_diagnosis_feature_order
        ),
        expected_rows=N_MIMIC,
        dataset_tokens=[
            "mimic",
        ],
        family_tokens=[
            "diagnosis",
            "pmh",
        ],
    )
)


mcmed_diagnosis_name, mcmed_diagnosis_source = (
    select_binary_feature_source(
        preferred_names=[
            "mcmed_s1_diagnosis_features_primary",
            "mcmed_s1_diagnosis_matrix",
            "mcmed_s1_pmh_diagnosis_features",
            "mcmed_diagnosis_feature_matrix",
        ],
        required_feature_order=(
            s1_diagnosis_feature_order
        ),
        expected_rows=N_MCMED,
        dataset_tokens=[
            "mcmed",
            "mc_med",
        ],
        family_tokens=[
            "diagnosis",
            "pmh",
        ],
    )
)


mimic_medication_name, mimic_medication_source = (
    select_binary_feature_source(
        preferred_names=[
            "mimic_s1_medication_features_primary",
            "mimic_s1_medication_matrix",
            "mimic_s1_pmh_medication_features",
            "mimic_medication_feature_matrix",
        ],
        required_feature_order=(
            s1_medication_feature_order
        ),
        expected_rows=N_MIMIC,
        dataset_tokens=[
            "mimic",
        ],
        family_tokens=[
            "medication",
            "medicine",
            "drug",
            "pmh",
        ],
    )
)


mcmed_medication_name, mcmed_medication_source = (
    select_binary_feature_source(
        preferred_names=[
            "mcmed_s1_medication_features_primary",
            "mcmed_s1_medication_candidate_matrix",
            "mcmed_s1_medication_matrix",
            "mcmed_s1_pmh_medication_features",
            "mcmed_medication_feature_matrix",
        ],
        required_feature_order=(
            s1_medication_feature_order
        ),
        expected_rows=N_MCMED,
        dataset_tokens=[
            "mcmed",
            "mc_med",
        ],
        family_tokens=[
            "medication",
            "medicine",
            "drug",
            "pmh",
        ],
    )
)



mimic_s1_diagnosis_aligned = align_feature_matrix(
    source=mimic_diagnosis_source,
    feature_order=s1_diagnosis_feature_order,
    cohort=mimic_s0_validated,
    id_column="stay_id",
    source_name=mimic_diagnosis_name,
)


mcmed_s1_diagnosis_aligned = align_feature_matrix(
    source=mcmed_diagnosis_source,
    feature_order=s1_diagnosis_feature_order,
    cohort=mcmed_s0_validated,
    id_column="CSN",
    source_name=mcmed_diagnosis_name,
)


mimic_s1_medication_aligned = align_feature_matrix(
    source=mimic_medication_source,
    feature_order=s1_medication_feature_order,
    cohort=mimic_s0_validated,
    id_column="stay_id",
    source_name=mimic_medication_name,
)


mcmed_s1_medication_aligned = align_feature_matrix(
    source=mcmed_medication_source,
    feature_order=s1_medication_feature_order,
    cohort=mcmed_s0_validated,
    id_column="CSN",
    source_name=mcmed_medication_name,
)



mimic_s1_matrix = pd.concat(
    [
        mimic_s1_diagnosis_aligned,
        mimic_s1_medication_aligned,
    ],
    axis=1,
)


mcmed_s1_matrix = pd.concat(
    [
        mcmed_s1_diagnosis_aligned,
        mcmed_s1_medication_aligned,
    ],
    axis=1,
)


assert mimic_s1_matrix.shape == (
    N_MIMIC,
    N_S1,
)


assert mcmed_s1_matrix.shape == (
    N_MCMED,
    N_S1,
)


assert (
    mimic_s1_matrix.columns.tolist()
    == s1_feature_order
)


assert (
    mcmed_s1_matrix.columns.tolist()
    == s1_feature_order
)


assert (
    mimic_s1_matrix.columns.tolist()
    == mcmed_s1_matrix.columns.tolist()
)



def prepare_s0_matrix(
    cohort,
    id_column,
    expected_rows,
):
    """
    Preserve the validated S0 values without fitting any
    preprocessing or imputation.
    """

    missing_features = [
        feature
        for feature in S0_FEATURE_ORDER
        if feature not in cohort.columns
    ]


    assert not missing_features, (
        "Validated S0 cohort is missing features: "
        f"{missing_features}"
    )


    assert len(cohort) == expected_rows

    assert cohort[id_column].notna().all()
    assert cohort[id_column].is_unique


    matrix = (
        cohort
        .set_index(
            id_column,
            drop=True,
        )
        .loc[
            :,
            S0_FEATURE_ORDER,
        ]
        .copy()
    )


    matrix.index.name = id_column


    return matrix


mimic_s0_matrix = prepare_s0_matrix(
    cohort=mimic_s0_validated,
    id_column="stay_id",
    expected_rows=N_MIMIC,
)


mcmed_s0_matrix = prepare_s0_matrix(
    cohort=mcmed_s0_validated,
    id_column="CSN",
    expected_rows=N_MCMED,
)



mimic_s2_matrix = (
    mimic_s2_objective_workup_matrix
    .copy()
)


mcmed_s2_matrix = (
    mcmed_s2_objective_workup_matrix
    .copy()
)


assert mimic_s2_matrix.shape == (
    N_MIMIC,
    N_S2,
)


assert mcmed_s2_matrix.shape == (
    N_MCMED,
    N_S2,
)


assert (
    mimic_s2_matrix.columns.tolist()
    ==
    list(
        s2_objective_workup_feature_order
    )
)


assert (
    mcmed_s2_matrix.columns.tolist()
    ==
    list(
        s2_objective_workup_feature_order
    )
)


assert mimic_s0_matrix.index.equals(
    mimic_s1_matrix.index
)


assert mimic_s0_matrix.index.equals(
    mimic_s2_matrix.index
)


assert mcmed_s0_matrix.index.equals(
    mcmed_s1_matrix.index
)


assert mcmed_s0_matrix.index.equals(
    mcmed_s2_matrix.index
)



constructed_full_feature_order = (
    S0_FEATURE_ORDER
    + s1_feature_order
    + list(
        s2_objective_workup_feature_order
    )
)


assert len(
    constructed_full_feature_order
) == N_FULL


assert len(
    set(
        constructed_full_feature_order
    )
) == N_FULL, (
    "Duplicate feature names exist across S0, S1 and S2."
)



legacy_order_path = (
    CONFIG_DIRECTORY
    / "legacy_path5_feature_order.csv"
)


if not legacy_order_path.exists():

    fallback_legacy_path = (
        Path.cwd()
        / "config"
        / "feature_vocabularies"
        / "legacy_path5_feature_order.csv"
    )

    if fallback_legacy_path.exists():
        legacy_order_path = fallback_legacy_path


legacy_order_used = False
legacy_order_audited = False
legacy_order_exact_match = False

legacy_s0_alias_audit = pd.DataFrame()
legacy_schema_change_audit = pd.DataFrame()
legacy_common_feature_order_audit = pd.DataFrame()
legacy_common_feature_order_summary = pd.DataFrame()



LEGACY_S0_NAME_ALIASES = {

    "gender": "sex_gender",
    "sex": "sex_gender",
    "sex_gender": "sex_gender",

    "means_of_arrival": "arrival_transport_grouped",
    "arrival_transport": "arrival_transport_grouped",
    "arrival_transport_grouped": (
        "arrival_transport_grouped"
    ),

    "temperature": "temperature_c",
    "temperature_c": "temperature_c",
    "triage_temperature": "temperature_c",
    "triage_temp": "temperature_c",

    "heartrate": "heart_rate",
    "heart_rate": "heart_rate",
    "triage_hr": "heart_rate",

    "resprate": "respiratory_rate",
    "respiratory_rate": "respiratory_rate",
    "triage_rr": "respiratory_rate",

    "o2sat": "oxygen_saturation",
    "spo2": "oxygen_saturation",
    "oxygen_saturation": "oxygen_saturation",
    "triage_spo2": "oxygen_saturation",

    "sbp": "systolic_blood_pressure",
    "systolic_blood_pressure": (
        "systolic_blood_pressure"
    ),
    "triage_sbp": "systolic_blood_pressure",

    "dbp": "diastolic_blood_pressure",
    "diastolic_blood_pressure": (
        "diastolic_blood_pressure"
    ),
    "triage_dbp": "diastolic_blood_pressure",
}


def reconcile_legacy_feature_name(
    feature_name,
):
    """
    Reconcile only genuine naming aliases.

    Clinically different features such as pain_score and age
    remain different.
    """

    original_name = str(
        feature_name
    ).strip()

    lookup_name = (
        original_name
        .lower()
    )

    return LEGACY_S0_NAME_ALIASES.get(
        lookup_name,
        original_name,
    )



if legacy_order_path.exists():

    legacy_order_table = pd.read_csv(
        legacy_order_path
    )


    legacy_feature_column = next(
        (
            column
            for column in [
                "feature_name",
                "final_feature_name",
                "model_feature_name",
                "column_name",
            ]
            if column in legacy_order_table.columns
        ),
        None,
    )


    if legacy_feature_column is None:

        if legacy_order_table.shape[1] == 1:

            legacy_feature_column = (
                legacy_order_table.columns[0]
            )

        else:

            raise ValueError(
                "No feature-name column could be identified "
                "in legacy_path5_feature_order.csv.\n"
                f"Available columns: "
                f"{legacy_order_table.columns.tolist()}"
            )


    legacy_feature_order_original = (
        legacy_order_table[
            legacy_feature_column
        ]
        .astype("string")
        .str.strip()
        .dropna()
        .tolist()
    )


    legacy_feature_order_original = [
        feature_name
        for feature_name
        in legacy_feature_order_original
        if feature_name != ""
    ]


    assert (
        len(
            legacy_feature_order_original
        )
        == N_FULL
    ), (
        "The legacy Pathway 5 feature order should contain "
        f"{N_FULL} features, but contains "
        f"{len(legacy_feature_order_original)}."
    )


    legacy_feature_order_reconciled = [
        reconcile_legacy_feature_name(
            feature_name
        )
        for feature_name
        in legacy_feature_order_original
    ]


    assert (
        len(
            set(
                legacy_feature_order_reconciled
            )
        )
        == N_FULL
    ), (
        "Legacy name reconciliation created duplicate "
        "feature names."
    )



    legacy_s0_alias_audit = pd.DataFrame(
        {
            "legacy_feature_name": (
                legacy_feature_order_original
            ),
            "current_feature_name": (
                legacy_feature_order_reconciled
            ),
        }
    )


    legacy_s0_alias_audit = (
        legacy_s0_alias_audit[
            legacy_s0_alias_audit[
                "legacy_feature_name"
            ]
            !=
            legacy_s0_alias_audit[
                "current_feature_name"
            ]
        ]
        .reset_index(
            drop=True
        )
    )



    missing_from_legacy = sorted(
        set(
            constructed_full_feature_order
        )
        -
        set(
            legacy_feature_order_reconciled
        )
    )


    extra_in_legacy = sorted(
        set(
            legacy_feature_order_reconciled
        )
        -
        set(
            constructed_full_feature_order
        )
    )


    legacy_schema_change_audit = pd.DataFrame(
        [
            {
                "comparison": (
                    "Current feature absent "
                    "from legacy schema"
                ),
                "feature_name": feature_name,
                "interpretation": (
                    "Added to the current validated S0 design"
                ),
            }
            for feature_name in missing_from_legacy
        ]
        +
        [
            {
                "comparison": (
                    "Legacy feature absent "
                    "from current schema"
                ),
                "feature_name": feature_name,
                "interpretation": (
                    "Removed from the current validated S0 design"
                ),
            }
            for feature_name in extra_in_legacy
        ]
    )


    print(
        "Legacy S0 feature-name reconciliation:"
    )

    display(
        legacy_s0_alias_audit
    )


    print(
        "\nDocumented legacy-to-current schema differences:"
    )

    display(
        legacy_schema_change_audit
    )



    assert missing_from_legacy == [
        "age",
    ], (
        "Unexpected current features are missing from the "
        "legacy schema:\n"
        f"{missing_from_legacy}"
    )


    assert extra_in_legacy == [
        "pain_score",
    ], (
        "Unexpected legacy-only features were detected:\n"
        f"{extra_in_legacy}"
    )



    common_feature_set = (
        set(
            constructed_full_feature_order
        )
        &
        set(
            legacy_feature_order_reconciled
        )
    )


    assert len(
        common_feature_set
    ) == N_FULL - 1, (
        "Expected exactly 2,494 features shared between "
        "the legacy and current schemas."
    )


    current_common_order = [
        feature_name
        for feature_name
        in constructed_full_feature_order
        if feature_name in common_feature_set
    ]


    legacy_common_order = [
        feature_name
        for feature_name
        in legacy_feature_order_reconciled
        if feature_name in common_feature_set
    ]


    assert len(
        current_common_order
    ) == N_FULL - 1


    assert len(
        legacy_common_order
    ) == N_FULL - 1


    assert set(
        current_common_order
    ) == set(
        legacy_common_order
    ), (
        "The legacy and current common-feature sets differ."
    )



    current_position_lookup = {
        feature_name: position
        for position, feature_name
        in enumerate(
            constructed_full_feature_order,
            start=1,
        )
    }


    legacy_position_lookup = {
        feature_name: position
        for position, feature_name
        in enumerate(
            legacy_feature_order_reconciled,
            start=1,
        )
    }


    legacy_common_feature_order_audit = (
        pd.DataFrame(
            {
                "feature_name": (
                    current_common_order
                ),
            }
        )
    )


    legacy_common_feature_order_audit[
        "current_position"
    ] = (
        legacy_common_feature_order_audit[
            "feature_name"
        ]
        .map(
            current_position_lookup
        )
    )


    legacy_common_feature_order_audit[
        "legacy_position"
    ] = (
        legacy_common_feature_order_audit[
            "feature_name"
        ]
        .map(
            legacy_position_lookup
        )
    )


    legacy_common_feature_order_audit[
        "position_difference"
    ] = (
        legacy_common_feature_order_audit[
            "current_position"
        ]
        -
        legacy_common_feature_order_audit[
            "legacy_position"
        ]
    )


    legacy_common_feature_order_audit[
        "same_position"
    ] = (
        legacy_common_feature_order_audit[
            "position_difference"
        ]
        .eq(0)
    )


    legacy_common_feature_order_summary = (
        pd.DataFrame(
            [
                {
                    "common_features": len(
                        legacy_common_feature_order_audit
                    ),
                    "features_in_same_position": int(
                        legacy_common_feature_order_audit[
                            "same_position"
                        ]
                        .sum()
                    ),
                    "features_in_different_position": int(
                        (
                            ~legacy_common_feature_order_audit[
                                "same_position"
                            ]
                        )
                        .sum()
                    ),
                    "common_relative_order_identical": (
                        current_common_order
                        ==
                        legacy_common_order
                    ),
                    "legacy_order_used_for_current_matrix": (
                        False
                    ),
                }
            ]
        )
    )


    print(
        "\nLegacy versus current common-feature "
        "position summary:"
    )

    display(
        legacy_common_feature_order_summary
    )


    print(
        "\nFirst common features with different positions:"
    )

    display(
        legacy_common_feature_order_audit[
            ~legacy_common_feature_order_audit[
                "same_position"
            ]
        ]
        .sort_values(
            "current_position"
        )
        .head(30)
        .reset_index(
            drop=True
        )
    )


    legacy_order_audited = True
    legacy_order_exact_match = False
    legacy_order_used = False


    full_feature_order = (
        constructed_full_feature_order
    )


else:

    print(
        "Legacy Pathway 5 order file was not found. "
        "Using the current constructed S0 + S1 + S2 order."
    )

    full_feature_order = (
        constructed_full_feature_order
    )



assert len(
    full_feature_order
) == N_FULL


assert len(
    set(
        full_feature_order
    )
) == N_FULL


assert (
    full_feature_order
    ==
    constructed_full_feature_order
), (
    "The current validated feature order should remain "
    "authoritative."
)


print(
    "\nLegacy Pathway 5 schema audited:",
    legacy_order_audited,
)


print(
    "Legacy schema exactly matches current schema:",
    legacy_order_exact_match,
)


print(
    "Legacy feature order directly used:",
    legacy_order_used,
)


print(
    "Current design substitution:",
    "pain_score removed; age added",
)

mimic_full_feature_matrix = pd.concat(
    [
        mimic_s0_matrix,
        mimic_s1_matrix,
        mimic_s2_matrix,
    ],
    axis=1,
)


mcmed_full_feature_matrix = pd.concat(
    [
        mcmed_s0_matrix,
        mcmed_s1_matrix,
        mcmed_s2_matrix,
    ],
    axis=1,
)


mimic_full_feature_matrix = (
    mimic_full_feature_matrix
    .loc[
        :,
        full_feature_order,
    ]
)


mcmed_full_feature_matrix = (
    mcmed_full_feature_matrix
    .loc[
        :,
        full_feature_order,
    ]
)



assert mimic_full_feature_matrix.shape == (
    N_MIMIC,
    N_FULL,
)


assert mcmed_full_feature_matrix.shape == (
    N_MCMED,
    N_FULL,
)


assert (
    mimic_full_feature_matrix.columns.tolist()
    == full_feature_order
)


assert (
    mcmed_full_feature_matrix.columns.tolist()
    == full_feature_order
)


assert (
    mimic_full_feature_matrix.columns.tolist()
    ==
    mcmed_full_feature_matrix.columns.tolist()
)


assert not (
    mimic_full_feature_matrix
    .columns
    .duplicated()
    .any()
)


assert not (
    mcmed_full_feature_matrix
    .columns
    .duplicated()
    .any()
)


binary_feature_order = (
    s1_feature_order
    + list(
        s2_objective_workup_feature_order
    )
)


for dataset_name, matrix in [
    (
        "MIMIC-IV",
        mimic_full_feature_matrix,
    ),
    (
        "MC-MED",
        mcmed_full_feature_matrix,
    ),
]:

    binary_block = matrix.loc[
        :,
        binary_feature_order,
    ]


    assert not binary_block.isna().any().any(), (
        f"{dataset_name} S1/S2 block contains "
        "missing values."
    )


    valid_binary = binary_block.isin(
        [
            0,
            1,
            False,
            True,
        ]
    )


    assert valid_binary.all().all(), (
        f"{dataset_name} S1/S2 block contains "
        "non-binary values."
    )



mimic_index_columns = [
    column
    for column in [
        "subject_id",
        "stay_id",
        "intime",
        "split",
        "outcome",
        "outcome_binary",
        "anchor_year",
        "anchor_year_group",
        "shifted_encounter_year",
        "anchor_to_encounter_year_offset",
        "approx_event_year_start",
        "approx_event_year_end",
        "approx_event_year",
        "approx_event_period",
    ]
    if column in mimic_s0_validated.columns
]


mcmed_index_columns = [
    column
    for column in [
        "MRN",
        "CSN",
        "split",
        "outcome",
        "outcome_binary",
    ]
    if column in mcmed_s0_validated.columns
]


mimic_model_index = (
    mimic_s0_validated[
        mimic_index_columns
    ]
    .copy()
    .set_index(
        "stay_id",
        drop=False,
    )
    .reindex(
        mimic_full_feature_matrix.index
    )
)


mcmed_model_index = (
    mcmed_s0_validated[
        mcmed_index_columns
    ]
    .copy()
    .set_index(
        "CSN",
        drop=False,
    )
    .reindex(
        mcmed_full_feature_matrix.index
    )
)


assert mimic_model_index.index.equals(
    mimic_full_feature_matrix.index
)


assert mcmed_model_index.index.equals(
    mcmed_full_feature_matrix.index
)



full_feature_order_sha256 = (
    hashlib.sha256(
        "\n".join(
            full_feature_order
        ).encode(
            "utf-8"
        )
    ).hexdigest()
)


family_labels = (
    ["S0"] * N_S0
    + ["S1_diagnosis"] * N_DIAGNOSIS
    + ["S1_medication_name"]
    * N_MEDICATION_NAME
    + ["S1_medication_class"]
    * N_MEDICATION_CLASS
    + ["S2_lab_result"] * 351
    + ["S2_imaging_result"] * 80
)


constructed_family_lookup = dict(
    zip(
        constructed_full_feature_order,
        family_labels,
    )
)


full_feature_manifest = pd.DataFrame(
    {
        "full_position": range(
            1,
            N_FULL + 1,
        ),
        "feature_name": (
            full_feature_order
        ),
    }
)


full_feature_manifest[
    "family"
] = (
    full_feature_manifest[
        "feature_name"
    ]
    .map(
        constructed_family_lookup
    )
)


assert full_feature_manifest[
    "family"
].notna().all()


full_feature_manifest[
    "family_position"
] = (
    full_feature_manifest
    .groupby(
        "family",
        sort=False,
    )
    .cumcount()
    + 1
)


full_feature_manifest[
    "feature_order_sha256"
] = (
    full_feature_order_sha256
)



def summarise_full_matrix(
    dataset_name,
    matrix,
):
    binary_block = matrix.loc[
        :,
        binary_feature_order,
    ]

    binary_support = (
        binary_block
        .sum(
            axis=0
        )
    )


    s0_block = matrix.loc[
        :,
        S0_FEATURE_ORDER,
    ]


    return {
        "dataset": dataset_name,
        "frozen_encounters": len(matrix),
        "s0_features": N_S0,
        "s1_diagnosis_features": N_DIAGNOSIS,
        "s1_medication_features": N_MEDICATION_TOTAL,
        "s2_features": N_S2,
        "total_features": matrix.shape[1],
        "binary_features_with_at_least_one_positive": int(
            binary_support.gt(0).sum()
        ),
        "binary_structural_zero_features_retained": int(
            binary_support.eq(0).sum()
        ),
        "s0_missing_cells_before_preprocessing": int(
            s0_block.isna().sum().sum()
        ),
        "encounters_with_any_s0_missingness": int(
            s0_block.isna().any(axis=1).sum()
        ),
        "feature_order_sha256": (
            full_feature_order_sha256
        ),
    }


full_feature_freeze_audit = pd.DataFrame(
    [
        summarise_full_matrix(
            dataset_name="MIMIC-IV",
            matrix=mimic_full_feature_matrix,
        ),
        summarise_full_matrix(
            dataset_name="MC-MED",
            matrix=mcmed_full_feature_matrix,
        ),
    ]
)


full_feature_source_selection = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "family": "S1 diagnosis",
            "selected_object": (
                mimic_diagnosis_name
            ),
        },
        {
            "dataset": "MC-MED",
            "family": "S1 diagnosis",
            "selected_object": (
                mcmed_diagnosis_name
            ),
        },
        {
            "dataset": "MIMIC-IV",
            "family": "S1 medication",
            "selected_object": (
                mimic_medication_name
            ),
        },
        {
            "dataset": "MC-MED",
            "family": "S1 medication",
            "selected_object": (
                mcmed_medication_name
            ),
        },
    ]
)



print(
    "Selected S1 source objects:"
)

display(
    full_feature_source_selection
)


print(
    "\nComplete feature-matrix freeze audit:"
)

display(
    full_feature_freeze_audit
)


print(
    "\nCOMPLETE MODELLING FEATURE MATRICES "
    "FROZEN IN MEMORY"
)


print(
    "MIMIC-IV:",
    mimic_full_feature_matrix.shape,
)


print(
    "MC-MED:",
    mcmed_full_feature_matrix.shape,
)


print(
    "S0 features:",
    N_S0,
)


print(
    "S1 features:",
    N_S1,
)


print(
    "S2 features:",
    N_S2,
)


print(
    "Total features:",
    N_FULL,
)


print(
    "Legacy Pathway 5 feature order used:",
    legacy_order_used,
)


print(
    "Full feature-order SHA-256:",
    full_feature_order_sha256,
)


print(
    "\nNo preprocessing has been fitted."
)


print(
    "No dataframe has been written to disk."
)


#### Save and verify the official Notebook 1 hand-off

This final section writes the raw frozen datasets and metadata required by internal modelling and external validation. Raw S0 missingness is preserved. No preprocessor, model, calibration object, or decision threshold is created.


In [ ]:


required_handoff_objects = [
    "mimic_full_feature_matrix",
    "mcmed_full_feature_matrix",
    "mimic_model_index",
    "mcmed_model_index",
    "full_feature_order",
    "full_feature_manifest",
    "full_feature_order_sha256",
    "full_feature_freeze_audit",
    "mimic_temporal_split_policy",
]

missing_handoff_objects = [
    object_name
    for object_name in required_handoff_objects
    if object_name not in globals()
]

if missing_handoff_objects:
    raise RuntimeError(
        "The complete matrix-freeze and temporal-split sections "
        "must run before the hand-off.\n"
        f"Missing objects: {missing_handoff_objects}"
    )


PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


FULL_FEATURE_ORDER = list(
    full_feature_order
)


assert len(
    FULL_FEATURE_ORDER
) == 2495

assert len(
    set(FULL_FEATURE_ORDER)
) == 2495

assert mimic_full_feature_matrix.shape == (
    8550,
    2495,
)

assert mcmed_full_feature_matrix.shape == (
    6707,
    2495,
)

assert (
    mimic_full_feature_matrix.columns.tolist()
    ==
    FULL_FEATURE_ORDER
)

assert (
    mcmed_full_feature_matrix.columns.tolist()
    ==
    FULL_FEATURE_ORDER
)



def normalise_split_labels(
    values: pd.Series,
) -> pd.Series:

    return (
        values
        .astype("string")
        .str.strip()
        .str.lower()
        .replace(
            {
                "training": "train",
                "tr": "train",
                "val": "validation",
                "valid": "validation",
                "dev": "validation",
                "development": "validation",
                "testing": "test",
            }
        )
    )


def assemble_handoff_dataset(
    metadata: pd.DataFrame,
    feature_matrix: pd.DataFrame,
    dataset_name: str,
) -> pd.DataFrame:

    assert metadata.index.equals(
        feature_matrix.index
    )


    duplicate_columns = sorted(
        set(metadata.columns).intersection(
            feature_matrix.columns
        )
    )


    if duplicate_columns:
        raise ValueError(
            f"{dataset_name} metadata overlap the model "
            "features:\n"
            f"{duplicate_columns[:20]}"
        )


    output = pd.concat(
        [
            metadata,
            feature_matrix,
        ],
        axis=1,
    )


    assert not output.columns.duplicated().any()


    assert (
        output.loc[
            :,
            FULL_FEATURE_ORDER,
        ].columns.tolist()
        ==
        FULL_FEATURE_ORDER
    )


    return output.reset_index(
        drop=True
    )


def outcome_support(
    dataframe: pd.DataFrame,
) -> dict[int, int]:

    return (
        pd.to_numeric(
            dataframe[
                "outcome_binary"
            ],
            errors="raise",
        )
        .astype("int8")
        .value_counts()
        .reindex(
            [
                0,
                1,
            ],
            fill_value=0,
        )
        .astype(int)
        .to_dict()
    )



mimic_metadata = (
    mimic_model_index
    .copy()
    .reindex(
        mimic_full_feature_matrix.index
    )
)


mcmed_metadata = (
    mcmed_model_index
    .copy()
    .reindex(
        mcmed_full_feature_matrix.index
    )
)


assert mimic_metadata.index.equals(
    mimic_full_feature_matrix.index
)

assert mcmed_metadata.index.equals(
    mcmed_full_feature_matrix.index
)


assert {
    "split",
    "outcome_binary",
}.issubset(
    mimic_metadata.columns
)


assert (
    "outcome_binary"
    in mcmed_metadata.columns
)


mimic_metadata[
    "split"
] = normalise_split_labels(
    mimic_metadata[
        "split"
    ]
)


mcmed_metadata[
    "split"
] = "external"


observed_mimic_splits = set(
    mimic_metadata[
        "split"
    ]
    .dropna()
    .unique()
)


assert observed_mimic_splits == {
    "train",
    "validation",
    "test",
}



mimic_all_frozen = (
    assemble_handoff_dataset(
        metadata=mimic_metadata,
        feature_matrix=(
            mimic_full_feature_matrix
        ),
        dataset_name="MIMIC-IV",
    )
)


mcmed_external_frozen = (
    assemble_handoff_dataset(
        metadata=mcmed_metadata,
        feature_matrix=(
            mcmed_full_feature_matrix
        ),
        dataset_name="MC-MED",
    )
)


mimic_train_frozen = (
    mimic_all_frozen.loc[
        mimic_all_frozen[
            "split"
        ].eq(
            "train"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


mimic_validation_frozen = (
    mimic_all_frozen.loc[
        mimic_all_frozen[
            "split"
        ].eq(
            "validation"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


mimic_test_frozen = (
    mimic_all_frozen.loc[
        mimic_all_frozen[
            "split"
        ].eq(
            "test"
        )
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


handoff_datasets = {
    "mimic_train": (
        mimic_train_frozen
    ),
    "mimic_validation": (
        mimic_validation_frozen
    ),
    "mimic_test": (
        mimic_test_frozen
    ),
    "mcmed_external": (
        mcmed_external_frozen
    ),
}



handoff_support_records = []


for dataset_name, dataframe in (
    handoff_datasets.items()
):

    support = outcome_support(
        dataframe
    )


    handoff_support_records.append(
        {
            "dataset": dataset_name,
            "rows": int(
                len(dataframe)
            ),
            "home": int(
                support[0]
            ),
            "admitted": int(
                support[1]
            ),
            "admitted_prevalence": float(
                support[1]
                /
                len(dataframe)
            ),
        }
    )


handoff_support = pd.DataFrame(
    handoff_support_records
)


assert int(
    handoff_support.loc[
        handoff_support[
            "dataset"
        ].str.startswith(
            "mimic_"
        ),
        "rows",
    ].sum()
) == 8550


assert outcome_support(
    mcmed_external_frozen
) == {
    0: 4872,
    1: 1835,
}



feature_manifest_to_save = (
    full_feature_manifest.copy()
)


family_to_timing = {
    "S0": "S0_TRIAGE",
    "S1_diagnosis": "S1_HISTORY",
    "S1_medication_name": "S1_HISTORY",
    "S1_medication_class": "S1_HISTORY",
    "S2_lab_result": (
        "S2_OBJECTIVE_WORKUP"
    ),
    "S2_imaging_result": (
        "S2_OBJECTIVE_WORKUP"
    ),
}


feature_manifest_to_save[
    "timing_stage"
] = (
    feature_manifest_to_save[
        "family"
    ]
    .map(
        family_to_timing
    )
)


categorical_features = {
    "sex_gender",
    "arrival_transport_grouped",
}


numeric_features = {
    "age",
    "temperature_c",
    "heart_rate",
    "respiratory_rate",
    "oxygen_saturation",
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
}


def feature_storage_type(
    feature_name: str,
) -> str:

    if feature_name in categorical_features:
        return "categorical_raw"

    if feature_name in numeric_features:
        return "numeric_raw"

    return "binary"


feature_manifest_to_save[
    "storage_type"
] = (
    feature_manifest_to_save[
        "feature_name"
    ]
    .map(
        feature_storage_type
    )
)


feature_manifest_to_save[
    "inclusion_status"
] = "PRIMARY_MODEL"


feature_manifest_to_save[
    "external_alignment"
] = "HARMONISED_SAME_COLUMN_ORDER"


feature_manifest_to_save[
    "feature_order_sha256"
] = full_feature_order_sha256


assert (
    feature_manifest_to_save[
        "feature_name"
    ].tolist()
    ==
    FULL_FEATURE_ORDER
)


assert feature_manifest_to_save[
    "timing_stage"
].notna().all()


family_counts = (
    feature_manifest_to_save[
        "family"
    ]
    .value_counts()
    .to_dict()
)


assert family_counts == {
    "S1_diagnosis": 879,
    "S1_medication_name": 812,
    "S1_medication_class": 364,
    "S2_lab_result": 351,
    "S2_imaging_result": 80,
    "S0": 9,
}


binary_feature_order = (
    feature_manifest_to_save.loc[
        feature_manifest_to_save[
            "storage_type"
        ].eq(
            "binary"
        ),
        "feature_name",
    ]
    .tolist()
)


for dataset_name, dataframe in (
    handoff_datasets.items()
):

    binary_block = dataframe.loc[
        :,
        binary_feature_order,
    ]


    assert not binary_block.isna().any().any(), (
        f"{dataset_name} contains missing binary features."
    )


    assert binary_block.isin(
        [
            0,
            1,
            False,
            True,
        ]
    ).all().all(), (
        f"{dataset_name} contains non-binary feature values."
    )



mimic_split_columns = [
    "subject_id",
    "stay_id",
    "split",
    "outcome",
    "outcome_binary",
    "anchor_year",
    "anchor_year_group",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
    "approx_event_period",
]


missing_mimic_split_columns = [
    column
    for column in mimic_split_columns
    if column not in mimic_metadata.columns
]


if missing_mimic_split_columns:
    raise ValueError(
        "MIMIC metadata are missing temporal split fields:\n"
        f"{missing_mimic_split_columns}"
    )


mimic_split_assignments_to_save = (
    mimic_metadata.loc[
        :,
        mimic_split_columns,
    ]
    .rename(
        columns={
            "subject_id": "patient_id",
            "stay_id": "encounter_id",
        }
    )
    .reset_index(
        drop=True
    )
)


mimic_split_assignments_to_save.insert(
    0,
    "dataset",
    "MIMIC-IV",
)


mcmed_patient_ids = (
    mcmed_metadata[
        "MRN"
    ]
    if "MRN" in mcmed_metadata.columns
    else pd.Series(
        pd.NA,
        index=mcmed_metadata.index,
        dtype="string",
    )
)


mcmed_outcomes = (
    mcmed_metadata[
        "outcome"
    ]
    if "outcome" in mcmed_metadata.columns
    else pd.Series(
        pd.NA,
        index=mcmed_metadata.index,
        dtype="string",
    )
)


mcmed_split_assignments_to_save = (
    pd.DataFrame(
        {
            "dataset": pd.Series(
                "MC-MED",
                index=mcmed_metadata.index,
                dtype="string",
            ),
            "patient_id": (
                mcmed_patient_ids
            ),
            "encounter_id": (
                mcmed_metadata[
                    "CSN"
                ]
            ),
            "split": pd.Series(
                "external",
                index=mcmed_metadata.index,
                dtype="string",
            ),
            "outcome": (
                mcmed_outcomes
            ),
            "outcome_binary": (
                mcmed_metadata[
                    "outcome_binary"
                ]
            ),
            "anchor_year": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "anchor_year_group": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="string",
            ),
            "shifted_encounter_year": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "anchor_to_encounter_year_offset": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "approx_event_year_start": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "approx_event_year_end": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "approx_event_year": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="Int64",
            ),
            "approx_event_period": pd.Series(
                pd.NA,
                index=mcmed_metadata.index,
                dtype="string",
            ),
        }
    )
    .reset_index(
        drop=True
    )
)


split_assignments = pd.concat(
    [
        mimic_split_assignments_to_save,
        mcmed_split_assignments_to_save,
    ],
    ignore_index=True,
)




split_string_columns = [
    "dataset",
    "patient_id",
    "encounter_id",
    "split",
    "outcome",
    "anchor_year_group",
    "approx_event_period",
]


for column in split_string_columns:

    split_assignments[
        column
    ] = (
        split_assignments[
            column
        ]
        .astype("string")
        .str.strip()
    )


split_integer_columns = [
    "anchor_year",
    "shifted_encounter_year",
    "anchor_to_encounter_year_offset",
    "approx_event_year_start",
    "approx_event_year_end",
    "approx_event_year",
]


for column in split_integer_columns:

    split_assignments[
        column
    ] = pd.to_numeric(
        split_assignments[
            column
        ],
        errors="coerce",
    ).astype(
        "Int64"
    )


split_assignments[
    "outcome_binary"
] = pd.to_numeric(
    split_assignments[
        "outcome_binary"
    ],
    errors="raise",
).astype(
    "int8"
)


assert len(
    split_assignments
) == (
    8550
    +
    6707
)


assert split_assignments[
    "dataset"
].notna().all()


assert split_assignments[
    "encounter_id"
].notna().all()


assert not split_assignments[
    [
        "dataset",
        "encounter_id",
    ]
].duplicated().any()


split_assignment_schema_audit = (
    pd.DataFrame(
        {
            "column": (
                split_assignments.columns
            ),
            "dtype": [
                str(
                    split_assignments[
                        column
                    ].dtype
                )
                for column
                in split_assignments.columns
            ],
            "missing_rows": [
                int(
                    split_assignments[
                        column
                    ]
                    .isna()
                    .sum()
                )
                for column
                in split_assignments.columns
            ],
        }
    )
)



def create_missingness_summary(
    dataframe: pd.DataFrame,
    partition_name: str,
) -> pd.DataFrame:

    missing_count = (
        dataframe.loc[
            :,
            FULL_FEATURE_ORDER,
        ]
        .isna()
        .sum()
        .reindex(
            FULL_FEATURE_ORDER
        )
    )


    summary = pd.DataFrame(
        {
            "partition": partition_name,
            "feature_name": (
                FULL_FEATURE_ORDER
            ),
            "rows": len(
                dataframe
            ),
            "missing_count": (
                missing_count.to_numpy()
            ),
        }
    )


    summary[
        "missing_percent"
    ] = (
        summary[
            "missing_count"
        ]
        /
        summary[
            "rows"
        ]
        *
        100
    ).round(
        6
    )


    return summary.merge(
        feature_manifest_to_save[
            [
                "feature_name",
                "family",
                "timing_stage",
                "storage_type",
            ]
        ],
        on="feature_name",
        how="left",
        validate="many_to_one",
    )


missingness_summary = pd.concat(
    [
        create_missingness_summary(
            mimic_train_frozen,
            "MIMIC-IV train",
        ),
        create_missingness_summary(
            mimic_validation_frozen,
            "MIMIC-IV validation",
        ),
        create_missingness_summary(
            mimic_test_frozen,
            "MIMIC-IV test",
        ),
        create_missingness_summary(
            mcmed_external_frozen,
            "MC-MED external",
        ),
    ],
    ignore_index=True,
)


assert len(
    missingness_summary
) == (
    4
    *
    2495
)



cohort_flow_handoff = pd.DataFrame(
    [
        {
            "dataset": "MIMIC-IV",
            "stage": (
                "Final HOME/ADMITTED cohort"
            ),
            "split": "all",
            "n_encounters": 8550,
            "home": 6254,
            "admitted": 2296,
        },
        *[
            {
                "dataset": "MIMIC-IV",
                "stage": (
                    "Approximate temporal split"
                ),
                "split": (
                    row.dataset.replace(
                        "mimic_",
                        "",
                    )
                ),
                "n_encounters": int(
                    row.rows
                ),
                "home": int(
                    row.home
                ),
                "admitted": int(
                    row.admitted
                ),
            }
            for row
            in handoff_support.loc[
                handoff_support[
                    "dataset"
                ].str.startswith(
                    "mimic_"
                )
            ].itertuples(
                index=False
            )
        ],
        {
            "dataset": "MC-MED",
            "stage": (
                "Final external HOME/ADMITTED cohort"
            ),
            "split": "external",
            "n_encounters": 6707,
            "home": 4872,
            "admitted": 1835,
        },
    ]
)


cohort_flow_parts = []


for object_name in [
    "mimic_final_flow_table",
    "mcmed_final_flow_table",
]:

    if object_name in globals():

        flow_table = (
            globals()[
                object_name
            ]
            .copy()
        )


        flow_table.insert(
            0,
            "source_object",
            object_name,
        )


        cohort_flow_parts.append(
            flow_table
        )


cohort_flow_to_save = pd.concat(
    [
        *cohort_flow_parts,
        cohort_flow_handoff.assign(
            source_object=(
                "final_handoff_summary"
            )
        ),
    ],
    ignore_index=True,
    sort=False,
)


data_dictionary = pd.DataFrame(
    [
        {
            "variable_or_group": (
                "outcome_binary"
            ),
            "role": "target",
            "definition": (
                "HOME = 0; ADMITTED = 1"
            ),
            "timing_stage": "OUTCOME",
        },
        {
            "variable_or_group": "split",
            "role": "partition",
            "definition": (
                "Approximate encounter-year MIMIC-IV train, "
                "validation and internal-test assignments; "
                "MC-MED marked external"
            ),
            "timing_stage": "PARTITION",
        },
        {
            "variable_or_group": (
                "approx_event_year"
            ),
            "role": (
                "partition_metadata"
            ),
            "definition": (
                "Midpoint of the aligned three-year encounter "
                "range derived from anchor metadata"
            ),
            "timing_stage": "METADATA",
        },
        {
            "variable_or_group": "S0",
            "role": "predictor_group",
            "definition": (
                "Age, sex/gender representation, arrival "
                "transport and initial triage vital signs"
            ),
            "timing_stage": "S0_TRIAGE",
        },
        {
            "variable_or_group": "S1",
            "role": "predictor_group",
            "definition": (
                "Previous diagnoses and previous or "
                "active-at-arrival medications"
            ),
            "timing_stage": "S1_HISTORY",
        },
        {
            "variable_or_group": "S2",
            "role": "predictor_group",
            "definition": (
                "Current-encounter abnormal laboratory and "
                "imaging-result features"
            ),
            "timing_stage": (
                "S2_OBJECTIVE_WORKUP"
            ),
        },
    ]
)



cohort_size_lookup = {
    row.dataset: int(
        row.rows
    )
    for row
    in handoff_support.itertuples(
        index=False
    )
}


split_support_config = {
    row.dataset: {
        "rows": int(
            row.rows
        ),
        "home": int(
            row.home
        ),
        "admitted": int(
            row.admitted
        ),
        "admitted_prevalence": float(
            row.admitted_prevalence
        ),
    }
    for row
    in handoff_support.itertuples(
        index=False
    )
}


pipeline_config = {
    "pipeline_stage": (
        "01_data_preparation_complete"
    ),
    "project": (
        "ESI-3 abdominal pain HOME versus ADMITTED"
    ),
    "project_root": str(
        PROJECT_ROOT
    ),
    "saved_at_utc": (
        datetime
        .now(
            timezone.utc
        )
        .isoformat()
    ),
    "python_version": (
        sys.version
    ),
    "platform": (
        platform.platform()
    ),
    "random_seed": int(
        RANDOM_STATE
    ),
    "cohort_sizes": (
        cohort_size_lookup
    ),
    "outcome_support": (
        split_support_config
    ),
    "outcome_mapping": {
        "HOME": 0,
        "ADMITTED": 1,
    },
    "mimic_temporal_split_policy": (
        mimic_temporal_split_policy
    ),
    "raw_feature_count": 2495,
    "feature_groups": {
        "S0": 9,
        "S1_diagnosis": 879,
        "S1_medication_name": 812,
        "S1_medication_class": 364,
        "S2_lab_result": 351,
        "S2_imaging_result": 80,
        "S1_total": 2055,
        "S2_total": 431,
    },
    "feature_order_sha256": (
        full_feature_order_sha256
    ),
    "official_notebook_1_handoff": (
        "Raw 2,495-feature Parquet datasets"
    ),
    "preprocessing_policy": (
        "Fit only in Notebook 2 using MIMIC-IV training rows; "
        "fit preprocessing independently inside CV folds"
    ),
    "validation_policy": (
        "Use the approximate later MIMIC-IV validation period "
        "for algorithm comparison, calibration and threshold "
        "selection"
    ),
    "internal_test_policy": (
        "Evaluate once after model, calibration and threshold "
        "locking"
    ),
    "external_validation_policy": (
        "Apply the locked MIMIC preprocessing and model to "
        "MC-MED without refitting"
    ),
}



output_paths = {
    "mimic_train": (
        PROCESSED_DATA_DIR
        / "mimic_train.parquet"
    ),
    "mimic_validation": (
        PROCESSED_DATA_DIR
        / "mimic_validation.parquet"
    ),
    "mimic_test": (
        PROCESSED_DATA_DIR
        / "mimic_test.parquet"
    ),
    "mcmed_external": (
        PROCESSED_DATA_DIR
        / "mcmed_external.parquet"
    ),
    "split_assignments": (
        PROCESSED_DATA_DIR
        / "split_assignments.parquet"
    ),
    "feature_manifest": (
        PROCESSED_DATA_DIR
        / "feature_manifest.csv"
    ),
    "missingness_summary": (
        PROCESSED_DATA_DIR
        / "missingness_summary.csv"
    ),
    "cohort_flow": (
        PROCESSED_DATA_DIR
        / "cohort_flow.csv"
    ),
    "data_dictionary": (
        PROCESSED_DATA_DIR
        / "data_dictionary.csv"
    ),
    "pipeline_config": (
        PROCESSED_DATA_DIR
        / "pipeline_config.yaml"
    ),
    "full_feature_freeze_audit": (
        PROCESSED_DATA_DIR
        / "full_feature_freeze_audit.csv"
    ),
    "artifact_checksums": (
        PROCESSED_DATA_DIR
        / "artifact_checksums.csv"
    ),
}



mimic_train_frozen.to_parquet(
    output_paths[
        "mimic_train"
    ],
    index=False,
    engine="pyarrow",
)


mimic_validation_frozen.to_parquet(
    output_paths[
        "mimic_validation"
    ],
    index=False,
    engine="pyarrow",
)


mimic_test_frozen.to_parquet(
    output_paths[
        "mimic_test"
    ],
    index=False,
    engine="pyarrow",
)


mcmed_external_frozen.to_parquet(
    output_paths[
        "mcmed_external"
    ],
    index=False,
    engine="pyarrow",
)


print(
    "\nSplit-assignment schema before Parquet export:"
)

display(
    split_assignment_schema_audit
)


split_assignments.to_parquet(
    output_paths[
        "split_assignments"
    ],
    index=False,
    engine="pyarrow",
)


feature_manifest_to_save.to_csv(
    output_paths[
        "feature_manifest"
    ],
    index=False,
)


missingness_summary.to_csv(
    output_paths[
        "missingness_summary"
    ],
    index=False,
)


cohort_flow_to_save.to_csv(
    output_paths[
        "cohort_flow"
    ],
    index=False,
)


data_dictionary.to_csv(
    output_paths[
        "data_dictionary"
    ],
    index=False,
)


full_feature_freeze_audit.to_csv(
    output_paths[
        "full_feature_freeze_audit"
    ],
    index=False,
)


with output_paths[
    "pipeline_config"
].open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        pipeline_config,
        file,
        indent=2,
        ensure_ascii=False,
    )



expected_parquet_rows = {
    "mimic_train": len(
        mimic_train_frozen
    ),
    "mimic_validation": len(
        mimic_validation_frozen
    ),
    "mimic_test": len(
        mimic_test_frozen
    ),
    "mcmed_external": len(
        mcmed_external_frozen
    ),
    "split_assignments": len(
        split_assignments
    ),
}


saved_parquet_audit = []


for artifact_name, expected_rows in (
    expected_parquet_rows.items()
):

    parquet_file = pq.ParquetFile(
        output_paths[
            artifact_name
        ]
    )


    observed_columns = (
        parquet_file.schema.names
    )


    saved_parquet_audit.append(
        {
            "artifact": artifact_name,
            "rows": int(
                parquet_file
                .metadata
                .num_rows
            ),
            "expected_rows": int(
                expected_rows
            ),
            "columns": len(
                observed_columns
            ),
            "exists": (
                output_paths[
                    artifact_name
                ].exists()
            ),
        }
    )


    assert (
        parquet_file
        .metadata
        .num_rows
        ==
        expected_rows
    )


saved_parquet_audit = pd.DataFrame(
    saved_parquet_audit
)



def file_sha256(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


checksum_records = []


for artifact_name, artifact_path in (
    output_paths.items()
):

    if artifact_name == (
        "artifact_checksums"
    ):
        continue


    if not artifact_path.exists():
        raise FileNotFoundError(
            "Expected hand-off artifact was not written: "
            f"{artifact_path}"
        )


    checksum_records.append(
        {
            "artifact": artifact_name,
            "filename": (
                artifact_path.name
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": (
                file_sha256(
                    artifact_path
                )
            ),
        }
    )


artifact_checksums = pd.DataFrame(
    checksum_records
)


artifact_checksums.to_csv(
    output_paths[
        "artifact_checksums"
    ],
    index=False,
)



handoff_file_audit = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": (
                artifact_path.exists()
            ),
            "size_mb": round(
                artifact_path
                .stat()
                .st_size
                /
                (
                    1024
                    **
                    2
                ),
                3,
            ),
            "path": str(
                artifact_path
            ),
        }
        for artifact_name, artifact_path
        in output_paths.items()
    ]
)


assert handoff_file_audit[
    "exists"
].all()


print(
    "Official Notebook 1 hand-off:"
)

display(
    handoff_support.round(
        {
            "admitted_prevalence": 4,
        }
    )
)


print(
    "\nSaved Parquet audit:"
)

display(
    saved_parquet_audit
)


print(
    "\nSaved artifact audit:"
)

display(
    handoff_file_audit
)


print(
    "\nNOTEBOOK 1 HAND-OFF COMPLETED"
)

print(
    "Raw feature count:",
    len(
        FULL_FEATURE_ORDER
    ),
)

print(
    "Feature-order SHA-256:",
    full_feature_order_sha256,
)

print(
    "Preprocessing fitted: NO"
)

print(
    "Model fitted: NO"
)

print(
    "MC-MED used for development decisions: NO"
)